In [ ]:
# ==========================================
# CFB1 — WEEKLY CONFIGURATION
# ==========================================

from google.colab import userdata

# ------------------------------------------
# 1. API KEYS
# ------------------------------------------

CFBD_API_KEY = userdata.get("CFBD_API_KEY")
ODDS_API_KEY = userdata.get("ODDS_API_KEY")

# Clean accidental whitespace/newlines from Colab Secrets
CFBD_API_KEY = CFBD_API_KEY.strip() if CFBD_API_KEY else None
ODDS_API_KEY = ODDS_API_KEY.strip() if ODDS_API_KEY else None


# ------------------------------------------
# 2. SEASON / WEEK DEFINITIONS
# ------------------------------------------

YEAR = 2026

# Completed data available to the model
DATA_THROUGH_WEEK = 5

# Upcoming slate being handicapped
TARGET_WEEK = 6

# Target game-date window
TARGET_START_DATE = "2026-10-06"
TARGET_END_DATE = "2026-10-10"


# ------------------------------------------
# 3. STARTUP VALIDATION
# ------------------------------------------

print("=" * 60)
print("CFB1 WEEKLY CONFIGURATION")
print("=" * 60)

print(f"Season:                   {YEAR}")
print(f"Data through week:        {DATA_THROUGH_WEEK}")
print(f"Target handicapping week: {TARGET_WEEK}")
print(f"Target slate dates:       {TARGET_START_DATE} through {TARGET_END_DATE}")

print("\nAPI keys:")
print("CFBD key loaded:          ", bool(CFBD_API_KEY))
print("Odds API key loaded:      ", bool(ODDS_API_KEY))

print("\n" + "-" * 60)

if (
    CFBD_API_KEY
    and ODDS_API_KEY
    and DATA_THROUGH_WEEK < TARGET_WEEK
    and TARGET_START_DATE <= TARGET_END_DATE
):
    print("CFB1 WEEKLY CONFIGURATION: PASS")
else:
    print("CFB1 WEEKLY CONFIGURATION: STOP")

print("=" * 60)

CFB1 WEEKLY CONFIGURATION
Season:                   2026
Data through week:        5
Target handicapping week: 6
Target slate dates:       2026-10-06 through 2026-10-10

API keys:
CFBD key loaded:           True
Odds API key loaded:       True

------------------------------------------------------------
CFB1 WEEKLY CONFIGURATION: PASS


In [ ]:
# ==========================================
# CELL 2 — CFBD WEEK-FROZEN DATA ACQUISITION
# ==========================================

import requests
import pandas as pd

print("=" * 65)
print("CFB1 CFBD WEEK-FROZEN DATA ACQUISITION")
print("=" * 65)

print(f"Season:            {YEAR}")
print(f"Data through week: {DATA_THROUGH_WEEK}")

cfbd_headers = {
    "Authorization": f"Bearer {CFBD_API_KEY}",
    "Accept": "application/json"
}


# ==========================================
# 1. PLAY-LEVEL PPA / EPA
# ==========================================
#
# The CFBD /ppa/teams endpoint does not allow
# a historical end-week cutoff.
#
# Therefore CFB1 rebuilds season-to-date team
# EPA directly from play-level PPA through the
# configured DATA_THROUGH_WEEK.
#
# This reproduces the historical team-level
# PPA values used by the original CFB1 model.
# ==========================================

ppa_play_rows = []
ppa_week_status = {}

print("\nPLAY-LEVEL PPA / EPA")

for wk in range(1, DATA_THROUGH_WEEK + 1):

    res = requests.get(
        "https://api.collegefootballdata.com/plays",
        headers=cfbd_headers,
        params={
            "year": YEAR,
            "week": wk,
            "seasonType": "regular"
        }
    )

    ppa_week_status[wk] = res.status_code

    print(
        f"Week {wk}: "
        f"HTTP {res.status_code}",
        end=""
    )

    if res.status_code == 200:

        rows = res.json()

        print(f" | plays returned: {len(rows)}")

        ppa_play_rows.extend(rows)

    else:

        print(" | ERROR")
        print(res.text)


# ------------------------------------------
# Build play dataframe
# ------------------------------------------

df_ppa_plays = pd.DataFrame(ppa_play_rows)

required_play_columns = [
    "offense",
    "defense",
    "ppa"
]

missing_play_columns = [
    col for col in required_play_columns
    if col not in df_ppa_plays.columns
]


if len(missing_play_columns) == 0:

    df_ppa_valid = (
        df_ppa_plays
        .dropna(
            subset=[
                "offense",
                "defense",
                "ppa"
            ]
        )
        .copy()
    )

    # Offensive EPA
    df_off_ppa = (
        df_ppa_valid
        .groupby(
            "offense",
            as_index=False
        )["ppa"]
        .mean()
        .rename(
            columns={
                "offense": "team",
                "ppa": "Off_EPA"
            }
        )
    )

    # Defensive EPA = average opponent PPA allowed
    df_def_ppa = (
        df_ppa_valid
        .groupby(
            "defense",
            as_index=False
        )["ppa"]
        .mean()
        .rename(
            columns={
                "defense": "team",
                "ppa": "Def_EPA"
            }
        )
    )

    df_ppa = pd.merge(
        df_off_ppa,
        df_def_ppa,
        on="team",
        how="outer"
    )

else:

    df_ppa_valid = pd.DataFrame()
    df_ppa = pd.DataFrame()


print("\nPPA / EPA reconstruction:")
print("Total plays returned:     ", len(df_ppa_plays))
print("Valid PPA plays:          ", len(df_ppa_valid))
print("Teams reconstructed:      ", len(df_ppa))
print("Missing required columns: ", missing_play_columns)


# ==========================================
# 2. ADVANCED TEAM STATISTICS
# ==========================================
#
# This CFBD endpoint supports explicit
# startWeek / endWeek filtering.
# ==========================================

print("\nADVANCED TEAM STATISTICS")

adv_res = requests.get(
    "https://api.collegefootballdata.com/stats/season/advanced",
    headers=cfbd_headers,
    params={
        "year": YEAR,
        "seasonType": "regular",
        "startWeek": 1,
        "endWeek": DATA_THROUGH_WEEK
    }
)

print("HTTP status:", adv_res.status_code)

if adv_res.status_code == 200:

    adv_json = adv_res.json()

    print("Teams returned:", len(adv_json))

else:

    adv_json = []

    print("ERROR:")
    print(adv_res.text)


# ==========================================
# 3. DATA ACQUISITION VALIDATION
# ==========================================

all_ppa_weeks_pass = all(
    status == 200
    for status in ppa_week_status.values()
)

ppa_structure_pass = (
    len(missing_play_columns) == 0
    and len(df_ppa) > 0
)

advanced_pass = (
    adv_res.status_code == 200
    and len(adv_json) > 0
)


print("\n" + "-" * 65)

if (
    all_ppa_weeks_pass
    and ppa_structure_pass
    and advanced_pass
):

    print("CFB1 CFBD WEEK-FROZEN DATA: PASS")

else:

    print("CFB1 CFBD WEEK-FROZEN DATA: STOP")

print("=" * 65)

CFB1 CFBD WEEK-FROZEN DATA ACQUISITION
Season:            2026
Data through week: 5

PLAY-LEVEL PPA / EPA
Week 1: HTTP 200 | plays returned: 35773
Week 2: HTTP 200 | plays returned: 23131
Week 3: HTTP 200 | plays returned: 22765
Week 4: HTTP 200 | plays returned: 21833
Week 5: HTTP 200 | plays returned: 20368

PPA / EPA reconstruction:
Total plays returned:      123870
Valid PPA plays:           93364
Teams reconstructed:       313
Missing required columns:  []

ADVANCED TEAM STATISTICS
HTTP status: 200
Teams returned: 138

-----------------------------------------------------------------
CFB1 CFBD WEEK-FROZEN DATA: PASS


In [ ]:
# ==========================================
# CELL 3 — BUILD + VALIDATE CFBD EFFICIENCY MATRIX
# ==========================================

print("=" * 65)
print("CFB1 CFBD EFFICIENCY MATRIX")
print("=" * 65)

print(f"Season:            {YEAR}")
print(f"Data through week: {DATA_THROUGH_WEEK}")


# ==========================================
# 1. BUILD ADVANCED STATS TABLE
# ==========================================

df_adv = pd.json_normalize(adv_json)

df_adv_clean = pd.DataFrame({
    "team": df_adv["team"],
    "Off_SuccessRate": df_adv["offense.successRate"],
    "Off_PassingDown_SR": df_adv["offense.passingDowns.successRate"],
    "Off_Explosiveness": df_adv["offense.explosiveness"],
    "Def_SuccessRate": df_adv["defense.successRate"],
    "Def_PassingDown_SR": df_adv["defense.passingDowns.successRate"],
    "Def_Havoc_Rate": df_adv["defense.havoc.total"],
    "Off_LineYards": df_adv["offense.lineYards"]
})


# ==========================================
# 2. MERGE WEEK-FROZEN CFBD TABLES
# ==========================================

df_cfbd = pd.merge(
    df_ppa,
    df_adv_clean,
    on="team",
    how="inner"
)


# ==========================================
# 3. CALCULATE NET EPA MARGIN
# ==========================================

df_cfbd["Net_EPA_Margin"] = (
    df_cfbd["Off_EPA"]
    - df_cfbd["Def_EPA"]
)


# ==========================================
# 4. VALIDATION
# ==========================================

required_columns = [
    "team",
    "Off_EPA",
    "Def_EPA",
    "Net_EPA_Margin",
    "Off_SuccessRate",
    "Off_PassingDown_SR",
    "Off_Explosiveness",
    "Def_SuccessRate",
    "Def_PassingDown_SR",
    "Def_Havoc_Rate",
    "Off_LineYards"
]

missing_columns = [
    col for col in required_columns
    if col not in df_cfbd.columns
]

duplicate_teams = df_cfbd[
    df_cfbd.duplicated(
        subset=["team"],
        keep=False
    )
]

critical_nulls = (
    df_cfbd[required_columns]
    .isnull()
    .sum()
)

total_critical_nulls = int(
    critical_nulls.sum()
)


# ==========================================
# 5. REPORT
# ==========================================

print("\nDATA COVERAGE")
print("-" * 65)

print(f"Teams in reconstructed PPA data: {len(df_ppa)}")
print(f"Teams in advanced stats:         {len(df_adv_clean)}")
print(f"Teams after inner merge:         {len(df_cfbd)}")

print("\nVALIDATION")
print("-" * 65)

print(f"Required columns missing: {len(missing_columns)}")

if missing_columns:
    print(missing_columns)

print(f"Duplicate team rows:       {len(duplicate_teams)}")
print(f"Critical missing values:   {total_critical_nulls}")

if total_critical_nulls > 0:
    print("\nMissing values by column:")
    print(
        critical_nulls[
            critical_nulls > 0
        ]
    )


# ==========================================
# 6. FINAL STATUS
# ==========================================

print("\n" + "-" * 65)

if (
    len(missing_columns) == 0
    and len(duplicate_teams) == 0
    and total_critical_nulls == 0
    and len(df_cfbd) > 0
):
    print("CFB1 EFFICIENCY MATRIX: PASS")
else:
    print("CFB1 EFFICIENCY MATRIX: STOP")

print("=" * 65)


# ==========================================
# 7. VISUAL CHECK
# ==========================================

display(
    df_cfbd.head(10)
)

CFB1 CFBD EFFICIENCY MATRIX
Season:            2026
Data through week: 5

DATA COVERAGE
-----------------------------------------------------------------
Teams in reconstructed PPA data: 313
Teams in advanced stats:         138
Teams after inner merge:         138

VALIDATION
-----------------------------------------------------------------
Required columns missing: 0
Duplicate team rows:       0
Critical missing values:   0

-----------------------------------------------------------------
CFB1 EFFICIENCY MATRIX: PASS


,team,Off_EPA,Def_EPA,Off_SuccessRate,Off_PassingDown_SR,Off_Explosiveness,Def_SuccessRate,Def_PassingDown_SR,Def_Havoc_Rate,Off_LineYards,Net_EPA_Margin
0,Air Force,0.189718,0.000603,0.452769,0.253165,1.085244,0.333333,0.187500,0.144,3.527711,0.189115
1,Akron,0.052222,0.282767,0.430303,0.313131,1.039954,0.444805,0.319149,0.133,3.692568,-0.230545
2,Alabama,0.349468,-0.112745,0.512121,0.379310,1.307878,0.343750,0.247788,0.200,3.163536,0.462213
3,App State,0.237502,0.079889,0.454861,0.446809,1.312445,0.422794,0.212500,0.176,2.576866,0.157613
4,Arizona,0.217433,-0.013423,0.495822,0.437500,1.155767,0.343137,0.284553,0.167,3.474359,0.230856
5,Arizona State,0.024659,0.140031,0.416667,0.311828,1.259956,0.380074,0.292929,0.137,2.826119,-0.115371
6,Arkansas,0.014616,0.166878,0.410765,0.279070,1.114178,0.426667,0.216867,0.123,2.655405,-0.152262
7,Arkansas State,0.154351,0.188218,0.472414,0.316456,1.153007,0.476636,0.356322,0.115,3.320548,-0.033867
8,Army,0.331186,0.229616,0.531136,0.316667,1.091834,0.450000,0.338462,0.123,3.650239,0.101571
9,Auburn,0.074267,0.008065,0.421652,0.358974,1.258493,0.384416,0.250000,0.166,3.321084,0.066203


In [ ]:
# ==========================================
# CELL 4 — ODDS API CONNECTION + SLATE ACQUISITION
# ==========================================

print("=" * 65)
print("CFB1 ODDS API CONNECTION + SLATE ACQUISITION")
print("=" * 65)

odds_url = (
    "https://api.the-odds-api.com/"
    "v4/sports/americanfootball_ncaaf/odds/"
)

odds_params = {
    "apiKey": ODDS_API_KEY,
    "regions": "us",
    "markets": "spreads,totals",
    "bookmakers": "draftkings,fanduel"
}

odds_res = requests.get(
    odds_url,
    params=odds_params
)

print("\nHTTP status:", odds_res.status_code)


# ==========================================
# 1. LOAD SLATE
# ==========================================

if odds_res.status_code == 200:

    odds_json = odds_res.json()

    print("Games returned:", len(odds_json))

    dk_games = 0
    fd_games = 0

    for game in odds_json:

        bookmaker_keys = [
            bm.get("key")
            for bm in game.get("bookmakers", [])
        ]

        if "draftkings" in bookmaker_keys:
            dk_games += 1

        if "fanduel" in bookmaker_keys:
            fd_games += 1

    print("Games with DraftKings:", dk_games)
    print("Games with FanDuel:   ", fd_games)

else:

    odds_json = []

    print("\nERROR:")
    print(odds_res.text)


# ==========================================
# 2. PREFLIGHT STATUS
# ==========================================

print("\n" + "-" * 65)

if (
    odds_res.status_code == 200
    and len(odds_json) > 0
):
    print("CFB1 ODDS API CONNECTION: PASS")
else:
    print("CFB1 ODDS API CONNECTION: STOP")

print("=" * 65)

CFB1 ODDS API CONNECTION + SLATE ACQUISITION

HTTP status: 200
Games returned: 66
Games with DraftKings: 57
Games with FanDuel:    66

-----------------------------------------------------------------
CFB1 ODDS API CONNECTION: PASS


In [ ]:
# ==========================================
# CELL 5 — BUILD FULL ODDS MARKET CARD
# ==========================================

print("=" * 65)
print("CFB1 FULL ODDS MARKET CARD")
print("=" * 65)

games_list = []

for game in odds_json:

    home = game.get("home_team")
    away = game.get("away_team")

    kickoff_utc = pd.to_datetime(
        game.get("commence_time"),
        utc=True
    )

    kickoff_et = kickoff_utc.tz_convert(
        "America/New_York"
    )

    dk_spread = None
    fd_spread = None
    dk_total = None
    fd_total = None

    for bm in game.get("bookmakers", []):

        bookmaker = bm.get("key")

        for market in bm.get("markets", []):

            market_key = market.get("key")

            # DraftKings
            if bookmaker == "draftkings":

                if market_key == "spreads":

                    for outcome in market.get("outcomes", []):

                        # Store HOME team's spread
                        if outcome.get("name") == home:
                            dk_spread = outcome.get("point")

                elif market_key == "totals":

                    for outcome in market.get("outcomes", []):

                        if outcome.get("name") == "Over":
                            dk_total = outcome.get("point")

            # FanDuel
            elif bookmaker == "fanduel":

                if market_key == "spreads":

                    for outcome in market.get("outcomes", []):

                        # Store HOME team's spread
                        if outcome.get("name") == home:
                            fd_spread = outcome.get("point")

                elif market_key == "totals":

                    for outcome in market.get("outcomes", []):

                        if outcome.get("name") == "Over":
                            fd_total = outcome.get("point")

    games_list.append({
        "Home_Team": home,
        "Away_Team": away,
        "Kickoff_UTC": kickoff_utc,
        "Kickoff_ET": kickoff_et,
        "Game_Date_ET": kickoff_et.date(),
        "DK_Spread": dk_spread,
        "FD_Spread": fd_spread,
        "DK_Total": dk_total,
        "FD_Total": fd_total
    })


df_market = pd.DataFrame(games_list)


# ==========================================
# VALIDATION
# ==========================================

required_columns = [
    "Home_Team",
    "Away_Team",
    "Kickoff_UTC",
    "Kickoff_ET",
    "Game_Date_ET",
    "DK_Spread",
    "FD_Spread",
    "DK_Total",
    "FD_Total"
]

missing_columns = [
    col for col in required_columns
    if col not in df_market.columns
]

duplicate_games = df_market[
    df_market.duplicated(
        subset=["Home_Team", "Away_Team"],
        keep=False
    )
]

missing_dk_spreads = df_market["DK_Spread"].isna().sum()
missing_fd_spreads = df_market["FD_Spread"].isna().sum()
missing_dk_totals = df_market["DK_Total"].isna().sum()
missing_fd_totals = df_market["FD_Total"].isna().sum()


# ==========================================
# REPORT
# ==========================================

print(f"\nGames loaded:             {len(df_market)}")
print(f"Required columns missing: {len(missing_columns)}")
print(f"Duplicate games:          {len(duplicate_games)}")

print("\nGames by Eastern date:")
print(
    df_market["Game_Date_ET"]
    .value_counts()
    .sort_index()
)

print("\nMissing sportsbook data:")
print(f"DK spreads missing:       {missing_dk_spreads}")
print(f"FD spreads missing:       {missing_fd_spreads}")
print(f"DK totals missing:        {missing_dk_totals}")
print(f"FD totals missing:        {missing_fd_totals}")

print("\n" + "-" * 65)

if (
    len(missing_columns) == 0
    and len(duplicate_games) == 0
    and len(df_market) > 0
):
    print("CFB1 FULL MARKET CARD: PASS")
else:
    print("CFB1 FULL MARKET CARD: STOP")

print("=" * 65)

CFB1 FULL ODDS MARKET CARD

Games loaded:             66
Required columns missing: 0
Duplicate games:          0

Games by Eastern date:
Game_Date_ET
2026-10-07     2
2026-10-08     4
2026-10-09     5
2026-10-10    46
2026-10-17     9
Name: count, dtype: int64

Missing sportsbook data:
DK spreads missing:       9
FD spreads missing:       0
DK totals missing:        9
FD totals missing:        0

-----------------------------------------------------------------
CFB1 FULL MARKET CARD: PASS


In [ ]:
# ==========================================
# CELL 6 — BUILD + VALIDATE TARGET-WEEK MARKET CARD
# ==========================================

print("=" * 65)
print("CFB1 TARGET-WEEK MARKET CARD")
print("=" * 65)

print(f"Target handicapping week: {TARGET_WEEK}")
print(f"Target dates:             {TARGET_START_DATE} through {TARGET_END_DATE}")


# ==========================================
# 1. CONVERT CONFIG DATES
# ==========================================

target_start = pd.to_datetime(
    TARGET_START_DATE
).date()

target_end = pd.to_datetime(
    TARGET_END_DATE
).date()


# ==========================================
# 2. FILTER FULL MARKET CARD
# ==========================================

df_target_market = (
    df_market[
        (df_market["Game_Date_ET"] >= target_start)
        &
        (df_market["Game_Date_ET"] <= target_end)
    ]
    .copy()
    .sort_values(
        by=["Kickoff_ET", "Away_Team", "Home_Team"]
    )
    .reset_index(drop=True)
)


# ==========================================
# 3. VALIDATION
# ==========================================

required_columns = [
    "Home_Team",
    "Away_Team",
    "Kickoff_ET",
    "Game_Date_ET",
    "DK_Spread",
    "FD_Spread",
    "DK_Total",
    "FD_Total"
]

missing_columns = [
    col for col in required_columns
    if col not in df_target_market.columns
]

duplicate_games = df_target_market[
    df_target_market.duplicated(
        subset=["Home_Team", "Away_Team"],
        keep=False
    )
]

outside_date_window = df_target_market[
    (df_target_market["Game_Date_ET"] < target_start)
    |
    (df_target_market["Game_Date_ET"] > target_end)
]


# ==========================================
# 4. REPORT
# ==========================================

print(f"\nGames in full Odds API card: {len(df_market)}")
print(f"Games in target-week card:  {len(df_target_market)}")

print("\nGames by target date:")
print(
    df_target_market["Game_Date_ET"]
    .value_counts()
    .sort_index()
)

print("\nValidation:")
print(f"Required columns missing:   {len(missing_columns)}")
print(f"Duplicate games:            {len(duplicate_games)}")
print(f"Games outside date window:  {len(outside_date_window)}")

print("\nMissing sportsbook data:")
print(
    "DK spreads missing:        ",
    df_target_market["DK_Spread"].isna().sum()
)
print(
    "FD spreads missing:        ",
    df_target_market["FD_Spread"].isna().sum()
)
print(
    "DK totals missing:         ",
    df_target_market["DK_Total"].isna().sum()
)
print(
    "FD totals missing:         ",
    df_target_market["FD_Total"].isna().sum()
)


# ==========================================
# 5. FINAL STATUS
# ==========================================

print("\n" + "-" * 65)

if (
    len(df_target_market) > 0
    and len(missing_columns) == 0
    and len(duplicate_games) == 0
    and len(outside_date_window) == 0
):
    print("CFB1 TARGET-WEEK MARKET CARD: PASS")
else:
    print("CFB1 TARGET-WEEK MARKET CARD: STOP")

print("=" * 65)


# ==========================================
# 6. VISUAL CHECK
# ==========================================

display(
    df_target_market[
        [
            "Game_Date_ET",
            "Kickoff_ET",
            "Away_Team",
            "Home_Team",
            "DK_Spread",
            "FD_Spread",
            "DK_Total",
            "FD_Total"
        ]
    ]
)

CFB1 TARGET-WEEK MARKET CARD
Target handicapping week: 6
Target dates:             2026-10-06 through 2026-10-10

Games in full Odds API card: 66
Games in target-week card:  57

Games by target date:
Game_Date_ET
2026-10-07     2
2026-10-08     4
2026-10-09     5
2026-10-10    46
Name: count, dtype: int64

Validation:
Required columns missing:   0
Duplicate games:            0
Games outside date window:  0

Missing sportsbook data:
DK spreads missing:         0
FD spreads missing:         0
DK totals missing:          0
FD totals missing:          0

-----------------------------------------------------------------
CFB1 TARGET-WEEK MARKET CARD: PASS


,Game_Date_ET,Kickoff_ET,Away_Team,Home_Team,DK_Spread,FD_Spread,DK_Total,FD_Total
0,2026-10-07,2026-10-07 19:00:00-04:00,Jacksonville State Gamecocks,Kennesaw State Owls,3.0,2.5,48.5,48.5
1,2026-10-07,2026-10-07 19:30:00-04:00,New Mexico State Aggies,Florida International Panthers,-6.5,-6.5,45.5,46.5
2,2026-10-08,2026-10-08 19:00:00-04:00,Missouri State Bears,Western Kentucky Hilltoppers,-2.5,-2.5,56.5,56.5
3,2026-10-08,2026-10-08 19:00:00-04:00,Sam Houston State Bearkats,Liberty Flames,-13.5,-13.5,52.5,52.5
4,2026-10-08,2026-10-08 19:30:00-04:00,South Alabama Jaguars,Arkansas State Red Wolves,-1.5,-1.5,58.5,58.5
5,2026-10-08,2026-10-08 19:30:00-04:00,South Florida Bulls,UTSA Roadrunners,-7.0,-7.5,49.5,50.5
6,2026-10-09,2026-10-09 19:00:00-04:00,Florida State Seminoles,Louisville Cardinals,-3.5,-3.5,59.5,59.5
7,2026-10-09,2026-10-09 21:00:00-04:00,Iowa Hawkeyes,Washington Huskies,-3.5,-3.5,41.5,41.5
8,2026-10-09,2026-10-09 21:00:00-04:00,Washington State Cougars,Utah State Aggies,-5.5,-5.5,44.5,43.5
9,2026-10-09,2026-10-09 21:00:00-04:00,Wyoming Cowboys,San Jose State Spartans,-6.0,-5.5,43.5,42.5


In [ ]:
# ==========================================
# CELL 10 — TEAM NAME NORMALIZATION + SAFETY CHECK
# ==========================================

import re

print("=" * 65)
print("CFB1 TEAM NAME NORMALIZATION + SAFETY CHECK")
print("=" * 65)


# ==========================================
# 1. ODDS API -> CFBD TEAM NAME ALIASES
# ==========================================

TEAM_ALIASES = {
    "Air Force Falcons": "Air Force",
    "Akron Zips": "Akron",
    "Alabama Crimson Tide": "Alabama",
    "Arizona State Sun Devils": "Arizona State",
    "Arizona Wildcats": "Arizona",
    "Arkansas Razorbacks": "Arkansas",
    "Arkansas State Red Wolves": "Arkansas State",
    "Army Black Knights": "Army",
    "Auburn Tigers": "Auburn",
    "BYU Cougars": "BYU",
    "Ball State Cardinals": "Ball State",
    "Baylor Bears": "Baylor",
    "Boise State Broncos": "Boise State",
    "Boston College Eagles": "Boston College",
    "Bowling Green Falcons": "Bowling Green",
    "Buffalo Bulls": "Buffalo",
    "California Golden Bears": "California",
    "Central Michigan Chippewas": "Central Michigan",
    "Charlotte 49ers": "Charlotte",
    "Cincinnati Bearcats": "Cincinnati",
    "Clemson Tigers": "Clemson",
    "Coastal Carolina Chanticleers": "Coastal Carolina",
    "Colorado Buffaloes": "Colorado",
    "Colorado State Rams": "Colorado State",
    "Delaware Blue Hens": "Delaware",
    "Eastern Michigan Eagles": "Eastern Michigan",
    "Florida Atlantic Owls": "Florida Atlantic",
    "Florida Gators": "Florida",
    "Florida State Seminoles": "Florida State",
    "Fresno State Bulldogs": "Fresno State",
    "Georgia Bulldogs": "Georgia",
    "Georgia Southern Eagles": "Georgia Southern",
    "Georgia State Panthers": "Georgia State",
    "Hawaii Rainbow Warriors": "Hawai'i",
    "Houston Cougars": "Houston",
    "Illinois Fighting Illini": "Illinois",
    "Indiana Hoosiers": "Indiana",
    "Iowa Hawkeyes": "Iowa",
    "Iowa State Cyclones": "Iowa State",
    "James Madison Dukes": "James Madison",
    "Kansas Jayhawks": "Kansas",
    "Kent State Golden Flashes": "Kent State",
    "Kentucky Wildcats": "Kentucky",
    "Liberty Flames": "Liberty",
    "Louisiana Ragin Cajuns": "Louisiana",
    "Louisiana Tech Bulldogs": "Louisiana Tech",
    "Louisville Cardinals": "Louisville",
    "LSU Tigers": "LSU",
    "Marshall Thundering Herd": "Marshall",
    "Maryland Terrapins": "Maryland",
    "Memphis Tigers": "Memphis",
    "Miami (OH) RedHawks": "Miami (OH)",
    "Miami Hurricanes": "Miami",
    "Michigan State Spartans": "Michigan State",
    "Michigan Wolverines": "Michigan",
    "Middle Tennessee Blue Raiders": "Middle Tennessee",
    "Minnesota Golden Gophers": "Minnesota",
    "Mississippi State Bulldogs": "Mississippi State",
    "Missouri Tigers": "Missouri",
    "NC State Wolfpack": "NC State",
    "Navy Midshipmen": "Navy",
    "Nebraska Cornhuskers": "Nebraska",
    "New Mexico Lobos": "New Mexico",
    "New Mexico State Aggies": "New Mexico State",
    "North Carolina Tar Heels": "North Carolina",
    "North Texas Mean Green": "North Texas",
    "Northwestern Wildcats": "Northwestern",
    "Notre Dame Fighting Irish": "Notre Dame",
    "Ohio Bobcats": "Ohio",
    "Ohio State Buckeyes": "Ohio State",
    "Old Dominion Monarchs": "Old Dominion",
    "Oregon State Beavers": "Oregon State",
    "Penn State Nittany Lions": "Penn State",
    "Pittsburgh Panthers": "Pittsburgh",
    "Purdue Boilermakers": "Purdue",
    "Rice Owls": "Rice",
    "Rutgers Scarlet Knights": "Rutgers",
    "SMU Mustangs": "SMU",
    "San Diego State Aztecs": "San Diego State",
    "San Jose State Spartans": "San José State",
    "South Alabama Jaguars": "South Alabama",
    "South Carolina Gamecocks": "South Carolina",
    "South Florida Bulls": "South Florida",
    "Stanford Cardinal": "Stanford",
    "Syracuse Orange": "Syracuse",
    "TCU Horned Frogs": "TCU",
    "Temple Owls": "Temple",
    "Tennessee Volunteers": "Tennessee",
    "Texas A&M Aggies": "Texas A&M",
    "Texas State Bobcats": "Texas State",
    "Texas Tech Red Raiders": "Texas Tech",
    "Toledo Rockets": "Toledo",
    "Tulsa Golden Hurricane": "Tulsa",
    "UAB Blazers": "UAB",
    "Appalachian State Mountaineers": "App State",
    "Duke Blue Devils": "Duke",
    "East Carolina Pirates": "East Carolina",
    "Florida International Panthers": "Florida International",
    "Georgia Tech Yellow Jackets": "Georgia Tech",
    "Jacksonville State Gamecocks": "Jacksonville State",
    "Kansas State Wildcats": "Kansas State",
    "Kennesaw State Owls": "Kennesaw State",
    "Missouri State Bears": "Missouri State",
    "Nevada Wolf Pack": "Nevada",
    "North Dakota State Bison": "North Dakota State",
    "Northern Illinois Huskies": "Northern Illinois",
    "Oklahoma Sooners": "Oklahoma",
    "Oklahoma State Cowboys": "Oklahoma State",
    "Ole Miss Rebels": "Ole Miss",
    "Oregon Ducks": "Oregon",
    "Sacramento State Hornets": "Sacramento State",
    "Sam Houston State Bearkats": "Sam Houston",
    "Southern Mississippi Golden Eagles": "Southern Miss",
    "Texas Longhorns": "Texas",
    "Troy Trojans": "Troy",
    "Tulane Green Wave": "Tulane",
    "UCLA Bruins": "UCLA",
    "Utah Utes": "Utah",
    "UCF Knights": "UCF",
    "UConn Huskies": "UConn",
    "UL Monroe Warhawks": "UL Monroe",
    "UMass Minutemen": "Massachusetts",
    "UNLV Rebels": "UNLV",
    "USC Trojans": "USC",
    "UTEP Miners": "UTEP",
    "UTSA Roadrunners": "UTSA",
    "Utah State Aggies": "Utah State",
    "Vanderbilt Commodores": "Vanderbilt",
    "Virginia Cavaliers": "Virginia",
    "Virginia Tech Hokies": "Virginia Tech",
    "Wake Forest Demon Deacons": "Wake Forest",
    "Washington Huskies": "Washington",
    "Washington State Cougars": "Washington State",
    "West Virginia Mountaineers": "West Virginia",
    "Western Kentucky Hilltoppers": "Western Kentucky",
    "Western Michigan Broncos": "Western Michigan",
    "Wisconsin Badgers": "Wisconsin",
    "Wyoming Cowboys": "Wyoming",
}


# ==========================================
# 2. NORMALIZATION FUNCTION
# ==========================================

def normalize_team_name(team_name):

    cleaned = re.sub(
        r"\s+",
        " ",
        str(team_name)
    ).strip()

    return TEAM_ALIASES.get(
        cleaned,
        cleaned
    )


# ==========================================
# 3. APPLY TO TARGET-WEEK MARKET CARD
# ==========================================

df_target_market[
    "Home_CFBD_Name"
] = (
    df_target_market["Home_Team"]
    .apply(normalize_team_name)
)

df_target_market[
    "Away_CFBD_Name"
] = (
    df_target_market["Away_Team"]
    .apply(normalize_team_name)
)


# ==========================================
# 4. CFBD TEAM SET
# ==========================================

cfbd_teams = set(
    df_cfbd["team"]
    .dropna()
    .astype(str)
    .str.strip()
)


# ==========================================
# 5. IDENTIFY MATCH STATUS
# ==========================================

df_target_market[
    "Home_CFBD_Matched"
] = (
    df_target_market[
        "Home_CFBD_Name"
    ].isin(cfbd_teams)
)

df_target_market[
    "Away_CFBD_Matched"
] = (
    df_target_market[
        "Away_CFBD_Name"
    ].isin(cfbd_teams)
)


home_unmatched = (
    df_target_market[
        ~df_target_market[
            "Home_CFBD_Matched"
        ]
    ][
        [
            "Home_Team",
            "Home_CFBD_Name"
        ]
    ]
    .drop_duplicates()
)

away_unmatched = (
    df_target_market[
        ~df_target_market[
            "Away_CFBD_Matched"
        ]
    ][
        [
            "Away_Team",
            "Away_CFBD_Name"
        ]
    ]
    .drop_duplicates()
)


total_unmatched = (
    len(home_unmatched)
    +
    len(away_unmatched)
)


# ==========================================
# 6. REPORT
# ==========================================

print(f"\nTarget-week games checked: {len(df_target_market)}")
print(f"Team appearances checked: {len(df_target_market) * 2}")
print(f"Unmatched home names:      {len(home_unmatched)}")
print(f"Unmatched away names:      {len(away_unmatched)}")
print(f"Total unmatched names:     {total_unmatched}")


if len(home_unmatched) > 0:

    print("\nUnmatched HOME names:")
    display(home_unmatched)


if len(away_unmatched) > 0:

    print("\nUnmatched AWAY names:")
    display(away_unmatched)


# ==========================================
# 7. FINAL STATUS
# ==========================================

print("\n" + "-" * 65)

if total_unmatched == 0:

    print("CFB1 TEAM NAME NORMALIZATION: PASS")

else:

    print("CFB1 TEAM NAME NORMALIZATION: STOP")
    print("Resolve unmatched names before CFBD merge.")

print("=" * 65)

CFB1 TEAM NAME NORMALIZATION + SAFETY CHECK

Target-week games checked: 57
Team appearances checked: 114
Unmatched home names:      0
Unmatched away names:      0
Total unmatched names:     0

-----------------------------------------------------------------
CFB1 TEAM NAME NORMALIZATION: PASS


In [ ]:
# ==========================================
# CELL 11 — MERGE TARGET-WEEK MARKET + CFBD DATA
# ==========================================

print("=" * 65)
print("CFB1 TARGET-WEEK MARKET + CFBD MERGE")
print("=" * 65)

print(f"Target handicapping week: {TARGET_WEEK}")
print(f"Target games entering merge: {len(df_target_market)}")


# ==========================================
# 1. PREPARE HOME TEAM CFBD DATA
# ==========================================

home_cfbd = df_cfbd.copy()

home_cfbd = home_cfbd.rename(columns={
    "team": "Home_CFBD_Name",
    "Off_EPA": "Home_Off_EPA",
    "Def_EPA": "Home_Def_EPA",
    "Net_EPA_Margin": "Home_Net_EPA_Margin",
    "Off_SuccessRate": "Home_Off_SuccessRate",
    "Off_PassingDown_SR": "Home_Off_PassingDown_SR",
    "Off_Explosiveness": "Home_Off_Explosiveness",
    "Def_SuccessRate": "Home_Def_SuccessRate",
    "Def_PassingDown_SR": "Home_Def_PassingDown_SR",
    "Def_Havoc_Rate": "Home_Def_Havoc_Rate",
    "Off_LineYards": "Home_Off_LineYards"
})


# ==========================================
# 2. PREPARE AWAY TEAM CFBD DATA
# ==========================================

away_cfbd = df_cfbd.copy()

away_cfbd = away_cfbd.rename(columns={
    "team": "Away_CFBD_Name",
    "Off_EPA": "Away_Off_EPA",
    "Def_EPA": "Away_Def_EPA",
    "Net_EPA_Margin": "Away_Net_EPA_Margin",
    "Off_SuccessRate": "Away_Off_SuccessRate",
    "Off_PassingDown_SR": "Away_Off_PassingDown_SR",
    "Off_Explosiveness": "Away_Off_Explosiveness",
    "Def_SuccessRate": "Away_Def_SuccessRate",
    "Def_PassingDown_SR": "Away_Def_PassingDown_SR",
    "Def_Havoc_Rate": "Away_Def_Havoc_Rate",
    "Off_LineYards": "Away_Off_LineYards"
})


# ==========================================
# 3. MERGE HOME + AWAY CFBD DATA
# ==========================================

df_master = df_target_market.merge(
    home_cfbd,
    on="Home_CFBD_Name",
    how="left",
    validate="many_to_one"
)

df_master = df_master.merge(
    away_cfbd,
    on="Away_CFBD_Name",
    how="left",
    validate="many_to_one"
)


# ==========================================
# 4. CREATE GAME-LEVEL EPA EDGE
# ==========================================
#
# Positive = HOME has stronger Net EPA Margin
# Negative = AWAY has stronger Net EPA Margin
# ==========================================

df_master["Net_EPA_Edge_Home"] = (
    df_master["Home_Net_EPA_Margin"]
    - df_master["Away_Net_EPA_Margin"]
)


# ==========================================
# 5. VALIDATION
# ==========================================

expected_games = len(df_target_market)

home_missing = df_master[
    df_master["Home_Net_EPA_Margin"].isna()
]

away_missing = df_master[
    df_master["Away_Net_EPA_Margin"].isna()
]

duplicate_games = df_master[
    df_master.duplicated(
        subset=["Home_Team", "Away_Team"],
        keep=False
    )
]


print("\nVALIDATION")
print("-" * 65)

print(f"Expected target-week games: {expected_games}")
print(f"Games after CFBD merge:     {len(df_master)}")
print(f"Missing home CFBD records:  {len(home_missing)}")
print(f"Missing away CFBD records:  {len(away_missing)}")
print(f"Duplicate games:            {len(duplicate_games)}")


if len(home_missing) > 0:

    print("\nHOME TEAMS MISSING CFBD DATA:")

    display(
        home_missing[
            [
                "Home_Team",
                "Home_CFBD_Name"
            ]
        ]
    )


if len(away_missing) > 0:

    print("\nAWAY TEAMS MISSING CFBD DATA:")

    display(
        away_missing[
            [
                "Away_Team",
                "Away_CFBD_Name"
            ]
        ]
    )


# ==========================================
# 6. FINAL STATUS
# ==========================================

print("\n" + "-" * 65)

if (
    len(df_master) == expected_games
    and len(home_missing) == 0
    and len(away_missing) == 0
    and len(duplicate_games) == 0
):

    print("CFB1 TARGET-WEEK CFBD MERGE: PASS")

else:

    print("CFB1 TARGET-WEEK CFBD MERGE: STOP")
    raise ValueError(
        "Target-week CFBD merge failed. "
        "Do not continue to model calculations."
    )

print("=" * 65)


# ==========================================
# 7. VISUAL CHECK
# ==========================================

display(
    df_master[
        [
            "Away_Team",
            "Home_Team",
            "DK_Spread",
            "FD_Spread",
            "Away_Net_EPA_Margin",
            "Home_Net_EPA_Margin",
            "Net_EPA_Edge_Home"
        ]
    ]
    .head(10)
)

CFB1 TARGET-WEEK MARKET + CFBD MERGE
Target handicapping week: 6
Target games entering merge: 57

VALIDATION
-----------------------------------------------------------------
Expected target-week games: 57
Games after CFBD merge:     57
Missing home CFBD records:  0
Missing away CFBD records:  0
Duplicate games:            0

-----------------------------------------------------------------
CFB1 TARGET-WEEK CFBD MERGE: PASS


,Away_Team,Home_Team,DK_Spread,FD_Spread,Away_Net_EPA_Margin,Home_Net_EPA_Margin,Net_EPA_Edge_Home
0,Jacksonville State Gamecocks,Kennesaw State Owls,3.0,2.5,0.165220,-0.054248,-0.219468
1,New Mexico State Aggies,Florida International Panthers,-6.5,-6.5,0.013578,0.012947,-0.000631
2,Missouri State Bears,Western Kentucky Hilltoppers,-2.5,-2.5,-0.126415,-0.274928,-0.148512
3,Sam Houston State Bearkats,Liberty Flames,-13.5,-13.5,0.036380,0.127749,0.091369
4,South Alabama Jaguars,Arkansas State Red Wolves,-1.5,-1.5,-0.053218,-0.033867,0.019351
5,South Florida Bulls,UTSA Roadrunners,-7.0,-7.5,0.189698,0.172703,-0.016996
6,Florida State Seminoles,Louisville Cardinals,-3.5,-3.5,0.151340,0.183498,0.032159
7,Iowa Hawkeyes,Washington Huskies,-3.5,-3.5,0.191521,0.203461,0.011939
8,Washington State Cougars,Utah State Aggies,-5.5,-5.5,-0.150280,-0.144463,0.005817
9,Wyoming Cowboys,San Jose State Spartans,-6.0,-5.5,-0.065317,0.039860,0.105177


In [ ]:
# ==========================================
# CELL 12 — NET EPA TRIGGER
# ==========================================

print("=" * 65)
print("CFB1 NET EPA TRIGGER")
print("=" * 65)

# ------------------------------------------
# Established CFB1 threshold
# ------------------------------------------

EPA_TRIGGER_THRESHOLD = 0.12


# ==========================================
# 1. IDENTIFY EPA EDGE TEAM
# ==========================================
#
# Net_EPA_Edge_Home:
#   positive -> HOME has stronger Net EPA
#   negative -> AWAY has stronger Net EPA
# ==========================================

df_master["EPA_Edge_Value"] = (
    df_master["Net_EPA_Edge_Home"].abs()
)

df_master["EPA_Edge_Team"] = df_master.apply(
    lambda row:
        row["Home_Team"]
        if row["Net_EPA_Edge_Home"] > 0
        else (
            row["Away_Team"]
            if row["Net_EPA_Edge_Home"] < 0
            else "EVEN"
        ),
    axis=1
)


# ==========================================
# 2. APPLY EPA TRIGGER
# ==========================================

df_master["EPA_Trigger"] = (
    df_master["EPA_Edge_Value"]
    >= EPA_TRIGGER_THRESHOLD
)


# ==========================================
# 3. VALIDATION
# ==========================================

trigger_count = int(
    df_master["EPA_Trigger"].sum()
)

below_count = (
    len(df_master)
    - trigger_count
)

missing_edge_values = int(
    df_master["EPA_Edge_Value"]
    .isna()
    .sum()
)

missing_edge_teams = int(
    df_master["EPA_Edge_Team"]
    .isna()
    .sum()
)


print(f"\nEPA trigger threshold:      {EPA_TRIGGER_THRESHOLD:.2f}")
print(f"Target-week games checked:  {len(df_master)}")
print(f"Games triggering EPA edge:  {trigger_count}")
print(f"Games below threshold:      {below_count}")
print(f"Missing EPA edge values:    {missing_edge_values}")
print(f"Missing EPA edge teams:     {missing_edge_teams}")


# ==========================================
# 4. FINAL STATUS
# ==========================================

print("\n" + "-" * 65)

if (
    missing_edge_values == 0
    and missing_edge_teams == 0
    and trigger_count + below_count == len(df_master)
):

    print("CFB1 NET EPA TRIGGER: PASS")

else:

    print("CFB1 NET EPA TRIGGER: STOP")
    raise ValueError(
        "Net EPA trigger validation failed."
    )

print("=" * 65)


# ==========================================
# 5. VISUAL CHECK — STRONGEST EPA EDGES
# ==========================================

display(
    df_master[
        [
            "Away_Team",
            "Home_Team",
            "DK_Spread",
            "FD_Spread",
            "Away_Net_EPA_Margin",
            "Home_Net_EPA_Margin",
            "Net_EPA_Edge_Home",
            "EPA_Edge_Team",
            "EPA_Edge_Value",
            "EPA_Trigger"
        ]
    ]
    .sort_values(
        "EPA_Edge_Value",
        ascending=False
    )
    .head(15)
)

CFB1 NET EPA TRIGGER

EPA trigger threshold:      0.12
Target-week games checked:  57
Games triggering EPA edge:  33
Games below threshold:      24
Missing EPA edge values:    0
Missing EPA edge teams:     0

-----------------------------------------------------------------
CFB1 NET EPA TRIGGER: PASS


,Away_Team,Home_Team,DK_Spread,FD_Spread,Away_Net_EPA_Margin,Home_Net_EPA_Margin,Net_EPA_Edge_Home,EPA_Edge_Team,EPA_Edge_Value,EPA_Trigger
32,Stanford Cardinal,Notre Dame Fighting Irish,-38.5,-37.5,-0.259711,0.384058,0.643768,Notre Dame Fighting Irish,0.643768,True
47,Air Force Falcons,Northern Illinois Huskies,7.5,7.5,0.189115,-0.279218,-0.468333,Air Force Falcons,0.468333,True
19,Ball State Cardinals,Northwestern Wildcats,-36.5,-36.5,-0.245019,0.222243,0.467262,Northwestern Wildcats,0.467262,True
40,Tennessee Volunteers,Arkansas Razorbacks,13.5,13.5,0.290179,-0.152262,-0.442441,Tennessee Volunteers,0.442441,True
25,Charlotte 49ers,North Texas Mean Green,-28.5,-28.5,-0.293648,0.148227,0.441874,North Texas Mean Green,0.441874,True
39,Maryland Terrapins,Ohio State Buckeyes,-34.5,-34.5,0.001977,0.434594,0.432617,Ohio State Buckeyes,0.432617,True
44,Nevada Wolf Pack,UTEP Miners,9.5,8.5,-0.055660,-0.467836,-0.412176,Nevada Wolf Pack,0.412176,True
41,San Diego State Aztecs,Oregon State Beavers,-14.5,-14.5,-0.180412,0.207422,0.387834,Oregon State Beavers,0.387834,True
49,James Madison Dukes,Georgia Southern Eagles,7.5,7.5,0.422448,0.047463,-0.374985,James Madison Dukes,0.374985,True
26,Duke Blue Devils,Georgia Tech Yellow Jackets,6.5,6.5,0.359083,-0.013925,-0.373008,Duke Blue Devils,0.373008,True


In [ ]:
# ==========================================
# CELL 13 — BCF TOYS WEEKLY INPUT + STRUCTURAL VALIDATION
# ==========================================

from io import StringIO

print("=" * 70)
print("CFB1 BCF TOYS WEEKLY INPUT")
print("=" * 70)

print(f"Target handicapping week: {TARGET_WEEK}")
print(f"BCF data through:          Week {DATA_THROUGH_WEEK}")


# ==========================================
# 1. WEEKLY BCF INPUT
# ==========================================
#
# Paste the Gem's CSV exactly as supplied.
# DO NOT manually rename BCF teams here.
# ==========================================

bcf_csv = """
Team,FEI,OFEI,DFEI,SFEI,OAY,DAY
Georgia,1.50,.67,.74,.14,.595,.247
Notre Dame,1.44,.67,.73,.08,.671,.284
Ohio State,1.40,.74,.68,-.02,.673,.326
Alabama,1.33,.67,.63,.07,.618,.375
Utah,1.23,.56,.57,.10,.659,.316
Miami,1.18,.71,.50,.01,.674,.360
Indiana,1.14,.61,.51,.05,.752,.398
Texas,1.08,.30,.76,.04,.493,.338
Oregon,1.06,.57,.41,.07,.537,.558
LSU,.89,.24,.66,.02,.587,.236
Tennessee,.88,.35,.47,.07,.465,.345
USC,.85,.75,.01,.06,.708,.464
Penn State,.77,.25,.45,.05,.510,.338
BYU,.74,.31,.40,.05,.498,.380
Texas A&M,.73,.19,.33,.17,.457,.435
Missouri,.71,.18,.48,.05,.535,.367
Ole Miss,.70,.54,.09,.07,.540,.538
Iowa,.68,-.11,.61,.16,.430,.339
Oklahoma,.66,-.07,.65,.08,.381,.309
Kansas State,.64,.35,.32,.00,.633,.388
Wisconsin,.63,-.01,.61,.01,.486,.334
Texas Tech,.62,.00,.56,.06,.455,.389
Mississippi State,.62,.46,.19,-.02,.585,.473
Boise State,.60,.30,.24,.07,.593,.527
Nebraska,.58,.33,.21,.03,.614,.369
Northwestern,.56,.23,.29,.06,.553,.495
Arizona,.56,.30,.25,.03,.611,.427
Florida State,.55,.41,.09,.03,.507,.538
Michigan,.53,.15,.40,-.02,.411,.346
Florida,.53,.43,.04,.09,.654,.548
Washington,.51,.05,.42,.03,.400,.384
Auburn,.51,-.01,.46,.07,.461,.442
Pittsburgh,.51,.07,.37,.06,.491,.360
UCLA,.50,.29,.12,.05,.612,.390
South Carolina,.50,.20,.25,.05,.525,.468
Iowa State,.49,.22,.19,.09,.481,.438
Kentucky,.49,.09,.38,.01,.498,.475
Wake Forest,.47,.24,.27,.01,.617,.423
Louisville,.45,.22,.23,.00,.522,.557
Baylor,.40,.22,.11,.07,.517,.443
Duke,.34,.16,.17,.04,.554,.361
Houston,.33,.34,.02,.00,.683,.450
Vanderbilt,.30,.24,.03,.03,.448,.445
SMU,.30,.20,.19,-.09,.600,.410
Illinois,.30,.37,-.10,.03,.642,.634
James Madison,.29,.07,.19,.04,.506,.390
Kansas,.26,.11,.09,.06,.439,.348
Minnesota,.26,-.12,.27,.10,.444,.344
New Mexico,.24,-.04,.21,.02,.580,.266
TCU,.22,-.13,.30,.03,.425,.381
Virginia,.22,-.03,.27,-.01,.459,.357
Virginia Tech,.21,.17,.00,.05,.564,.355
UCF,.17,-.17,.34,.02,.373,.476
Clemson,.17,-.27,.37,.06,.377,.427
Cincinnati,.15,.14,-.01,.06,.494,.539
Memphis,.11,.25,-.14,-.02,.593,.549
Arizona State,.10,-.14,.23,.01,.395,.480
NC State,.10,.11,.02,.01,.495,.524
Oklahoma State,.07,-.11,.16,.02,.606,.450
West Virginia,.01,.16,-.16,.01,.543,.538
North Carolina,.01,.00,-.02,.04,.435,.528
Michigan State,-.01,-.07,-.07,.12,.478,.499
Oregon State,-.02,.10,-.05,-.08,.571,.473
Purdue,-.03,.10,-.18,.04,.527,.682
Fresno State,-.03,-.27,.20,.05,.477,.369
Arkansas,-.03,.05,-.12,.01,.384,.540
Toledo,-.04,.01,-.01,-.03,.581,.469
Georgia Tech,-.05,.01,-.15,.08,.474,.573
Liberty,-.07,-.02,-.09,.01,.635,.365
North Texas,-.09,.29,-.25,-.10,.587,.541
Washington State,-.09,-.29,.19,-.03,.280,.579
UTSA,-.10,-.07,.01,-.03,.434,.478
South Florida,-.10,-.17,.12,-.03,.427,.431
Army,-.13,.11,-.28,.05,.581,.663
Maryland,-.14,-.25,.09,.02,.444,.509
East Carolina,-.17,-.28,.15,-.05,.323,.427
Tulane,-.17,-.18,-.01,.07,.407,.501
Utah State,-.17,-.15,-.01,-.02,.389,.528
Colorado,-.21,-.34,.22,-.11,.381,.454
Georgia State,-.24,-.04,-.17,-.02,.543,.426
Boston College,-.24,-.32,.11,-.04,.398,.448
UNLV,-.25,-.21,.00,-.05,.459,.428
California,-.26,-.25,-.05,.02,.392,.505
Texas State,-.28,.11,-.25,-.15,.582,.565
San Diego State,-.30,-.55,.20,.05,.318,.509
Western Michigan,-.30,-.28,-.04,.03,.451,.466
Air Force,-.30,.07,-.28,-.09,.495,.475
Navy,-.32,-.23,-.06,-.06,.400,.467
Syracuse,-.32,-.22,-.04,-.06,.526,.522
Louisiana Tech,-.33,-.27,-.01,-.05,.423,.585
North Dakota State,-.33,-.33,-.10,.10,.550,.329
Troy,-.34,-.39,.10,-.05,.396,.417
Rutgers,-.37,-.04,-.33,.01,.464,.629
Temple,-.40,-.09,-.28,-.08,.505,.595
Tulsa,-.41,-.30,.00,-.11,.406,.499
Louisiana,-.44,-.34,.01,-.07,.455,.350
Ohio,-.44,-.26,-.27,.08,.504,.611
South Alabama,-.45,.02,-.46,-.02,.548,.606
Connecticut,-.45,-.08,-.38,.01,.474,.575
Appalachian State,-.47,-.18,-.28,-.02,.470,.450
Old Dominion,-.50,-.54,.05,-.04,.259,.498
Miami (OH),-.50,-.21,-.31,.03,.509,.578
Florida Atlantic,-.53,-.17,-.34,-.01,.520,.537
Georgia Southern,-.53,-.06,-.32,-.15,.444,.486
Stanford,-.56,-.24,-.32,.01,.395,.565
Marshall,-.56,-.30,-.27,-.01,.442,.609
Jacksonville State,-.59,-.35,-.22,-.02,.545,.527
Nevada,-.59,-.40,-.18,-.02,.548,.446
San Jose State,-.62,-.48,-.07,-.08,.351,.449
Arkansas State,-.65,-.30,-.36,.02,.514,.574
Missouri State,-.65,-.18,-.44,-.02,.407,.724
UAB,-.68,-.33,-.27,-.08,.445,.481
Coastal Carolina,-.68,-.44,-.24,-.04,.428,.561
Colorado State,-.69,-.07,-.59,-.05,.492,.666
Florida International,-.72,-.61,-.01,-.11,.493,.334
Hawaii,-.73,-.46,-.32,.04,.423,.477
Delaware,-.74,-.50,-.26,.01,.328,.542
Central Michigan,-.74,-.32,-.34,-.05,.398,.611
New Mexico State,-.76,-.43,-.24,-.07,.464,.509
Rice,-.76,-.38,-.31,-.08,.375,.574
Kennesaw State,-.77,-.38,-.33,-.06,.359,.672
Sam Houston,-.78,-.48,-.25,-.07,.356,.453
Eastern Michigan,-.81,-.40,-.34,-.09,.359,.474
Western Kentucky,-.81,-.50,-.26,-.06,.271,.616
Bowling Green,-.81,-.56,-.11,-.11,.337,.566
Southern Mississippi,-.81,-.54,-.21,-.07,.317,.542
Buffalo,-.84,-.58,-.27,-.02,.305,.602
Wyoming,-.88,-.65,-.25,.01,.432,.547
Middle Tennessee,-.89,-.49,-.38,-.04,.388,.551
Northern Illinois,-.91,-.41,-.34,-.17,.366,.649
Kent State,-.97,-.52,-.41,-.05,.325,.567
Ball State,-1.06,-.49,-.49,-.08,.332,.658
Massachusetts,-1.11,-.58,-.52,-.03,.504,.478
Charlotte,-1.11,-.72,-.29,-.07,.317,.575
UL Monroe,-1.19,-.49,-.58,-.14,.417,.734
Akron,-1.29,-.63,-.51,-.16,.321,.621
UTEP,-1.30,-.67,-.48,-.15,.186,.742
Sacramento State,-1.47,-.86,-.64,.01,.322,.534
"""


# ==========================================
# 2. BASIC INPUT CHECK
# ==========================================

bcf_text = bcf_csv.strip()

if (
    not bcf_text
    or bcf_text == "PASTE THE GEM CSV HERE"
):
    raise ValueError(
        "BCF INPUT MISSING — paste the Gem CSV "
        "between the triple quotes before continuing."
    )


# ==========================================
# 3. LOAD RAW BCF DATA
# ==========================================

df_bcf_raw = pd.read_csv(
    StringIO(bcf_text)
)


# ==========================================
# 4. REQUIRED COLUMN VALIDATION
# ==========================================

required_bcf_columns = [
    "Team",
    "FEI",
    "OFEI",
    "DFEI",
    "SFEI",
    "OAY",
    "DAY"
]

missing_bcf_columns = [
    col
    for col in required_bcf_columns
    if col not in df_bcf_raw.columns
]


# ==========================================
# 5. DUPLICATE TEAM CHECK
# ==========================================

duplicate_bcf_teams = pd.DataFrame()

if "Team" in df_bcf_raw.columns:

    duplicate_bcf_teams = df_bcf_raw[
        df_bcf_raw.duplicated(
            subset=["Team"],
            keep=False
        )
    ]


# ==========================================
# 6. NUMERIC VALIDATION
# ==========================================

numeric_bcf_columns = [
    "FEI",
    "OFEI",
    "DFEI",
    "SFEI",
    "OAY",
    "DAY"
]

numeric_problems = {}

if len(missing_bcf_columns) == 0:

    for col in numeric_bcf_columns:

        converted = pd.to_numeric(
            df_bcf_raw[col],
            errors="coerce"
        )

        numeric_problems[col] = int(
            converted.isna().sum()
        )

        df_bcf_raw[col] = converted


total_numeric_problems = (
    sum(numeric_problems.values())
    if numeric_problems
    else 0
)


# ==========================================
# 7. CLEAN WHITESPACE ONLY
# ==========================================
#
# Preserve BCF naming.
# Do not convert to CFBD names yet.
# ==========================================

if "Team" in df_bcf_raw.columns:

    df_bcf_raw["Team"] = (
        df_bcf_raw["Team"]
        .astype(str)
        .str.strip()
    )


# ==========================================
# 8. REPORT
# ==========================================

print("\nSTRUCTURAL VALIDATION")
print("-" * 70)

print(f"BCF rows loaded:              {len(df_bcf_raw)}")
print(f"Required columns missing:     {len(missing_bcf_columns)}")
print(f"Duplicate BCF team rows:      {len(duplicate_bcf_teams)}")
print(f"Numeric conversion problems:  {total_numeric_problems}")


if missing_bcf_columns:

    print("\nMissing required columns:")
    print(missing_bcf_columns)


if len(duplicate_bcf_teams) > 0:

    print("\nDuplicate BCF teams:")
    display(
        duplicate_bcf_teams[
            required_bcf_columns
        ]
    )


if total_numeric_problems > 0:

    print("\nNumeric problems by column:")
    print(numeric_problems)


# ==========================================
# 9. FINAL STATUS
# ==========================================

print("\n" + "-" * 70)

if (
    len(df_bcf_raw) > 0
    and len(missing_bcf_columns) == 0
    and len(duplicate_bcf_teams) == 0
    and total_numeric_problems == 0
):

    print("CFB1 BCF RAW INPUT: PASS")

else:

    print("CFB1 BCF RAW INPUT: STOP")

    raise ValueError(
        "BCF raw input validation failed. "
        "Do not continue to BCF normalization."
    )

print("=" * 70)


# ==========================================
# 10. VISUAL CHECK
# ==========================================

display(
    df_bcf_raw.head(10)
)

CFB1 BCF TOYS WEEKLY INPUT
Target handicapping week: 6
BCF data through:          Week 5

STRUCTURAL VALIDATION
----------------------------------------------------------------------
BCF rows loaded:              138
Required columns missing:     0
Duplicate BCF team rows:      0
Numeric conversion problems:  0

----------------------------------------------------------------------
CFB1 BCF RAW INPUT: PASS


,Team,FEI,OFEI,DFEI,SFEI,OAY,DAY
0,Georgia,1.50,0.67,0.74,0.14,0.595,0.247
1,Notre Dame,1.44,0.67,0.73,0.08,0.671,0.284
2,Ohio State,1.40,0.74,0.68,-0.02,0.673,0.326
3,Alabama,1.33,0.67,0.63,0.07,0.618,0.375
4,Utah,1.23,0.56,0.57,0.10,0.659,0.316
5,Miami,1.18,0.71,0.50,0.01,0.674,0.360
6,Indiana,1.14,0.61,0.51,0.05,0.752,0.398
7,Texas,1.08,0.30,0.76,0.04,0.493,0.338
8,Oregon,1.06,0.57,0.41,0.07,0.537,0.558
9,LSU,0.89,0.24,0.66,0.02,0.587,0.236


In [ ]:
# ==========================================
# CELL 14 — BCF -> CFBD NAME NORMALIZATION
# ==========================================

print("=" * 70)
print("CFB1 BCF -> CFBD NAME NORMALIZATION")
print("=" * 70)


# ==========================================
# 1. BCF-SPECIFIC NAME MAP
# ==========================================
#
# Preserve the raw BCF name in df_bcf_raw.
# Create a separate CFBD-compatible name.
# ==========================================

BCF_TO_CFBD = {
    "Appalachian State": "App State",
    "Connecticut": "UConn",
    "Hawaii": "Hawai'i",
    "San Jose State": "San José State",
    "Southern Mississippi": "Southern Miss",
}


# ==========================================
# 2. CREATE NORMALIZED BCF TABLE
# ==========================================

df_bcf = df_bcf_raw.copy()

df_bcf["CFBD_Team"] = (
    df_bcf["Team"]
    .map(BCF_TO_CFBD)
    .fillna(df_bcf["Team"])
)


# ==========================================
# 3. CHECK AGAINST CFBD UNIVERSE
# ==========================================

bcf_unmatched = df_bcf[
    ~df_bcf["CFBD_Team"].isin(cfbd_teams)
].copy()


# ==========================================
# 4. CHECK TARGET-WEEK COVERAGE
# ==========================================

target_cfbd_teams = sorted(
    set(
        df_target_market["Home_CFBD_Name"].tolist()
        +
        df_target_market["Away_CFBD_Name"].tolist()
    )
)

normalized_bcf_team_set = set(
    df_bcf["CFBD_Team"]
)

missing_target_bcf_teams = [
    team
    for team in target_cfbd_teams
    if team not in normalized_bcf_team_set
]


# ==========================================
# 5. DUPLICATE NORMALIZED NAME CHECK
# ==========================================

duplicate_normalized_names = df_bcf[
    df_bcf.duplicated(
        subset=["CFBD_Team"],
        keep=False
    )
]


# ==========================================
# 6. REPORT
# ==========================================

print(f"\nBCF rows normalized:              {len(df_bcf)}")
print(f"BCF names unmatched to CFBD:      {len(bcf_unmatched)}")
print(f"Duplicate normalized BCF names:   {len(duplicate_normalized_names)}")
print(f"Target-week teams missing in BCF: {len(missing_target_bcf_teams)}")


if len(bcf_unmatched) > 0:

    print("\nBCF NAMES NOT MATCHING CFBD:")
    display(
        bcf_unmatched[
            [
                "Team",
                "CFBD_Team"
            ]
        ]
    )


if len(duplicate_normalized_names) > 0:

    print("\nDUPLICATE NORMALIZED BCF NAMES:")
    display(
        duplicate_normalized_names[
            [
                "Team",
                "CFBD_Team"
            ]
        ]
    )


if len(missing_target_bcf_teams) > 0:

    print("\nTARGET-WEEK TEAMS MISSING FROM NORMALIZED BCF:")
    for team in missing_target_bcf_teams:
        print(" -", team)


# ==========================================
# 7. FINAL STATUS
# ==========================================

print("\n" + "-" * 70)

if (
    len(bcf_unmatched) == 0
    and len(duplicate_normalized_names) == 0
    and len(missing_target_bcf_teams) == 0
):

    print("CFB1 BCF NAME NORMALIZATION: PASS")

else:

    print("CFB1 BCF NAME NORMALIZATION: STOP")
    raise ValueError(
        "BCF name normalization failed. "
        "Do not continue to BCF merge."
    )

print("=" * 70)

CFB1 BCF -> CFBD NAME NORMALIZATION

BCF rows normalized:              138
BCF names unmatched to CFBD:      0
Duplicate normalized BCF names:   0
Target-week teams missing in BCF: 0

----------------------------------------------------------------------
CFB1 BCF NAME NORMALIZATION: PASS


In [ ]:
# ==========================================
# OLD CELL 19 — MERGE BCF DATA INTO MASTER CARD
# ==========================================

print("=" * 70)
print("CFB1 BCF -> MASTER CARD MERGE")
print("=" * 70)


# ==========================================
# 1. PREPARE HOME BCF DATA
# ==========================================

bcf_home = df_bcf[
    [
        "CFBD_Team",
        "FEI",
        "OFEI",
        "DFEI",
        "SFEI",
        "OAY",
        "DAY"
    ]
].copy()

bcf_home = bcf_home.rename(columns={
    "CFBD_Team": "Home_CFBD_Name",
    "FEI": "Home_FEI",
    "OFEI": "Home_OFEI",
    "DFEI": "Home_DFEI",
    "SFEI": "Home_SFEI",
    "OAY": "Home_OAY",
    "DAY": "Home_DAY"
})


# ==========================================
# 2. PREPARE AWAY BCF DATA
# ==========================================

bcf_away = df_bcf[
    [
        "CFBD_Team",
        "FEI",
        "OFEI",
        "DFEI",
        "SFEI",
        "OAY",
        "DAY"
    ]
].copy()

bcf_away = bcf_away.rename(columns={
    "CFBD_Team": "Away_CFBD_Name",
    "FEI": "Away_FEI",
    "OFEI": "Away_OFEI",
    "DFEI": "Away_DFEI",
    "SFEI": "Away_SFEI",
    "OAY": "Away_OAY",
    "DAY": "Away_DAY"
})


# ==========================================
# 3. MERGE HOME BCF DATA
# ==========================================

df_master_bcf = df_master.merge(
    bcf_home,
    on="Home_CFBD_Name",
    how="left",
    validate="many_to_one"
)


# ==========================================
# 4. MERGE AWAY BCF DATA
# ==========================================

df_master_bcf = df_master_bcf.merge(
    bcf_away,
    on="Away_CFBD_Name",
    how="left",
    validate="many_to_one"
)


# ==========================================
# 5. VALIDATION
# ==========================================

expected_games = len(df_master)

missing_home_bcf = df_master_bcf[
    df_master_bcf["Home_FEI"].isna()
]

missing_away_bcf = df_master_bcf[
    df_master_bcf["Away_FEI"].isna()
]

duplicate_games = df_master_bcf[
    df_master_bcf.duplicated(
        subset=["Home_Team", "Away_Team"],
        keep=False
    )
]


print(f"\nExpected games:              {expected_games}")
print(f"Games after BCF merge:       {len(df_master_bcf)}")
print(f"Missing home BCF records:    {len(missing_home_bcf)}")
print(f"Missing away BCF records:    {len(missing_away_bcf)}")
print(f"Duplicate games after merge: {len(duplicate_games)}")


if len(missing_home_bcf) > 0:

    print("\nHOME TEAMS MISSING BCF DATA:")

    display(
        missing_home_bcf[
            [
                "Home_Team",
                "Home_CFBD_Name"
            ]
        ]
    )


if len(missing_away_bcf) > 0:

    print("\nAWAY TEAMS MISSING BCF DATA:")

    display(
        missing_away_bcf[
            [
                "Away_Team",
                "Away_CFBD_Name"
            ]
        ]
    )


# ==========================================
# 6. FINAL STATUS
# ==========================================

print("\n" + "-" * 70)

if (
    len(df_master_bcf) == expected_games
    and len(missing_home_bcf) == 0
    and len(missing_away_bcf) == 0
    and len(duplicate_games) == 0
):

    print("CFB1 BCF -> MASTER CARD MERGE: PASS")

else:

    print("CFB1 BCF -> MASTER CARD MERGE: STOP")

    raise ValueError(
        "BCF merge validation failed. "
        "Do not continue to FEI calculations."
    )

print("=" * 70)


# ==========================================
# 7. VISUAL CHECK
# ==========================================

display(
    df_master_bcf[
        [
            "Away_Team",
            "Home_Team",
            "Away_FEI",
            "Home_FEI",
            "Away_OFEI",
            "Home_OFEI",
            "Away_DFEI",
            "Home_DFEI"
        ]
    ]
    .head(10)
)

CFB1 BCF -> MASTER CARD MERGE

Expected games:              57
Games after BCF merge:       57
Missing home BCF records:    0
Missing away BCF records:    0
Duplicate games after merge: 0

----------------------------------------------------------------------
CFB1 BCF -> MASTER CARD MERGE: PASS


,Away_Team,Home_Team,Away_FEI,Home_FEI,Away_OFEI,Home_OFEI,Away_DFEI,Home_DFEI
0,Jacksonville State Gamecocks,Kennesaw State Owls,-0.59,-0.77,-0.35,-0.38,-0.22,-0.33
1,New Mexico State Aggies,Florida International Panthers,-0.76,-0.72,-0.43,-0.61,-0.24,-0.01
2,Missouri State Bears,Western Kentucky Hilltoppers,-0.65,-0.81,-0.18,-0.50,-0.44,-0.26
3,Sam Houston State Bearkats,Liberty Flames,-0.78,-0.07,-0.48,-0.02,-0.25,-0.09
4,South Alabama Jaguars,Arkansas State Red Wolves,-0.45,-0.65,0.02,-0.30,-0.46,-0.36
5,South Florida Bulls,UTSA Roadrunners,-0.10,-0.10,-0.17,-0.07,0.12,0.01
6,Florida State Seminoles,Louisville Cardinals,0.55,0.45,0.41,0.22,0.09,0.23
7,Iowa Hawkeyes,Washington Huskies,0.68,0.51,-0.11,0.05,0.61,0.42
8,Washington State Cougars,Utah State Aggies,-0.09,-0.17,-0.29,-0.15,0.19,-0.01
9,Wyoming Cowboys,San Jose State Spartans,-0.88,-0.62,-0.65,-0.48,-0.25,-0.07


In [ ]:
# ==========================================
# OLD CELL 20 — FEI MODEL DIFFERENTIAL + POLARITY
# ==========================================

print("=" * 70)
print("CFB1 FEI MODEL DIFFERENTIAL + POLARITY")
print("=" * 70)


# ==========================================
# 1. ESTABLISHED CFB1 FEI CONVERSION
# ==========================================

FEI_MULTIPLIER = 30


# ==========================================
# 2. CALCULATE HOME-PERSPECTIVE FEI DIFFERENTIAL
# ==========================================
#
# Positive = HOME team stronger
# Negative = AWAY team stronger
# ==========================================

df_master_bcf["FEI_Model_Diff_Home"] = (
    (
        df_master_bcf["Home_FEI"]
        - df_master_bcf["Away_FEI"]
    )
    * FEI_MULTIPLIER
).round(1)


# ==========================================
# 3. IDENTIFY FEI MODEL FAVORITE
# ==========================================

def fei_favorite(row):

    if row["FEI_Model_Diff_Home"] > 0:
        return row["Home_Team"]

    if row["FEI_Model_Diff_Home"] < 0:
        return row["Away_Team"]

    return "EVEN"


df_master_bcf["FEI_Model_Favorite"] = (
    df_master_bcf.apply(
        fei_favorite,
        axis=1
    )
)


# ==========================================
# 4. ABSOLUTE FEI MODEL MARGIN
# ==========================================

df_master_bcf["FEI_Model_Margin"] = (
    df_master_bcf[
        "FEI_Model_Diff_Home"
    ].abs()
)


# ==========================================
# 5. VALIDATION
# ==========================================

missing_fei_model = df_master_bcf[
    df_master_bcf[
        "FEI_Model_Diff_Home"
    ].isna()
]

polarity_errors = df_master_bcf[
    (
        (
            df_master_bcf[
                "FEI_Model_Diff_Home"
            ] > 0
        )
        &
        (
            df_master_bcf[
                "FEI_Model_Favorite"
            ]
            !=
            df_master_bcf[
                "Home_Team"
            ]
        )
    )
    |
    (
        (
            df_master_bcf[
                "FEI_Model_Diff_Home"
            ] < 0
        )
        &
        (
            df_master_bcf[
                "FEI_Model_Favorite"
            ]
            !=
            df_master_bcf[
                "Away_Team"
            ]
        )
    )
]

even_errors = df_master_bcf[
    (
        df_master_bcf[
            "FEI_Model_Diff_Home"
        ] == 0
    )
    &
    (
        df_master_bcf[
            "FEI_Model_Favorite"
        ] != "EVEN"
    )
]


# ==========================================
# 6. REPORT
# ==========================================

print(f"\nFEI multiplier:              {FEI_MULTIPLIER}")
print(f"Games tested:                {len(df_master_bcf)}")
print(f"Missing FEI model values:    {len(missing_fei_model)}")
print(f"Polarity errors:             {len(polarity_errors)}")
print(f"Even-game labeling errors:   {len(even_errors)}")


# ==========================================
# 7. FINAL STATUS
# ==========================================

print("\n" + "-" * 70)

if (
    len(missing_fei_model) == 0
    and len(polarity_errors) == 0
    and len(even_errors) == 0
):

    print("CFB1 FEI MODEL POLARITY: PASS")

else:

    print("CFB1 FEI MODEL POLARITY: STOP")

    raise ValueError(
        "FEI model polarity validation failed."
    )

print("=" * 70)


# ==========================================
# 8. VISUAL CHECK
# ==========================================

display(
    df_master_bcf[
        [
            "Away_Team",
            "Home_Team",
            "Away_FEI",
            "Home_FEI",
            "FEI_Model_Diff_Home",
            "FEI_Model_Favorite",
            "FEI_Model_Margin"
        ]
    ]
    .head(12)
)

CFB1 FEI MODEL DIFFERENTIAL + POLARITY

FEI multiplier:              30
Games tested:                57
Missing FEI model values:    0
Polarity errors:             0
Even-game labeling errors:   0

----------------------------------------------------------------------
CFB1 FEI MODEL POLARITY: PASS


,Away_Team,Home_Team,Away_FEI,Home_FEI,FEI_Model_Diff_Home,FEI_Model_Favorite,FEI_Model_Margin
0,Jacksonville State Gamecocks,Kennesaw State Owls,-0.59,-0.77,-5.4,Jacksonville State Gamecocks,5.4
1,New Mexico State Aggies,Florida International Panthers,-0.76,-0.72,1.2,Florida International Panthers,1.2
2,Missouri State Bears,Western Kentucky Hilltoppers,-0.65,-0.81,-4.8,Missouri State Bears,4.8
3,Sam Houston State Bearkats,Liberty Flames,-0.78,-0.07,21.3,Liberty Flames,21.3
4,South Alabama Jaguars,Arkansas State Red Wolves,-0.45,-0.65,-6.0,South Alabama Jaguars,6.0
5,South Florida Bulls,UTSA Roadrunners,-0.10,-0.10,0.0,EVEN,0.0
6,Florida State Seminoles,Louisville Cardinals,0.55,0.45,-3.0,Florida State Seminoles,3.0
7,Iowa Hawkeyes,Washington Huskies,0.68,0.51,-5.1,Iowa Hawkeyes,5.1
8,Washington State Cougars,Utah State Aggies,-0.09,-0.17,-2.4,Washington State Cougars,2.4
9,Wyoming Cowboys,San Jose State Spartans,-0.88,-0.62,7.8,San Jose State Spartans,7.8


In [ ]:
# ==========================================
# OLD CELL 21 — FEI VS MARKET DIAGNOSTIC
# ==========================================

print("=" * 72)
print("CFB1 FEI VS MARKET DIAGNOSTIC")
print("=" * 72)


# ==========================================
# 1. CONVERT FEI DIFFERENTIAL TO
#    SPORTSBOOK-STYLE HOME SPREAD
# ==========================================
#
# FEI_Model_Diff_Home:
#   positive = HOME stronger
#   negative = AWAY stronger
#
# Sportsbook home spread:
#   negative = HOME favored
#   positive = HOME underdog
#
# Example:
#   FEI_Model_Diff_Home = +7
#   FEI_Model_Home_Spread = -7
# ==========================================

df_master_bcf["FEI_Model_Home_Spread"] = (
    -df_master_bcf["FEI_Model_Diff_Home"]
).round(1)


# ==========================================
# 2. COMPARE FEI MODEL TO EACH SPORTSBOOK
# ==========================================
#
# Positive edge = value on HOME
# Negative edge = value on AWAY
#
# Example:
#   Model home spread = -7
#   Market home spread = -3
#
#   -3 - (-7) = +4
#
#   Model sees 4 points of HOME value.
# ==========================================

df_master_bcf["DK_FEI_Edge_Home"] = (
    df_master_bcf["DK_Spread"]
    - df_master_bcf["FEI_Model_Home_Spread"]
).round(1)

df_master_bcf["FD_FEI_Edge_Home"] = (
    df_master_bcf["FD_Spread"]
    - df_master_bcf["FEI_Model_Home_Spread"]
).round(1)


# ==========================================
# 3. EDGE MAGNITUDE
# ==========================================

df_master_bcf["DK_FEI_Edge_Value"] = (
    df_master_bcf["DK_FEI_Edge_Home"].abs()
)

df_master_bcf["FD_FEI_Edge_Value"] = (
    df_master_bcf["FD_FEI_Edge_Home"].abs()
)


# ==========================================
# 4. IDENTIFY EDGE TEAM
# ==========================================

def edge_team(row, edge_col):

    edge = row[edge_col]

    if edge > 0:
        return row["Home_Team"]

    if edge < 0:
        return row["Away_Team"]

    return "EVEN"


df_master_bcf["DK_FEI_Edge_Team"] = (
    df_master_bcf.apply(
        lambda row:
            edge_team(
                row,
                "DK_FEI_Edge_Home"
            ),
        axis=1
    )
)

df_master_bcf["FD_FEI_Edge_Team"] = (
    df_master_bcf.apply(
        lambda row:
            edge_team(
                row,
                "FD_FEI_Edge_Home"
            ),
        axis=1
    )
)


# ==========================================
# 5. BOOK AGREEMENT
# ==========================================

df_master_bcf["FEI_Books_Agree"] = (
    df_master_bcf["DK_FEI_Edge_Team"]
    ==
    df_master_bcf["FD_FEI_Edge_Team"]
)


# ==========================================
# 6. VALIDATION
# ==========================================

required_market_fields = [
    "FEI_Model_Home_Spread",
    "DK_Spread",
    "FD_Spread",
    "DK_FEI_Edge_Home",
    "FD_FEI_Edge_Home"
]

missing_market_comparisons = df_master_bcf[
    df_master_bcf[
        required_market_fields
    ].isna().any(axis=1)
]

agreement_count = int(
    df_master_bcf[
        "FEI_Books_Agree"
    ].sum()
)

disagreement_count = (
    len(df_master_bcf)
    - agreement_count
)


# ==========================================
# 7. REPORT
# ==========================================

print(f"\nGames tested:                  {len(df_master_bcf)}")
print(f"Missing market comparisons:    {len(missing_market_comparisons)}")
print(f"DK / FD directional agreement: {agreement_count}")
print(f"DK / FD directional conflict:  {disagreement_count}")


# ==========================================
# 8. FINAL STATUS
# ==========================================

print("\n" + "-" * 72)

if len(missing_market_comparisons) == 0:

    print("CFB1 FEI MARKET COMPARISON: PASS")

else:

    print("CFB1 FEI MARKET COMPARISON: STOP")

    display(
        missing_market_comparisons[
            [
                "Away_Team",
                "Home_Team",
                "FEI_Model_Home_Spread",
                "DK_Spread",
                "FD_Spread"
            ]
        ]
    )

    raise ValueError(
        "FEI market comparison contains missing values."
    )

print("=" * 72)


# ==========================================
# 9. VISUAL CHECK
# ==========================================

display(
    df_master_bcf[
        [
            "Away_Team",
            "Home_Team",
            "FEI_Model_Home_Spread",
            "DK_Spread",
            "FD_Spread",
            "DK_FEI_Edge_Home",
            "FD_FEI_Edge_Home",
            "DK_FEI_Edge_Team",
            "FD_FEI_Edge_Team",
            "FEI_Books_Agree"
        ]
    ]
    .head(15)
)

CFB1 FEI VS MARKET DIAGNOSTIC

Games tested:                  57
Missing market comparisons:    0
DK / FD directional agreement: 57
DK / FD directional conflict:  0

------------------------------------------------------------------------
CFB1 FEI MARKET COMPARISON: PASS


,Away_Team,Home_Team,FEI_Model_Home_Spread,DK_Spread,FD_Spread,DK_FEI_Edge_Home,FD_FEI_Edge_Home,DK_FEI_Edge_Team,FD_FEI_Edge_Team,FEI_Books_Agree
0,Jacksonville State Gamecocks,Kennesaw State Owls,5.4,3.0,2.5,-2.4,-2.9,Jacksonville State Gamecocks,Jacksonville State Gamecocks,True
1,New Mexico State Aggies,Florida International Panthers,-1.2,-6.5,-6.5,-5.3,-5.3,New Mexico State Aggies,New Mexico State Aggies,True
2,Missouri State Bears,Western Kentucky Hilltoppers,4.8,-2.5,-2.5,-7.3,-7.3,Missouri State Bears,Missouri State Bears,True
3,Sam Houston State Bearkats,Liberty Flames,-21.3,-13.5,-13.5,7.8,7.8,Liberty Flames,Liberty Flames,True
4,South Alabama Jaguars,Arkansas State Red Wolves,6.0,-1.5,-1.5,-7.5,-7.5,South Alabama Jaguars,South Alabama Jaguars,True
5,South Florida Bulls,UTSA Roadrunners,-0.0,-7.0,-7.5,-7.0,-7.5,South Florida Bulls,South Florida Bulls,True
6,Florida State Seminoles,Louisville Cardinals,3.0,-3.5,-3.5,-6.5,-6.5,Florida State Seminoles,Florida State Seminoles,True
7,Iowa Hawkeyes,Washington Huskies,5.1,-3.5,-3.5,-8.6,-8.6,Iowa Hawkeyes,Iowa Hawkeyes,True
8,Washington State Cougars,Utah State Aggies,2.4,-5.5,-5.5,-7.9,-7.9,Washington State Cougars,Washington State Cougars,True
9,Wyoming Cowboys,San Jose State Spartans,-7.8,-6.0,-5.5,1.8,2.3,San Jose State Spartans,San Jose State Spartans,True


In [ ]:
# ==========================================
# OLD CELL 23 — FEI VS MARKET FAVORITE
# ==========================================

print("=" * 72)
print("CFB1 FEI VS MARKET FAVORITE")
print("=" * 72)


# ==========================================
# 1. IDENTIFY MARKET FAVORITE
# ==========================================
#
# Home spread:
#   negative = HOME favorite
#   positive = AWAY favorite
#   zero     = pick'em
# ==========================================

def market_favorite_from_home_spread(row, spread_col):

    spread = row[spread_col]

    if spread < 0:
        return row["Home_Team"]

    if spread > 0:
        return row["Away_Team"]

    return "PICKEM"


df_master_bcf["DK_Market_Favorite"] = (
    df_master_bcf.apply(
        lambda row:
            market_favorite_from_home_spread(
                row,
                "DK_Spread"
            ),
        axis=1
    )
)

df_master_bcf["FD_Market_Favorite"] = (
    df_master_bcf.apply(
        lambda row:
            market_favorite_from_home_spread(
                row,
                "FD_Spread"
            ),
        axis=1
    )
)


# ==========================================
# 2. FEI VS MARKET FAVORITE AGREEMENT
# ==========================================

df_master_bcf["DK_FEI_Market_Agree"] = (
    df_master_bcf["FEI_Model_Favorite"]
    ==
    df_master_bcf["DK_Market_Favorite"]
)

df_master_bcf["FD_FEI_Market_Agree"] = (
    df_master_bcf["FEI_Model_Favorite"]
    ==
    df_master_bcf["FD_Market_Favorite"]
)


# ==========================================
# 3. VALIDATION
# ==========================================

required_fields = [
    "DK_Market_Favorite",
    "FD_Market_Favorite",
    "DK_FEI_Market_Agree",
    "FD_FEI_Market_Agree"
]

missing_values = df_master_bcf[
    df_master_bcf[
        required_fields
    ].isna().any(axis=1)
]


dk_agree = int(
    df_master_bcf[
        "DK_FEI_Market_Agree"
    ].sum()
)

fd_agree = int(
    df_master_bcf[
        "FD_FEI_Market_Agree"
    ].sum()
)

dk_disagree = (
    len(df_master_bcf)
    - dk_agree
)

fd_disagree = (
    len(df_master_bcf)
    - fd_agree
)


# ==========================================
# 4. REPORT
# ==========================================

print(f"\nGames tested:                   {len(df_master_bcf)}")
print(f"Missing required values:       {len(missing_values)}")

print("\nDRAFTKINGS")
print(f"FEI / market same favorite:     {dk_agree}")
print(f"FEI / market different favorite:{dk_disagree}")

print("\nFANDUEL")
print(f"FEI / market same favorite:     {fd_agree}")
print(f"FEI / market different favorite:{fd_disagree}")


# ==========================================
# 5. FINAL STATUS
# ==========================================

print("\n" + "-" * 72)

if len(missing_values) == 0:

    print("CFB1 FEI VS MARKET FAVORITE: PASS")

else:

    print("CFB1 FEI VS MARKET FAVORITE: STOP")

    raise ValueError(
        "FEI vs market favorite validation failed."
    )

print("=" * 72)


# ==========================================
# 6. VISUAL CHECK — DK DISAGREEMENTS
# ==========================================

display(
    df_master_bcf[
        ~df_master_bcf[
            "DK_FEI_Market_Agree"
        ]
    ][
        [
            "Away_Team",
            "Home_Team",
            "FEI_Model_Favorite",
            "FEI_Model_Home_Spread",
            "DK_Market_Favorite",
            "DK_Spread",
            "DK_FEI_Edge_Team",
            "DK_FEI_Edge_Value"
        ]
    ]
    .sort_values(
        "DK_FEI_Edge_Value",
        ascending=False
    )
)

CFB1 FEI VS MARKET FAVORITE

Games tested:                   57
Missing required values:       0

DRAFTKINGS
FEI / market same favorite:     46
FEI / market different favorite:11

FANDUEL
FEI / market same favorite:     46
FEI / market different favorite:11

------------------------------------------------------------------------
CFB1 FEI VS MARKET FAVORITE: PASS


,Away_Team,Home_Team,FEI_Model_Favorite,FEI_Model_Home_Spread,DK_Market_Favorite,DK_Spread,DK_FEI_Edge_Team,DK_FEI_Edge_Value
17,UCF Knights,Oklahoma State Cowboys,UCF Knights,3.0,Oklahoma State Cowboys,-10.5,UCF Knights,13.5
7,Iowa Hawkeyes,Washington Huskies,Iowa Hawkeyes,5.1,Washington Huskies,-3.5,Iowa Hawkeyes,8.6
8,Washington State Cougars,Utah State Aggies,Washington State Cougars,2.4,Utah State Aggies,-5.5,Washington State Cougars,7.9
4,South Alabama Jaguars,Arkansas State Red Wolves,South Alabama Jaguars,6.0,Arkansas State Red Wolves,-1.5,South Alabama Jaguars,7.5
2,Missouri State Bears,Western Kentucky Hilltoppers,Missouri State Bears,4.8,Western Kentucky Hilltoppers,-2.5,Missouri State Bears,7.3
5,South Florida Bulls,UTSA Roadrunners,EVEN,-0.0,UTSA Roadrunners,-7.0,South Florida Bulls,7.0
48,Georgia Bulldogs,Alabama Crimson Tide,Georgia Bulldogs,5.1,Alabama Crimson Tide,-1.5,Georgia Bulldogs,6.6
6,Florida State Seminoles,Louisville Cardinals,Florida State Seminoles,3.0,Louisville Cardinals,-3.5,Florida State Seminoles,6.5
45,North Dakota State Bison,UNLV Rebels,UNLV Rebels,-2.4,North Dakota State Bison,3.0,UNLV Rebels,5.4
15,Texas A&M Aggies,Missouri Tigers,Texas A&M Aggies,0.6,Missouri Tigers,-3.5,Texas A&M Aggies,4.1


In [ ]:
# ==========================================
# OLD CELL 24 — HAVOC TRAP SIGNAL
# ==========================================

print("=" * 72)
print("CFB1 HAVOC TRAP SIGNAL")
print("=" * 72)


# ==========================================
# 1. ESTABLISHED CFB1 THRESHOLDS
# ==========================================

PASSING_DOWN_SR_THRESHOLD = 0.273
DEF_HAVOC_THRESHOLD = 0.192


# ==========================================
# 2. AWAY OFFENSE VS HOME DEFENSE
# ==========================================

df_master_bcf["Away_Havoc_Trap"] = (
    (
        df_master_bcf["Away_Off_PassingDown_SR"]
        <= PASSING_DOWN_SR_THRESHOLD
    )
    &
    (
        df_master_bcf["Home_Def_Havoc_Rate"]
        >= DEF_HAVOC_THRESHOLD
    )
)


# ==========================================
# 3. HOME OFFENSE VS AWAY DEFENSE
# ==========================================

df_master_bcf["Home_Havoc_Trap"] = (
    (
        df_master_bcf["Home_Off_PassingDown_SR"]
        <= PASSING_DOWN_SR_THRESHOLD
    )
    &
    (
        df_master_bcf["Away_Def_Havoc_Rate"]
        >= DEF_HAVOC_THRESHOLD
    )
)


# ==========================================
# 4. VALIDATION
# ==========================================

missing_havoc_inputs = df_master_bcf[
    df_master_bcf[
        [
            "Away_Off_PassingDown_SR",
            "Home_Def_Havoc_Rate",
            "Home_Off_PassingDown_SR",
            "Away_Def_Havoc_Rate"
        ]
    ].isna().any(axis=1)
]

missing_havoc_outputs = df_master_bcf[
    df_master_bcf[
        [
            "Away_Havoc_Trap",
            "Home_Havoc_Trap"
        ]
    ].isna().any(axis=1)
]


# ==========================================
# 5. COUNTS
# ==========================================

away_traps = int(
    df_master_bcf["Away_Havoc_Trap"].sum()
)

home_traps = int(
    df_master_bcf["Home_Havoc_Trap"].sum()
)

games_with_any_trap = int(
    df_master_bcf[
        [
            "Away_Havoc_Trap",
            "Home_Havoc_Trap"
        ]
    ].any(axis=1).sum()
)


# ==========================================
# 6. REPORT
# ==========================================

print(f"\nGames tested:                  {len(df_master_bcf)}")
print(f"Away offenses in Havoc Trap:  {away_traps}")
print(f"Home offenses in Havoc Trap:  {home_traps}")
print(f"Games with at least one trap: {games_with_any_trap}")

print(f"\nPassing Down SR threshold:     {PASSING_DOWN_SR_THRESHOLD:.3f}")
print(f"Defensive Havoc threshold:     {DEF_HAVOC_THRESHOLD:.3f}")

print(f"\nMissing Havoc input rows:      {len(missing_havoc_inputs)}")
print(f"Missing Havoc output rows:     {len(missing_havoc_outputs)}")


# ==========================================
# 7. FINAL STATUS
# ==========================================

print("\n" + "-" * 72)

if (
    len(missing_havoc_inputs) == 0
    and len(missing_havoc_outputs) == 0
):

    print("CFB1 HAVOC TRAP SIGNAL: PASS")

else:

    print("CFB1 HAVOC TRAP SIGNAL: STOP")

    raise ValueError(
        "Havoc Trap validation failed."
    )

print("=" * 72)


# ==========================================
# 8. READABLE TRIGGER TABLE
# ==========================================

havoc_rows = []

for _, row in df_master_bcf.iterrows():

    if row["Away_Havoc_Trap"]:

        havoc_rows.append({
            "Game":
                f"{row['Away_Team']} at {row['Home_Team']}",
            "Offense_In_Trap":
                row["Away_Team"],
            "PassingDown_SR":
                row["Away_Off_PassingDown_SR"],
            "Opp_Def_Havoc":
                row["Home_Def_Havoc_Rate"]
        })

    if row["Home_Havoc_Trap"]:

        havoc_rows.append({
            "Game":
                f"{row['Away_Team']} at {row['Home_Team']}",
            "Offense_In_Trap":
                row["Home_Team"],
            "PassingDown_SR":
                row["Home_Off_PassingDown_SR"],
            "Opp_Def_Havoc":
                row["Away_Def_Havoc_Rate"]
        })


havoc_diagnostic = pd.DataFrame(havoc_rows)

if len(havoc_diagnostic) > 0:

    havoc_diagnostic = havoc_diagnostic.sort_values(
        [
            "Opp_Def_Havoc",
            "PassingDown_SR"
        ],
        ascending=[
            False,
            True
        ]
    )

    display(havoc_diagnostic)

else:

    print(
        f"\nNo Week {TARGET_WEEK} "
        "Havoc Trap matchups found."
    )

CFB1 HAVOC TRAP SIGNAL

Games tested:                  57
Away offenses in Havoc Trap:  2
Home offenses in Havoc Trap:  1
Games with at least one trap: 3

Passing Down SR threshold:     0.273
Defensive Havoc threshold:     0.192

Missing Havoc input rows:      0
Missing Havoc output rows:     0

------------------------------------------------------------------------
CFB1 HAVOC TRAP SIGNAL: PASS


,Game,Offense_In_Trap,PassingDown_SR,Opp_Def_Havoc
1,Virginia Tech Hokies at California Golden Bears,California Golden Bears,0.272000,0.252
0,Stanford Cardinal at Notre Dame Fighting Irish,Stanford Cardinal,0.231405,0.236
2,Rice Owls at East Carolina Pirates,Rice Owls,0.225806,0.194


In [ ]:
CFBD_API_KEY = CFBD_API_KEY.strip()

print("CFBD key cleaned:", bool(CFBD_API_KEY))

CFBD key cleaned: True


In [ ]:
# ==========================================
# OLD CELL 25 — ADD DEFENSIVE LINE YARDS
# ==========================================

print("=" * 70)
print("CFB1 DEFENSIVE LINE YARDS")
print("=" * 70)

print(
    f"\nUsing existing CFBD advanced data "
    f"frozen through Week {DATA_THROUGH_WEEK}."
)


# ==========================================
# 1. EXTRACT FROM EXISTING FROZEN ADV DATA
# ==========================================
#
# IMPORTANT:
# adv_json was already downloaded earlier using:
#
# startWeek = 1
# endWeek   = DATA_THROUGH_WEEK
#
# Do NOT make a new season-wide API request here.
# ==========================================

def_lineyards_rows = []

for team_row in adv_json:

    team = team_row.get("team")

    defense = (
        team_row.get("defense")
        or {}
    )

    def_lineyards = defense.get(
        "lineYards"
    )

    def_lineyards_rows.append({
        "team": team,
        "Def_LineYards": def_lineyards
    })


df_def_lineyards = pd.DataFrame(
    def_lineyards_rows
)


# ==========================================
# 2. VALIDATE SOURCE TABLE
# ==========================================

duplicate_teams = int(
    df_def_lineyards.duplicated(
        subset=["team"],
        keep=False
    ).sum()
)

missing_source_values = int(
    df_def_lineyards[
        "Def_LineYards"
    ].isna().sum()
)


# ==========================================
# 3. MERGE INTO EXISTING CFBD TABLE
# ==========================================

df_cfbd = df_cfbd.drop(
    columns=["Def_LineYards"],
    errors="ignore"
)

df_cfbd = df_cfbd.merge(
    df_def_lineyards,
    on="team",
    how="left",
    validate="one_to_one"
)

missing_after_merge = int(
    df_cfbd[
        "Def_LineYards"
    ].isna().sum()
)


# ==========================================
# 4. REPORT
# ==========================================

print(f"\nTeams in frozen advanced data: {len(df_def_lineyards)}")
print(f"Duplicate teams:               {duplicate_teams}")
print(f"Missing source Line Yards:      {missing_source_values}")
print(f"Missing after CFBD merge:       {missing_after_merge}")


# ==========================================
# 5. FINAL STATUS
# ==========================================

print("\n" + "-" * 70)

if (
    duplicate_teams == 0
    and missing_source_values == 0
    and missing_after_merge == 0
):

    print("CFB1 DEFENSIVE LINE YARDS: PASS")

else:

    print("CFB1 DEFENSIVE LINE YARDS: STOP")

    raise ValueError(
        "Defensive Line Yards validation failed."
    )

print("=" * 70)


# ==========================================
# 6. VISUAL CHECK
# ==========================================

display(
    df_cfbd[
        [
            "team",
            "Off_LineYards",
            "Def_LineYards"
        ]
    ]
    .head(15)
)

CFB1 DEFENSIVE LINE YARDS

Using existing CFBD advanced data frozen through Week 5.

Teams in frozen advanced data: 138
Duplicate teams:               0
Missing source Line Yards:      0
Missing after CFBD merge:       0

----------------------------------------------------------------------
CFB1 DEFENSIVE LINE YARDS: PASS


,team,Off_LineYards,Def_LineYards
0,Air Force,3.527711,2.931624
1,Akron,3.692568,3.212422
2,Alabama,3.163536,2.034932
3,App State,2.576866,2.620312
4,Arizona,3.474359,2.660169
5,Arizona State,2.826119,2.935938
6,Arkansas,2.655405,3.159259
7,Arkansas State,3.320548,3.518519
8,Army,3.650239,3.215254
9,Auburn,3.321084,2.797436


In [ ]:
# ==========================================
# OLD CELL 26 — REFRESH DEFENSIVE LINE YARDS
# INTO MASTER CARD
# ==========================================

print("=" * 70)
print("CFB1 DEFENSIVE LINE YARDS -> MASTER CARD")
print("=" * 70)


# ==========================================
# 1. BUILD HOME LOOKUP
# ==========================================

lineyards_home = df_cfbd[
    [
        "team",
        "Def_LineYards"
    ]
].copy()

lineyards_home = lineyards_home.rename(
    columns={
        "team": "Home_CFBD_Name",
        "Def_LineYards": "Home_Def_LineYards"
    }
)


# ==========================================
# 2. BUILD AWAY LOOKUP
# ==========================================

lineyards_away = df_cfbd[
    [
        "team",
        "Def_LineYards"
    ]
].copy()

lineyards_away = lineyards_away.rename(
    columns={
        "team": "Away_CFBD_Name",
        "Def_LineYards": "Away_Def_LineYards"
    }
)


# ==========================================
# 3. REMOVE OLD VERSIONS IF PRESENT
# ==========================================

df_master_bcf = df_master_bcf.drop(
    columns=[
        "Home_Def_LineYards",
        "Away_Def_LineYards"
    ],
    errors="ignore"
)


# ==========================================
# 4. MERGE HOME DEFENSIVE LINE YARDS
# ==========================================

df_master_bcf = df_master_bcf.merge(
    lineyards_home,
    on="Home_CFBD_Name",
    how="left",
    validate="many_to_one"
)


# ==========================================
# 5. MERGE AWAY DEFENSIVE LINE YARDS
# ==========================================

df_master_bcf = df_master_bcf.merge(
    lineyards_away,
    on="Away_CFBD_Name",
    how="left",
    validate="many_to_one"
)


# ==========================================
# 6. VALIDATION
# ==========================================

expected_games = len(df_target_market)

missing_home = int(
    df_master_bcf[
        "Home_Def_LineYards"
    ].isna().sum()
)

missing_away = int(
    df_master_bcf[
        "Away_Def_LineYards"
    ].isna().sum()
)

duplicate_games = int(
    df_master_bcf.duplicated(
        subset=[
            "Home_Team",
            "Away_Team"
        ],
        keep=False
    ).sum()
)


# ==========================================
# 7. REPORT
# ==========================================

print(f"\nExpected games:                 {expected_games}")
print(f"Games after Line Yards merge:   {len(df_master_bcf)}")
print(f"Missing Home Def Line Yards:    {missing_home}")
print(f"Missing Away Def Line Yards:    {missing_away}")
print(f"Duplicate games after merge:    {duplicate_games}")


# ==========================================
# 8. FINAL STATUS
# ==========================================

print("\n" + "-" * 70)

if (
    len(df_master_bcf) == expected_games
    and missing_home == 0
    and missing_away == 0
    and duplicate_games == 0
):

    print("CFB1 DEFENSIVE LINE YARDS MASTER MERGE: PASS")

else:

    print("CFB1 DEFENSIVE LINE YARDS MASTER MERGE: STOP")

    raise ValueError(
        "Defensive Line Yards master-card merge failed."
    )

print("=" * 70)


# ==========================================
# 9. VISUAL CHECK
# ==========================================

display(
    df_master_bcf[
        [
            "Away_Team",
            "Home_Team",
            "Away_Off_LineYards",
            "Home_Def_LineYards",
            "Home_Off_LineYards",
            "Away_Def_LineYards"
        ]
    ]
    .head(10)
)

CFB1 DEFENSIVE LINE YARDS -> MASTER CARD

Expected games:                 57
Games after Line Yards merge:   57
Missing Home Def Line Yards:    0
Missing Away Def Line Yards:    0
Duplicate games after merge:    0

----------------------------------------------------------------------
CFB1 DEFENSIVE LINE YARDS MASTER MERGE: PASS


,Away_Team,Home_Team,Away_Off_LineYards,Home_Def_LineYards,Home_Off_LineYards,Away_Def_LineYards
0,Jacksonville State Gamecocks,Kennesaw State Owls,3.122424,3.160284,3.313636,2.914973
1,New Mexico State Aggies,Florida International Panthers,3.063158,2.231092,2.826316,3.004403
2,Missouri State Bears,Western Kentucky Hilltoppers,2.993893,3.432164,2.617791,3.331902
3,Sam Houston State Bearkats,Liberty Flames,2.799123,3.030612,3.282927,3.074825
4,South Alabama Jaguars,Arkansas State Red Wolves,3.223288,3.518519,3.320548,3.115441
5,South Florida Bulls,UTSA Roadrunners,3.011585,3.037931,2.957055,2.611765
6,Florida State Seminoles,Louisville Cardinals,3.408889,2.944211,2.865663,2.972611
7,Iowa Hawkeyes,Washington Huskies,3.263758,2.360606,3.012121,2.700613
8,Washington State Cougars,Utah State Aggies,2.727972,2.754545,3.058904,2.826190
9,Wyoming Cowboys,San Jose State Spartans,3.449068,2.980952,3.029193,3.327807


In [ ]:
# ==========================================
# OLD CELL 27 — LINE YARDS PUSH SIGNAL
# ==========================================

print("=" * 72)
print("CFB1 LINE YARDS PUSH SIGNAL")
print("=" * 72)


# ==========================================
# 1. ESTABLISHED CFB1 THRESHOLDS
# ==========================================

OFF_LINE_YARDS_THRESHOLD = 3.5
DEF_LINE_YARDS_THRESHOLD = 3.3


# ==========================================
# 2. AWAY OFFENSE VS HOME DEFENSE
# ==========================================

df_master_bcf["Away_LineYards_Push"] = (
    (
        df_master_bcf["Away_Off_LineYards"]
        >= OFF_LINE_YARDS_THRESHOLD
    )
    &
    (
        df_master_bcf["Home_Def_LineYards"]
        >= DEF_LINE_YARDS_THRESHOLD
    )
)


# ==========================================
# 3. HOME OFFENSE VS AWAY DEFENSE
# ==========================================

df_master_bcf["Home_LineYards_Push"] = (
    (
        df_master_bcf["Home_Off_LineYards"]
        >= OFF_LINE_YARDS_THRESHOLD
    )
    &
    (
        df_master_bcf["Away_Def_LineYards"]
        >= DEF_LINE_YARDS_THRESHOLD
    )
)


# ==========================================
# 4. VALIDATION
# ==========================================

missing_lineyards_inputs = df_master_bcf[
    df_master_bcf[
        [
            "Away_Off_LineYards",
            "Home_Def_LineYards",
            "Home_Off_LineYards",
            "Away_Def_LineYards"
        ]
    ].isna().any(axis=1)
]

missing_lineyards_outputs = df_master_bcf[
    df_master_bcf[
        [
            "Away_LineYards_Push",
            "Home_LineYards_Push"
        ]
    ].isna().any(axis=1)
]


# ==========================================
# 5. COUNTS
# ==========================================

away_pushes = int(
    df_master_bcf["Away_LineYards_Push"].sum()
)

home_pushes = int(
    df_master_bcf["Home_LineYards_Push"].sum()
)

games_with_any_push = int(
    df_master_bcf[
        [
            "Away_LineYards_Push",
            "Home_LineYards_Push"
        ]
    ].any(axis=1).sum()
)


# ==========================================
# 6. REPORT
# ==========================================

print(f"\nGames tested:                    {len(df_master_bcf)}")
print(f"Away offenses triggering push:  {away_pushes}")
print(f"Home offenses triggering push:  {home_pushes}")
print(f"Games with at least one push:    {games_with_any_push}")

print(f"\nOff Line Yards threshold:        {OFF_LINE_YARDS_THRESHOLD:.1f}")
print(f"Opp Def Line Yards threshold:    {DEF_LINE_YARDS_THRESHOLD:.1f}")

print(f"\nMissing Line Yards input rows:   {len(missing_lineyards_inputs)}")
print(f"Missing Line Yards output rows:  {len(missing_lineyards_outputs)}")


# ==========================================
# 7. FINAL STATUS
# ==========================================

print("\n" + "-" * 72)

if (
    len(missing_lineyards_inputs) == 0
    and len(missing_lineyards_outputs) == 0
):

    print("CFB1 LINE YARDS PUSH SIGNAL: PASS")

else:

    print("CFB1 LINE YARDS PUSH SIGNAL: STOP")

    raise ValueError(
        "Line Yards Push validation failed."
    )

print("=" * 72)


# ==========================================
# 8. READABLE TRIGGER TABLE
# ==========================================

lineyards_rows = []

for _, row in df_master_bcf.iterrows():

    if row["Away_LineYards_Push"]:

        lineyards_rows.append({
            "Game":
                f"{row['Away_Team']} at {row['Home_Team']}",
            "Offense_With_Push":
                row["Away_Team"],
            "Off_LineYards":
                row["Away_Off_LineYards"],
            "Opp_Def_LineYards":
                row["Home_Def_LineYards"]
        })

    if row["Home_LineYards_Push"]:

        lineyards_rows.append({
            "Game":
                f"{row['Away_Team']} at {row['Home_Team']}",
            "Offense_With_Push":
                row["Home_Team"],
            "Off_LineYards":
                row["Home_Off_LineYards"],
            "Opp_Def_LineYards":
                row["Away_Def_LineYards"]
        })


lineyards_diagnostic = pd.DataFrame(lineyards_rows)

if len(lineyards_diagnostic) > 0:

    lineyards_diagnostic = lineyards_diagnostic.sort_values(
        [
            "Off_LineYards",
            "Opp_Def_LineYards"
        ],
        ascending=[
            False,
            False
        ]
    )

    display(lineyards_diagnostic)

else:

    print(
        f"\nNo Week {TARGET_WEEK} "
        "Line Yards Push matchups found."
    )

CFB1 LINE YARDS PUSH SIGNAL

Games tested:                    57
Away offenses triggering push:  1
Home offenses triggering push:  1
Games with at least one push:    2

Off Line Yards threshold:        3.5
Opp Def Line Yards threshold:    3.3

Missing Line Yards input rows:   0
Missing Line Yards output rows:  0

------------------------------------------------------------------------
CFB1 LINE YARDS PUSH SIGNAL: PASS


,Game,Offense_With_Push,Off_LineYards,Opp_Def_LineYards
0,Coastal Carolina Chanticleers at Marshall Thun...,Coastal Carolina Chanticleers,3.530769,3.434375
1,UAB Blazers at Memphis Tigers,Memphis Tigers,3.529293,3.402083


In [ ]:
# ==========================================
# OLD CELL 28 — EXPLOSIVENESS FRAUD SIGNAL
# ==========================================

print("=" * 72)
print("CFB1 EXPLOSIVENESS FRAUD SIGNAL")
print("=" * 72)


# ==========================================
# 1. ESTABLISHED CFB1 THRESHOLDS
# ==========================================

EXPLOSIVENESS_THRESHOLD = 1.40
OFF_SUCCESS_RATE_MAX = 0.42
DEF_SUCCESS_RATE_MAX = 0.33


# ==========================================
# 2. AWAY OFFENSE VS HOME DEFENSE
# ==========================================

df_master_bcf["Away_Explosiveness_Fraud"] = (
    (
        df_master_bcf["Away_Off_Explosiveness"]
        >= EXPLOSIVENESS_THRESHOLD
    )
    &
    (
        df_master_bcf["Away_Off_SuccessRate"]
        <= OFF_SUCCESS_RATE_MAX
    )
    &
    (
        df_master_bcf["Home_Def_SuccessRate"]
        <= DEF_SUCCESS_RATE_MAX
    )
)


# ==========================================
# 3. HOME OFFENSE VS AWAY DEFENSE
# ==========================================

df_master_bcf["Home_Explosiveness_Fraud"] = (
    (
        df_master_bcf["Home_Off_Explosiveness"]
        >= EXPLOSIVENESS_THRESHOLD
    )
    &
    (
        df_master_bcf["Home_Off_SuccessRate"]
        <= OFF_SUCCESS_RATE_MAX
    )
    &
    (
        df_master_bcf["Away_Def_SuccessRate"]
        <= DEF_SUCCESS_RATE_MAX
    )
)


# ==========================================
# 4. VALIDATION
# ==========================================

missing_fraud_inputs = df_master_bcf[
    df_master_bcf[
        [
            "Away_Off_Explosiveness",
            "Away_Off_SuccessRate",
            "Home_Def_SuccessRate",
            "Home_Off_Explosiveness",
            "Home_Off_SuccessRate",
            "Away_Def_SuccessRate"
        ]
    ].isna().any(axis=1)
]

missing_fraud_outputs = df_master_bcf[
    df_master_bcf[
        [
            "Away_Explosiveness_Fraud",
            "Home_Explosiveness_Fraud"
        ]
    ].isna().any(axis=1)
]


# ==========================================
# 5. COUNTS
# ==========================================

away_frauds = int(
    df_master_bcf[
        "Away_Explosiveness_Fraud"
    ].sum()
)

home_frauds = int(
    df_master_bcf[
        "Home_Explosiveness_Fraud"
    ].sum()
)

games_with_any_fraud = int(
    df_master_bcf[
        [
            "Away_Explosiveness_Fraud",
            "Home_Explosiveness_Fraud"
        ]
    ].any(axis=1).sum()
)


# ==========================================
# 6. REPORT
# ==========================================

print(f"\nGames tested:                         {len(df_master_bcf)}")
print(f"Away offenses triggering fraud:      {away_frauds}")
print(f"Home offenses triggering fraud:      {home_frauds}")
print(f"Games with at least one fraud signal:{games_with_any_fraud}")

print(f"\nExplosiveness threshold:              {EXPLOSIVENESS_THRESHOLD:.2f}")
print(f"Off Success Rate max:                 {OFF_SUCCESS_RATE_MAX:.2f}")
print(f"Opp Def Success Rate max:             {DEF_SUCCESS_RATE_MAX:.2f}")

print(f"\nMissing Explosiveness input rows:     {len(missing_fraud_inputs)}")
print(f"Missing Explosiveness output rows:    {len(missing_fraud_outputs)}")


# ==========================================
# 7. FINAL STATUS
# ==========================================

print("\n" + "-" * 72)

if (
    len(missing_fraud_inputs) == 0
    and len(missing_fraud_outputs) == 0
):

    print("CFB1 EXPLOSIVENESS FRAUD SIGNAL: PASS")

else:

    print("CFB1 EXPLOSIVENESS FRAUD SIGNAL: STOP")

    raise ValueError(
        "Explosiveness Fraud validation failed."
    )

print("=" * 72)


# ==========================================
# 8. READABLE TRIGGER TABLE
# ==========================================

fraud_rows = []

for _, row in df_master_bcf.iterrows():

    if row["Away_Explosiveness_Fraud"]:

        fraud_rows.append({
            "Game":
                f"{row['Away_Team']} at {row['Home_Team']}",
            "Offense_Flagged":
                row["Away_Team"],
            "Explosiveness":
                row["Away_Off_Explosiveness"],
            "Off_SuccessRate":
                row["Away_Off_SuccessRate"],
            "Opp_Def_SuccessRate":
                row["Home_Def_SuccessRate"]
        })

    if row["Home_Explosiveness_Fraud"]:

        fraud_rows.append({
            "Game":
                f"{row['Away_Team']} at {row['Home_Team']}",
            "Offense_Flagged":
                row["Home_Team"],
            "Explosiveness":
                row["Home_Off_Explosiveness"],
            "Off_SuccessRate":
                row["Home_Off_SuccessRate"],
            "Opp_Def_SuccessRate":
                row["Away_Def_SuccessRate"]
        })


fraud_diagnostic = pd.DataFrame(
    fraud_rows
)

if len(fraud_diagnostic) > 0:

    fraud_diagnostic = fraud_diagnostic.sort_values(
        [
            "Explosiveness",
            "Off_SuccessRate"
        ],
        ascending=[
            False,
            True
        ]
    )

    display(
        fraud_diagnostic
    )

else:

    print(
        f"\nNo Week {TARGET_WEEK} "
        "Explosiveness Fraud matchups found."
    )

CFB1 EXPLOSIVENESS FRAUD SIGNAL

Games tested:                         57
Away offenses triggering fraud:      0
Home offenses triggering fraud:      0
Games with at least one fraud signal:0

Explosiveness threshold:              1.40
Off Success Rate max:                 0.42
Opp Def Success Rate max:             0.33

Missing Explosiveness input rows:     0
Missing Explosiveness output rows:    0

------------------------------------------------------------------------
CFB1 EXPLOSIVENESS FRAUD SIGNAL: PASS

No Week 6 Explosiveness Fraud matchups found.


In [ ]:
# ==========================================
# OLD CELL 30 — OAY / DAY EDGE SIGNAL
# ==========================================

print("=" * 72)
print("CFB1 OAY / DAY EDGE SIGNAL")
print("=" * 72)


# ==========================================
# 1. ESTABLISHED CFB1 THRESHOLDS
# ==========================================

OAY_HIGH_THRESHOLD = 0.54125
DAY_WEAK_THRESHOLD = 0.56825


# ==========================================
# 2. AWAY OFFENSE VS HOME DEFENSE
# ==========================================

df_master_bcf["Away_OAY_DAY_Edge"] = (
    (
        df_master_bcf["Away_OAY"]
        >= OAY_HIGH_THRESHOLD
    )
    &
    (
        df_master_bcf["Home_DAY"]
        >= DAY_WEAK_THRESHOLD
    )
)


# ==========================================
# 3. HOME OFFENSE VS AWAY DEFENSE
# ==========================================

df_master_bcf["Home_OAY_DAY_Edge"] = (
    (
        df_master_bcf["Home_OAY"]
        >= OAY_HIGH_THRESHOLD
    )
    &
    (
        df_master_bcf["Away_DAY"]
        >= DAY_WEAK_THRESHOLD
    )
)


# ==========================================
# 4. VALIDATION
# ==========================================

missing_oay_day_inputs = df_master_bcf[
    df_master_bcf[
        [
            "Away_OAY",
            "Home_DAY",
            "Home_OAY",
            "Away_DAY"
        ]
    ].isna().any(axis=1)
]

missing_oay_day_outputs = df_master_bcf[
    df_master_bcf[
        [
            "Away_OAY_DAY_Edge",
            "Home_OAY_DAY_Edge"
        ]
    ].isna().any(axis=1)
]


# ==========================================
# 5. COUNTS
# ==========================================

away_edges = int(
    df_master_bcf[
        "Away_OAY_DAY_Edge"
    ].sum()
)

home_edges = int(
    df_master_bcf[
        "Home_OAY_DAY_Edge"
    ].sum()
)

games_with_any_edge = int(
    df_master_bcf[
        [
            "Away_OAY_DAY_Edge",
            "Home_OAY_DAY_Edge"
        ]
    ].any(axis=1).sum()
)


# ==========================================
# 6. REPORT
# ==========================================

print(f"\nGames tested:                    {len(df_master_bcf)}")
print(f"Away offenses triggering edge:  {away_edges}")
print(f"Home offenses triggering edge:  {home_edges}")
print(f"Games with at least one edge:    {games_with_any_edge}")

print(f"\nHigh OAY threshold:              {OAY_HIGH_THRESHOLD:.5f}")
print(f"Weak DAY threshold:              {DAY_WEAK_THRESHOLD:.5f}")

print(f"\nMissing OAY/DAY input rows:      {len(missing_oay_day_inputs)}")
print(f"Missing OAY/DAY output rows:     {len(missing_oay_day_outputs)}")


# ==========================================
# 7. FINAL STATUS
# ==========================================

print("\n" + "-" * 72)

if (
    len(missing_oay_day_inputs) == 0
    and len(missing_oay_day_outputs) == 0
):

    print("CFB1 OAY / DAY EDGE SIGNAL: PASS")

else:

    print("CFB1 OAY / DAY EDGE SIGNAL: STOP")

    raise ValueError(
        "OAY / DAY Edge validation failed."
    )

print("=" * 72)


# ==========================================
# 8. READABLE TRIGGER TABLE
# ==========================================

oay_day_rows = []

for _, row in df_master_bcf.iterrows():

    if row["Away_OAY_DAY_Edge"]:

        oay_day_rows.append({
            "Game":
                f"{row['Away_Team']} at {row['Home_Team']}",
            "Offense_With_Edge":
                row["Away_Team"],
            "OAY":
                row["Away_OAY"],
            "Opp_DAY":
                row["Home_DAY"]
        })

    if row["Home_OAY_DAY_Edge"]:

        oay_day_rows.append({
            "Game":
                f"{row['Away_Team']} at {row['Home_Team']}",
            "Offense_With_Edge":
                row["Home_Team"],
            "OAY":
                row["Home_OAY"],
            "Opp_DAY":
                row["Away_DAY"]
        })


oay_day_diagnostic = pd.DataFrame(
    oay_day_rows
)

if len(oay_day_diagnostic) > 0:

    oay_day_diagnostic = oay_day_diagnostic.sort_values(
        [
            "OAY",
            "Opp_DAY"
        ],
        ascending=[
            False,
            False
        ]
    )

    display(
        oay_day_diagnostic
    )

else:

    print(
        f"\nNo Week {TARGET_WEEK} "
        "OAY / DAY edge matchups found."
    )

CFB1 OAY / DAY EDGE SIGNAL

Games tested:                    57
Away offenses triggering edge:  4
Home offenses triggering edge:  3
Games with at least one edge:    7

High OAY threshold:              0.54125
Weak DAY threshold:              0.56825

Missing OAY/DAY input rows:      0
Missing OAY/DAY output rows:     0

------------------------------------------------------------------------
CFB1 OAY / DAY EDGE SIGNAL: PASS


,Game,Offense_With_Edge,OAY,Opp_DAY
4,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,0.587,0.575
3,Buffalo Bulls at Toledo Rockets,Toledo Rockets,0.581,0.602
5,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,0.554,0.573
2,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,0.553,0.658
6,Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,0.548,0.742
1,South Alabama Jaguars at Arkansas State Red Wo...,South Alabama Jaguars,0.548,0.574
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,0.545,0.672


In [ ]:
# ==========================================
# OLD CELL 31 — MODULE 3 SIGNAL BOARD
# ==========================================

print("=" * 78)
print("CFB1 MODULE 3 SIGNAL BOARD")
print("=" * 78)


# ==========================================
# 1. BUILD SIGNAL BOARD
# ==========================================

module3_rows = []

for _, row in df_master_bcf.iterrows():

    away_signals = []
    home_signals = []


    # --------------------------------------
    # HAVOC TRAP
    # Negative for the offense
    # --------------------------------------

    if row["Away_Havoc_Trap"]:
        away_signals.append(
            "HAVOC_TRAP_NEGATIVE"
        )

    if row["Home_Havoc_Trap"]:
        home_signals.append(
            "HAVOC_TRAP_NEGATIVE"
        )


    # --------------------------------------
    # LINE YARDS PUSH
    # Positive offensive matchup
    # --------------------------------------

    if row["Away_LineYards_Push"]:
        away_signals.append(
            "LINE_YARDS_POSITIVE"
        )

    if row["Home_LineYards_Push"]:
        home_signals.append(
            "LINE_YARDS_POSITIVE"
        )


    # --------------------------------------
    # EXPLOSIVENESS FRAUD
    # Negative offensive matchup
    # --------------------------------------

    if row["Away_Explosiveness_Fraud"]:
        away_signals.append(
            "EXPLOSIVENESS_FRAUD_NEGATIVE"
        )

    if row["Home_Explosiveness_Fraud"]:
        home_signals.append(
            "EXPLOSIVENESS_FRAUD_NEGATIVE"
        )


    # --------------------------------------
    # OAY / DAY EDGE
    # Positive offensive matchup
    # --------------------------------------

    if row["Away_OAY_DAY_Edge"]:
        away_signals.append(
            "OAY_DAY_POSITIVE"
        )

    if row["Home_OAY_DAY_Edge"]:
        home_signals.append(
            "OAY_DAY_POSITIVE"
        )


    module3_rows.append({
        "Game":
            f"{row['Away_Team']} at {row['Home_Team']}",

        "Away_Team":
            row["Away_Team"],

        "Away_Signals":
            ", ".join(away_signals),

        "Away_Signal_Count":
            len(away_signals),

        "Home_Team":
            row["Home_Team"],

        "Home_Signals":
            ", ".join(home_signals),

        "Home_Signal_Count":
            len(home_signals),

        "Total_Module3_Signals":
            len(away_signals)
            + len(home_signals)
    })


df_module3_board = pd.DataFrame(
    module3_rows
)


# ==========================================
# 2. ACTIVE MODULE 3 GAMES
# ==========================================

active_module3 = df_module3_board[
    df_module3_board[
        "Total_Module3_Signals"
    ] > 0
].copy()


# ==========================================
# 3. VALIDATION
# ==========================================

expected_games = len(df_master_bcf)

duplicate_games = int(
    df_module3_board.duplicated(
        subset=["Game"],
        keep=False
    ).sum()
)

missing_required = int(
    df_module3_board[
        [
            "Game",
            "Away_Team",
            "Home_Team",
            "Away_Signal_Count",
            "Home_Signal_Count",
            "Total_Module3_Signals"
        ]
    ].isna().any(axis=1).sum()
)

signal_count_errors = int(
    (
        df_module3_board[
            "Total_Module3_Signals"
        ]
        !=
        (
            df_module3_board[
                "Away_Signal_Count"
            ]
            +
            df_module3_board[
                "Home_Signal_Count"
            ]
        )
    ).sum()
)


# ==========================================
# 4. REPORT
# ==========================================

print(f"\nTarget-week games:               {expected_games}")
print(f"Module 3 board rows:             {len(df_module3_board)}")
print(f"Games with Module 3 signal:      {len(active_module3)}")
print(
    f"Games with 2+ Module 3 signals: "
    f"{int((active_module3['Total_Module3_Signals'] >= 2).sum())}"
)

print(f"\nDuplicate game rows:             {duplicate_games}")
print(f"Missing required values:         {missing_required}")
print(f"Signal-count errors:             {signal_count_errors}")


# ==========================================
# 5. FINAL STATUS
# ==========================================

print("\n" + "-" * 78)

if (
    len(df_module3_board) == expected_games
    and duplicate_games == 0
    and missing_required == 0
    and signal_count_errors == 0
):

    print("CFB1 MODULE 3 SIGNAL BOARD: PASS")

else:

    print("CFB1 MODULE 3 SIGNAL BOARD: STOP")

    raise ValueError(
        "Module 3 Signal Board validation failed."
    )

print("=" * 78)


# ==========================================
# 6. ACTIVE MODULE 3 MATCHUPS
# ==========================================

if len(active_module3) > 0:

    display(
        active_module3.sort_values(
            [
                "Total_Module3_Signals",
                "Game"
            ],
            ascending=[
                False,
                True
            ]
        )
    )

else:

    print(
        f"\nNo Week {TARGET_WEEK} "
        "Module 3 signals found."
    )

CFB1 MODULE 3 SIGNAL BOARD

Target-week games:               57
Module 3 board rows:             57
Games with Module 3 signal:      12
Games with 2+ Module 3 signals: 0

Duplicate game rows:             0
Missing required values:         0
Signal-count errors:             0

------------------------------------------------------------------------------
CFB1 MODULE 3 SIGNAL BOARD: PASS


,Game,Away_Team,Away_Signals,Away_Signal_Count,Home_Team,Home_Signals,Home_Signal_Count,Total_Module3_Signals
19,Ball State Cardinals at Northwestern Wildcats,Ball State Cardinals,,0,Northwestern Wildcats,OAY_DAY_POSITIVE,1,1
23,Buffalo Bulls at Toledo Rockets,Buffalo Bulls,,0,Toledo Rockets,OAY_DAY_POSITIVE,1,1
25,Charlotte 49ers at North Texas Mean Green,Charlotte 49ers,,0,North Texas Mean Green,OAY_DAY_POSITIVE,1,1
42,Coastal Carolina Chanticleers at Marshall Thun...,Coastal Carolina Chanticleers,LINE_YARDS_POSITIVE,1,Marshall Thundering Herd,,0,1
26,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,OAY_DAY_POSITIVE,1,Georgia Tech Yellow Jackets,,0,1
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,OAY_DAY_POSITIVE,1,Kennesaw State Owls,,0,1
44,Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,OAY_DAY_POSITIVE,1,UTEP Miners,,0,1
38,Rice Owls at East Carolina Pirates,Rice Owls,HAVOC_TRAP_NEGATIVE,1,East Carolina Pirates,,0,1
4,South Alabama Jaguars at Arkansas State Red Wo...,South Alabama Jaguars,OAY_DAY_POSITIVE,1,Arkansas State Red Wolves,,0,1
32,Stanford Cardinal at Notre Dame Fighting Irish,Stanford Cardinal,HAVOC_TRAP_NEGATIVE,1,Notre Dame Fighting Irish,,0,1


In [ ]:
# ==========================================
# OLD CELL 34 — CORRECTED MODULE 3
# DIRECTIONAL CONVERGENCE
# ==========================================

print("=" * 88)
print("CFB1 CORRECTED DIRECTIONAL CONVERGENCE")
print("=" * 88)


# ==========================================
# 1. BUILD CORRECTED CONVERGENCE TABLE
# ==========================================

corrected_rows = []

for _, row in df_master_bcf.iterrows():

    away = row["Away_Team"]
    home = row["Home_Team"]

    epa_team = row["EPA_Edge_Team"]
    fei_team = row["DK_FEI_Edge_Team"]

    epa_fei_agree = (
        epa_team == fei_team
        and epa_team != "EVEN"
    )

    shared_side = (
        epa_team
        if epa_fei_agree
        else ""
    )


    # --------------------------------------
    # COLLECT MODULE 3 SIGNALS
    # --------------------------------------

    away_positive = []
    away_negative = []

    home_positive = []
    home_negative = []


    if row["Away_LineYards_Push"]:
        away_positive.append("LINE_YARDS")

    if row["Away_OAY_DAY_Edge"]:
        away_positive.append("OAY_DAY")

    if row["Away_Havoc_Trap"]:
        away_negative.append("HAVOC_TRAP")

    if row["Away_Explosiveness_Fraud"]:
        away_negative.append(
            "EXPLOSIVENESS_FRAUD"
        )


    if row["Home_LineYards_Push"]:
        home_positive.append("LINE_YARDS")

    if row["Home_OAY_DAY_Edge"]:
        home_positive.append("OAY_DAY")

    if row["Home_Havoc_Trap"]:
        home_negative.append("HAVOC_TRAP")

    if row["Home_Explosiveness_Fraud"]:
        home_negative.append(
            "EXPLOSIVENESS_FRAUD"
        )


    # --------------------------------------
    # CLASSIFY MODULE 3 SUPPORT / CONFLICT
    # RELATIVE TO SHARED EPA + FEI SIDE
    # --------------------------------------

    m3_support = []
    m3_conflict = []


    # Shared side = AWAY
    if shared_side == away:

        for signal in away_positive:
            m3_support.append(
                f"{away} +{signal}"
            )

        for signal in home_negative:
            m3_support.append(
                f"{home} -{signal}"
            )

        for signal in away_negative:
            m3_conflict.append(
                f"{away} -{signal}"
            )

        for signal in home_positive:
            m3_conflict.append(
                f"{home} +{signal}"
            )


    # Shared side = HOME
    elif shared_side == home:

        for signal in home_positive:
            m3_support.append(
                f"{home} +{signal}"
            )

        for signal in away_negative:
            m3_support.append(
                f"{away} -{signal}"
            )

        for signal in home_negative:
            m3_conflict.append(
                f"{home} -{signal}"
            )

        for signal in away_positive:
            m3_conflict.append(
                f"{away} +{signal}"
            )


    corrected_rows.append({

        "Game":
            f"{away} at {home}",

        "EPA_Team":
            epa_team,

        "EPA_Value":
            row["EPA_Edge_Value"],

        "FEI_Team":
            fei_team,

        "FEI_Value":
            row["DK_FEI_Edge_Value"],

        "EPA_FEI_Agree":
            epa_fei_agree,

        "Shared_Side":
            shared_side,

        "M3_Support":
            ", ".join(m3_support),

        "M3_Conflict":
            ", ".join(m3_conflict),

        "M3_Support_Count":
            len(m3_support),

        "M3_Conflict_Count":
            len(m3_conflict)
    })


df_convergence_corrected = pd.DataFrame(
    corrected_rows
)


# ==========================================
# 2. VALIDATION
# ==========================================

expected_games = len(df_master_bcf)

duplicate_games = int(
    df_convergence_corrected.duplicated(
        subset=["Game"],
        keep=False
    ).sum()
)

missing_required = int(
    df_convergence_corrected[
        [
            "Game",
            "EPA_Team",
            "EPA_Value",
            "FEI_Team",
            "FEI_Value",
            "EPA_FEI_Agree",
            "Shared_Side",
            "M3_Support_Count",
            "M3_Conflict_Count"
        ]
    ].isna().any(axis=1).sum()
)

shared_side_errors = int(
    (
        df_convergence_corrected[
            "EPA_FEI_Agree"
        ]
        &
        (
            df_convergence_corrected[
                "Shared_Side"
            ] == ""
        )
    ).sum()
)


# ==========================================
# 3. SUMMARY
# ==========================================

agreed = df_convergence_corrected[
    df_convergence_corrected[
        "EPA_FEI_Agree"
    ]
].copy()

with_support = int(
    (
        agreed[
            "M3_Support_Count"
        ] > 0
    ).sum()
)

with_conflict = int(
    (
        agreed[
            "M3_Conflict_Count"
        ] > 0
    ).sum()
)


# ==========================================
# 4. REPORT
# ==========================================

print(f"\nTarget-week games:                    {expected_games}")
print(f"Convergence rows:                    {len(df_convergence_corrected)}")
print(f"EPA + FEI same side:                 {len(agreed)}")
print(f"Same side + Module 3 support:        {with_support}")
print(f"Same side + Module 3 conflict:       {with_conflict}")

print(f"\nDuplicate game rows:                 {duplicate_games}")
print(f"Missing required values:             {missing_required}")
print(f"Shared-side assignment errors:       {shared_side_errors}")


# ==========================================
# 5. FINAL STATUS
# ==========================================

print("\n" + "-" * 88)

if (
    len(df_convergence_corrected) == expected_games
    and duplicate_games == 0
    and missing_required == 0
    and shared_side_errors == 0
):

    print("CFB1 CORRECTED DIRECTIONAL CONVERGENCE: PASS")

else:

    print("CFB1 CORRECTED DIRECTIONAL CONVERGENCE: STOP")

    raise ValueError(
        "Corrected convergence validation failed."
    )

print("=" * 88)


# ==========================================
# 6. ACTIVE AGREEMENT + MODULE 3
# ==========================================

m3_active = agreed[
    (
        agreed["M3_Support_Count"] > 0
    )
    |
    (
        agreed["M3_Conflict_Count"] > 0
    )
].copy()

if len(m3_active) > 0:

    display(
        m3_active[
            [
                "Game",
                "EPA_Team",
                "EPA_Value",
                "FEI_Team",
                "FEI_Value",
                "Shared_Side",
                "M3_Support",
                "M3_Conflict"
            ]
        ]
    )

else:

    print(
        f"\nNo Week {TARGET_WEEK} "
        "EPA+FEI agreement games with Module 3 involvement."
    )

CFB1 CORRECTED DIRECTIONAL CONVERGENCE

Target-week games:                    57
Convergence rows:                    57
EPA + FEI same side:                 37
Same side + Module 3 support:        10
Same side + Module 3 conflict:       0

Duplicate game rows:                 0
Missing required values:             0
Shared-side assignment errors:       0

----------------------------------------------------------------------------------------
CFB1 CORRECTED DIRECTIONAL CONVERGENCE: PASS


,Game,EPA_Team,EPA_Value,FEI_Team,FEI_Value,Shared_Side,M3_Support,M3_Conflict
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,0.219468,Jacksonville State Gamecocks,2.4,Jacksonville State Gamecocks,Jacksonville State Gamecocks +OAY_DAY,
19,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,0.467262,Northwestern Wildcats,12.1,Northwestern Wildcats,Northwestern Wildcats +OAY_DAY,
23,Buffalo Bulls at Toledo Rockets,Toledo Rockets,0.363014,Toledo Rockets,3.0,Toledo Rockets,Toledo Rockets +OAY_DAY,
25,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,0.441874,North Texas Mean Green,2.1,North Texas Mean Green,North Texas Mean Green +OAY_DAY,
26,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,0.373008,Duke Blue Devils,5.2,Duke Blue Devils,Duke Blue Devils +OAY_DAY,
32,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,0.643768,Notre Dame Fighting Irish,21.5,Notre Dame Fighting Irish,Stanford Cardinal -HAVOC_TRAP,
36,Virginia Tech Hokies at California Golden Bears,Virginia Tech Hokies,0.274383,Virginia Tech Hokies,4.1,Virginia Tech Hokies,California Golden Bears -HAVOC_TRAP,
38,Rice Owls at East Carolina Pirates,East Carolina Pirates,0.158938,East Carolina Pirates,7.7,East Carolina Pirates,Rice Owls -HAVOC_TRAP,
44,Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,0.412176,Nevada Wolf Pack,11.8,Nevada Wolf Pack,Nevada Wolf Pack +OAY_DAY,
46,UAB Blazers at Memphis Tigers,Memphis Tigers,0.053547,Memphis Tigers,9.7,Memphis Tigers,Memphis Tigers +LINE_YARDS,


In [ ]:
# ==========================================
# OLD CELL 36 — MARKET ROUTING LOGIC
# ==========================================

print("=" * 96)
print("CFB1 MARKET ROUTING BOARD")
print("=" * 96)


# ==========================================
# 1. BUILD MARKET ROUTES
# ==========================================

routing_rows = []

for _, row in df_convergence_corrected.iterrows():

    # Only route games where EPA + FEI
    # agree on the same model side.
    if not row["EPA_FEI_Agree"]:
        continue

    support_text = row["M3_Support"]
    conflict_text = row["M3_Conflict"]

    routes = []
    spread_status = "STANDARD_REVIEW"


    # --------------------------------------
    # MODULE 3 SIDE SUPPORT
    # --------------------------------------

    if row["M3_Support_Count"] > 0:

        routes.append(
            "FULL_GAME_SPREAD"
        )


    # --------------------------------------
    # MODULE 3 SIDE CONFLICT
    # --------------------------------------

    if row["M3_Conflict_Count"] > 0:

        spread_status = (
            "DOWNGRADE_FULL_GAME_SPREAD"
        )

        routes.append(
            "DERIVATIVES_FIRST"
        )


    # --------------------------------------
    # POSITIVE OFFENSIVE SUPPORT
    #
    # LINE_YARDS or OAY_DAY supporting
    # the model side
    # --------------------------------------

    if (
        "LINE_YARDS" in support_text
        or
        "OAY_DAY" in support_text
    ):

        routes.append(
            "TEAM_TOTAL_OVER"
        )

        routes.append(
            "1H_TEAM_TOTAL_OVER"
        )

        routes.append(
            "1H_SPREAD_REVIEW"
        )


    # --------------------------------------
    # OPPONENT OFFENSIVE WEAKNESS
    #
    # HAVOC_TRAP or EXPLOSIVENESS_FRAUD
    # supporting the model side
    # --------------------------------------

    if (
        "HAVOC_TRAP" in support_text
        or
        "EXPLOSIVENESS_FRAUD" in support_text
    ):

        routes.append(
            "OPP_TEAM_TOTAL_UNDER"
        )

        routes.append(
            "OPP_1H_TEAM_TOTAL_UNDER"
        )

        routes.append(
            "1H_UNDER_REVIEW"
        )


    # --------------------------------------
    # REMOVE DUPLICATES
    # PRESERVE ROUTE ORDER
    # --------------------------------------

    routes = list(
        dict.fromkeys(routes)
    )


    routing_rows.append({

        "Game":
            row["Game"],

        "Model_Side":
            row["Shared_Side"],

        "EPA_Value":
            row["EPA_Value"],

        "FEI_Value":
            row["FEI_Value"],

        "M3_Support":
            support_text,

        "M3_Conflict":
            conflict_text,

        "Spread_Status":
            spread_status,

        "Eligible_Market_Routes":
            ", ".join(routes),

        "Final_Bet_Status":
            "UNPRICED — DO NOT BET YET"
    })


df_market_routes = pd.DataFrame(
    routing_rows
)


# ==========================================
# 2. VALIDATION
# ==========================================

expected_games = int(
    df_convergence_corrected[
        "EPA_FEI_Agree"
    ].sum()
)

duplicate_games = int(
    df_market_routes.duplicated(
        subset=["Game"],
        keep=False
    ).sum()
)

missing_required = int(
    df_market_routes[
        [
            "Game",
            "Model_Side",
            "EPA_Value",
            "FEI_Value",
            "Spread_Status",
            "Eligible_Market_Routes",
            "Final_Bet_Status"
        ]
    ].isna().any(axis=1).sum()
)

blank_model_side = int(
    (
        df_market_routes[
            "Model_Side"
        ] == ""
    ).sum()
)


# ==========================================
# 3. REPORT
# ==========================================

print(f"\nEPA + FEI agreement games:       {expected_games}")
print(f"Games routed:                   {len(df_market_routes)}")

print(f"\nDuplicate game rows:            {duplicate_games}")
print(f"Missing required values:        {missing_required}")
print(f"Blank model-side rows:          {blank_model_side}")


# ==========================================
# 4. FINAL STATUS
# ==========================================

print("\n" + "-" * 96)

if (
    len(df_market_routes) == expected_games
    and duplicate_games == 0
    and missing_required == 0
    and blank_model_side == 0
):

    print("CFB1 MARKET ROUTING BOARD: PASS")

else:

    print("CFB1 MARKET ROUTING BOARD: STOP")

    raise ValueError(
        "Market Routing Board validation failed."
    )

print("=" * 96)


# ==========================================
# 5. DISPLAY ROUTING BOARD
# ==========================================

display(
    df_market_routes[
        [
            "Game",
            "Model_Side",
            "EPA_Value",
            "FEI_Value",
            "M3_Support",
            "M3_Conflict",
            "Spread_Status",
            "Eligible_Market_Routes",
            "Final_Bet_Status"
        ]
    ]
)

CFB1 MARKET ROUTING BOARD

EPA + FEI agreement games:       37
Games routed:                   37

Duplicate game rows:            0
Missing required values:        0
Blank model-side rows:          0

------------------------------------------------------------------------------------------------
CFB1 MARKET ROUTING BOARD: PASS


,Game,Model_Side,EPA_Value,FEI_Value,M3_Support,M3_Conflict,Spread_Status,Eligible_Market_Routes,Final_Bet_Status
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,0.219468,2.4,Jacksonville State Gamecocks +OAY_DAY,,STANDARD_REVIEW,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",UNPRICED — DO NOT BET YET
1,New Mexico State Aggies at Florida Internation...,New Mexico State Aggies,0.000631,5.3,,,STANDARD_REVIEW,,UNPRICED — DO NOT BET YET
2,Missouri State Bears at Western Kentucky Hillt...,Missouri State Bears,0.148512,7.3,,,STANDARD_REVIEW,,UNPRICED — DO NOT BET YET
3,Sam Houston State Bearkats at Liberty Flames,Liberty Flames,0.091369,7.8,,,STANDARD_REVIEW,,UNPRICED — DO NOT BET YET
4,South Florida Bulls at UTSA Roadrunners,South Florida Bulls,0.016996,7.0,,,STANDARD_REVIEW,,UNPRICED — DO NOT BET YET
5,Wyoming Cowboys at San Jose State Spartans,San Jose State Spartans,0.105177,1.8,,,STANDARD_REVIEW,,UNPRICED — DO NOT BET YET
6,Arizona Wildcats at West Virginia Mountaineers,Arizona Wildcats,0.184628,13.5,,,STANDARD_REVIEW,,UNPRICED — DO NOT BET YET
7,Indiana Hoosiers at Nebraska Cornhuskers,Indiana Hoosiers,0.113786,9.3,,,STANDARD_REVIEW,,UNPRICED — DO NOT BET YET
8,North Carolina Tar Heels at Pittsburgh Panthers,Pittsburgh Panthers,0.241220,11.5,,,STANDARD_REVIEW,,UNPRICED — DO NOT BET YET
9,Wake Forest Demon Deacons at NC State Wolfpack,Wake Forest Demon Deacons,0.144548,7.6,,,STANDARD_REVIEW,,UNPRICED — DO NOT BET YET


In [ ]:
# ==========================================
# OLD CELL 37 — SPREAD / KEY NUMBER STRUCTURE
# ==========================================

print("=" * 84)
print("CFB1 SPREAD / KEY NUMBER STRUCTURE")
print("=" * 84)


# ==========================================
# 1. ESTABLISHED KEY NUMBERS
# ==========================================

SPREAD_KEY_NUMBERS = [3, 7, 10, 14]


# ==========================================
# 2. VALIDATE MARKET INPUTS
# ==========================================

missing_spread_inputs = df_master_bcf[
    df_master_bcf[
        [
            "DK_Spread",
            "FD_Spread"
        ]
    ].isna().any(axis=1)
]


# ==========================================
# 3. CONSENSUS HOME SPREAD
# Average DK and FD home-team spread
# ==========================================

df_master_bcf["Consensus_Spread"] = (
    (
        df_master_bcf["DK_Spread"]
        +
        df_master_bcf["FD_Spread"]
    ) / 2
).round(2)

df_master_bcf["Consensus_Spread_Abs"] = (
    df_master_bcf[
        "Consensus_Spread"
    ].abs()
)


# ==========================================
# 4. NEAREST KEY NUMBER
# ==========================================

def nearest_key_number(spread):

    magnitude = abs(spread)

    nearest = min(
        SPREAD_KEY_NUMBERS,
        key=lambda key: abs(
            magnitude - key
        )
    )

    distance = abs(
        magnitude - nearest
    )

    return nearest, round(
        distance,
        2
    )


key_results = df_master_bcf[
    "Consensus_Spread"
].apply(
    nearest_key_number
)


df_master_bcf["Nearest_Key_Number"] = [
    result[0]
    for result in key_results
]

df_master_bcf["Distance_From_Key"] = [
    result[1]
    for result in key_results
]


# ==========================================
# 5. KEY-NUMBER PROXIMITY
# ==========================================

df_master_bcf["Within_0.5_Key"] = (
    df_master_bcf[
        "Distance_From_Key"
    ] <= 0.5
)

df_master_bcf["Within_1.0_Key"] = (
    df_master_bcf[
        "Distance_From_Key"
    ] <= 1.0
)


# ==========================================
# 6. SPREAD SIZE BUCKETS
# ==========================================

def spread_bucket(magnitude):

    if magnitude < 3:
        return "UNDER_3"

    elif magnitude < 7:
        return "3_TO_6.5"

    elif magnitude < 10:
        return "7_TO_9.5"

    elif magnitude < 14:
        return "10_TO_13.5"

    elif magnitude < 17:
        return "14_TO_16.5"

    elif magnitude < 21:
        return "17_TO_20.5"

    else:
        return "21_PLUS"


df_master_bcf["Spread_Size_Bucket"] = (
    df_master_bcf[
        "Consensus_Spread_Abs"
    ].apply(
        spread_bucket
    )
)


# ==========================================
# 7. VALIDATION
# ==========================================

missing_outputs = df_master_bcf[
    df_master_bcf[
        [
            "Consensus_Spread",
            "Consensus_Spread_Abs",
            "Nearest_Key_Number",
            "Distance_From_Key",
            "Within_0.5_Key",
            "Within_1.0_Key",
            "Spread_Size_Bucket"
        ]
    ].isna().any(axis=1)
]


print(
    f"\nGames tested:                   "
    f"{len(df_master_bcf)}"
)

print(
    f"Missing DK/FD spread inputs:    "
    f"{len(missing_spread_inputs)}"
)

print(
    f"Missing derived values:         "
    f"{len(missing_outputs)}"
)

print(
    f"Within 0.5 of key number:       "
    f"{int(df_master_bcf['Within_0.5_Key'].sum())}"
)

print(
    f"Within 1.0 of key number:       "
    f"{int(df_master_bcf['Within_1.0_Key'].sum())}"
)


# ==========================================
# 8. FINAL STATUS
# ==========================================

print("\n" + "-" * 84)

if (
    len(missing_spread_inputs) == 0
    and len(missing_outputs) == 0
):

    print(
        "CFB1 SPREAD / KEY NUMBER STRUCTURE: PASS"
    )

else:

    print(
        "CFB1 SPREAD / KEY NUMBER STRUCTURE: STOP"
    )

    raise ValueError(
        "Spread / Key Number validation failed."
    )

print("=" * 84)


# ==========================================
# 9. SPREAD SIZE DISTRIBUTION
# ==========================================

print("\nSPREAD SIZE DISTRIBUTION")
print("-" * 40)

print(
    df_master_bcf[
        "Spread_Size_Bucket"
    ]
    .value_counts()
    .sort_index()
)


# ==========================================
# 10. ROUTED GAMES — MARKET STRUCTURE
# ==========================================

routed_games = set(
    df_market_routes.loc[
        df_market_routes[
            "Eligible_Market_Routes"
        ] != "",
        "Game"
    ]
)

market_structure_rows = []

for _, row in df_master_bcf.iterrows():

    game = (
        f"{row['Away_Team']} "
        f"at {row['Home_Team']}"
    )

    if game in routed_games:

        market_structure_rows.append({

            "Game":
                game,

            "DK_Spread":
                row["DK_Spread"],

            "FD_Spread":
                row["FD_Spread"],

            "Consensus_Spread":
                row["Consensus_Spread"],

            "Nearest_Key_Number":
                row["Nearest_Key_Number"],

            "Distance_From_Key":
                row["Distance_From_Key"],

            "Spread_Size_Bucket":
                row["Spread_Size_Bucket"]
        })


df_routed_market_structure = pd.DataFrame(
    market_structure_rows
)


print(
    f"\nGames with active market routes: "
    f"{len(df_routed_market_structure)}"
)

if len(df_routed_market_structure) > 0:

    display(
        df_routed_market_structure
    )

CFB1 SPREAD / KEY NUMBER STRUCTURE

Games tested:                   57
Missing DK/FD spread inputs:    0
Missing derived values:         0
Within 0.5 of key number:       41
Within 1.0 of key number:       42

------------------------------------------------------------------------------------
CFB1 SPREAD / KEY NUMBER STRUCTURE: PASS

SPREAD SIZE DISTRIBUTION
----------------------------------------
Spread_Size_Bucket
10_TO_13.5     8
14_TO_16.5     3
17_TO_20.5     2
21_PLUS        4
3_TO_6.5      15
7_TO_9.5      12
UNDER_3       13
Name: count, dtype: int64

Games with active market routes: 10


,Game,DK_Spread,FD_Spread,Consensus_Spread,Nearest_Key_Number,Distance_From_Key,Spread_Size_Bucket
0,Jacksonville State Gamecocks at Kennesaw State...,3.0,2.5,2.75,3,0.25,UNDER_3
1,Ball State Cardinals at Northwestern Wildcats,-36.5,-36.5,-36.50,14,22.50,21_PLUS
2,Buffalo Bulls at Toledo Rockets,-21.0,-20.5,-20.75,14,6.75,17_TO_20.5
3,Charlotte 49ers at North Texas Mean Green,-28.5,-28.5,-28.50,14,14.50,21_PLUS
4,Duke Blue Devils at Georgia Tech Yellow Jackets,6.5,6.5,6.50,7,0.50,3_TO_6.5
5,Stanford Cardinal at Notre Dame Fighting Irish,-38.5,-37.5,-38.00,14,24.00,21_PLUS
6,Virginia Tech Hokies at California Golden Bears,10.0,9.5,9.75,10,0.25,7_TO_9.5
7,Rice Owls at East Carolina Pirates,-10.0,-9.5,-9.75,10,0.25,7_TO_9.5
8,Nevada Wolf Pack at UTEP Miners,9.5,8.5,9.00,10,1.00,7_TO_9.5
9,UAB Blazers at Memphis Tigers,-14.0,-14.5,-14.25,14,0.25,14_TO_16.5


In [ ]:
# ==========================================
# OLD CELL 38 — LARGE-SPREAD CLASSIFICATION
# ==========================================

print("=" * 86)
print("CFB1 LARGE-SPREAD CLASSIFICATION")
print("=" * 86)


# ==========================================
# 1. ESTABLISHED CFB1 THRESHOLDS
# ==========================================

LARGE_SPREAD_THRESHOLD = 16.5
VERY_LARGE_SPREAD_THRESHOLD = 20.5


# ==========================================
# 2. IDENTIFY MARKET FAVORITE
# ==========================================

def market_favorite_info(row):

    spread = row["Consensus_Spread"]

    if spread < 0:

        return pd.Series([
            row["Home_Team"],
            abs(spread)
        ])

    elif spread > 0:

        return pd.Series([
            row["Away_Team"],
            abs(spread)
        ])

    else:

        return pd.Series([
            "PICKEM",
            0.0
        ])


df_master_bcf[
    [
        "Consensus_Favorite",
        "Favorite_Spread_Size"
    ]
] = df_master_bcf.apply(
    market_favorite_info,
    axis=1
)


# ==========================================
# 3. LARGE-SPREAD FLAGS
# ==========================================

df_master_bcf["Large_Spread_16_5"] = (
    df_master_bcf[
        "Favorite_Spread_Size"
    ] >= LARGE_SPREAD_THRESHOLD
)

df_master_bcf["Very_Large_Spread_20_5"] = (
    df_master_bcf[
        "Favorite_Spread_Size"
    ] >= VERY_LARGE_SPREAD_THRESHOLD
)


# ==========================================
# 4. VALIDATION
# ==========================================

missing_inputs = df_master_bcf[
    df_master_bcf[
        [
            "Consensus_Spread",
            "Home_Team",
            "Away_Team"
        ]
    ].isna().any(axis=1)
]

missing_outputs = df_master_bcf[
    df_master_bcf[
        [
            "Consensus_Favorite",
            "Favorite_Spread_Size",
            "Large_Spread_16_5",
            "Very_Large_Spread_20_5"
        ]
    ].isna().any(axis=1)
]

invalid_size = int(
    (
        df_master_bcf[
            "Favorite_Spread_Size"
        ] < 0
    ).sum()
)


# ==========================================
# 5. FULL-SLATE SUMMARY
# ==========================================

large_count = int(
    df_master_bcf[
        "Large_Spread_16_5"
    ].sum()
)

very_large_count = int(
    df_master_bcf[
        "Very_Large_Spread_20_5"
    ].sum()
)


print(
    f"\nGames tested:                         "
    f"{len(df_master_bcf)}"
)

print(
    f"Favorite laying 16.5 or more:         "
    f"{large_count}"
)

print(
    f"Favorite laying 20.5 or more:         "
    f"{very_large_count}"
)

print(
    f"\nMissing large-spread input rows:       "
    f"{len(missing_inputs)}"
)

print(
    f"Missing large-spread output rows:      "
    f"{len(missing_outputs)}"
)

print(
    f"Invalid negative spread sizes:         "
    f"{invalid_size}"
)


# ==========================================
# 6. FINAL STATUS
# ==========================================

print("\n" + "-" * 86)

if (
    len(missing_inputs) == 0
    and len(missing_outputs) == 0
    and invalid_size == 0
):

    print(
        "CFB1 LARGE-SPREAD CLASSIFICATION: PASS"
    )

else:

    print(
        "CFB1 LARGE-SPREAD CLASSIFICATION: STOP"
    )

    raise ValueError(
        "Large-spread classification validation failed."
    )

print("=" * 86)


# ==========================================
# 7. ACTIVE ROUTED GAMES
# ==========================================

routed_games = set(
    df_market_routes.loc[
        df_market_routes[
            "Eligible_Market_Routes"
        ] != "",
        "Game"
    ]
)

large_spread_rows = []

for _, row in df_master_bcf.iterrows():

    game = (
        f"{row['Away_Team']} "
        f"at {row['Home_Team']}"
    )

    if game not in routed_games:
        continue

    if (
        row["Favorite_Spread_Size"]
        >= VERY_LARGE_SPREAD_THRESHOLD
    ):

        spread_class = (
            "VERY_LARGE_20.5_PLUS"
        )

    elif (
        row["Favorite_Spread_Size"]
        >= LARGE_SPREAD_THRESHOLD
    ):

        spread_class = (
            "LARGE_16.5_TO_20"
        )

    else:

        spread_class = (
            "BELOW_16.5"
        )


    large_spread_rows.append({

        "Game":
            game,

        "Consensus_Favorite":
            row["Consensus_Favorite"],

        "Favorite_Spread_Size":
            row["Favorite_Spread_Size"],

        "Spread_Class":
            spread_class,

        "Nearest_Key_Number":
            row["Nearest_Key_Number"],

        "Distance_From_Key":
            row["Distance_From_Key"]
    })


df_large_spread_routed = pd.DataFrame(
    large_spread_rows
)


print(
    f"\nGames with active market routes:      "
    f"{len(df_large_spread_routed)}"
)

if len(df_large_spread_routed) > 0:

    display(
        df_large_spread_routed
    )

CFB1 LARGE-SPREAD CLASSIFICATION

Games tested:                         57
Favorite laying 16.5 or more:         6
Favorite laying 20.5 or more:         6

Missing large-spread input rows:       0
Missing large-spread output rows:      0
Invalid negative spread sizes:         0

--------------------------------------------------------------------------------------
CFB1 LARGE-SPREAD CLASSIFICATION: PASS

Games with active market routes:      10


,Game,Consensus_Favorite,Favorite_Spread_Size,Spread_Class,Nearest_Key_Number,Distance_From_Key
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,2.75,BELOW_16.5,3,0.25
1,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,36.50,VERY_LARGE_20.5_PLUS,14,22.50
2,Buffalo Bulls at Toledo Rockets,Toledo Rockets,20.75,VERY_LARGE_20.5_PLUS,14,6.75
3,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,28.50,VERY_LARGE_20.5_PLUS,14,14.50
4,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,6.50,BELOW_16.5,7,0.50
5,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,38.00,VERY_LARGE_20.5_PLUS,14,24.00
6,Virginia Tech Hokies at California Golden Bears,Virginia Tech Hokies,9.75,BELOW_16.5,10,0.25
7,Rice Owls at East Carolina Pirates,East Carolina Pirates,9.75,BELOW_16.5,10,0.25
8,Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,9.00,BELOW_16.5,10,1.00
9,UAB Blazers at Memphis Tigers,Memphis Tigers,14.25,BELOW_16.5,14,0.25


In [ ]:
# ==========================================
# OLD CELL 39 — LARGE-SPREAD MARKET ROUTING
# ==========================================

print("=" * 100)
print("CFB1 LARGE-SPREAD MARKET ROUTING")
print("=" * 100)


# ==========================================
# 1. BUILD MARKET LOOKUP
# ==========================================

market_route_lookup = df_master_bcf[
    [
        "Away_Team",
        "Home_Team",
        "Favorite_Spread_Size"
    ]
].copy()

market_route_lookup["Game"] = (
    market_route_lookup["Away_Team"]
    + " at "
    + market_route_lookup["Home_Team"]
)

market_route_lookup = market_route_lookup[
    [
        "Game",
        "Favorite_Spread_Size"
    ]
]


# ==========================================
# 2. MERGE SPREAD SIZE INTO ROUTING BOARD
# ==========================================

routing_base = df_market_routes.merge(
    market_route_lookup,
    on="Game",
    how="left",
    validate="one_to_one"
)


# ==========================================
# 3. APPLY LARGE-SPREAD ROUTING
# ==========================================

final_route_rows = []

for _, route_row in routing_base.iterrows():

    game = route_row["Game"]
    spread_size = route_row["Favorite_Spread_Size"]

    routes = []

    if route_row["Eligible_Market_Routes"]:

        routes = [
            x.strip()
            for x in route_row[
                "Eligible_Market_Routes"
            ].split(",")
        ]

    spread_status = route_row[
        "Spread_Status"
    ]


    # --------------------------------------
    # LARGE-SPREAD ROUTING
    # --------------------------------------

    if spread_size >= VERY_LARGE_SPREAD_THRESHOLD:

        spread_status = (
            "DERIVATIVES_FIRST_20.5_PLUS"
        )

        routes.append(
            "1H_SPREAD_REVIEW"
        )

        routes.append(
            "TEAM_TOTAL_REVIEW"
        )


    elif spread_size >= LARGE_SPREAD_THRESHOLD:

        spread_status = (
            "LARGE_SPREAD_CAUTION"
        )

        routes.append(
            "1H_SPREAD_REVIEW"
        )

        routes.append(
            "TEAM_TOTAL_REVIEW"
        )


    # --------------------------------------
    # MODULE 3 CONFLICT OVERRIDE
    # --------------------------------------

    if route_row["M3_Conflict"] != "":

        spread_status = (
            "DOWNGRADE_FULL_GAME_SPREAD"
        )

        routes.append(
            "DERIVATIVES_FIRST"
        )


    # --------------------------------------
    # REMOVE DUPLICATES
    # PRESERVE ROUTE ORDER
    # --------------------------------------

    routes = list(
        dict.fromkeys(routes)
    )


    final_route_rows.append({

        "Game":
            game,

        "Model_Side":
            route_row["Model_Side"],

        "EPA_Value":
            route_row["EPA_Value"],

        "FEI_Value":
            route_row["FEI_Value"],

        "M3_Support":
            route_row["M3_Support"],

        "M3_Conflict":
            route_row["M3_Conflict"],

        "Favorite_Spread_Size":
            spread_size,

        "Spread_Status":
            spread_status,

        "Eligible_Market_Routes":
            ", ".join(routes),

        "Final_Bet_Status":
            "UNPRICED — DO NOT BET YET"
    })


df_final_routes = pd.DataFrame(
    final_route_rows
)


# ==========================================
# 4. ACTIVE ROUTES
# ==========================================

active_routes = df_final_routes[
    df_final_routes[
        "Eligible_Market_Routes"
    ] != ""
].copy()


# ==========================================
# 5. VALIDATION
# ==========================================

expected_games = len(
    df_market_routes
)

duplicate_games = int(
    df_final_routes.duplicated(
        subset=["Game"],
        keep=False
    ).sum()
)

missing_spread_size = int(
    df_final_routes[
        "Favorite_Spread_Size"
    ].isna().sum()
)

missing_required = int(
    df_final_routes[
        [
            "Game",
            "Model_Side",
            "EPA_Value",
            "FEI_Value",
            "Favorite_Spread_Size",
            "Spread_Status",
            "Eligible_Market_Routes",
            "Final_Bet_Status"
        ]
    ].isna().any(axis=1).sum()
)


# ==========================================
# 6. REPORT
# ==========================================

print(
    f"\nEPA + FEI agreement games:      "
    f"{expected_games}"
)

print(
    f"Final routing rows:             "
    f"{len(df_final_routes)}"
)

print(
    f"Games with active routes:       "
    f"{len(active_routes)}"
)

print(
    f"\nDuplicate game rows:            "
    f"{duplicate_games}"
)

print(
    f"Missing spread-size values:     "
    f"{missing_spread_size}"
)

print(
    f"Missing required values:        "
    f"{missing_required}"
)


# ==========================================
# 7. FINAL STATUS
# ==========================================

print("\n" + "-" * 100)

if (
    len(df_final_routes) == expected_games
    and duplicate_games == 0
    and missing_spread_size == 0
    and missing_required == 0
):

    print(
        "CFB1 LARGE-SPREAD MARKET ROUTING: PASS"
    )

else:

    print(
        "CFB1 LARGE-SPREAD MARKET ROUTING: STOP"
    )

    raise ValueError(
        "Large-spread market routing validation failed."
    )

print("=" * 100)


# ==========================================
# 8. SPREAD STATUS COUNTS
# ==========================================

print("\nSPREAD STATUS COUNTS")
print("-" * 40)

print(
    active_routes[
        "Spread_Status"
    ].value_counts()
)


# ==========================================
# 9. ACTIVE ROUTED GAMES
# ==========================================

if len(active_routes) > 0:

    display(
        active_routes[
            [
                "Game",
                "Model_Side",
                "EPA_Value",
                "FEI_Value",
                "M3_Support",
                "M3_Conflict",
                "Favorite_Spread_Size",
                "Spread_Status",
                "Eligible_Market_Routes",
                "Final_Bet_Status"
            ]
        ]
    )

CFB1 LARGE-SPREAD MARKET ROUTING

EPA + FEI agreement games:      37
Final routing rows:             37
Games with active routes:       12

Duplicate game rows:            0
Missing spread-size values:     0
Missing required values:        0

----------------------------------------------------------------------------------------------------
CFB1 LARGE-SPREAD MARKET ROUTING: PASS

SPREAD STATUS COUNTS
----------------------------------------
Spread_Status
STANDARD_REVIEW                6
DERIVATIVES_FIRST_20.5_PLUS    6
Name: count, dtype: int64


,Game,Model_Side,EPA_Value,FEI_Value,M3_Support,M3_Conflict,Favorite_Spread_Size,Spread_Status,Eligible_Market_Routes,Final_Bet_Status
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,0.219468,2.4,Jacksonville State Gamecocks +OAY_DAY,,2.75,STANDARD_REVIEW,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",UNPRICED — DO NOT BET YET
10,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,0.467262,12.1,Northwestern Wildcats +OAY_DAY,,36.50,DERIVATIVES_FIRST_20.5_PLUS,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",UNPRICED — DO NOT BET YET
11,Buffalo Bulls at Toledo Rockets,Toledo Rockets,0.363014,3.0,Toledo Rockets +OAY_DAY,,20.75,DERIVATIVES_FIRST_20.5_PLUS,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",UNPRICED — DO NOT BET YET
13,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,0.441874,2.1,North Texas Mean Green +OAY_DAY,,28.50,DERIVATIVES_FIRST_20.5_PLUS,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",UNPRICED — DO NOT BET YET
14,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,0.373008,5.2,Duke Blue Devils +OAY_DAY,,6.50,STANDARD_REVIEW,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",UNPRICED — DO NOT BET YET
20,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,0.643768,21.5,Stanford Cardinal -HAVOC_TRAP,,38.00,DERIVATIVES_FIRST_20.5_PLUS,"FULL_GAME_SPREAD, OPP_TEAM_TOTAL_UNDER, OPP_1H...",UNPRICED — DO NOT BET YET
22,Virginia Tech Hokies at California Golden Bears,Virginia Tech Hokies,0.274383,4.1,California Golden Bears -HAVOC_TRAP,,9.75,STANDARD_REVIEW,"FULL_GAME_SPREAD, OPP_TEAM_TOTAL_UNDER, OPP_1H...",UNPRICED — DO NOT BET YET
24,Rice Owls at East Carolina Pirates,East Carolina Pirates,0.158938,7.7,Rice Owls -HAVOC_TRAP,,9.75,STANDARD_REVIEW,"FULL_GAME_SPREAD, OPP_TEAM_TOTAL_UNDER, OPP_1H...",UNPRICED — DO NOT BET YET
25,Maryland Terrapins at Ohio State Buckeyes,Ohio State Buckeyes,0.432617,11.7,,,34.50,DERIVATIVES_FIRST_20.5_PLUS,"1H_SPREAD_REVIEW, TEAM_TOTAL_REVIEW",UNPRICED — DO NOT BET YET
28,Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,0.412176,11.8,Nevada Wolf Pack +OAY_DAY,,9.00,STANDARD_REVIEW,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",UNPRICED — DO NOT BET YET


In [ ]:
# ==========================================
# OLD CELL 40 — EVIDENCE VS ROUTING SEPARATION
# ==========================================

print("=" * 104)
print("CFB1 EVIDENCE VS ROUTING SEPARATION")
print("=" * 104)


# ==========================================
# 1. BUILD EVIDENCE / ROUTING TABLE
# ==========================================

evidence_route_rows = []

for _, row in df_final_routes.iterrows():

    evidence = []
    routing_modifiers = []


    # --------------------------------------
    # MODEL EVIDENCE
    # --------------------------------------

    evidence.append(
        "EPA_FEI_DIRECTIONAL_AGREEMENT"
    )

    if row["M3_Support"] != "":
        evidence.append(
            "MODULE3_SUPPORT"
        )

    if row["M3_Conflict"] != "":
        evidence.append(
            "MODULE3_CONFLICT"
        )


    # --------------------------------------
    # ROUTING MODIFIERS
    # These do NOT count as evidence.
    # --------------------------------------

    if (
        row["Favorite_Spread_Size"]
        >= VERY_LARGE_SPREAD_THRESHOLD
    ):

        routing_modifiers.append(
            "VERY_LARGE_SPREAD_20.5_PLUS"
        )

    elif (
        row["Favorite_Spread_Size"]
        >= LARGE_SPREAD_THRESHOLD
    ):

        routing_modifiers.append(
            "LARGE_SPREAD_16.5_TO_20"
        )


    if (
        row["Spread_Status"]
        == "DOWNGRADE_FULL_GAME_SPREAD"
    ):

        routing_modifiers.append(
            "FULL_GAME_SPREAD_DOWNGRADE"
        )


    # --------------------------------------
    # EVIDENCE CLASS
    # --------------------------------------

    if row["M3_Support"] != "":

        evidence_class = (
            "MODEL_PLUS_MATCHUP_SUPPORT"
        )

    elif row["M3_Conflict"] != "":

        evidence_class = (
            "MODEL_WITH_MATCHUP_CONFLICT"
        )

    else:

        evidence_class = (
            "MODEL_AGREEMENT_ONLY"
        )


    evidence_route_rows.append({

        "Game":
            row["Game"],

        "Model_Side":
            row["Model_Side"],

        "EPA_Value":
            row["EPA_Value"],

        "FEI_Value":
            row["FEI_Value"],

        "Evidence_Class":
            evidence_class,

        "Evidence":
            ", ".join(evidence),

        "Routing_Modifiers":
            ", ".join(routing_modifiers),

        "Spread_Status":
            row["Spread_Status"],

        "Eligible_Market_Routes":
            row["Eligible_Market_Routes"],

        "Final_Bet_Status":
            "UNPRICED — DO NOT BET YET"
    })


df_evidence_routes = pd.DataFrame(
    evidence_route_rows
)


# ==========================================
# 2. VALIDATION
# ==========================================

expected_games = len(
    df_final_routes
)

duplicate_games = int(
    df_evidence_routes.duplicated(
        subset=["Game"],
        keep=False
    ).sum()
)

missing_required = int(
    df_evidence_routes[
        [
            "Game",
            "Model_Side",
            "EPA_Value",
            "FEI_Value",
            "Evidence_Class",
            "Evidence",
            "Spread_Status",
            "Eligible_Market_Routes",
            "Final_Bet_Status"
        ]
    ].isna().any(axis=1).sum()
)

blank_evidence = int(
    (
        df_evidence_routes[
            "Evidence"
        ] == ""
    ).sum()
)


# ==========================================
# 3. REPORT
# ==========================================

print(
    f"\nExpected rows:                  "
    f"{expected_games}"
)

print(
    f"Evidence/routing rows:          "
    f"{len(df_evidence_routes)}"
)

print(
    f"Duplicate game rows:            "
    f"{duplicate_games}"
)

print(
    f"Missing required values:        "
    f"{missing_required}"
)

print(
    f"Blank evidence rows:            "
    f"{blank_evidence}"
)


# ==========================================
# 4. FINAL STATUS
# ==========================================

print("\n" + "-" * 104)

if (
    len(df_evidence_routes) == expected_games
    and duplicate_games == 0
    and missing_required == 0
    and blank_evidence == 0
):

    print(
        "CFB1 EVIDENCE VS ROUTING SEPARATION: PASS"
    )

else:

    print(
        "CFB1 EVIDENCE VS ROUTING SEPARATION: STOP"
    )

    raise ValueError(
        "Evidence vs Routing validation failed."
    )

print("=" * 104)


# ==========================================
# 5. EVIDENCE CLASS COUNTS
# ==========================================

print("\nEVIDENCE CLASS COUNTS")
print("-" * 45)

print(
    df_evidence_routes[
        "Evidence_Class"
    ].value_counts()
)


# ==========================================
# 6. MODULE 3 SUPPORT / CONFLICT
# ==========================================

module3_evidence = df_evidence_routes[
    df_evidence_routes[
        "Evidence_Class"
    ] != "MODEL_AGREEMENT_ONLY"
].copy()

print(
    f"\nGames with Module 3 evidence:   "
    f"{len(module3_evidence)}"
)

if len(module3_evidence) > 0:

    display(
        module3_evidence[
            [
                "Game",
                "Model_Side",
                "EPA_Value",
                "FEI_Value",
                "Evidence_Class",
                "Evidence",
                "Routing_Modifiers",
                "Spread_Status",
                "Eligible_Market_Routes"
            ]
        ]
    )


# ==========================================
# 7. MODEL-ONLY + ROUTING MODIFIERS
# ==========================================

model_only_routed = df_evidence_routes[
    (
        df_evidence_routes[
            "Evidence_Class"
        ]
        == "MODEL_AGREEMENT_ONLY"
    )
    &
    (
        df_evidence_routes[
            "Routing_Modifiers"
        ] != ""
    )
].copy()

print(
    f"\nModel-only games with routing "
    f"modifiers: {len(model_only_routed)}"
)

if len(model_only_routed) > 0:

    display(
        model_only_routed[
            [
                "Game",
                "Model_Side",
                "EPA_Value",
                "FEI_Value",
                "Evidence_Class",
                "Routing_Modifiers",
                "Spread_Status",
                "Eligible_Market_Routes"
            ]
        ]
    )

CFB1 EVIDENCE VS ROUTING SEPARATION

Expected rows:                  37
Evidence/routing rows:          37
Duplicate game rows:            0
Missing required values:        0
Blank evidence rows:            0

--------------------------------------------------------------------------------------------------------
CFB1 EVIDENCE VS ROUTING SEPARATION: PASS

EVIDENCE CLASS COUNTS
---------------------------------------------
Evidence_Class
MODEL_AGREEMENT_ONLY          27
MODEL_PLUS_MATCHUP_SUPPORT    10
Name: count, dtype: int64

Games with Module 3 evidence:   10


,Game,Model_Side,EPA_Value,FEI_Value,Evidence_Class,Evidence,Routing_Modifiers,Spread_Status,Eligible_Market_Routes
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,0.219468,2.4,MODEL_PLUS_MATCHUP_SUPPORT,"EPA_FEI_DIRECTIONAL_AGREEMENT, MODULE3_SUPPORT",,STANDARD_REVIEW,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT..."
10,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,0.467262,12.1,MODEL_PLUS_MATCHUP_SUPPORT,"EPA_FEI_DIRECTIONAL_AGREEMENT, MODULE3_SUPPORT",VERY_LARGE_SPREAD_20.5_PLUS,DERIVATIVES_FIRST_20.5_PLUS,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT..."
11,Buffalo Bulls at Toledo Rockets,Toledo Rockets,0.363014,3.0,MODEL_PLUS_MATCHUP_SUPPORT,"EPA_FEI_DIRECTIONAL_AGREEMENT, MODULE3_SUPPORT",VERY_LARGE_SPREAD_20.5_PLUS,DERIVATIVES_FIRST_20.5_PLUS,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT..."
13,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,0.441874,2.1,MODEL_PLUS_MATCHUP_SUPPORT,"EPA_FEI_DIRECTIONAL_AGREEMENT, MODULE3_SUPPORT",VERY_LARGE_SPREAD_20.5_PLUS,DERIVATIVES_FIRST_20.5_PLUS,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT..."
14,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,0.373008,5.2,MODEL_PLUS_MATCHUP_SUPPORT,"EPA_FEI_DIRECTIONAL_AGREEMENT, MODULE3_SUPPORT",,STANDARD_REVIEW,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT..."
20,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,0.643768,21.5,MODEL_PLUS_MATCHUP_SUPPORT,"EPA_FEI_DIRECTIONAL_AGREEMENT, MODULE3_SUPPORT",VERY_LARGE_SPREAD_20.5_PLUS,DERIVATIVES_FIRST_20.5_PLUS,"FULL_GAME_SPREAD, OPP_TEAM_TOTAL_UNDER, OPP_1H..."
22,Virginia Tech Hokies at California Golden Bears,Virginia Tech Hokies,0.274383,4.1,MODEL_PLUS_MATCHUP_SUPPORT,"EPA_FEI_DIRECTIONAL_AGREEMENT, MODULE3_SUPPORT",,STANDARD_REVIEW,"FULL_GAME_SPREAD, OPP_TEAM_TOTAL_UNDER, OPP_1H..."
24,Rice Owls at East Carolina Pirates,East Carolina Pirates,0.158938,7.7,MODEL_PLUS_MATCHUP_SUPPORT,"EPA_FEI_DIRECTIONAL_AGREEMENT, MODULE3_SUPPORT",,STANDARD_REVIEW,"FULL_GAME_SPREAD, OPP_TEAM_TOTAL_UNDER, OPP_1H..."
28,Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,0.412176,11.8,MODEL_PLUS_MATCHUP_SUPPORT,"EPA_FEI_DIRECTIONAL_AGREEMENT, MODULE3_SUPPORT",,STANDARD_REVIEW,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT..."
29,UAB Blazers at Memphis Tigers,Memphis Tigers,0.053547,9.7,MODEL_PLUS_MATCHUP_SUPPORT,"EPA_FEI_DIRECTIONAL_AGREEMENT, MODULE3_SUPPORT",,STANDARD_REVIEW,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT..."



Model-only games with routing modifiers: 2


,Game,Model_Side,EPA_Value,FEI_Value,Evidence_Class,Routing_Modifiers,Spread_Status,Eligible_Market_Routes
25,Maryland Terrapins at Ohio State Buckeyes,Ohio State Buckeyes,0.432617,11.7,MODEL_AGREEMENT_ONLY,VERY_LARGE_SPREAD_20.5_PLUS,DERIVATIVES_FIRST_20.5_PLUS,"1H_SPREAD_REVIEW, TEAM_TOTAL_REVIEW"
36,Hawaii Rainbow Warriors at Arizona State Sun D...,Arizona State Sun Devils,0.050585,4.4,MODEL_AGREEMENT_ONLY,VERY_LARGE_SPREAD_20.5_PLUS,DERIVATIVES_FIRST_20.5_PLUS,"1H_SPREAD_REVIEW, TEAM_TOTAL_REVIEW"


In [ ]:
# ==========================================
# OLD CELL 41 — PHASE 1 SHORTLIST
# ==========================================

print("=" * 92)
print("CFB1 PHASE 1 SHORTLIST")
print("=" * 92)


# ==========================================
# 1. CONSERVATIVE FEI EDGE
# Use smaller DK / FD edge.
# ==========================================

df_master_bcf["FEI_Edge_Conservative"] = (
    df_master_bcf[
        [
            "DK_FEI_Edge_Value",
            "FD_FEI_Edge_Value"
        ]
    ].min(axis=1)
)


# ==========================================
# 2. WEEKLY MODEL-STRENGTH CUT POINTS
# ==========================================

EPA_75 = df_master_bcf[
    "EPA_Edge_Value"
].quantile(0.75)

EPA_90 = df_master_bcf[
    "EPA_Edge_Value"
].quantile(0.90)

FEI_75 = df_master_bcf[
    "FEI_Edge_Conservative"
].quantile(0.75)

FEI_90 = df_master_bcf[
    "FEI_Edge_Conservative"
].quantile(0.90)


print(
    f"\nWeek {TARGET_WEEK} model-strength cut points"
)

print("-" * 50)

print(
    f"EPA top 25% begins at:           "
    f"{EPA_75:.3f}"
)

print(
    f"EPA top 10% begins at:           "
    f"{EPA_90:.3f}"
)

print(
    f"\nFEI top 25% begins at:           "
    f"{FEI_75:.1f} points"
)

print(
    f"FEI top 10% begins at:           "
    f"{FEI_90:.1f} points"
)


# ==========================================
# 3. BUILD PHASE 1 CANDIDATE UNIVERSE
# ==========================================

phase1_rows = []

for _, row in df_master_bcf.iterrows():

    game = (
        f"{row['Away_Team']} "
        f"at {row['Home_Team']}"
    )

    evidence_match = df_evidence_routes[
        df_evidence_routes[
            "Game"
        ] == game
    ]

    # Only EPA + FEI agreement games exist
    # in the evidence-routing universe.
    if len(evidence_match) == 0:
        continue

    evidence_row = evidence_match.iloc[0]


    # --------------------------------------
    # MODEL AGREEMENT
    # --------------------------------------

    model_agreement = (
        row["EPA_Edge_Team"]
        == row["DK_FEI_Edge_Team"]
        and
        row["EPA_Edge_Team"] != "EVEN"
    )


    # --------------------------------------
    # MODEL-STRENGTH TESTS
    # --------------------------------------

    epa_top25 = (
        row["EPA_Edge_Value"]
        >= EPA_75
    )

    fei_top25 = (
        row["FEI_Edge_Conservative"]
        >= FEI_75
    )

    dual_top25 = (
        model_agreement
        and epa_top25
        and fei_top25
    )


    # --------------------------------------
    # MODULE 3 EVIDENCE
    # --------------------------------------

    module3_support = (
        evidence_row["Evidence_Class"]
        == "MODEL_PLUS_MATCHUP_SUPPORT"
    )

    module3_conflict = (
        evidence_row["Evidence_Class"]
        == "MODEL_WITH_MATCHUP_CONFLICT"
    )


    # --------------------------------------
    # PHASE 1 REASONS
    # --------------------------------------

    reasons = []

    if module3_support:
        reasons.append(
            "MODULE3_SUPPORT"
        )

    if dual_top25:
        reasons.append(
            "DUAL_TOP25_MODEL_STRENGTH"
        )

    if module3_conflict:
        reasons.append(
            "MODULE3_CONFLICT"
        )


    phase1_candidate = (
        module3_support
        or dual_top25
        or module3_conflict
    )


    phase1_rows.append({

        "Game":
            game,

        "Model_Side":
            evidence_row["Model_Side"],

        "EPA_Value":
            row["EPA_Edge_Value"],

        "EPA_Top25":
            epa_top25,

        "FEI_Value_Conservative":
            row["FEI_Edge_Conservative"],

        "FEI_Top25":
            fei_top25,

        "Dual_Top25":
            dual_top25,

        "Evidence_Class":
            evidence_row["Evidence_Class"],

        "Candidate_Reason":
            ", ".join(reasons),

        "Phase1_Candidate":
            phase1_candidate
    })


df_phase1_diagnostic = pd.DataFrame(
    phase1_rows
)


# ==========================================
# 4. PHASE 1 CANDIDATE BOARD
# ==========================================

candidate_board = df_phase1_diagnostic[
    df_phase1_diagnostic[
        "Phase1_Candidate"
    ]
].copy()


# ==========================================
# 5. VALIDATION
# ==========================================

expected_agreement_games = len(
    df_evidence_routes
)

duplicate_games = int(
    df_phase1_diagnostic.duplicated(
        subset=["Game"],
        keep=False
    ).sum()
)

missing_required = int(
    df_phase1_diagnostic[
        [
            "Game",
            "Model_Side",
            "EPA_Value",
            "EPA_Top25",
            "FEI_Value_Conservative",
            "FEI_Top25",
            "Dual_Top25",
            "Evidence_Class",
            "Phase1_Candidate"
        ]
    ].isna().any(axis=1).sum()
)


# ==========================================
# 6. CANDIDATE COUNTS
# ==========================================

module3_count = int(
    candidate_board[
        "Candidate_Reason"
    ]
    .str.contains(
        "MODULE3_SUPPORT",
        regex=False
    )
    .sum()
)

dual_count = int(
    candidate_board[
        "Candidate_Reason"
    ]
    .str.contains(
        "DUAL_TOP25_MODEL_STRENGTH",
        regex=False
    )
    .sum()
)

conflict_count = int(
    candidate_board[
        "Candidate_Reason"
    ]
    .str.contains(
        "MODULE3_CONFLICT",
        regex=False
    )
    .sum()
)


# ==========================================
# 7. REPORT
# ==========================================

print(
    f"\nEPA + FEI agreement games:           "
    f"{expected_agreement_games}"
)

print(
    f"Phase 1 evaluation rows:             "
    f"{len(df_phase1_diagnostic)}"
)

print(
    f"Phase 1 candidates:                  "
    f"{len(candidate_board)}"
)

print(
    f"With Module 3 support:               "
    f"{module3_count}"
)

print(
    f"With dual top-25% model strength:    "
    f"{dual_count}"
)

print(
    f"With Module 3 conflict:              "
    f"{conflict_count}"
)

print(
    f"\nDuplicate game rows:                 "
    f"{duplicate_games}"
)

print(
    f"Missing required values:             "
    f"{missing_required}"
)


# ==========================================
# 8. FINAL STATUS
# ==========================================

print("\n" + "-" * 92)

if (
    len(df_phase1_diagnostic)
    == expected_agreement_games
    and duplicate_games == 0
    and missing_required == 0
):

    print(
        "CFB1 PHASE 1 SHORTLIST: PASS"
    )

else:

    print(
        "CFB1 PHASE 1 SHORTLIST: STOP"
    )

    raise ValueError(
        "Phase 1 shortlist validation failed."
    )

print("=" * 92)


# ==========================================
# 9. DISPLAY PHASE 1 CANDIDATES
# ==========================================

if len(candidate_board) > 0:

    display(
        candidate_board[
            [
                "Game",
                "Model_Side",
                "EPA_Value",
                "EPA_Top25",
                "FEI_Value_Conservative",
                "FEI_Top25",
                "Dual_Top25",
                "Evidence_Class",
                "Candidate_Reason"
            ]
        ].sort_values(
            [
                "Dual_Top25",
                "EPA_Value",
                "FEI_Value_Conservative"
            ],
            ascending=[
                False,
                False,
                False
            ]
        )
    )

CFB1 PHASE 1 SHORTLIST

Week 6 model-strength cut points
--------------------------------------------------
EPA top 25% begins at:           0.267
EPA top 10% begins at:           0.420

FEI top 25% begins at:           10.6 points
FEI top 10% begins at:           13.2 points

EPA + FEI agreement games:           37
Phase 1 evaluation rows:             37
Phase 1 candidates:                  14
With Module 3 support:               10
With dual top-25% model strength:    7
With Module 3 conflict:              0

Duplicate game rows:                 0
Missing required values:             0

--------------------------------------------------------------------------------------------
CFB1 PHASE 1 SHORTLIST: PASS


,Game,Model_Side,EPA_Value,EPA_Top25,FEI_Value_Conservative,FEI_Top25,Dual_Top25,Evidence_Class,Candidate_Reason
20,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,0.643768,True,21.5,True,True,MODEL_PLUS_MATCHUP_SUPPORT,"MODULE3_SUPPORT, DUAL_TOP25_MODEL_STRENGTH"
30,Air Force Falcons at Northern Illinois Huskies,Air Force Falcons,0.468333,True,10.8,True,True,MODEL_AGREEMENT_ONLY,DUAL_TOP25_MODEL_STRENGTH
10,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,0.467262,True,12.1,True,True,MODEL_PLUS_MATCHUP_SUPPORT,"MODULE3_SUPPORT, DUAL_TOP25_MODEL_STRENGTH"
26,Tennessee Volunteers at Arkansas Razorbacks,Tennessee Volunteers,0.442441,True,13.8,True,True,MODEL_AGREEMENT_ONLY,DUAL_TOP25_MODEL_STRENGTH
25,Maryland Terrapins at Ohio State Buckeyes,Ohio State Buckeyes,0.432617,True,11.7,True,True,MODEL_AGREEMENT_ONLY,DUAL_TOP25_MODEL_STRENGTH
28,Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,0.412176,True,11.8,True,True,MODEL_PLUS_MATCHUP_SUPPORT,"MODULE3_SUPPORT, DUAL_TOP25_MODEL_STRENGTH"
32,James Madison Dukes at Georgia Southern Eagles,James Madison Dukes,0.374985,True,17.1,True,True,MODEL_AGREEMENT_ONLY,DUAL_TOP25_MODEL_STRENGTH
13,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,0.441874,True,2.1,False,False,MODEL_PLUS_MATCHUP_SUPPORT,MODULE3_SUPPORT
14,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,0.373008,True,5.2,False,False,MODEL_PLUS_MATCHUP_SUPPORT,MODULE3_SUPPORT
11,Buffalo Bulls at Toledo Rockets,Toledo Rockets,0.363014,True,3.0,False,False,MODEL_PLUS_MATCHUP_SUPPORT,MODULE3_SUPPORT


In [ ]:
# ==========================================
# OLD CELL 42 — PHASE 1 REVIEW BOARD
# ==========================================

print("=" * 100)
print("CFB1 PHASE 1 — LATE-WEEK REVIEW BOARD")
print("=" * 100)


# ==========================================
# 1. BUILD REVIEW BOARD
# ==========================================

phase1_final_rows = []

for _, row in candidate_board.iterrows():

    reasons = row["Candidate_Reason"]


    # --------------------------------------
    # REVIEW CLASS
    # --------------------------------------

    if (
        "MODULE3_SUPPORT" in reasons
        and
        "DUAL_TOP25_MODEL_STRENGTH" in reasons
    ):

        review_class = (
            "MODEL_PLUS_MATCHUP"
        )

    elif (
        "DUAL_TOP25_MODEL_STRENGTH"
        in reasons
    ):

        review_class = (
            "STRONG_MODEL_AGREEMENT"
        )

    elif (
        "MODULE3_SUPPORT"
        in reasons
    ):

        review_class = (
            "MATCHUP_SUPPORTED"
        )

    elif (
        "MODULE3_CONFLICT"
        in reasons
    ):

        review_class = (
            "CONFLICT_REVIEW"
        )

    else:

        review_class = (
            "UNCLASSIFIED"
        )


    phase1_final_rows.append({

        "Game":
            row["Game"],

        "Model_Side":
            row["Model_Side"],

        "EPA_Value":
            row["EPA_Value"],

        "FEI_Value_Conservative":
            row[
                "FEI_Value_Conservative"
            ],

        "Evidence_Class":
            row["Evidence_Class"],

        "Review_Class":
            review_class,

        "Phase1_Reason":
            reasons,

        "Late_Week_Action":
            "COLLECT_COVERS_LINE_MOVEMENT"
    })


df_phase1_review = pd.DataFrame(
    phase1_final_rows
)


# ==========================================
# 2. VALIDATION
# ==========================================

expected_games = len(
    candidate_board
)

duplicate_games = int(
    df_phase1_review.duplicated(
        subset=["Game"],
        keep=False
    ).sum()
)

missing_required = int(
    df_phase1_review[
        [
            "Game",
            "Model_Side",
            "EPA_Value",
            "FEI_Value_Conservative",
            "Evidence_Class",
            "Review_Class",
            "Phase1_Reason",
            "Late_Week_Action"
        ]
    ].isna().any(axis=1).sum()
)

unclassified_games = int(
    (
        df_phase1_review[
            "Review_Class"
        ] == "UNCLASSIFIED"
    ).sum()
)


# ==========================================
# 3. REPORT
# ==========================================

print(
    f"\nPhase 1 candidates:             "
    f"{expected_games}"
)

print(
    f"Review-board rows:             "
    f"{len(df_phase1_review)}"
)

print(
    f"Duplicate game rows:           "
    f"{duplicate_games}"
)

print(
    f"Missing required values:       "
    f"{missing_required}"
)

print(
    f"Unclassified games:            "
    f"{unclassified_games}"
)


# ==========================================
# 4. FINAL STATUS
# ==========================================

print("\n" + "-" * 100)

if (
    len(df_phase1_review) == expected_games
    and duplicate_games == 0
    and missing_required == 0
    and unclassified_games == 0
):

    print(
        "CFB1 PHASE 1 REVIEW BOARD: PASS"
    )

else:

    print(
        "CFB1 PHASE 1 REVIEW BOARD: STOP"
    )

    raise ValueError(
        "Phase 1 Review Board validation failed."
    )

print("=" * 100)


# ==========================================
# 5. REVIEW CLASS COUNTS
# ==========================================

print("\nREVIEW CLASS COUNTS")
print("-" * 45)

print(
    df_phase1_review[
        "Review_Class"
    ].value_counts()
)


# ==========================================
# 6. PHASE 1 REVIEW GAMES
# ==========================================

if len(df_phase1_review) > 0:

    display(
        df_phase1_review[
            [
                "Game",
                "Model_Side",
                "EPA_Value",
                "FEI_Value_Conservative",
                "Review_Class",
                "Phase1_Reason",
                "Late_Week_Action"
            ]
        ]
    )

CFB1 PHASE 1 — LATE-WEEK REVIEW BOARD

Phase 1 candidates:             14
Review-board rows:             14
Duplicate game rows:           0
Missing required values:       0
Unclassified games:            0

----------------------------------------------------------------------------------------------------
CFB1 PHASE 1 REVIEW BOARD: PASS

REVIEW CLASS COUNTS
---------------------------------------------
Review_Class
MATCHUP_SUPPORTED         7
STRONG_MODEL_AGREEMENT    4
MODEL_PLUS_MATCHUP        3
Name: count, dtype: int64


,Game,Model_Side,EPA_Value,FEI_Value_Conservative,Review_Class,Phase1_Reason,Late_Week_Action
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,0.219468,2.4,MATCHUP_SUPPORTED,MODULE3_SUPPORT,COLLECT_COVERS_LINE_MOVEMENT
1,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,0.467262,12.1,MODEL_PLUS_MATCHUP,"MODULE3_SUPPORT, DUAL_TOP25_MODEL_STRENGTH",COLLECT_COVERS_LINE_MOVEMENT
2,Buffalo Bulls at Toledo Rockets,Toledo Rockets,0.363014,3.0,MATCHUP_SUPPORTED,MODULE3_SUPPORT,COLLECT_COVERS_LINE_MOVEMENT
3,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,0.441874,2.1,MATCHUP_SUPPORTED,MODULE3_SUPPORT,COLLECT_COVERS_LINE_MOVEMENT
4,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,0.373008,5.2,MATCHUP_SUPPORTED,MODULE3_SUPPORT,COLLECT_COVERS_LINE_MOVEMENT
5,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,0.643768,21.5,MODEL_PLUS_MATCHUP,"MODULE3_SUPPORT, DUAL_TOP25_MODEL_STRENGTH",COLLECT_COVERS_LINE_MOVEMENT
6,Virginia Tech Hokies at California Golden Bears,Virginia Tech Hokies,0.274383,4.1,MATCHUP_SUPPORTED,MODULE3_SUPPORT,COLLECT_COVERS_LINE_MOVEMENT
7,Rice Owls at East Carolina Pirates,East Carolina Pirates,0.158938,7.7,MATCHUP_SUPPORTED,MODULE3_SUPPORT,COLLECT_COVERS_LINE_MOVEMENT
8,Maryland Terrapins at Ohio State Buckeyes,Ohio State Buckeyes,0.432617,11.7,STRONG_MODEL_AGREEMENT,DUAL_TOP25_MODEL_STRENGTH,COLLECT_COVERS_LINE_MOVEMENT
9,Tennessee Volunteers at Arkansas Razorbacks,Tennessee Volunteers,0.442441,13.8,STRONG_MODEL_AGREEMENT,DUAL_TOP25_MODEL_STRENGTH,COLLECT_COVERS_LINE_MOVEMENT


In [ ]:
# ==========================================
# OLD CELL 43 — COVERS SPREAD MOVEMENT AUDIT
# ==========================================

from io import StringIO

print("=" * 92)
print("CFB1 COVERS LATE-WEEK MARKET AUDIT")
print("=" * 92)

print(
    f"\nTarget Week: {TARGET_WEEK}"
)

print(
    f"Phase 1 games requiring Covers review: "
    f"{len(df_phase1_review)}"
)


# ==========================================
# 1. WEEKLY COVERS INPUT
#
# Replace ONLY the rows inside this CSV block
# with the current week's Covers data.
#
# IMPORTANT:
# - One row per Phase 1 game
# - Home-team spread perspective
# - Preserve opening/current spreads exactly
# - Do not add non-Phase-1 games
# ==========================================

covers_csv = """
Game,Away_Team,Home_Team,Open_Home_Spread,Current_Home_Spread,Open_Total,Current_Total,Open_Home_Price,Current_Home_Price,Open_Away_Price,Current_Away_Price,Source,Data_Status
Jacksonville State Gamecocks at Kennesaw State Owls,Jacksonville State Gamecocks,Kennesaw State Owls,+6.5,+8.0,50,50,-110,-110,-110,-110,SYNTHETIC_TEST,FICTIONAL
Ball State Cardinals at Northwestern Wildcats,Ball State Cardinals,Northwestern Wildcats,-6.5,-7.5,50,50,-110,-110,-110,-110,SYNTHETIC_TEST,FICTIONAL
Buffalo Bulls at Toledo Rockets,Buffalo Bulls,Toledo Rockets,-10.0,-10.0,50,50,-110,-110,-110,-110,SYNTHETIC_TEST,FICTIONAL
Charlotte 49ers at North Texas Mean Green,Charlotte 49ers,North Texas Mean Green,-13.5,-16.0,50,50,-110,-110,-110,-110,SYNTHETIC_TEST,FICTIONAL
Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,Georgia Tech Yellow Jackets,-2.5,-4.0,50,50,-110,-110,-110,-110,SYNTHETIC_TEST,FICTIONAL
Stanford Cardinal at Notre Dame Fighting Irish,Stanford Cardinal,Notre Dame Fighting Irish,-20.5,-24.0,50,50,-110,-110,-110,-110,SYNTHETIC_TEST,FICTIONAL
Virginia Tech Hokies at California Golden Bears,Virginia Tech Hokies,California Golden Bears,-1.0,+1.5,50,50,-110,-110,-110,-110,SYNTHETIC_TEST,FICTIONAL
Rice Owls at East Carolina Pirates,Rice Owls,East Carolina Pirates,-6.5,-7.5,50,50,-110,-110,-110,-110,SYNTHETIC_TEST,FICTIONAL
Maryland Terrapins at Ohio State Buckeyes,Maryland Terrapins,Ohio State Buckeyes,-21.5,-20.5,50,50,-110,-110,-110,-110,SYNTHETIC_TEST,FICTIONAL
Tennessee Volunteers at Arkansas Razorbacks,Tennessee Volunteers,Arkansas Razorbacks,+3.5,+2.5,50,50,-110,-110,-110,-110,SYNTHETIC_TEST,FICTIONAL
Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,UTEP Miners,-2.5,-1.5,50,50,-110,-110,-110,-110,SYNTHETIC_TEST,FICTIONAL
UAB Blazers at Memphis Tigers,UAB Blazers,Memphis Tigers,-14.5,-14.5,50,50,-110,-110,-110,-110,SYNTHETIC_TEST,FICTIONAL
Air Force Falcons at Northern Illinois Huskies,Air Force Falcons,Northern Illinois Huskies,+7.5,+6.5,50,50,-110,-110,-110,-110,SYNTHETIC_TEST,FICTIONAL
James Madison Dukes at Georgia Southern Eagles,James Madison Dukes,Georgia Southern Eagles,+2.5,+4.0,50,50,-110,-110,-110,-110,SYNTHETIC_TEST,FICTIONAL
"""


# ==========================================
# 2. LOAD COVERS DATA
# ==========================================

df_covers = pd.read_csv(
    StringIO(covers_csv.strip()),
    skipinitialspace=True
)


# ==========================================
# 3. CLEAN TEXT FIELDS
# ==========================================

for col in df_covers.select_dtypes(
    include="object"
).columns:

    df_covers[col] = (
        df_covers[col]
        .astype(str)
        .str.strip()
    )


# ==========================================
# 4. CONVERT SPREADS TO NUMERIC
# ==========================================

spread_cols = [
    "Open_Home_Spread",
    "Current_Home_Spread"
]

for col in spread_cols:

    df_covers[col] = pd.to_numeric(
        df_covers[col],
        errors="coerce"
    )


# ==========================================
# 5. VALIDATE AGAINST PHASE 1 BOARD
# ==========================================

expected_games = set(
    df_phase1_review["Game"]
)

covers_games = set(
    df_covers["Game"]
)

missing_games = sorted(
    expected_games - covers_games
)

extra_games = sorted(
    covers_games - expected_games
)

duplicate_games = int(
    df_covers.duplicated(
        subset=["Game"],
        keep=False
    ).sum()
)

missing_spread_data = int(
    df_covers[
        [
            "Open_Home_Spread",
            "Current_Home_Spread"
        ]
    ].isna().any(axis=1).sum()
)


print("\n" + "-" * 92)

print(
    f"Phase 1 games expected:          "
    f"{len(expected_games)}"
)

print(
    f"Covers games received:           "
    f"{len(df_covers)}"
)

print(
    f"Missing Phase 1 games:           "
    f"{len(missing_games)}"
)

print(
    f"Unexpected games:                "
    f"{len(extra_games)}"
)

print(
    f"Duplicate games:                 "
    f"{duplicate_games}"
)

print(
    f"Missing spread observations:     "
    f"{missing_spread_data}"
)


if missing_games:

    print("\nMISSING GAMES:")

    for game in missing_games:
        print(game)


if extra_games:

    print("\nUNEXPECTED GAMES:")

    for game in extra_games:
        print(game)


# ==========================================
# 6. PREFLIGHT STATUS
# ==========================================

if (
    len(df_covers) == len(expected_games)
    and len(missing_games) == 0
    and len(extra_games) == 0
    and duplicate_games == 0
    and missing_spread_data == 0
):

    print(
        "\nCFB1 COVERS INTAKE: PASS"
    )

else:

    print(
        "\nCFB1 COVERS INTAKE: STOP"
    )

    raise ValueError(
        "Current-week Covers data must be "
        "completed before the market audit can run."
    )


# ==========================================
# 7. MERGE PHASE 1 MODEL CONTEXT
# ==========================================

covers_audit = df_covers.merge(
    df_phase1_review[
        [
            "Game",
            "Model_Side",
            "Review_Class"
        ]
    ],
    on="Game",
    how="left",
    validate="one_to_one"
)


# ==========================================
# 8. CALCULATE SPREAD MOVEMENT
#
# HOME-TEAM PERSPECTIVE:
#
# Negative movement = toward HOME
# Positive movement = toward AWAY
#
# Example:
# +7 -> +2.5 = -4.5 = toward HOME
# -6 -> -6.5 = -0.5 = toward HOME
# ==========================================

covers_audit["Spread_Move"] = (
    covers_audit[
        "Current_Home_Spread"
    ]
    -
    covers_audit[
        "Open_Home_Spread"
    ]
).round(2)


def movement_side(row):

    move = row["Spread_Move"]

    if move < 0:
        return row["Home_Team"]

    elif move > 0:
        return row["Away_Team"]

    return "NONE"


covers_audit["Movement_Toward"] = (
    covers_audit.apply(
        movement_side,
        axis=1
    )
)


# ==========================================
# 9. MODEL / MARKET RELATIONSHIP
# ==========================================

def market_relationship(row):

    if row["Movement_Toward"] == "NONE":
        return "NEUTRAL"

    if (
        row["Movement_Toward"]
        == row["Model_Side"]
    ):
        return "CONFIRMS_MODEL"

    return "CONFLICTS_WITH_MODEL"


covers_audit["Market_Relationship"] = (
    covers_audit.apply(
        market_relationship,
        axis=1
    )
)


# ==========================================
# 10. KEY-NUMBER CROSSING
# ==========================================

SIGNED_KEY_NUMBERS = [
    -14, -10, -7, -3,
      3,   7, 10, 14
]


def crossed_keys(row):

    opening = row[
        "Open_Home_Spread"
    ]

    current = row[
        "Current_Home_Spread"
    ]

    low = min(
        opening,
        current
    )

    high = max(
        opening,
        current
    )

    crossed = []

    for key in SIGNED_KEY_NUMBERS:

        if low < key < high:

            crossed.append(
                abs(key)
            )

    return ",".join(
        str(key)
        for key in sorted(
            set(crossed)
        )
    )


covers_audit[
    "Key_Numbers_Crossed"
] = covers_audit.apply(
    crossed_keys,
    axis=1
)

covers_audit["Key_Cross"] = (
    covers_audit[
        "Key_Numbers_Crossed"
    ] != ""
)


# ==========================================
# 11. MOVEMENT SIZE / CLASS
# ==========================================

covers_audit["Movement_Size"] = (
    covers_audit[
        "Spread_Move"
    ].abs()
)


def movement_class(size):

    if size == 0:
        return "NO_MOVE"

    elif size < 1:
        return "SMALL"

    elif size < 2:
        return "MODERATE"

    return "MAJOR"


covers_audit["Movement_Class"] = (
    covers_audit[
        "Movement_Size"
    ].apply(
        movement_class
    )
)


# ==========================================
# 12. FINAL AUDIT VALIDATION
# ==========================================

audit_missing = int(
    covers_audit[
        [
            "Model_Side",
            "Review_Class",
            "Spread_Move",
            "Movement_Toward",
            "Market_Relationship",
            "Movement_Size",
            "Movement_Class"
        ]
    ].isna().any(axis=1).sum()
)


print("\n" + "=" * 92)
print("CFB1 LATE-WEEK MARKET MOVEMENT AUDIT")
print("=" * 92)

print(
    f"\nAudit rows:                     "
    f"{len(covers_audit)}"
)

print(
    f"Missing audit values:           "
    f"{audit_missing}"
)

print(
    f"Games crossing a key number:    "
    f"{int(covers_audit['Key_Cross'].sum())}"
)


print("\nMARKET RELATIONSHIP COUNTS")
print("-" * 40)

print(
    covers_audit[
        "Market_Relationship"
    ].value_counts()
)


print("\n" + "-" * 92)

if (
    len(covers_audit)
    == len(df_phase1_review)
    and audit_missing == 0
):

    print(
        "CFB1 COVERS MARKET AUDIT: PASS"
    )

else:

    print(
        "CFB1 COVERS MARKET AUDIT: STOP"
    )

    raise ValueError(
        "Covers market audit validation failed."
    )

print("=" * 92)


# ==========================================
# 13. MARKET MOVEMENT BOARD
# ==========================================

display(
    covers_audit[
        [
            "Game",
            "Model_Side",
            "Review_Class",
            "Open_Home_Spread",
            "Current_Home_Spread",
            "Spread_Move",
            "Movement_Toward",
            "Movement_Size",
            "Movement_Class",
            "Key_Numbers_Crossed",
            "Market_Relationship"
        ]
    ]
)

CFB1 COVERS LATE-WEEK MARKET AUDIT

Target Week: 6
Phase 1 games requiring Covers review: 14

--------------------------------------------------------------------------------------------
Phase 1 games expected:          14
Covers games received:           14
Missing Phase 1 games:           0
Unexpected games:                0
Duplicate games:                 0
Missing spread observations:     0

CFB1 COVERS INTAKE: PASS

CFB1 LATE-WEEK MARKET MOVEMENT AUDIT

Audit rows:                     14
Missing audit values:           0
Games crossing a key number:    8

MARKET RELATIONSHIP COUNTS
----------------------------------------
Market_Relationship
CONFIRMS_MODEL          8
CONFLICTS_WITH_MODEL    4
NEUTRAL                 2
Name: count, dtype: int64

--------------------------------------------------------------------------------------------
CFB1 COVERS MARKET AUDIT: PASS


,Game,Model_Side,Review_Class,Open_Home_Spread,Current_Home_Spread,Spread_Move,Movement_Toward,Movement_Size,Movement_Class,Key_Numbers_Crossed,Market_Relationship
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,MATCHUP_SUPPORTED,6.5,8.0,1.5,Jacksonville State Gamecocks,1.5,MODERATE,7,CONFIRMS_MODEL
1,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,MODEL_PLUS_MATCHUP,-6.5,-7.5,-1.0,Northwestern Wildcats,1.0,MODERATE,7,CONFIRMS_MODEL
2,Buffalo Bulls at Toledo Rockets,Toledo Rockets,MATCHUP_SUPPORTED,-10.0,-10.0,0.0,NONE,0.0,NO_MOVE,,NEUTRAL
3,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,MATCHUP_SUPPORTED,-13.5,-16.0,-2.5,North Texas Mean Green,2.5,MAJOR,14,CONFIRMS_MODEL
4,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,MATCHUP_SUPPORTED,-2.5,-4.0,-1.5,Georgia Tech Yellow Jackets,1.5,MODERATE,3,CONFLICTS_WITH_MODEL
5,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,MODEL_PLUS_MATCHUP,-20.5,-24.0,-3.5,Notre Dame Fighting Irish,3.5,MAJOR,,CONFIRMS_MODEL
6,Virginia Tech Hokies at California Golden Bears,Virginia Tech Hokies,MATCHUP_SUPPORTED,-1.0,1.5,2.5,Virginia Tech Hokies,2.5,MAJOR,,CONFIRMS_MODEL
7,Rice Owls at East Carolina Pirates,East Carolina Pirates,MATCHUP_SUPPORTED,-6.5,-7.5,-1.0,East Carolina Pirates,1.0,MODERATE,7,CONFIRMS_MODEL
8,Maryland Terrapins at Ohio State Buckeyes,Ohio State Buckeyes,STRONG_MODEL_AGREEMENT,-21.5,-20.5,1.0,Maryland Terrapins,1.0,MODERATE,,CONFLICTS_WITH_MODEL
9,Tennessee Volunteers at Arkansas Razorbacks,Tennessee Volunteers,STRONG_MODEL_AGREEMENT,3.5,2.5,-1.0,Arkansas Razorbacks,1.0,MODERATE,3,CONFLICTS_WITH_MODEL


In [ ]:
# ==========================================
# OLD CELL 44 — FINAL DECISION BOARD
# ==========================================

print("=" * 110)
print("CFB1 FINAL DECISION BOARD")
print("=" * 110)


# ==========================================
# 1. COMBINE PHASE 1 + ROUTING + MARKET
# ==========================================

df_final_decision = (
    df_phase1_review[
        [
            "Game",
            "Model_Side",
            "Review_Class",
            "Phase1_Reason"
        ]
    ]
    .merge(
        df_final_routes[
            [
                "Game",
                "M3_Support",
                "M3_Conflict",
                "Favorite_Spread_Size",
                "Spread_Status",
                "Eligible_Market_Routes"
            ]
        ],
        on="Game",
        how="left",
        validate="one_to_one"
    )
    .merge(
        covers_audit[
            [
                "Game",
                "Open_Home_Spread",
                "Current_Home_Spread",
                "Movement_Toward",
                "Movement_Size",
                "Movement_Class",
                "Key_Numbers_Crossed",
                "Key_Cross",
                "Market_Relationship"
            ]
        ],
        on="Game",
        how="left",
        validate="one_to_one"
    )
)


# ==========================================
# 2. FINAL DECISION STATUS
# ==========================================

def final_decision_status(row):

    relationship = row[
        "Market_Relationship"
    ]

    review_class = row[
        "Review_Class"
    ]

    spread_status = row[
        "Spread_Status"
    ]

    movement_class = row[
        "Movement_Class"
    ]

    key_cross = row[
        "Key_Cross"
    ]


    # --------------------------------------
    # ORIGINAL MODULE 3 CONFLICT
    # --------------------------------------

    if review_class == "CONFLICT_REVIEW":

        if relationship == "CONFIRMS_MODEL":

            return (
                "CONFLICT_REVIEW_MARKET_SUPPORTS_MODEL"
            )

        elif relationship == "CONFLICTS_WITH_MODEL":

            return (
                "CONFLICT_REVIEW_MARKET_REJECTS_MODEL"
            )

        else:

            return (
                "CONFLICT_REVIEW_UNRESOLVED"
            )


    # --------------------------------------
    # MARKET CONFLICT
    # --------------------------------------

    if relationship == "CONFLICTS_WITH_MODEL":

        if (
            movement_class == "MAJOR"
            or key_cross
        ):

            return "STRONG_MARKET_CONFLICT"

        return "MARKET_CONFLICT"


    # --------------------------------------
    # NEUTRAL MARKET
    # --------------------------------------

    if relationship == "NEUTRAL":

        return "NEUTRAL_MARKET"


    # --------------------------------------
    # MARKET CONFIRMS MODEL
    # --------------------------------------

    if relationship == "CONFIRMS_MODEL":

        if spread_status in [
            "LARGE_SPREAD_CAUTION",
            "DERIVATIVES_FIRST_20.5_PLUS"
        ]:

            return (
                "CONFIRMED_BUT_LARGE_SPREAD"
            )

        return "CONFIRMED"


    return "UNCLASSIFIED"


df_final_decision[
    "Decision_Status"
] = df_final_decision.apply(
    final_decision_status,
    axis=1
)


# ==========================================
# 3. HUMAN-READABLE DECISION NOTE
# ==========================================

def decision_note(row):

    status = row[
        "Decision_Status"
    ]

    notes = {

        "CONFIRMED":
            "Market movement supports the model side.",

        "CONFIRMED_BUT_LARGE_SPREAD":
            "Market supports the model side, but spread size requires derivative review.",

        "NEUTRAL_MARKET":
            "Market movement provides no additional confirmation or conflict.",

        "MARKET_CONFLICT":
            "Market movement is against the model side.",

        "STRONG_MARKET_CONFLICT":
            "Major and/or key-number movement is against the model side.",

        "CONFLICT_REVIEW_MARKET_SUPPORTS_MODEL":
            "Module 3 conflicted with the model, but late market movement supports the model side.",

        "CONFLICT_REVIEW_MARKET_REJECTS_MODEL":
            "Module 3 conflicted with the model and the late market also moved against the model side.",

        "CONFLICT_REVIEW_UNRESOLVED":
            "Module 3 conflicts with the model and the market did not resolve the disagreement."
    }

    return notes.get(
        status,
        "Requires manual review."
    )


df_final_decision[
    "Decision_Note"
] = df_final_decision.apply(
    decision_note,
    axis=1
)


# ==========================================
# 4. VALIDATION
# ==========================================

expected_games = len(
    df_phase1_review
)

duplicate_games = int(
    df_final_decision.duplicated(
        subset=["Game"],
        keep=False
    ).sum()
)

missing_required = int(
    df_final_decision[
        [
            "Game",
            "Model_Side",
            "Review_Class",
            "Phase1_Reason",
            "Favorite_Spread_Size",
            "Spread_Status",
            "Open_Home_Spread",
            "Current_Home_Spread",
            "Movement_Toward",
            "Movement_Size",
            "Movement_Class",
            "Market_Relationship",
            "Decision_Status",
            "Decision_Note"
        ]
    ].isna().any(axis=1).sum()
)

unclassified_games = int(
    (
        df_final_decision[
            "Decision_Status"
        ] == "UNCLASSIFIED"
    ).sum()
)


# ==========================================
# 5. REPORT
# ==========================================

print(
    f"\nPhase 1 games expected:         "
    f"{expected_games}"
)

print(
    f"Final decision rows:            "
    f"{len(df_final_decision)}"
)

print(
    f"Duplicate game rows:            "
    f"{duplicate_games}"
)

print(
    f"Missing required values:        "
    f"{missing_required}"
)

print(
    f"Unclassified games:             "
    f"{unclassified_games}"
)


# ==========================================
# 6. FINAL STATUS
# ==========================================

print("\n" + "-" * 110)

if (
    len(df_final_decision) == expected_games
    and duplicate_games == 0
    and missing_required == 0
    and unclassified_games == 0
):

    print(
        "CFB1 FINAL DECISION BOARD: PASS"
    )

else:

    print(
        "CFB1 FINAL DECISION BOARD: STOP"
    )

    raise ValueError(
        "Final Decision Board validation failed."
    )

print("=" * 110)


# ==========================================
# 7. DECISION STATUS COUNTS
# ==========================================

print("\nDECISION STATUS COUNTS")
print("-" * 50)

print(
    df_final_decision[
        "Decision_Status"
    ].value_counts()
)


# ==========================================
# 8. FINAL LATE-WEEK REVIEW BOARD
# ==========================================

display(
    df_final_decision[
        [
            "Game",
            "Model_Side",
            "Review_Class",
            "M3_Support",
            "M3_Conflict",
            "Favorite_Spread_Size",
            "Spread_Status",
            "Open_Home_Spread",
            "Current_Home_Spread",
            "Movement_Toward",
            "Movement_Class",
            "Key_Numbers_Crossed",
            "Market_Relationship",
            "Decision_Status",
            "Eligible_Market_Routes",
            "Decision_Note"
        ]
    ]
)

CFB1 FINAL DECISION BOARD

Phase 1 games expected:         14
Final decision rows:            14
Duplicate game rows:            0
Missing required values:        0
Unclassified games:             0

--------------------------------------------------------------------------------------------------------------
CFB1 FINAL DECISION BOARD: PASS

DECISION STATUS COUNTS
--------------------------------------------------
Decision_Status
CONFIRMED                     5
CONFIRMED_BUT_LARGE_SPREAD    3
STRONG_MARKET_CONFLICT        3
NEUTRAL_MARKET                2
MARKET_CONFLICT               1
Name: count, dtype: int64


,Game,Model_Side,Review_Class,M3_Support,M3_Conflict,Favorite_Spread_Size,Spread_Status,Open_Home_Spread,Current_Home_Spread,Movement_Toward,Movement_Class,Key_Numbers_Crossed,Market_Relationship,Decision_Status,Eligible_Market_Routes,Decision_Note
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,MATCHUP_SUPPORTED,Jacksonville State Gamecocks +OAY_DAY,,2.75,STANDARD_REVIEW,6.5,8.0,Jacksonville State Gamecocks,MODERATE,7,CONFIRMS_MODEL,CONFIRMED,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",Market movement supports the model side.
1,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,MODEL_PLUS_MATCHUP,Northwestern Wildcats +OAY_DAY,,36.50,DERIVATIVES_FIRST_20.5_PLUS,-6.5,-7.5,Northwestern Wildcats,MODERATE,7,CONFIRMS_MODEL,CONFIRMED_BUT_LARGE_SPREAD,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...","Market supports the model side, but spread siz..."
2,Buffalo Bulls at Toledo Rockets,Toledo Rockets,MATCHUP_SUPPORTED,Toledo Rockets +OAY_DAY,,20.75,DERIVATIVES_FIRST_20.5_PLUS,-10.0,-10.0,NONE,NO_MOVE,,NEUTRAL,NEUTRAL_MARKET,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",Market movement provides no additional confirm...
3,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,MATCHUP_SUPPORTED,North Texas Mean Green +OAY_DAY,,28.50,DERIVATIVES_FIRST_20.5_PLUS,-13.5,-16.0,North Texas Mean Green,MAJOR,14,CONFIRMS_MODEL,CONFIRMED_BUT_LARGE_SPREAD,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...","Market supports the model side, but spread siz..."
4,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,MATCHUP_SUPPORTED,Duke Blue Devils +OAY_DAY,,6.50,STANDARD_REVIEW,-2.5,-4.0,Georgia Tech Yellow Jackets,MODERATE,3,CONFLICTS_WITH_MODEL,STRONG_MARKET_CONFLICT,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",Major and/or key-number movement is against th...
5,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,MODEL_PLUS_MATCHUP,Stanford Cardinal -HAVOC_TRAP,,38.00,DERIVATIVES_FIRST_20.5_PLUS,-20.5,-24.0,Notre Dame Fighting Irish,MAJOR,,CONFIRMS_MODEL,CONFIRMED_BUT_LARGE_SPREAD,"FULL_GAME_SPREAD, OPP_TEAM_TOTAL_UNDER, OPP_1H...","Market supports the model side, but spread siz..."
6,Virginia Tech Hokies at California Golden Bears,Virginia Tech Hokies,MATCHUP_SUPPORTED,California Golden Bears -HAVOC_TRAP,,9.75,STANDARD_REVIEW,-1.0,1.5,Virginia Tech Hokies,MAJOR,,CONFIRMS_MODEL,CONFIRMED,"FULL_GAME_SPREAD, OPP_TEAM_TOTAL_UNDER, OPP_1H...",Market movement supports the model side.
7,Rice Owls at East Carolina Pirates,East Carolina Pirates,MATCHUP_SUPPORTED,Rice Owls -HAVOC_TRAP,,9.75,STANDARD_REVIEW,-6.5,-7.5,East Carolina Pirates,MODERATE,7,CONFIRMS_MODEL,CONFIRMED,"FULL_GAME_SPREAD, OPP_TEAM_TOTAL_UNDER, OPP_1H...",Market movement supports the model side.
8,Maryland Terrapins at Ohio State Buckeyes,Ohio State Buckeyes,STRONG_MODEL_AGREEMENT,,,34.50,DERIVATIVES_FIRST_20.5_PLUS,-21.5,-20.5,Maryland Terrapins,MODERATE,,CONFLICTS_WITH_MODEL,MARKET_CONFLICT,"1H_SPREAD_REVIEW, TEAM_TOTAL_REVIEW",Market movement is against the model side.
9,Tennessee Volunteers at Arkansas Razorbacks,Tennessee Volunteers,STRONG_MODEL_AGREEMENT,,,13.50,STANDARD_REVIEW,3.5,2.5,Arkansas Razorbacks,MODERATE,3,CONFLICTS_WITH_MODEL,STRONG_MARKET_CONFLICT,,Major and/or key-number movement is against th...


In [ ]:
# ==========================================
# OLD CELL 45 — GEM-READY FINAL EXPORT
# ==========================================

print("=" * 105)
print("CFB1 GEM-READY FINAL EXPORT")
print("=" * 105)


# ==========================================
# 1. BUILD COMPACT EXPORT
# ==========================================

df_gem_export = df_final_decision[
    [
        "Game",
        "Model_Side",
        "Review_Class",
        "Phase1_Reason",
        "M3_Support",
        "M3_Conflict",
        "Favorite_Spread_Size",
        "Spread_Status",
        "Open_Home_Spread",
        "Current_Home_Spread",
        "Movement_Toward",
        "Movement_Class",
        "Key_Numbers_Crossed",
        "Market_Relationship",
        "Decision_Status",
        "Eligible_Market_Routes",
        "Decision_Note"
    ]
].copy()


# ==========================================
# 2. CLEAN TEXT FIELDS
# ==========================================

for col in df_gem_export.select_dtypes(
    include="object"
).columns:

    df_gem_export[col] = (
        df_gem_export[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )


# ==========================================
# 3. DISPLAY ORDER
#
# This is NOT a betting rank.
# It only groups similar decision statuses.
# ==========================================

STATUS_ORDER = {
    "CONFIRMED": 1,
    "CONFIRMED_BUT_LARGE_SPREAD": 2,
    "NEUTRAL_MARKET": 3,
    "CONFLICT_REVIEW_MARKET_SUPPORTS_MODEL": 4,
    "CONFLICT_REVIEW_UNRESOLVED": 5,
    "MARKET_CONFLICT": 6,
    "STRONG_MARKET_CONFLICT": 7,
    "CONFLICT_REVIEW_MARKET_REJECTS_MODEL": 8,
    "UNCLASSIFIED": 9
}

df_gem_export["Display_Order"] = (
    df_gem_export[
        "Decision_Status"
    ]
    .map(STATUS_ORDER)
    .fillna(99)
)

df_gem_export = (
    df_gem_export
    .sort_values(
        [
            "Display_Order",
            "Game"
        ]
    )
    .drop(
        columns=[
            "Display_Order"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ==========================================
# 4. VALIDATION
# ==========================================

expected_games = len(
    df_final_decision
)

duplicate_games = int(
    df_gem_export[
        "Game"
    ].duplicated().sum()
)

missing_status = int(
    (
        df_gem_export[
            "Decision_Status"
        ] == ""
    ).sum()
)

missing_model_side = int(
    (
        df_gem_export[
            "Model_Side"
        ] == ""
    ).sum()
)


print(
    f"\nExpected games:              "
    f"{expected_games}"
)

print(
    f"Exported games:              "
    f"{len(df_gem_export)}"
)

print(
    f"Duplicate games:             "
    f"{duplicate_games}"
)

print(
    f"Missing decision statuses:   "
    f"{missing_status}"
)

print(
    f"Missing model sides:         "
    f"{missing_model_side}"
)


# ==========================================
# 5. FINAL STATUS
# ==========================================

print("\n" + "-" * 105)

if (
    len(df_gem_export) == expected_games
    and duplicate_games == 0
    and missing_status == 0
    and missing_model_side == 0
):

    print(
        "CFB1 GEM EXPORT: PASS"
    )

else:

    print(
        "CFB1 GEM EXPORT: STOP"
    )

    raise ValueError(
        "Gem export validation failed."
    )

print("=" * 105)


# ==========================================
# 6. DISPLAY COMPACT BOARD
# ==========================================

display(
    df_gem_export
)


# ==========================================
# 7. COPY / PASTE CSV
# ==========================================

gem_csv = df_gem_export.to_csv(
    index=False
)

print(
    "\n"
    + "=" * 105
)

print(
    "COPY EVERYTHING BETWEEN THE MARKERS "
    "INTO THE CFB1 GEM"
)

print(
    "=" * 105
)

print(
    "----- BEGIN CFB1 FINAL BOARD CSV -----"
)

print(
    gem_csv
)

print(
    "----- END CFB1 FINAL BOARD CSV -----"
)

CFB1 GEM-READY FINAL EXPORT

Expected games:              14
Exported games:              14
Duplicate games:             0
Missing decision statuses:   0
Missing model sides:         0

---------------------------------------------------------------------------------------------------------
CFB1 GEM EXPORT: PASS


,Game,Model_Side,Review_Class,Phase1_Reason,M3_Support,M3_Conflict,Favorite_Spread_Size,Spread_Status,Open_Home_Spread,Current_Home_Spread,Movement_Toward,Movement_Class,Key_Numbers_Crossed,Market_Relationship,Decision_Status,Eligible_Market_Routes,Decision_Note
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,MATCHUP_SUPPORTED,MODULE3_SUPPORT,Jacksonville State Gamecocks +OAY_DAY,,2.75,STANDARD_REVIEW,6.5,8.0,Jacksonville State Gamecocks,MODERATE,7,CONFIRMS_MODEL,CONFIRMED,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",Market movement supports the model side.
1,James Madison Dukes at Georgia Southern Eagles,James Madison Dukes,STRONG_MODEL_AGREEMENT,DUAL_TOP25_MODEL_STRENGTH,,,7.50,STANDARD_REVIEW,2.5,4.0,James Madison Dukes,MODERATE,3,CONFIRMS_MODEL,CONFIRMED,,Market movement supports the model side.
2,Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,MODEL_PLUS_MATCHUP,"MODULE3_SUPPORT, DUAL_TOP25_MODEL_STRENGTH",Nevada Wolf Pack +OAY_DAY,,9.00,STANDARD_REVIEW,-2.5,-1.5,Nevada Wolf Pack,MODERATE,,CONFIRMS_MODEL,CONFIRMED,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",Market movement supports the model side.
3,Rice Owls at East Carolina Pirates,East Carolina Pirates,MATCHUP_SUPPORTED,MODULE3_SUPPORT,Rice Owls -HAVOC_TRAP,,9.75,STANDARD_REVIEW,-6.5,-7.5,East Carolina Pirates,MODERATE,7,CONFIRMS_MODEL,CONFIRMED,"FULL_GAME_SPREAD, OPP_TEAM_TOTAL_UNDER, OPP_1H...",Market movement supports the model side.
4,Virginia Tech Hokies at California Golden Bears,Virginia Tech Hokies,MATCHUP_SUPPORTED,MODULE3_SUPPORT,California Golden Bears -HAVOC_TRAP,,9.75,STANDARD_REVIEW,-1.0,1.5,Virginia Tech Hokies,MAJOR,,CONFIRMS_MODEL,CONFIRMED,"FULL_GAME_SPREAD, OPP_TEAM_TOTAL_UNDER, OPP_1H...",Market movement supports the model side.
5,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,MODEL_PLUS_MATCHUP,"MODULE3_SUPPORT, DUAL_TOP25_MODEL_STRENGTH",Northwestern Wildcats +OAY_DAY,,36.50,DERIVATIVES_FIRST_20.5_PLUS,-6.5,-7.5,Northwestern Wildcats,MODERATE,7,CONFIRMS_MODEL,CONFIRMED_BUT_LARGE_SPREAD,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...","Market supports the model side, but spread siz..."
6,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,MATCHUP_SUPPORTED,MODULE3_SUPPORT,North Texas Mean Green +OAY_DAY,,28.50,DERIVATIVES_FIRST_20.5_PLUS,-13.5,-16.0,North Texas Mean Green,MAJOR,14,CONFIRMS_MODEL,CONFIRMED_BUT_LARGE_SPREAD,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...","Market supports the model side, but spread siz..."
7,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,MODEL_PLUS_MATCHUP,"MODULE3_SUPPORT, DUAL_TOP25_MODEL_STRENGTH",Stanford Cardinal -HAVOC_TRAP,,38.00,DERIVATIVES_FIRST_20.5_PLUS,-20.5,-24.0,Notre Dame Fighting Irish,MAJOR,,CONFIRMS_MODEL,CONFIRMED_BUT_LARGE_SPREAD,"FULL_GAME_SPREAD, OPP_TEAM_TOTAL_UNDER, OPP_1H...","Market supports the model side, but spread siz..."
8,Buffalo Bulls at Toledo Rockets,Toledo Rockets,MATCHUP_SUPPORTED,MODULE3_SUPPORT,Toledo Rockets +OAY_DAY,,20.75,DERIVATIVES_FIRST_20.5_PLUS,-10.0,-10.0,NONE,NO_MOVE,,NEUTRAL,NEUTRAL_MARKET,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",Market movement provides no additional confirm...
9,UAB Blazers at Memphis Tigers,Memphis Tigers,MATCHUP_SUPPORTED,MODULE3_SUPPORT,Memphis Tigers +LINE_YARDS,,14.25,STANDARD_REVIEW,-14.5,-14.5,NONE,NO_MOVE,,NEUTRAL,NEUTRAL_MARKET,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",Market movement provides no additional confirm...



COPY EVERYTHING BETWEEN THE MARKERS INTO THE CFB1 GEM
----- BEGIN CFB1 FINAL BOARD CSV -----
Game,Model_Side,Review_Class,Phase1_Reason,M3_Support,M3_Conflict,Favorite_Spread_Size,Spread_Status,Open_Home_Spread,Current_Home_Spread,Movement_Toward,Movement_Class,Key_Numbers_Crossed,Market_Relationship,Decision_Status,Eligible_Market_Routes,Decision_Note
Jacksonville State Gamecocks at Kennesaw State Owls,Jacksonville State Gamecocks,MATCHUP_SUPPORTED,MODULE3_SUPPORT,Jacksonville State Gamecocks +OAY_DAY,,2.75,STANDARD_REVIEW,6.5,8.0,Jacksonville State Gamecocks,MODERATE,7,CONFIRMS_MODEL,CONFIRMED,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOTAL_OVER, 1H_SPREAD_REVIEW",Market movement supports the model side.
James Madison Dukes at Georgia Southern Eagles,James Madison Dukes,STRONG_MODEL_AGREEMENT,DUAL_TOP25_MODEL_STRENGTH,,,7.5,STANDARD_REVIEW,2.5,4.0,James Madison Dukes,MODERATE,3,CONFIRMS_MODEL,CONFIRMED,,Market movement supports the model side.
Nevada Wolf Pack at UTEP Miners,Nevad

In [ ]:
# ==========================================
# CELL 46 — PRICING INPUT + VALIDATION
# RUN-ALL SAFE
# ==========================================

from io import StringIO
import pandas as pd
import numpy as np

print("=" * 100)
print("CFB1 CURRENT PRICING INPUT")
print("=" * 100)


# --------------------------------------------------
# 1. REQUIRED PRICING FORMAT
# --------------------------------------------------
#
# One row = one actual sportsbook wager option.
#
# REQUIRED COLUMNS:
#
# Game
# Market_Type
# Team
# Bet_Direction
# Line
# American_Odds
# Sportsbook
#
#
# ACCEPTED MARKET TYPES:
#
# FULL_GAME_SPREAD
# TEAM_TOTAL
# 1H_TEAM_TOTAL
# 1H_SPREAD
# 1H_TOTAL
#
#
# BET DIRECTION:
#
# SPREAD
# OVER
# UNDER
#
#
# EXAMPLES ONLY:
#
# FULL_GAME_SPREAD:
# Miami Hurricanes at Clemson Tigers,
# FULL_GAME_SPREAD,
# Miami Hurricanes,
# SPREAD,
# -16.5,
# -110,
# FanDuel
#
# TEAM TOTAL:
# Purdue Boilermakers at Illinois Fighting Illini,
# TEAM_TOTAL,
# Illinois Fighting Illini,
# OVER,
# 34.5,
# -110,
# FanDuel
#
# --------------------------------------------------


# --------------------------------------------------
# 2. PASTE CURRENT PRICING CSV HERE
#
# Leave only the header until current prices
# have been collected.
# --------------------------------------------------

pricing_csv = """
Game,Market_Type,Team,Bet_Direction,Line,American_Odds,Sportsbook
"""


# --------------------------------------------------
# 3. LOAD INPUT
# --------------------------------------------------

df_pricing = pd.read_csv(
    StringIO(pricing_csv.strip()),
    skipinitialspace=True
)


# Clean text columns
for col in df_pricing.select_dtypes(include='object').columns:
    df_pricing[col] = (
        df_pricing[col]
        .fillna('')
        .astype(str)
        .str.strip()
    )


# --------------------------------------------------
# 4. EMPTY INPUT = SAFE STOP POINT
# --------------------------------------------------

if len(df_pricing) == 0:

    print("\nPRICING STATUS: AWAITING PRICING INPUT")

    print("""
Paste current sportsbook markets into pricing_csv
using this exact column order:

Game,Market_Type,Team,Bet_Direction,Line,American_Odds,Sportsbook
""")

    print("Accepted Market_Type values:")
    print("  FULL_GAME_SPREAD")
    print("  TEAM_TOTAL")
    print("  1H_TEAM_TOTAL")
    print("  1H_SPREAD")
    print("  1H_TOTAL")

    print("\nAccepted Bet_Direction values:")
    print("  SPREAD")
    print("  OVER")
    print("  UNDER")

else:

    # --------------------------------------------------
    # 5. REQUIRED COLUMN VALIDATION
    # --------------------------------------------------

    REQUIRED_COLUMNS = [
        'Game',
        'Market_Type',
        'Team',
        'Bet_Direction',
        'Line',
        'American_Odds',
        'Sportsbook'
    ]

    missing_columns = [
        col for col in REQUIRED_COLUMNS
        if col not in df_pricing.columns
    ]

    if missing_columns:

        print("\nPRICING STATUS: STOP")
        print("Missing columns:", missing_columns)

        raise ValueError(
            "Pricing CSV is missing required columns."
        )


    # --------------------------------------------------
    # 6. STANDARDIZE TEXT
    # --------------------------------------------------

    df_pricing['Market_Type'] = (
        df_pricing['Market_Type']
        .str.upper()
        .str.strip()
    )

    df_pricing['Bet_Direction'] = (
        df_pricing['Bet_Direction']
        .str.upper()
        .str.strip()
    )


    # --------------------------------------------------
    # 7. NUMERIC CONVERSION
    # --------------------------------------------------

    df_pricing['Line'] = pd.to_numeric(
        df_pricing['Line'],
        errors='coerce'
    )

    df_pricing['American_Odds'] = pd.to_numeric(
        df_pricing['American_Odds'],
        errors='coerce'
    )


    # --------------------------------------------------
    # 8. BASIC VALIDATION
    # --------------------------------------------------

    VALID_MARKETS = {
        'FULL_GAME_SPREAD',
        'TEAM_TOTAL',
        '1H_TEAM_TOTAL',
        '1H_SPREAD',
        '1H_TOTAL'
    }

    VALID_DIRECTIONS = {
        'SPREAD',
        'OVER',
        'UNDER'
    }


    invalid_market = ~df_pricing[
        'Market_Type'
    ].isin(VALID_MARKETS)

    invalid_direction = ~df_pricing[
        'Bet_Direction'
    ].isin(VALID_DIRECTIONS)

    missing_line = (
        df_pricing['Line']
        .isna()
    )

    missing_odds = (
        df_pricing['American_Odds']
        .isna()
    )

    missing_sportsbook = (
        df_pricing['Sportsbook']
        == ''
    )

    missing_game = (
        df_pricing['Game']
        == ''
    )


    # --------------------------------------------------
    # 9. AMERICAN ODDS VALIDATION
    #
    # Standard American odds should normally be
    # <= -100 or >= +100.
    # --------------------------------------------------

    invalid_american_odds = (
        (df_pricing['American_Odds'] > -100)
        &
        (df_pricing['American_Odds'] < 100)
    )


    # --------------------------------------------------
    # 10. GAME VALIDATION
    # --------------------------------------------------

    if 'df_final_decision' not in globals():

        raise NameError(
            "df_final_decision does not exist. "
            "Run through Cell 44 first."
        )

    valid_games = set(
        df_final_decision['Game']
    )

    invalid_game = ~df_pricing[
        'Game'
    ].isin(valid_games)


    # --------------------------------------------------
    # 11. TEAM VALIDATION
    #
    # Spread and team-total wagers require a team.
    # 1H total can be a game total and may leave
    # Team blank.
    # --------------------------------------------------

    team_required = df_pricing[
        'Market_Type'
    ].isin(
        [
            'FULL_GAME_SPREAD',
            'TEAM_TOTAL',
            '1H_TEAM_TOTAL',
            '1H_SPREAD'
        ]
    )

    missing_required_team = (
        team_required
        &
        (df_pricing['Team'] == '')
    )


    # --------------------------------------------------
    # 12. MARKET / DIRECTION CONSISTENCY
    # --------------------------------------------------

    spread_market = df_pricing[
        'Market_Type'
    ].isin(
        [
            'FULL_GAME_SPREAD',
            '1H_SPREAD'
        ]
    )

    total_market = df_pricing[
        'Market_Type'
    ].isin(
        [
            'TEAM_TOTAL',
            '1H_TEAM_TOTAL',
            '1H_TOTAL'
        ]
    )

    bad_spread_direction = (
        spread_market
        &
        (df_pricing['Bet_Direction'] != 'SPREAD')
    )

    bad_total_direction = (
        total_market
        &
        ~df_pricing['Bet_Direction'].isin(
            ['OVER', 'UNDER']
        )
    )


    # --------------------------------------------------
    # 13. DUPLICATE CHECK
    # --------------------------------------------------

    duplicate_mask = df_pricing.duplicated(
        subset=[
            'Game',
            'Market_Type',
            'Team',
            'Bet_Direction',
            'Line',
            'American_Odds',
            'Sportsbook'
        ],
        keep=False
    )


    # --------------------------------------------------
    # 14. BUILD ERROR FLAG
    # --------------------------------------------------

    df_pricing['Validation_Error'] = ''

    def add_error(mask, message):

        df_pricing.loc[
            mask,
            'Validation_Error'
        ] += message + '; '


    add_error(
        invalid_market,
        'INVALID_MARKET_TYPE'
    )

    add_error(
        invalid_direction,
        'INVALID_BET_DIRECTION'
    )

    add_error(
        missing_line,
        'MISSING_LINE'
    )

    add_error(
        missing_odds,
        'MISSING_ODDS'
    )

    add_error(
        invalid_american_odds,
        'INVALID_AMERICAN_ODDS'
    )

    add_error(
        missing_sportsbook,
        'MISSING_SPORTSBOOK'
    )

    add_error(
        missing_game,
        'MISSING_GAME'
    )

    add_error(
        invalid_game,
        'GAME_NOT_ON_FINAL_BOARD'
    )

    add_error(
        missing_required_team,
        'TEAM_REQUIRED'
    )

    add_error(
        bad_spread_direction,
        'SPREAD_REQUIRES_SPREAD_DIRECTION'
    )

    add_error(
        bad_total_direction,
        'TOTAL_REQUIRES_OVER_OR_UNDER'
    )

    add_error(
        duplicate_mask,
        'DUPLICATE_MARKET'
    )


    df_pricing['Validation_Error'] = (
        df_pricing['Validation_Error']
        .str.rstrip('; ')
    )


    # --------------------------------------------------
    # 15. VALIDATION SUMMARY
    # --------------------------------------------------

    bad_rows = df_pricing[
        df_pricing['Validation_Error'] != ''
    ].copy()


    print(f"\nPricing rows received:        {len(df_pricing)}")
    print(f"Valid pricing rows:           {len(df_pricing) - len(bad_rows)}")
    print(f"Rows requiring review:        {len(bad_rows)}")


    if len(bad_rows) == 0:

        print("\nPRICING INPUT STATUS: PASS")

    else:

        print("\nPRICING INPUT STATUS: STOP")

        print("\nROWS REQUIRING REVIEW:")

        display(
            bad_rows[
                [
                    'Game',
                    'Market_Type',
                    'Team',
                    'Bet_Direction',
                    'Line',
                    'American_Odds',
                    'Sportsbook',
                    'Validation_Error'
                ]
            ]
        )

        raise ValueError(
            "Pricing input failed validation."
        )


    # --------------------------------------------------
    # 16. CLEAN PRICING TABLE
    # --------------------------------------------------

    df_pricing_valid = (
        df_pricing[
            df_pricing['Validation_Error'] == ''
        ]
        .drop(columns=['Validation_Error'])
        .copy()
        .reset_index(drop=True)
    )


    print("\n" + "=" * 100)
    print("VALIDATED CURRENT PRICING BOARD")
    print("=" * 100)

    display(
        df_pricing_valid
    )


    print(
        "\nPRICING VALIDATION COMPLETE — "
        "NO EV OR STAKE CALCULATIONS PERFORMED."
    )

CFB1 CURRENT PRICING INPUT

PRICING STATUS: AWAITING PRICING INPUT

Paste current sportsbook markets into pricing_csv
using this exact column order:

Game,Market_Type,Team,Bet_Direction,Line,American_Odds,Sportsbook

Accepted Market_Type values:
  FULL_GAME_SPREAD
  TEAM_TOTAL
  1H_TEAM_TOTAL
  1H_SPREAD
  1H_TOTAL

Accepted Bet_Direction values:
  SPREAD
  OVER
  UNDER


In [ ]:
# ==========================================
# CELL 47 — BLENDED MODEL + SHRINKAGE
# DIAGNOSTIC ONLY
# ==========================================

import numpy as np
import pandas as pd

print("=" * 105)
print("CFB1 BLENDED MODEL + SHRINKAGE DIAGNOSTIC")
print("=" * 105)


# --------------------------------------------------
# 1. REQUIRED INPUT CHECK
# --------------------------------------------------

required_cols = [
    'Home_Team',
    'Away_Team',
    'Net_EPA_Edge_Home',
    'FEI_Model_Home_Spread',
    'DK_Spread',
    'FD_Spread'
]

missing_cols = [
    col for col in required_cols
    if col not in df_master_bcf.columns
]

if missing_cols:
    raise ValueError(
        f"Cell 47 missing required columns: {missing_cols}"
    )


# --------------------------------------------------
# 2. CONSENSUS MARKET LINE
#
# Home-team perspective:
# negative = home favored
# positive = home underdog
# --------------------------------------------------

df_prob_model = df_master_bcf.copy()

df_prob_model['Consensus_Home_Spread'] = (
    df_prob_model[
        ['DK_Spread', 'FD_Spread']
    ].mean(axis=1)
)

# Convert sportsbook notation into expected
# HOME scoring margin:
#
# Home -7  -> +7 home margin
# Home +7  -> -7 home margin
#
df_prob_model['Market_Home_Margin'] = (
    -df_prob_model['Consensus_Home_Spread']
)


# --------------------------------------------------
# 3. EPA -> POINT MARGIN CALIBRATION
#
# EPA itself is not measured in scoreboard points.
# For V1, estimate:
#
# Market_Home_Margin =
# intercept + beta * Net_EPA_Edge_Home
#
# The intercept also captures the average home-field
# component embedded in this week's market.
# --------------------------------------------------

epa_x = df_prob_model[
    'Net_EPA_Edge_Home'
].astype(float)

market_y = df_prob_model[
    'Market_Home_Margin'
].astype(float)

valid_mask = (
    epa_x.notna()
    &
    market_y.notna()
)

x = epa_x[valid_mask].values
y = market_y[valid_mask].values

# OLS: y = intercept + beta*x
beta, intercept = np.polyfit(
    x,
    y,
    1
)

print("\nEPA -> POINT CALIBRATION")
print("-" * 45)

print(
    f"Estimated EPA points multiplier: "
    f"{beta:.2f}"
)

print(
    f"Estimated home-field/intercept:  "
    f"{intercept:.2f} points"
)


# --------------------------------------------------
# 4. EPA-IMPLIED MODEL LINE
# --------------------------------------------------

df_prob_model['EPA_Model_Home_Margin'] = (
    intercept
    +
    beta
    * df_prob_model['Net_EPA_Edge_Home']
)

df_prob_model['EPA_Model_Home_Spread'] = (
    -df_prob_model['EPA_Model_Home_Margin']
)


# --------------------------------------------------
# 5. FEI MODEL MARGIN
# --------------------------------------------------

df_prob_model['FEI_Model_Home_Margin'] = (
    -df_prob_model['FEI_Model_Home_Spread']
)


# --------------------------------------------------
# 6. RAW BLENDED MODEL
#
# Start with equal weighting.
#
# This is intentionally easy to audit.
# We can change the weights later after
# historical validation.
# --------------------------------------------------

EPA_WEIGHT = 0.50
FEI_WEIGHT = 0.50

df_prob_model['Blended_Home_Margin_Raw'] = (
    EPA_WEIGHT
    * df_prob_model['EPA_Model_Home_Margin']
    +
    FEI_WEIGHT
    * df_prob_model['FEI_Model_Home_Margin']
)

df_prob_model['Blended_Home_Spread_Raw'] = (
    -df_prob_model['Blended_Home_Margin_Raw']
)


# --------------------------------------------------
# 7. RAW MODEL EDGE VS MARKET
#
# Positive value = raw blended model sees
# value on HOME.
#
# Negative value = raw blended model sees
# value on AWAY.
# --------------------------------------------------

df_prob_model['Blended_Edge_Home_Raw'] = (
    df_prob_model['Consensus_Home_Spread']
    -
    df_prob_model['Blended_Home_Spread_Raw']
)


# --------------------------------------------------
# 8. SHRINKAGE / EDGE RETENTION TEST
#
# IMPORTANT:
#
# "Retention" means how much of the raw
# model-vs-market disagreement we keep.
#
# 25% = very conservative
# 40% = conservative
# 50% = moderate
# 60% = more aggressive
# --------------------------------------------------

RETENTION_LEVELS = [
    0.25,
    0.40,
    0.50,
    0.60
]

for retention in RETENTION_LEVELS:

    label = int(retention * 100)

    df_prob_model[
        f'Edge_Retained_{label}'
    ] = (
        df_prob_model[
            'Blended_Edge_Home_Raw'
        ]
        * retention
    )

    df_prob_model[
        f'Shrunk_Home_Spread_{label}'
    ] = (
        df_prob_model[
            'Consensus_Home_Spread'
        ]
        -
        df_prob_model[
            f'Edge_Retained_{label}'
        ]
    )


# --------------------------------------------------
# 9. ADD PHASE 1 INFORMATION
# --------------------------------------------------

if 'df_phase1_review' in globals():

    df_prob_model = df_prob_model.merge(
        df_phase1_review[
            [
                'Game',
                'Model_Side',
                'Review_Class'
            ]
        ],
        left_on=(
            df_prob_model['Away_Team']
            + ' at '
            + df_prob_model['Home_Team']
        ),
        right_on='Game',
        how='left'
    )

else:

    df_prob_model['Game'] = (
        df_prob_model['Away_Team']
        + ' at '
        + df_prob_model['Home_Team']
    )

    df_prob_model['Model_Side'] = ''
    df_prob_model['Review_Class'] = ''


# --------------------------------------------------
# 10. PHASE 1 DIAGNOSTIC BOARD
# --------------------------------------------------

df_phase1_shrinkage = df_prob_model[
    df_prob_model['Game'].isin(
        set(df_phase1_review['Game'])
    )
].copy()


print("\n" + "=" * 105)
print("PHASE 1 SHRINKAGE COMPARISON")
print("=" * 105)

display(
    df_phase1_shrinkage[
        [
            'Game',
            'Model_Side',
            'Consensus_Home_Spread',
            'EPA_Model_Home_Spread',
            'FEI_Model_Home_Spread',
            'Blended_Home_Spread_Raw',
            'Blended_Edge_Home_Raw',

            'Shrunk_Home_Spread_25',
            'Shrunk_Home_Spread_40',
            'Shrunk_Home_Spread_50',
            'Shrunk_Home_Spread_60'
        ]
    ].round(2)
)


# --------------------------------------------------
# 11. EXTREME EDGE CHECK
# --------------------------------------------------

df_phase1_shrinkage[
    'Raw_Edge_Size'
] = (
    df_phase1_shrinkage[
        'Blended_Edge_Home_Raw'
    ].abs()
)

print("\n" + "=" * 105)
print("LARGEST RAW BLENDED MODEL DISAGREEMENTS")
print("=" * 105)

display(
    df_phase1_shrinkage[
        [
            'Game',
            'Consensus_Home_Spread',
            'Blended_Home_Spread_Raw',
            'Raw_Edge_Size',
            'Shrunk_Home_Spread_25',
            'Shrunk_Home_Spread_40',
            'Shrunk_Home_Spread_50',
            'Shrunk_Home_Spread_60'
        ]
    ]
    .sort_values(
        'Raw_Edge_Size',
        ascending=False
    )
    .round(2)
)


print(
    "\nDIAGNOSTIC COMPLETE — "
    "NO RETENTION FACTOR LOCKED YET."
)

print(
    "NO COVER PROBABILITY, EV, OR KELLY "
    "CALCULATIONS PERFORMED."
)

CFB1 BLENDED MODEL + SHRINKAGE DIAGNOSTIC

EPA -> POINT CALIBRATION
---------------------------------------------
Estimated EPA points multiplier: 38.04
Estimated home-field/intercept:  4.75 points

PHASE 1 SHRINKAGE COMPARISON


,Game,Model_Side,Consensus_Home_Spread,EPA_Model_Home_Spread,FEI_Model_Home_Spread,Blended_Home_Spread_Raw,Blended_Edge_Home_Raw,Shrunk_Home_Spread_25,Shrunk_Home_Spread_40,Shrunk_Home_Spread_50,Shrunk_Home_Spread_60
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,2.75,3.60,5.4,4.50,-1.75,3.19,3.45,3.62,3.80
19,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,-36.50,-22.52,-48.6,-35.56,-0.94,-36.27,-36.12,-36.03,-35.94
23,Buffalo Bulls at Toledo Rockets,Toledo Rockets,-20.75,-18.56,-24.0,-21.28,0.53,-20.88,-20.96,-21.01,-21.07
25,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,-28.50,-21.56,-30.6,-26.08,-2.42,-27.89,-27.53,-27.29,-27.05
26,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,6.50,9.44,11.7,10.57,-4.07,7.52,8.13,8.53,8.94
32,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,-38.00,-29.24,-60.0,-44.62,6.62,-39.65,-40.65,-41.31,-41.97
36,Virginia Tech Hokies at California Golden Bears,Virginia Tech Hokies,9.75,5.69,14.1,9.89,-0.14,9.79,9.81,9.82,9.84
38,Rice Owls at East Carolina Pirates,East Carolina Pirates,-9.75,-10.80,-17.7,-14.25,4.50,-10.87,-11.55,-12.00,-12.45
39,Maryland Terrapins at Ohio State Buckeyes,Ohio State Buckeyes,-34.50,-21.21,-46.2,-33.70,-0.80,-34.30,-34.18,-34.10,-34.02
40,Tennessee Volunteers at Arkansas Razorbacks,Tennessee Volunteers,13.50,12.08,27.3,19.69,-6.19,15.05,15.98,16.60,17.21



LARGEST RAW BLENDED MODEL DISAGREEMENTS


,Game,Consensus_Home_Spread,Blended_Home_Spread_Raw,Raw_Edge_Size,Shrunk_Home_Spread_25,Shrunk_Home_Spread_40,Shrunk_Home_Spread_50,Shrunk_Home_Spread_60
49,James Madison Dukes at Georgia Southern Eagles,7.50,17.06,9.56,9.89,11.32,12.28,13.23
47,Air Force Falcons at Northern Illinois Huskies,7.50,15.68,8.18,9.55,10.77,11.59,12.41
44,Nevada Wolf Pack at UTEP Miners,9.00,16.12,7.12,10.78,11.85,12.56,13.27
32,Stanford Cardinal at Notre Dame Fighting Irish,-38.00,-44.62,6.62,-39.65,-40.65,-41.31,-41.97
40,Tennessee Volunteers at Arkansas Razorbacks,13.50,19.69,6.19,15.05,15.98,16.60,17.21
38,Rice Owls at East Carolina Pirates,-9.75,-14.25,4.50,-10.87,-11.55,-12.00,-12.45
26,Duke Blue Devils at Georgia Tech Yellow Jackets,6.50,10.57,4.07,7.52,8.13,8.53,8.94
25,Charlotte 49ers at North Texas Mean Green,-28.50,-26.08,2.42,-27.89,-27.53,-27.29,-27.05
0,Jacksonville State Gamecocks at Kennesaw State...,2.75,4.50,1.75,3.19,3.45,3.62,3.80
46,UAB Blazers at Memphis Tigers,-14.25,-15.24,0.99,-14.50,-14.65,-14.75,-14.85



DIAGNOSTIC COMPLETE — NO RETENTION FACTOR LOCKED YET.
NO COVER PROBABILITY, EV, OR KELLY CALCULATIONS PERFORMED.


In [ ]:
# ==========================================
# CELL 48 — KEY-NUMBER LINE VALUE DIAGNOSTIC
# ==========================================

import numpy as np
import pandas as pd

print("=" * 105)
print("CFB1 KEY-NUMBER LINE VALUE DIAGNOSTIC")
print("=" * 105)

# --------------------------------------------------
# 1. LOCK V1 RETENTION FACTOR
# --------------------------------------------------

MODEL_EDGE_RETENTION = 0.40

print(
    f"\nCFB1 V1 model-edge retention: "
    f"{MODEL_EDGE_RETENTION:.0%}"
)


# --------------------------------------------------
# 2. REQUIRED INPUT CHECK
# --------------------------------------------------

if 'df_phase1_shrinkage' not in globals():
    raise NameError(
        "df_phase1_shrinkage does not exist. "
        "Run Cell 47 first."
    )

if 'df_final_decision' not in globals():
    raise NameError(
        "df_final_decision does not exist. "
        "Run through Cell 44 first."
    )


# --------------------------------------------------
# 3. MERGE CURRENT MARKET LINE
#
# Covers current line is used here because this
# diagnostic is about late-week executable line value.
# --------------------------------------------------

df_key_value = df_phase1_shrinkage[
    [
        'Game',
        'Model_Side',
        'Home_Team',
        'Away_Team',
        'Shrunk_Home_Spread_40'
    ]
].copy()

df_key_value = df_key_value.merge(
    df_final_decision[
        [
            'Game',
            'Current_Home_Spread',
            'Decision_Status',
            'Eligible_Market_Routes'
        ]
    ],
    on='Game',
    how='left',
    validate='one_to_one'
)


# --------------------------------------------------
# 4. CONVERT EVERYTHING TO MODEL-SIDE SPREAD
#
# Example:
#
# Northwestern home +2.5
# Penn State is model side
#
# Penn State market spread = -2.5
#
# Shrunk home line = +5.2
# Penn State retained model line = -5.2
# --------------------------------------------------

def model_side_market_spread(row):

    if row['Model_Side'] == row['Home_Team']:
        return row['Current_Home_Spread']

    elif row['Model_Side'] == row['Away_Team']:
        return -row['Current_Home_Spread']

    else:
        return np.nan


def model_side_retained_spread(row):

    if row['Model_Side'] == row['Home_Team']:
        return row['Shrunk_Home_Spread_40']

    elif row['Model_Side'] == row['Away_Team']:
        return -row['Shrunk_Home_Spread_40']

    else:
        return np.nan


df_key_value['Market_Model_Spread'] = (
    df_key_value.apply(
        model_side_market_spread,
        axis=1
    )
)

df_key_value['Retained_Model_Spread'] = (
    df_key_value.apply(
        model_side_retained_spread,
        axis=1
    )
)


# --------------------------------------------------
# 5. MODEL CUSHION
#
# Positive = sportsbook gives us a better number
# than the retained model line.
#
# Example:
#
# market  -2.5
# model   -5.2
#
# cushion = 2.7 points
# --------------------------------------------------

df_key_value['Model_Cushion_Points'] = (
    df_key_value['Market_Model_Spread']
    -
    df_key_value['Retained_Model_Spread']
).round(2)


# --------------------------------------------------
# 6. KEY NUMBERS
# --------------------------------------------------

SIGNED_KEY_NUMBERS = [
    -14, -10, -7, -3,
      3,   7, 10, 14
]


def keys_between_market_and_model(row):

    market = row['Market_Model_Spread']
    model = row['Retained_Model_Spread']

    if pd.isna(market) or pd.isna(model):
        return ''

    low = min(market, model)
    high = max(market, model)

    keys = []

    for key in SIGNED_KEY_NUMBERS:

        if low < key < high:
            keys.append(abs(key))

    return ','.join(
        str(k)
        for k in sorted(set(keys))
    )


df_key_value['Keys_Between_Market_And_Model'] = (
    df_key_value.apply(
        keys_between_market_and_model,
        axis=1
    )
)

df_key_value['Key_Barrier_Present'] = (
    df_key_value[
        'Keys_Between_Market_And_Model'
    ] != ''
)


# --------------------------------------------------
# 7. NEXT ADVERSE KEY NUMBER
#
# For a favorite, identify the next key number
# that would be encountered by laying more points.
#
# Example:
#
# Penn State -2.5 -> next adverse key = 3
# --------------------------------------------------

KEY_MAGNITUDES = [3, 7, 10, 14]


def next_adverse_key(row):

    spread = row['Market_Model_Spread']

    if pd.isna(spread):
        return np.nan

    # Only relevant here when model side is favored
    if spread >= 0:
        return np.nan

    magnitude = abs(spread)

    for key in KEY_MAGNITUDES:

        if key > magnitude:
            return key

    return np.nan


df_key_value['Next_Adverse_Key'] = (
    df_key_value.apply(
        next_adverse_key,
        axis=1
    )
)


# --------------------------------------------------
# 8. LINE VALUE STATUS
# --------------------------------------------------

def line_value_status(row):

    cushion = row['Model_Cushion_Points']

    if pd.isna(cushion):
        return 'CHECK'

    if cushion <= 0:
        return 'NO_RETAINED_MODEL_EDGE'

    if row['Key_Barrier_Present']:
        return 'EDGE_WITH_KEY_BARRIER'

    return 'EDGE_NO_KEY_BARRIER'


df_key_value['Line_Value_Status'] = (
    df_key_value.apply(
        line_value_status,
        axis=1
    )
)


# --------------------------------------------------
# 9. KEY-NUMBER TRADEOFF RULE
# --------------------------------------------------

def tradeoff_note(row):

    status = row['Line_Value_Status']
    keys = row['Keys_Between_Market_And_Model']

    if status == 'NO_RETAINED_MODEL_EDGE':
        return (
            'Retained model does not show a '
            'positive line cushion.'
        )

    if status == 'EDGE_WITH_KEY_BARRIER':
        return (
            f"Model shows line value, but key number(s) "
            f"{keys} lie between market and retained "
            f"model line. Do not trade through a key "
            f"solely for better juice without explicit "
            f"price-value analysis."
        )

    if status == 'EDGE_NO_KEY_BARRIER':
        return (
            'Retained model shows line cushion with no '
            'listed key number between market and model.'
        )

    return 'Requires review.'


df_key_value['Key_Tradeoff_Note'] = (
    df_key_value.apply(
        tradeoff_note,
        axis=1
    )
)


# --------------------------------------------------
# 10. DISPLAY
# --------------------------------------------------

print("\n" + "=" * 105)
print("PHASE 1 KEY-NUMBER VALUE BOARD")
print("=" * 105)

display(
    df_key_value[
        [
            'Game',
            'Model_Side',
            'Market_Model_Spread',
            'Retained_Model_Spread',
            'Model_Cushion_Points',
            'Next_Adverse_Key',
            'Keys_Between_Market_And_Model',
            'Key_Barrier_Present',
            'Line_Value_Status',
            'Decision_Status',
            'Key_Tradeoff_Note'
        ]
    ].round(2)
)


# --------------------------------------------------
# 11. SUMMARY
# --------------------------------------------------

print("\nLINE VALUE STATUS COUNTS")
print("-" * 45)

print(
    df_key_value[
        'Line_Value_Status'
    ].value_counts()
)


print("\n" + "=" * 105)

print(
    "KEY-NUMBER DIAGNOSTIC COMPLETE — "
    "NO BET SELECTION OR EV CALCULATION PERFORMED."
)

CFB1 KEY-NUMBER LINE VALUE DIAGNOSTIC

CFB1 V1 model-edge retention: 40%

PHASE 1 KEY-NUMBER VALUE BOARD


,Game,Model_Side,Market_Model_Spread,Retained_Model_Spread,Model_Cushion_Points,Next_Adverse_Key,Keys_Between_Market_And_Model,Key_Barrier_Present,Line_Value_Status,Decision_Status,Key_Tradeoff_Note
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,-8.0,-3.45,-4.55,10.0,7,True,NO_RETAINED_MODEL_EDGE,CONFIRMED,Retained model does not show a positive line c...
1,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,-7.5,-36.12,28.62,10.0,"10,14",True,EDGE_WITH_KEY_BARRIER,CONFIRMED_BUT_LARGE_SPREAD,"Model shows line value, but key number(s) 10,1..."
2,Buffalo Bulls at Toledo Rockets,Toledo Rockets,-10.0,-20.96,10.96,14.0,14,True,EDGE_WITH_KEY_BARRIER,NEUTRAL_MARKET,"Model shows line value, but key number(s) 14 l..."
3,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,-16.0,-27.53,11.53,NaN,,False,EDGE_NO_KEY_BARRIER,CONFIRMED_BUT_LARGE_SPREAD,Retained model shows line cushion with no list...
4,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,4.0,-8.13,12.13,NaN,"3,7",True,EDGE_WITH_KEY_BARRIER,STRONG_MARKET_CONFLICT,"Model shows line value, but key number(s) 3,7 ..."
5,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,-24.0,-40.65,16.65,NaN,,False,EDGE_NO_KEY_BARRIER,CONFIRMED_BUT_LARGE_SPREAD,Retained model shows line cushion with no list...
6,Virginia Tech Hokies at California Golden Bears,Virginia Tech Hokies,-1.5,-9.81,8.31,3.0,"3,7",True,EDGE_WITH_KEY_BARRIER,CONFIRMED,"Model shows line value, but key number(s) 3,7 ..."
7,Rice Owls at East Carolina Pirates,East Carolina Pirates,-7.5,-11.55,4.05,10.0,10,True,EDGE_WITH_KEY_BARRIER,CONFIRMED,"Model shows line value, but key number(s) 10 l..."
8,Maryland Terrapins at Ohio State Buckeyes,Ohio State Buckeyes,-20.5,-34.18,13.68,NaN,,False,EDGE_NO_KEY_BARRIER,MARKET_CONFLICT,Retained model shows line cushion with no list...
9,Tennessee Volunteers at Arkansas Razorbacks,Tennessee Volunteers,-2.5,-15.98,13.48,3.0,"3,7,10,14",True,EDGE_WITH_KEY_BARRIER,STRONG_MARKET_CONFLICT,"Model shows line value, but key number(s) 3,7,..."



LINE VALUE STATUS COUNTS
---------------------------------------------
Line_Value_Status
EDGE_WITH_KEY_BARRIER     9
EDGE_NO_KEY_BARRIER       4
NO_RETAINED_MODEL_EDGE    1
Name: count, dtype: int64

KEY-NUMBER DIAGNOSTIC COMPLETE — NO BET SELECTION OR EV CALCULATION PERFORMED.


In [ ]:
# ==========================================
# CELL 51 — HISTORICAL SPREAD-ERROR DATASET
# ==========================================

import requests
import pandas as pd
import numpy as np

print("=" * 105)
print("CFB1 HISTORICAL SPREAD-ERROR CALIBRATION")
print("=" * 105)

# --------------------------------------------------
# 1. SETTINGS
# --------------------------------------------------

HISTORICAL_YEARS = [
    2021,
    2022,
    2023,
    2024,
    2025
]

PROVIDER_PRIORITY = [
    "DraftKings",
    "ESPN Bet",
    "Bovada"
]

CFBD_BASE_URL = "https://api.collegefootballdata.com"

headers = {
    "Authorization": f"Bearer {CFBD_API_KEY.strip()}"
}


# --------------------------------------------------
# 2. PROVIDER SELECTION
# --------------------------------------------------

def choose_historical_line(game):

    lines = game.get("lines", [])

    for provider in PROVIDER_PRIORITY:

        for line in lines:

            if (
                line.get("provider") == provider
                and line.get("spread") is not None
            ):
                return line

    return None


# --------------------------------------------------
# 3. DOWNLOAD HISTORICAL SEASONS
# --------------------------------------------------

historical_rows = []

for year in HISTORICAL_YEARS:

    print(f"Loading {year}...")

    params = {
        "year": year,
        "seasonType": "regular"
    }

    response = requests.get(
        f"{CFBD_BASE_URL}/lines",
        headers=headers,
        params=params,
        timeout=30
    )

    if response.status_code != 200:

        raise ValueError(
            f"CFBD historical line request failed "
            f"for {year}: HTTP {response.status_code}"
        )

    games = response.json()

    print(
        f"  Games returned: {len(games)}"
    )

    for game in games:

        # ------------------------------------------
        # FBS VS FBS ONLY
        # ------------------------------------------

        home_class = str(
            game.get("homeClassification", "")
        ).lower()

        away_class = str(
            game.get("awayClassification", "")
        ).lower()

        if (
            home_class != "fbs"
            or away_class != "fbs"
        ):
            continue


        # ------------------------------------------
        # FINAL SCORE REQUIRED
        # ------------------------------------------

        home_score = game.get("homeScore")
        away_score = game.get("awayScore")

        if (
            home_score is None
            or away_score is None
        ):
            continue


        # ------------------------------------------
        # SELECT HISTORICAL LINE
        # ------------------------------------------

        line = choose_historical_line(game)

        if line is None:
            continue


        closing_home_spread = line.get("spread")

        if closing_home_spread is None:
            continue


        # ------------------------------------------
        # CALCULATIONS
        # ------------------------------------------

        actual_home_margin = (
            home_score
            -
            away_score
        )

        market_expected_home_margin = (
            -closing_home_spread
        )

        spread_error = (
            actual_home_margin
            -
            market_expected_home_margin
        )

        # Equivalent:
        # actual_home_margin + closing_home_spread


        if spread_error > 0:
            ats_result = "HOME_COVER"

        elif spread_error < 0:
            ats_result = "AWAY_COVER"

        else:
            ats_result = "PUSH"


        historical_rows.append({

            "Season":
                year,

            "Week":
                game.get("week"),

            "Game_ID":
                game.get("id"),

            "Away_Team":
                game.get("awayTeam"),

            "Home_Team":
                game.get("homeTeam"),

            "Away_Score":
                away_score,

            "Home_Score":
                home_score,

            "Provider":
                line.get("provider"),

            "Closing_Home_Spread":
                closing_home_spread,

            "Actual_Home_Margin":
                actual_home_margin,

            "Market_Expected_Home_Margin":
                market_expected_home_margin,

            "Spread_Error":
                spread_error,

            "ATS_Result":
                ats_result
        })


# --------------------------------------------------
# 4. CREATE DATAFRAME
# --------------------------------------------------

df_hist_spread = pd.DataFrame(
    historical_rows
)


# --------------------------------------------------
# 5. BASIC VALIDATION
# --------------------------------------------------

print("\n" + "=" * 105)
print("HISTORICAL DATASET VALIDATION")
print("=" * 105)

print(
    f"Historical FBS-vs-FBS games: "
    f"{len(df_hist_spread)}"
)

print(
    f"Duplicate Game IDs: "
    f"{df_hist_spread['Game_ID'].duplicated().sum()}"
)

print(
    f"Missing spread errors: "
    f"{df_hist_spread['Spread_Error'].isna().sum()}"
)


print("\nPROVIDER COUNTS")
print("-" * 45)

print(
    df_hist_spread[
        'Provider'
    ].value_counts()
)


print("\nSEASON COUNTS")
print("-" * 45)

print(
    df_hist_spread[
        'Season'
    ].value_counts().sort_index()
)


# --------------------------------------------------
# 6. ATS BALANCE CHECK
# --------------------------------------------------

print("\nATS RESULT COUNTS")
print("-" * 45)

print(
    df_hist_spread[
        'ATS_Result'
    ].value_counts()
)


# --------------------------------------------------
# 7. SPREAD ERROR DISTRIBUTION
# --------------------------------------------------

spread_error = (
    df_hist_spread[
        'Spread_Error'
    ].astype(float)
)

print("\n" + "=" * 105)
print("HISTORICAL SPREAD-ERROR DISTRIBUTION")
print("=" * 105)

print(
    f"Mean error:          "
    f"{spread_error.mean():.3f}"
)

print(
    f"Median error:        "
    f"{spread_error.median():.3f}"
)

print(
    f"Std deviation:       "
    f"{spread_error.std(ddof=1):.3f}"
)

print(
    f"Mean absolute error: "
    f"{spread_error.abs().mean():.3f}"
)


print("\nSPREAD-ERROR QUANTILES")
print("-" * 45)

quantiles = spread_error.quantile(
    [
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95
    ]
)

print(
    quantiles.round(2)
)


# --------------------------------------------------
# 8. SANITY CHECK BY YEAR
# --------------------------------------------------

print("\n" + "=" * 105)
print("SPREAD ERROR BY SEASON")
print("=" * 105)

season_summary = (
    df_hist_spread
    .groupby('Season')
    .agg(
        Games=('Game_ID', 'count'),
        Mean_Error=('Spread_Error', 'mean'),
        Median_Error=('Spread_Error', 'median'),
        Std_Error=('Spread_Error', 'std'),
        MAE=('Spread_Error',
             lambda x: x.abs().mean())
    )
    .round(3)
)

display(
    season_summary
)


# --------------------------------------------------
# 9. SAMPLE ROWS
# --------------------------------------------------

print("\n" + "=" * 105)
print("SAMPLE CALIBRATION ROWS")
print("=" * 105)

display(
    df_hist_spread[
        [
            'Season',
            'Away_Team',
            'Home_Team',
            'Provider',
            'Closing_Home_Spread',
            'Actual_Home_Margin',
            'Spread_Error',
            'ATS_Result'
        ]
    ].head(20)
)


print("\n" + "=" * 105)

print(
    "HISTORICAL CALIBRATION DATASET COMPLETE — "
    "NO CFB1 COVER PROBABILITIES CALCULATED YET."
)

CFB1 HISTORICAL SPREAD-ERROR CALIBRATION
Loading 2021...
  Games returned: 849
Loading 2022...
  Games returned: 1417
Loading 2023...
  Games returned: 1350
Loading 2024...
  Games returned: 1523
Loading 2025...
  Games returned: 1547

HISTORICAL DATASET VALIDATION
Historical FBS-vs-FBS games: 3730
Duplicate Game IDs: 0
Missing spread errors: 0

PROVIDER COUNTS
---------------------------------------------
Provider
DraftKings    2160
Bovada        1511
ESPN Bet        59
Name: count, dtype: int64

SEASON COUNTS
---------------------------------------------
Season
2021    732
2022    734
2023    750
2024    752
2025    762
Name: count, dtype: int64

ATS RESULT COUNTS
---------------------------------------------
ATS_Result
HOME_COVER    1849
AWAY_COVER    1812
PUSH            69
Name: count, dtype: int64

HISTORICAL SPREAD-ERROR DISTRIBUTION
Mean error:          0.056
Median error:        0.000
Std deviation:       15.349
Mean absolute error: 12.146

SPREAD-ERROR QUANTILES
-------------

,Games,Mean_Error,Median_Error,Std_Error,MAE
Season,,,,,
2021,732,-0.474,0.5,15.707,12.608
2022,734,-0.631,0.0,15.294,12.043
2023,750,-0.075,0.0,15.078,12.011
2024,752,0.680,0.0,15.586,12.235
2025,762,0.741,0.5,15.069,11.844



SAMPLE CALIBRATION ROWS


,Season,Away_Team,Home_Team,Provider,Closing_Home_Spread,Actual_Home_Margin,Spread_Error,ATS_Result
0,2021,Liberty,UAB,Bovada,-3.0,-24,-27.0,AWAY_COVER
1,2021,UNLV,Air Force,Bovada,-18.0,34,16.0,HOME_COVER
2,2021,BYU,USC,Bovada,7.0,-4,3.0,HOME_COVER
3,2021,Hawai'i,Wyoming,Bovada,-11.5,-24,-35.5,AWAY_COVER
4,2021,Nevada,Colorado State,Bovada,3.0,-42,-39.0,AWAY_COVER
5,2021,BYU,Georgia Southern,Bovada,20.0,-17,3.0,HOME_COVER
6,2021,Kent State,Akron,Bovada,14.0,-38,-24.0,AWAY_COVER
7,2021,Texas State,Coastal Carolina,Bovada,-24.5,14,-10.5,AWAY_COVER
8,2021,Arkansas State,Georgia State,Bovada,-16.0,8,-8.0,AWAY_COVER
9,2021,App State,Troy,Bovada,10.0,-38,-28.0,AWAY_COVER



HISTORICAL CALIBRATION DATASET COMPLETE — NO CFB1 COVER PROBABILITIES CALCULATED YET.


In [ ]:
# ==========================================
# CELL 52 — EMPIRICAL COVER PROBABILITY
# FULL-GAME SPREADS ONLY
# ==========================================

import numpy as np
import pandas as pd

print("=" * 108)
print("CFB1 EMPIRICAL FULL-GAME COVER PROBABILITY")
print("=" * 108)


# --------------------------------------------------
# 1. REQUIRED INPUTS
# --------------------------------------------------

if 'df_hist_spread' not in globals():
    raise NameError(
        "df_hist_spread does not exist. "
        "Run Cell 51 first."
    )

if 'df_key_value' not in globals():
    raise NameError(
        "df_key_value does not exist. "
        "Run Cell 48 first."
    )


# --------------------------------------------------
# 2. HISTORICAL ERROR ARRAY
#
# Spread_Error > 0 = home beat market expectation
# Spread_Error < 0 = away beat market expectation
# --------------------------------------------------

hist_errors = (
    df_hist_spread['Spread_Error']
    .dropna()
    .astype(float)
)


# --------------------------------------------------
# 3. SPREAD-MAGNITUDE BUCKETS
#
# Used only as a secondary/local comparison.
# --------------------------------------------------

def spread_bucket(spread_abs):

    if spread_abs < 3:
        return 'UNDER_3'

    elif spread_abs < 7:
        return '3_TO_6.5'

    elif spread_abs < 10:
        return '7_TO_9.5'

    elif spread_abs < 14:
        return '10_TO_13.5'

    elif spread_abs < 17:
        return '14_TO_16.5'

    elif spread_abs < 21:
        return '17_TO_20.5'

    else:
        return '21_PLUS'


df_hist_spread['Spread_Bucket'] = (
    df_hist_spread[
        'Closing_Home_Spread'
    ]
    .abs()
    .apply(spread_bucket)
)


# --------------------------------------------------
# 4. EMPIRICAL PROBABILITY FUNCTION
#
# cushion > 0 means retained model expects the
# model side to outperform the market by cushion.
#
# We orient historical error from MODEL SIDE:
#
# If model side is HOME:
#     oriented_error = Spread_Error
#
# If model side is AWAY:
#     oriented_error = -Spread_Error
#
# Model side covers if:
#
# oriented_error + cushion > 0
#
# Push if exactly 0
# --------------------------------------------------

def empirical_probability(
    cushion,
    model_is_home,
    errors
):

    if pd.isna(cushion):
        return np.nan, np.nan, np.nan

    oriented = (
        errors
        if model_is_home
        else -errors
    )

    adjusted = (
        oriented
        + cushion
    )

    win_prob = (
        (adjusted > 0).mean()
    )

    push_prob = (
        (adjusted == 0).mean()
    )

    loss_prob = (
        (adjusted < 0).mean()
    )

    return (
        win_prob,
        push_prob,
        loss_prob
    )


# --------------------------------------------------
# 5. CALCULATE PROBABILITIES
# --------------------------------------------------

prob_rows = []

for _, row in df_key_value.iterrows():

    game = row['Game']

    model_side = row['Model_Side']
    home_team = row['Home_Team']
    away_team = row['Away_Team']

    market_spread = row[
        'Market_Model_Spread'
    ]

    cushion = row[
        'Model_Cushion_Points'
    ]

    model_is_home = (
        model_side == home_team
    )

    # --------------------------------------
    # GLOBAL SAMPLE
    # --------------------------------------

    global_win, global_push, global_loss = (
        empirical_probability(
            cushion,
            model_is_home,
            hist_errors
        )
    )


    # --------------------------------------
    # LOCAL SPREAD BUCKET
    # --------------------------------------

    bucket = spread_bucket(
        abs(market_spread)
    )

    bucket_errors = (
        df_hist_spread.loc[
            df_hist_spread[
                'Spread_Bucket'
            ] == bucket,
            'Spread_Error'
        ]
        .dropna()
        .astype(float)
    )

    local_win, local_push, local_loss = (
        empirical_probability(
            cushion,
            model_is_home,
            bucket_errors
        )
    )


    prob_rows.append({

        'Game':
            game,

        'Model_Side':
            model_side,

        'Market_Model_Spread':
            market_spread,

        'Retained_Model_Spread':
            row[
                'Retained_Model_Spread'
            ],

        'Model_Cushion_Points':
            cushion,

        'Spread_Bucket':
            bucket,

        'Historical_Bucket_N':
            len(bucket_errors),

        'Global_Cover_Prob':
            global_win,

        'Global_Push_Prob':
            global_push,

        'Global_Loss_Prob':
            global_loss,

        'Bucket_Cover_Prob':
            local_win,

        'Bucket_Push_Prob':
            local_push,

        'Bucket_Loss_Prob':
            local_loss,

        'Line_Value_Status':
            row[
                'Line_Value_Status'
            ],

        'Decision_Status':
            row[
                'Decision_Status'
            ]
    })


df_cover_prob = pd.DataFrame(
    prob_rows
)


# --------------------------------------------------
# 6. CONSERVATIVE V1 PROBABILITY
#
# Use the LOWER of:
# - global empirical cover probability
# - comparable-spread bucket probability
#
# This prevents us from choosing whichever
# estimate looks more attractive.
# --------------------------------------------------

df_cover_prob[
    'CFB1_Cover_Prob_V1'
] = (
    df_cover_prob[
        [
            'Global_Cover_Prob',
            'Bucket_Cover_Prob'
        ]
    ]
    .min(axis=1)
)


# --------------------------------------------------
# 7. FORMAT FOR DISPLAY
# --------------------------------------------------

display_df = df_cover_prob.copy()

prob_cols = [
    'Global_Cover_Prob',
    'Global_Push_Prob',
    'Bucket_Cover_Prob',
    'Bucket_Push_Prob',
    'CFB1_Cover_Prob_V1'
]

for col in prob_cols:

    display_df[col] = (
        display_df[col] * 100
    ).round(1)


# --------------------------------------------------
# 8. DISPLAY
# --------------------------------------------------

print("\n" + "=" * 108)
print("FULL-GAME SPREAD PROBABILITY BOARD")
print("=" * 108)

display(
    display_df[
        [
            'Game',
            'Model_Side',
            'Market_Model_Spread',
            'Retained_Model_Spread',
            'Model_Cushion_Points',
            'Spread_Bucket',
            'Historical_Bucket_N',
            'Global_Cover_Prob',
            'Bucket_Cover_Prob',
            'CFB1_Cover_Prob_V1',
            'Line_Value_Status',
            'Decision_Status'
        ]
    ]
)


# --------------------------------------------------
# 9. PROBABILITY SANITY CHECK
# --------------------------------------------------

print("\n" + "=" * 108)
print("PROBABILITY SANITY CHECK")
print("=" * 108)

print(
    f"Minimum bucket sample size: "
    f"{df_cover_prob['Historical_Bucket_N'].min()}"
)

print(
    f"Maximum V1 probability: "
    f"{df_cover_prob['CFB1_Cover_Prob_V1'].max():.3f}"
)

print(
    f"Minimum V1 probability: "
    f"{df_cover_prob['CFB1_Cover_Prob_V1'].min():.3f}"
)


# --------------------------------------------------
# 10. IMPORTANT LIMITATION
# --------------------------------------------------

print("\n" + "=" * 108)

print(
    "FULL-GAME SPREAD CALIBRATION COMPLETE."
)

print(
    "These probabilities apply only to full-game "
    "spread evaluation."
)

print(
    "Do NOT reuse them for team totals, "
    "1H spreads, or 1H totals."
)

print(
    "NO EV OR KELLY STAKES CALCULATED YET."
)

CFB1 EMPIRICAL FULL-GAME COVER PROBABILITY

FULL-GAME SPREAD PROBABILITY BOARD


,Game,Model_Side,Market_Model_Spread,Retained_Model_Spread,Model_Cushion_Points,Spread_Bucket,Historical_Bucket_N,Global_Cover_Prob,Bucket_Cover_Prob,CFB1_Cover_Prob_V1,Line_Value_Status,Decision_Status
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,-8.0,-3.449837,-4.55,7_TO_9.5,516,37.1,35.9,35.9,NO_RETAINED_MODEL_EDGE,CONFIRMED
1,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,-7.5,-36.124942,28.62,7_TO_9.5,516,96.6,96.7,96.6,EDGE_WITH_KEY_BARRIER,CONFIRMED_BUT_LARGE_SPREAD
2,Buffalo Bulls at Toledo Rockets,Toledo Rockets,-10.0,-20.961807,10.96,10_TO_13.5,513,76.7,75.0,75.0,EDGE_WITH_KEY_BARRIER,NEUTRAL_MARKET
3,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,-16.0,-27.531789,11.53,14_TO_16.5,323,78.4,78.3,78.3,EDGE_NO_KEY_BARRIER,CONFIRMED_BUT_LARGE_SPREAD
4,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,4.0,-8.127997,12.13,3_TO_6.5,1031,78.3,78.9,78.3,EDGE_WITH_KEY_BARRIER,STRONG_MARKET_CONFLICT
5,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,-24.0,-40.647840,16.65,21_PLUS,546,86.9,87.5,86.9,EDGE_NO_KEY_BARRIER,CONFIRMED_BUT_LARGE_SPREAD
6,Virginia Tech Hokies at California Golden Bears,Virginia Tech Hokies,-1.5,-9.807644,8.31,UNDER_3,513,70.5,71.5,70.5,EDGE_WITH_KEY_BARRIER,CONFIRMED
7,Rice Owls at East Carolina Pirates,East Carolina Pirates,-7.5,-11.549153,4.05,7_TO_9.5,516,61.6,62.6,61.6,EDGE_WITH_KEY_BARRIER,CONFIRMED
8,Maryland Terrapins at Ohio State Buckeyes,Ohio State Buckeyes,-20.5,-34.181357,13.68,17_TO_20.5,288,81.7,79.2,79.2,EDGE_NO_KEY_BARRIER,MARKET_CONFLICT
9,Tennessee Volunteers at Arkansas Razorbacks,Tennessee Volunteers,-2.5,-15.976263,13.48,UNDER_3,513,80.3,80.7,80.3,EDGE_WITH_KEY_BARRIER,STRONG_MARKET_CONFLICT



PROBABILITY SANITY CHECK
Minimum bucket sample size: 288
Maximum V1 probability: 0.966
Minimum V1 probability: 0.359

FULL-GAME SPREAD CALIBRATION COMPLETE.
These probabilities apply only to full-game spread evaluation.
Do NOT reuse them for team totals, 1H spreads, or 1H totals.
NO EV OR KELLY STAKES CALCULATED YET.


In [ ]:
# ==========================================
# CELL 53 — AUTOMATIC SPORTSBOOK MARKET DISCOVERY
# FanDuel + DraftKings
# ==========================================

import requests
import pandas as pd
import numpy as np

print("=" * 110)
print("CFB1 AUTOMATIC SPORTSBOOK MARKET DISCOVERY")
print("=" * 110)


# --------------------------------------------------
# 1. REQUIRED INPUTS
# --------------------------------------------------

if 'df_key_value' not in globals():
    raise NameError(
        "df_key_value does not exist. "
        "Run through Cell 48 first."
    )

if not ODDS_API_KEY:
    raise ValueError(
        "ODDS_API_KEY is missing."
    )


# --------------------------------------------------
# 2. SETTINGS
# --------------------------------------------------

SPORT_KEY = "americanfootball_ncaaf"

BOOKMAKERS_WANTED = {
    "fanduel": "FanDuel",
    "draftkings": "DraftKings"
}

ODDS_BASE_URL = (
    "https://api.the-odds-api.com/v4"
)

api_key = ODDS_API_KEY.strip()


# --------------------------------------------------
# 3. GET CURRENT EVENT LIST
#
# The events endpoint does NOT include odds.
# It gives us event IDs needed for detailed
# market discovery.
# --------------------------------------------------

events_url = (
    f"{ODDS_BASE_URL}/sports/"
    f"{SPORT_KEY}/events"
)

events_response = requests.get(
    events_url,
    params={
        "apiKey": api_key,
        "dateFormat": "iso"
    },
    timeout=30
)

print(
    f"\nEvents endpoint HTTP status: "
    f"{events_response.status_code}"
)

if events_response.status_code != 200:
    raise ValueError(
        "Unable to retrieve NCAA football events.\n"
        f"Response: {events_response.text[:500]}"
    )

events_raw = events_response.json()

print(
    f"Current NCAA football events returned: "
    f"{len(events_raw)}"
)


# --------------------------------------------------
# 4. BUILD EVENT LOOKUP
# --------------------------------------------------

event_rows = []

for event in events_raw:

    away = event.get("away_team")
    home = event.get("home_team")

    if away is None or home is None:
        continue

    game = f"{away} at {home}"

    event_rows.append({

        "Game":
            game,

        "Event_ID":
            event.get("id"),

        "Away_Team":
            away,

        "Home_Team":
            home,

        "Commence_Time":
            event.get("commence_time")
    })


df_odds_events = pd.DataFrame(
    event_rows
)


# --------------------------------------------------
# 5. MATCH OUR 10 CFB1 GAMES TO EVENT IDs
# --------------------------------------------------

target_games = (
    df_key_value[
        [
            "Game",
            "Away_Team",
            "Home_Team"
        ]
    ]
    .drop_duplicates()
    .copy()
)

df_event_match = target_games.merge(
    df_odds_events[
        [
            "Game",
            "Event_ID",
            "Commence_Time"
        ]
    ],
    on="Game",
    how="left",
    validate="one_to_one"
)


# --------------------------------------------------
# 6. VALIDATE EVENT MATCHING
# --------------------------------------------------

missing_events = df_event_match[
    df_event_match[
        "Event_ID"
    ].isna()
]

print("\nEVENT MATCH VALIDATION")
print("-" * 50)

print(
    f"CFB1 target games: "
    f"{len(target_games)}"
)

print(
    f"Matched Odds API events: "
    f"{df_event_match['Event_ID'].notna().sum()}"
)

print(
    f"Unmatched events: "
    f"{len(missing_events)}"
)

if len(missing_events) > 0:

    print(
        "\nUNMATCHED GAMES"
    )

    display(
        missing_events[
            [
                "Game",
                "Away_Team",
                "Home_Team"
            ]
        ]
    )


# --------------------------------------------------
# 7. MARKET DISCOVERY
#
# Query each matched event's /markets endpoint.
#
# This tells us which markets have recently been
# observed for each bookmaker.
# --------------------------------------------------

market_rows = []

request_log = []

for _, game_row in df_event_match.dropna(
    subset=["Event_ID"]
).iterrows():

    game = game_row["Game"]
    event_id = game_row["Event_ID"]

    markets_url = (
        f"{ODDS_BASE_URL}/sports/"
        f"{SPORT_KEY}/events/"
        f"{event_id}/markets"
    )

    response = requests.get(
        markets_url,
        params={
            "apiKey": api_key,
            "regions": "us",
            "dateFormat": "iso"
        },
        timeout=30
    )

    request_log.append({

        "Game":
            game,

        "HTTP_Status":
            response.status_code,

        "Requests_Remaining":
            response.headers.get(
                "x-requests-remaining"
            ),

        "Requests_Used":
            response.headers.get(
                "x-requests-used"
            ),

        "Last_Request_Cost":
            response.headers.get(
                "x-requests-last"
            )
    })

    if response.status_code != 200:
        continue

    market_data = response.json()

    bookmakers = market_data.get(
        "bookmakers",
        []
    )

    for book in bookmakers:

        book_key = book.get("key")

        if book_key not in BOOKMAKERS_WANTED:
            continue

        book_name = BOOKMAKERS_WANTED[
            book_key
        ]

        markets = book.get(
            "markets",
            []
        )

        for market in markets:

            # Depending on endpoint response,
            # market may be a dictionary with key
            # or occasionally another simple structure.
            if isinstance(market, dict):

                market_key = (
                    market.get("key")
                    or market.get("market_key")
                )

            else:

                market_key = str(market)

            if market_key is None:
                continue

            market_rows.append({

                "Game":
                    game,

                "Event_ID":
                    event_id,

                "Bookmaker":
                    book_name,

                "Bookmaker_Key":
                    book_key,

                "Market_Key":
                    market_key
            })


df_market_discovery = pd.DataFrame(
    market_rows
)

df_market_request_log = pd.DataFrame(
    request_log
)


# --------------------------------------------------
# 8. TARGET MARKET CATEGORIES
# --------------------------------------------------

MARKETS_OF_INTEREST = [

    # Full game
    "spreads",
    "alternate_spreads",
    "totals",
    "alternate_totals",
    "team_totals",
    "alternate_team_totals",

    # First half
    "spreads_h1",
    "alternate_spreads_h1",
    "totals_h1",
    "alternate_totals_h1",
    "team_totals_h1",
    "alternate_team_totals_h1"
]


# --------------------------------------------------
# 9. MARKET AVAILABILITY MATRIX
# --------------------------------------------------

if len(df_market_discovery) == 0:

    print(
        "\nMARKET DISCOVERY STATUS: CHECK"
    )

    print(
        "No FanDuel or DraftKings market keys "
        "were returned."
    )

else:

    df_market_discovery[
        "Is_CFB1_Target_Market"
    ] = (
        df_market_discovery[
            "Market_Key"
        ].isin(
            MARKETS_OF_INTEREST
        )
    )

    target_market_data = (
        df_market_discovery[
            df_market_discovery[
                "Is_CFB1_Target_Market"
            ]
        ]
        .copy()
    )


    # ----------------------------------------------
    # Pivot to readable matrix
    # ----------------------------------------------

    availability = (
        target_market_data
        .assign(Available="YES")
        .pivot_table(
            index=[
                "Game",
                "Bookmaker"
            ],
            columns="Market_Key",
            values="Available",
            aggfunc="first",
            fill_value=""
        )
        .reset_index()
    )


    print(
        "\n" + "=" * 110
    )

    print(
        "CFB1 MARKET AVAILABILITY MATRIX"
    )

    print(
        "=" * 110
    )

    display(
        availability
    )


# --------------------------------------------------
# 10. ALL MARKET KEYS OBSERVED
#
# Helpful in case the API uses a market key we
# weren't expecting.
# --------------------------------------------------

print(
    "\n" + "=" * 110
)

print(
    "ALL MARKET KEYS OBSERVED — "
    "FANDUEL / DRAFTKINGS"
)

print(
    "=" * 110
)

if len(df_market_discovery) > 0:

    all_market_summary = (
        df_market_discovery
        .groupby(
            "Bookmaker"
        )["Market_Key"]
        .apply(
            lambda x:
            sorted(
                set(x)
            )
        )
    )

    for bookmaker, markets in (
        all_market_summary.items()
    ):

        print(
            f"\n{bookmaker}"
        )

        print("-" * 50)

        for market in markets:
            print(market)


# --------------------------------------------------
# 11. REQUEST / QUOTA AUDIT
# --------------------------------------------------

print(
    "\n" + "=" * 110
)

print(
    "API REQUEST AUDIT"
)

print(
    "=" * 110
)

display(
    df_market_request_log
)


# --------------------------------------------------
# 12. CFB1 ROUTE COVERAGE SUMMARY
# --------------------------------------------------

print(
    "\n" + "=" * 110
)

print(
    "CFB1 PRICING DISCOVERY SUMMARY"
)

print(
    "=" * 110
)

if len(df_market_discovery) > 0:

    for game in df_event_match["Game"]:

        game_data = (
            df_market_discovery[
                df_market_discovery[
                    "Game"
                ] == game
            ]
        )

        print(
            f"\n{game}"
        )

        if len(game_data) == 0:

            print(
                "  No FanDuel/DraftKings "
                "markets discovered."
            )

            continue

        for bookmaker in [
            "FanDuel",
            "DraftKings"
        ]:

            book_data = (
                game_data[
                    game_data[
                        "Bookmaker"
                    ] == bookmaker
                ]
            )

            markets = sorted(
                set(
                    book_data[
                        "Market_Key"
                    ]
                )
            )

            relevant = [
                m
                for m in markets
                if m in MARKETS_OF_INTEREST
            ]

            print(
                f"  {bookmaker}: "
                f"{', '.join(relevant) if relevant else 'No target markets'}"
            )


print(
    "\n" + "=" * 110
)

print(
    "MARKET DISCOVERY COMPLETE."
)

print(
    "NO BETS, EV, OR KELLY STAKES "
    "CALCULATED."
)

CFB1 AUTOMATIC SPORTSBOOK MARKET DISCOVERY

Events endpoint HTTP status: 200
Current NCAA football events returned: 66

EVENT MATCH VALIDATION
--------------------------------------------------
CFB1 target games: 14
Matched Odds API events: 14
Unmatched events: 0

CFB1 MARKET AVAILABILITY MATRIX


Market_Key,Game,Bookmaker,alternate_spreads,alternate_spreads_h1,alternate_team_totals,alternate_totals,alternate_totals_h1,spreads,spreads_h1,team_totals,team_totals_h1,totals,totals_h1
0,Air Force Falcons at Northern Illinois Huskies,DraftKings,YES,YES,YES,YES,YES,YES,YES,,YES,YES,YES
1,Air Force Falcons at Northern Illinois Huskies,FanDuel,YES,,YES,YES,,YES,YES,YES,YES,YES,YES
2,Ball State Cardinals at Northwestern Wildcats,DraftKings,YES,YES,YES,YES,YES,YES,YES,,YES,YES,YES
3,Ball State Cardinals at Northwestern Wildcats,FanDuel,YES,,YES,YES,,YES,YES,YES,YES,YES,YES
4,Buffalo Bulls at Toledo Rockets,DraftKings,YES,YES,YES,YES,YES,YES,YES,,YES,YES,YES
5,Buffalo Bulls at Toledo Rockets,FanDuel,YES,,YES,YES,,YES,YES,YES,YES,YES,YES
6,Charlotte 49ers at North Texas Mean Green,DraftKings,YES,YES,YES,YES,YES,YES,YES,,YES,YES,YES
7,Charlotte 49ers at North Texas Mean Green,FanDuel,YES,,YES,YES,,YES,YES,YES,YES,YES,YES
8,Duke Blue Devils at Georgia Tech Yellow Jackets,DraftKings,YES,YES,YES,YES,YES,YES,YES,,YES,YES,YES
9,Duke Blue Devils at Georgia Tech Yellow Jackets,FanDuel,YES,,YES,YES,,YES,YES,YES,YES,YES,YES



ALL MARKET KEYS OBSERVED — FANDUEL / DRAFTKINGS

DraftKings
--------------------------------------------------
alternate_spreads
alternate_spreads_h1
alternate_spreads_q1
alternate_team_totals
alternate_totals
alternate_totals_h1
alternate_totals_q1
correct_score
h2h
h2h_3_way_h1
h2h_3_way_h2
h2h_h1
h2h_h2
h2h_q1
h2h_q2
h2h_q3
h2h_q4
halftime_fulltime
last_team_to_score
odd_even
odd_even_h1
odd_even_h2
overtime
player_1st_td
player_anytime_td
player_anytime_td_h1
player_anytime_td_q1
player_pass_attempts_alternate
player_pass_completions_alternate
player_pass_rush_yds_alternate
player_pass_tds_alternate
player_pass_yds_alternate
player_reception_yds_alternate
player_rush_yds_alternate
player_rush_yds_alternate_h1
player_tds_over
spreads
spreads_h1
spreads_h2
spreads_q1
spreads_q2
spreads_q3
spreads_q4
team_totals_h1
team_totals_h2
team_totals_q1
total_tds
totals
totals_h1
totals_h2
totals_q1
totals_q2
totals_q3
totals_q4

FanDuel
--------------------------------------------------
alte

,Game,HTTP_Status,Requests_Remaining,Requests_Used,Last_Request_Cost
0,Jacksonville State Gamecocks at Kennesaw State...,200,115,385,1
1,Ball State Cardinals at Northwestern Wildcats,200,114,386,1
2,Buffalo Bulls at Toledo Rockets,200,113,387,1
3,Charlotte 49ers at North Texas Mean Green,200,112,388,1
4,Duke Blue Devils at Georgia Tech Yellow Jackets,200,111,389,1
5,Stanford Cardinal at Notre Dame Fighting Irish,200,110,390,1
6,Virginia Tech Hokies at California Golden Bears,200,109,391,1
7,Rice Owls at East Carolina Pirates,200,108,392,1
8,Maryland Terrapins at Ohio State Buckeyes,200,107,393,1
9,Tennessee Volunteers at Arkansas Razorbacks,200,106,394,1



CFB1 PRICING DISCOVERY SUMMARY

Jacksonville State Gamecocks at Kennesaw State Owls
  FanDuel: alternate_spreads, alternate_team_totals, alternate_totals, spreads, spreads_h1, team_totals, team_totals_h1, totals, totals_h1
  DraftKings: alternate_spreads, alternate_spreads_h1, alternate_team_totals, alternate_totals, alternate_totals_h1, spreads, spreads_h1, team_totals_h1, totals, totals_h1

Ball State Cardinals at Northwestern Wildcats
  FanDuel: alternate_spreads, alternate_team_totals, alternate_totals, spreads, spreads_h1, team_totals, team_totals_h1, totals, totals_h1
  DraftKings: alternate_spreads, alternate_spreads_h1, alternate_team_totals, alternate_totals, alternate_totals_h1, spreads, spreads_h1, team_totals_h1, totals, totals_h1

Buffalo Bulls at Toledo Rockets
  FanDuel: alternate_spreads, alternate_team_totals, alternate_totals, spreads, spreads_h1, team_totals, team_totals_h1, totals, totals_h1
  DraftKings: alternate_spreads, alternate_spreads_h1, alternate_team_tota

In [ ]:
# ==========================================
# CELL 54 — AUTOMATIC SPORTSBOOK PRICING PULL
# FanDuel + DraftKings
# ==========================================

import requests
import pandas as pd
import numpy as np

print("=" * 110)
print("CFB1 AUTOMATIC SPORTSBOOK PRICING PULL")
print("=" * 110)


# --------------------------------------------------
# 1. REQUIRED INPUTS
# --------------------------------------------------

if 'df_event_match' not in globals():
    raise NameError(
        "df_event_match does not exist. "
        "Run Cell 53 first."
    )

if not ODDS_API_KEY:
    raise ValueError(
        "ODDS_API_KEY is missing."
    )


# --------------------------------------------------
# 2. SETTINGS
# --------------------------------------------------

SPORT_KEY = "americanfootball_ncaaf"

ODDS_BASE_URL = (
    "https://api.the-odds-api.com/v4"
)

api_key = ODDS_API_KEY.strip()

BOOKMAKERS = [
    "fanduel",
    "draftkings"
]


# --------------------------------------------------
# 3. MARKETS TO PULL
#
# These cover:
# - full-game spreads
# - alternate spreads
# - team totals
# - first-half spreads
# - first-half team totals
# - first-half totals
#
# We'll only use markets actually returned.
# --------------------------------------------------

MARKETS_TO_PULL = [
    "spreads",
    "alternate_spreads",
    "team_totals",
    "alternate_team_totals",
    "spreads_h1",
    "alternate_spreads_h1",
    "team_totals_h1",
    "totals_h1",
    "alternate_totals_h1"
]


# --------------------------------------------------
# 4. ROUTE MAP
#
# Translate Odds API market keys into CFB1
# standardized market labels.
# --------------------------------------------------

MARKET_TYPE_MAP = {

    "spreads":
        "FULL_GAME_SPREAD",

    "alternate_spreads":
        "FULL_GAME_SPREAD",

    "team_totals":
        "TEAM_TOTAL",

    "alternate_team_totals":
        "TEAM_TOTAL",

    "spreads_h1":
        "1H_SPREAD",

    "alternate_spreads_h1":
        "1H_SPREAD",

    "team_totals_h1":
        "1H_TEAM_TOTAL",

    "totals_h1":
        "1H_TOTAL",

    "alternate_totals_h1":
        "1H_TOTAL"
}


# --------------------------------------------------
# 5. PULL ODDS EVENT BY EVENT
#
# Official Odds API event-odds endpoint accepts
# additional markets one event at a time.
# --------------------------------------------------

pricing_rows = []
pricing_request_log = []

markets_param = ",".join(
    MARKETS_TO_PULL
)

bookmakers_param = ",".join(
    BOOKMAKERS
)


for _, event_row in df_event_match.dropna(
    subset=["Event_ID"]
).iterrows():

    game = event_row["Game"]
    event_id = event_row["Event_ID"]

    event_url = (
        f"{ODDS_BASE_URL}/sports/"
        f"{SPORT_KEY}/events/"
        f"{event_id}/odds"
    )

    response = requests.get(
        event_url,
        params={
            "apiKey": api_key,
            "bookmakers": bookmakers_param,
            "markets": markets_param,
            "oddsFormat": "american",
            "dateFormat": "iso"
        },
        timeout=30
    )


    pricing_request_log.append({

        "Game":
            game,

        "HTTP_Status":
            response.status_code,

        "Requests_Remaining":
            response.headers.get(
                "x-requests-remaining"
            ),

        "Requests_Used":
            response.headers.get(
                "x-requests-used"
            ),

        "Last_Request_Cost":
            response.headers.get(
                "x-requests-last"
            )
    })


    if response.status_code != 200:

        print(
            f"\nCHECK: {game}"
        )

        print(
            f"HTTP {response.status_code}"
        )

        print(
            response.text[:300]
        )

        continue


    data = response.json()

    home_team = data.get("home_team")
    away_team = data.get("away_team")

    bookmakers = data.get(
        "bookmakers",
        []
    )


    # --------------------------------------------------
    # 6. FLATTEN BOOKMAKER -> MARKET -> OUTCOME
    # --------------------------------------------------

    for book in bookmakers:

        book_key = book.get("key")
        book_name = book.get("title")

        markets = book.get(
            "markets",
            []
        )

        for market in markets:

            market_key = market.get(
                "key"
            )

            if market_key not in MARKET_TYPE_MAP:
                continue

            market_type = MARKET_TYPE_MAP[
                market_key
            ]

            market_last_update = market.get(
                "last_update"
            )

            outcomes = market.get(
                "outcomes",
                []
            )


            for outcome in outcomes:

                outcome_name = outcome.get(
                    "name"
                )

                description = outcome.get(
                    "description"
                )

                point = outcome.get(
                    "point"
                )

                price = outcome.get(
                    "price"
                )


                # --------------------------------------
                # Determine team / direction
                # --------------------------------------

                team = None
                direction = None


                # Spread markets
                if market_type in [
                    "FULL_GAME_SPREAD",
                    "1H_SPREAD"
                ]:

                    team = outcome_name
                    direction = "SPREAD"


                # Team totals
                elif market_type in [
                    "TEAM_TOTAL",
                    "1H_TEAM_TOTAL"
                ]:

                    # In team-total markets:
                    # name usually = Over / Under
                    # description usually = team
                    team = description
                    direction = str(
                        outcome_name
                    ).upper()


                # Game first-half total
                elif market_type == "1H_TOTAL":

                    team = ""
                    direction = str(
                        outcome_name
                    ).upper()


                pricing_rows.append({

                    "Game":
                        game,

                    "Away_Team":
                        away_team,

                    "Home_Team":
                        home_team,

                    "Sportsbook":
                        book_name,

                    "Sportsbook_Key":
                        book_key,

                    "Source_Market_Key":
                        market_key,

                    "Market_Type":
                        market_type,

                    "Team":
                        team,

                    "Bet_Direction":
                        direction,

                    "Line":
                        point,

                    "American_Odds":
                        price,

                    "Market_Last_Update":
                        market_last_update
                })


# --------------------------------------------------
# 7. CREATE PRICING DATAFRAME
# --------------------------------------------------

df_auto_pricing = pd.DataFrame(
    pricing_rows
)

df_pricing_pull_log = pd.DataFrame(
    pricing_request_log
)


# --------------------------------------------------
# 8. BASIC VALIDATION
# --------------------------------------------------

print(
    "\n" + "=" * 110
)

print(
    "PRICING PULL VALIDATION"
)

print(
    "=" * 110
)

print(
    f"Pricing rows returned: "
    f"{len(df_auto_pricing)}"
)

if len(df_auto_pricing) == 0:

    raise ValueError(
        "No pricing rows returned."
    )


print(
    f"Games represented: "
    f"{df_auto_pricing['Game'].nunique()}"
)

print(
    f"Sportsbooks represented: "
    f"{sorted(df_auto_pricing['Sportsbook'].dropna().unique())}"
)

print(
    f"Missing line values: "
    f"{df_auto_pricing['Line'].isna().sum()}"
)

print(
    f"Missing American odds: "
    f"{df_auto_pricing['American_Odds'].isna().sum()}"
)


# --------------------------------------------------
# 9. MARKET ROW COUNTS
# --------------------------------------------------

print(
    "\nMARKET ROW COUNTS"
)

print("-" * 50)

market_counts = (
    df_auto_pricing
    .groupby(
        [
            "Sportsbook",
            "Source_Market_Key"
        ]
    )
    .size()
    .rename("Rows")
)

print(
    market_counts
)


# --------------------------------------------------
# 10. FULL-GAME SPREAD BOARD
#
# Main + alternate spread lines only.
# --------------------------------------------------

df_full_game_spreads = (
    df_auto_pricing[
        df_auto_pricing[
            "Market_Type"
        ] == "FULL_GAME_SPREAD"
    ]
    .copy()
)


print(
    "\n" + "=" * 110
)

print(
    "FULL-GAME SPREAD PRICE BOARD"
)

print(
    "=" * 110
)

display(
    df_full_game_spreads[
        [
            "Game",
            "Sportsbook",
            "Source_Market_Key",
            "Team",
            "Line",
            "American_Odds"
        ]
    ]
    .sort_values(
        [
            "Game",
            "Team",
            "Line",
            "Sportsbook"
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------
# 11. TEAM TOTAL BOARD
# --------------------------------------------------

df_team_total_prices = (
    df_auto_pricing[
        df_auto_pricing[
            "Market_Type"
        ] == "TEAM_TOTAL"
    ]
    .copy()
)


print(
    "\n" + "=" * 110
)

print(
    "TEAM TOTAL PRICE BOARD"
)

print(
    "=" * 110
)

display(
    df_team_total_prices[
        [
            "Game",
            "Sportsbook",
            "Source_Market_Key",
            "Team",
            "Bet_Direction",
            "Line",
            "American_Odds"
        ]
    ]
    .sort_values(
        [
            "Game",
            "Team",
            "Line",
            "Bet_Direction",
            "Sportsbook"
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------
# 12. FIRST-HALF BOARD
# --------------------------------------------------

df_first_half_prices = (
    df_auto_pricing[
        df_auto_pricing[
            "Market_Type"
        ].isin(
            [
                "1H_SPREAD",
                "1H_TEAM_TOTAL",
                "1H_TOTAL"
            ]
        )
    ]
    .copy()
)


print(
    "\n" + "=" * 110
)

print(
    "FIRST-HALF PRICE BOARD"
)

print(
    "=" * 110
)

display(
    df_first_half_prices[
        [
            "Game",
            "Sportsbook",
            "Source_Market_Key",
            "Market_Type",
            "Team",
            "Bet_Direction",
            "Line",
            "American_Odds"
        ]
    ]
    .sort_values(
        [
            "Game",
            "Market_Type",
            "Team",
            "Line",
            "Sportsbook"
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------
# 13. API REQUEST AUDIT
# --------------------------------------------------

print(
    "\n" + "=" * 110
)

print(
    "PRICING API REQUEST AUDIT"
)

print(
    "=" * 110
)

display(
    df_pricing_pull_log
)


print(
    "\n" + "=" * 110
)

print(
    "AUTOMATIC PRICING PULL COMPLETE."
)

print(
    "NO BET SELECTION, EV, OR KELLY "
    "CALCULATIONS PERFORMED."
)

CFB1 AUTOMATIC SPORTSBOOK PRICING PULL

CHECK: UAB Blazers at Memphis Tigers
HTTP 401
{"message":"Usage quota has been reached. See usage plans at https://the-odds-api.com","error_code":"OUT_OF_USAGE_CREDITS","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#out-of-usage-credits"}


CHECK: Air Force Falcons at Northern Illinois Huskies
HTTP 401
{"message":"Usage quota has been reached. See usage plans at https://the-odds-api.com","error_code":"OUT_OF_USAGE_CREDITS","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#out-of-usage-credits"}


CHECK: James Madison Dukes at Georgia Southern Eagles
HTTP 401
{"message":"Usage quota has been reached. See usage plans at https://the-odds-api.com","error_code":"OUT_OF_USAGE_CREDITS","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#out-of-usage-credits"}


PRICING PULL VALIDATION
Pricing rows returned: 5416
Games represented: 11
Sportsbooks represented: ['DraftK

,Game,Sportsbook,Source_Market_Key,Team,Line,American_Odds
0,Ball State Cardinals at Northwestern Wildcats,FanDuel,alternate_spreads,Ball State Cardinals,9.5,1800
1,Ball State Cardinals at Northwestern Wildcats,FanDuel,alternate_spreads,Ball State Cardinals,10.5,1500
2,Ball State Cardinals at Northwestern Wildcats,FanDuel,alternate_spreads,Ball State Cardinals,11.5,1500
3,Ball State Cardinals at Northwestern Wildcats,FanDuel,alternate_spreads,Ball State Cardinals,12.5,1400
4,Ball State Cardinals at Northwestern Wildcats,FanDuel,alternate_spreads,Ball State Cardinals,13.5,1300
...,...,...,...,...,...,...
2101,Virginia Tech Hokies at California Golden Bears,FanDuel,alternate_spreads,Virginia Tech Hokies,12.5,-3500
2102,Virginia Tech Hokies at California Golden Bears,FanDuel,alternate_spreads,Virginia Tech Hokies,13.5,-4000
2103,Virginia Tech Hokies at California Golden Bears,FanDuel,alternate_spreads,Virginia Tech Hokies,14.5,-5000
2104,Virginia Tech Hokies at California Golden Bears,FanDuel,alternate_spreads,Virginia Tech Hokies,15.5,-6000



TEAM TOTAL PRICE BOARD


,Game,Sportsbook,Source_Market_Key,Team,Bet_Direction,Line,American_Odds
0,Ball State Cardinals at Northwestern Wildcats,FanDuel,alternate_team_totals,Ball State Cardinals,OVER,0.5,-590
1,Ball State Cardinals at Northwestern Wildcats,FanDuel,alternate_team_totals,Ball State Cardinals,UNDER,0.5,390
2,Ball State Cardinals at Northwestern Wildcats,FanDuel,alternate_team_totals,Ball State Cardinals,OVER,1.5,-590
3,Ball State Cardinals at Northwestern Wildcats,FanDuel,alternate_team_totals,Ball State Cardinals,UNDER,1.5,390
4,Ball State Cardinals at Northwestern Wildcats,FanDuel,alternate_team_totals,Ball State Cardinals,OVER,2.5,-590
...,...,...,...,...,...,...,...
2601,Virginia Tech Hokies at California Golden Bears,FanDuel,alternate_team_totals,Virginia Tech Hokies,UNDER,49.5,-3500
2602,Virginia Tech Hokies at California Golden Bears,FanDuel,alternate_team_totals,Virginia Tech Hokies,OVER,50.5,1060
2603,Virginia Tech Hokies at California Golden Bears,FanDuel,alternate_team_totals,Virginia Tech Hokies,UNDER,50.5,-3500
2604,Virginia Tech Hokies at California Golden Bears,FanDuel,alternate_team_totals,Virginia Tech Hokies,OVER,51.5,1200



FIRST-HALF PRICE BOARD


,Game,Sportsbook,Source_Market_Key,Market_Type,Team,Bet_Direction,Line,American_Odds
0,Ball State Cardinals at Northwestern Wildcats,DraftKings,alternate_spreads_h1,1H_SPREAD,Ball State Cardinals,SPREAD,15.5,208
1,Ball State Cardinals at Northwestern Wildcats,DraftKings,alternate_spreads_h1,1H_SPREAD,Ball State Cardinals,SPREAD,16.5,196
2,Ball State Cardinals at Northwestern Wildcats,DraftKings,alternate_spreads_h1,1H_SPREAD,Ball State Cardinals,SPREAD,17.5,149
3,Ball State Cardinals at Northwestern Wildcats,DraftKings,alternate_spreads_h1,1H_SPREAD,Ball State Cardinals,SPREAD,18.5,127
4,Ball State Cardinals at Northwestern Wildcats,DraftKings,alternate_spreads_h1,1H_SPREAD,Ball State Cardinals,SPREAD,19.5,122
...,...,...,...,...,...,...,...,...
699,Virginia Tech Hokies at California Golden Bears,DraftKings,alternate_totals_h1,1H_TOTAL,,UNDER,31.5,-283
700,Virginia Tech Hokies at California Golden Bears,DraftKings,alternate_totals_h1,1H_TOTAL,,OVER,32.5,207
701,Virginia Tech Hokies at California Golden Bears,DraftKings,alternate_totals_h1,1H_TOTAL,,UNDER,32.5,-291
702,Virginia Tech Hokies at California Golden Bears,DraftKings,alternate_totals_h1,1H_TOTAL,,OVER,33.5,221



PRICING API REQUEST AUDIT


,Game,HTTP_Status,Requests_Remaining,Requests_Used,Last_Request_Cost
0,Jacksonville State Gamecocks at Kennesaw State...,200,93,407,9
1,Ball State Cardinals at Northwestern Wildcats,200,84,416,9
2,Buffalo Bulls at Toledo Rockets,200,75,425,9
3,Charlotte 49ers at North Texas Mean Green,200,66,434,9
4,Duke Blue Devils at Georgia Tech Yellow Jackets,200,57,443,9
5,Stanford Cardinal at Notre Dame Fighting Irish,200,48,452,9
6,Virginia Tech Hokies at California Golden Bears,200,39,461,9
7,Rice Owls at East Carolina Pirates,200,30,470,9
8,Maryland Terrapins at Ohio State Buckeyes,200,21,479,9
9,Tennessee Volunteers at Arkansas Razorbacks,200,12,488,9



AUTOMATIC PRICING PULL COMPLETE.
NO BET SELECTION, EV, OR KELLY CALCULATIONS PERFORMED.


In [ ]:
# ==========================================
# CELL 55 — SMART FULL-GAME SPREAD FILTER
# ==========================================

import numpy as np
import pandas as pd

print("=" * 110)
print("CFB1 SMART FULL-GAME SPREAD FILTER")
print("=" * 110)


# --------------------------------------------------
# 1. REQUIRED INPUTS
# --------------------------------------------------

if 'df_auto_pricing' not in globals():
    raise NameError(
        "df_auto_pricing missing. Run Cell 54."
    )

if 'df_key_value' not in globals():
    raise NameError(
        "df_key_value missing. Run Cell 48."
    )

if 'df_hist_spread' not in globals():
    raise NameError(
        "df_hist_spread missing. Run Cell 51."
    )


# --------------------------------------------------
# 2. ONLY FULL-GAME SPREADS
# --------------------------------------------------

spread_prices = df_auto_pricing[
    df_auto_pricing[
        'Market_Type'
    ] == 'FULL_GAME_SPREAD'
].copy()


# --------------------------------------------------
# 3. MERGE MODEL INFORMATION
# --------------------------------------------------

model_info = df_key_value[
    [
        'Game',
        'Model_Side',
        'Market_Model_Spread',
        'Retained_Model_Spread',
        'Line_Value_Status',
        'Decision_Status',
        'Eligible_Market_Routes'
    ]
].copy()

spread_prices = spread_prices.merge(
    model_info,
    on='Game',
    how='inner'
)


# --------------------------------------------------
# 4. KEEP ONLY MODEL SIDE
# --------------------------------------------------

spread_prices = spread_prices[
    spread_prices['Team']
    ==
    spread_prices['Model_Side']
].copy()


# --------------------------------------------------
# 5. ONLY GAMES ROUTED TO FULL-GAME SPREAD
# --------------------------------------------------

spread_prices = spread_prices[
    spread_prices[
        'Eligible_Market_Routes'
    ]
    .astype(str)
    .str.contains(
        'FULL_GAME_SPREAD',
        case=False,
        na=False
    )
].copy()


# --------------------------------------------------
# 6. REMOVE EXTREME PRICES
#
# We do not need absurd -1000 / +1500 type
# alternate lines for normal CFB1 evaluation.
# --------------------------------------------------

spread_prices = spread_prices[
    (
        spread_prices['American_Odds'] >= -250
    )
    &
    (
        spread_prices['American_Odds'] <= 300
    )
].copy()


# --------------------------------------------------
# 7. MODEL CUSHION AT EACH ACTUAL LINE
#
# Example:
# retained model = -5.2
#
# -2.5 candidate:
# cushion = 2.7
#
# -3.5 candidate:
# cushion = 1.7
# --------------------------------------------------

spread_prices[
    'Candidate_Cushion'
] = (
    spread_prices['Line']
    -
    spread_prices['Retained_Model_Spread']
)


# --------------------------------------------------
# 8. KEEP LINES WITH SOME MODEL ROOM
#
# Allow down to -0.5 temporarily so we can
# see lines right around fair value.
# --------------------------------------------------

spread_prices = spread_prices[
    spread_prices[
        'Candidate_Cushion'
    ] >= -0.5
].copy()


# --------------------------------------------------
# 9. EMPIRICAL COVER PROBABILITY
# --------------------------------------------------

def spread_bucket(spread_abs):

    if spread_abs < 3:
        return 'UNDER_3'
    elif spread_abs < 7:
        return '3_TO_6.5'
    elif spread_abs < 10:
        return '7_TO_9.5'
    elif spread_abs < 14:
        return '10_TO_13.5'
    elif spread_abs < 17:
        return '14_TO_16.5'
    elif spread_abs < 21:
        return '17_TO_20.5'
    else:
        return '21_PLUS'


def empirical_cover_prob(
    cushion,
    errors
):

    adjusted = (
        errors.astype(float)
        + cushion
    )

    return (
        adjusted > 0
    ).mean()


global_errors = (
    df_hist_spread[
        'Spread_Error'
    ]
    .dropna()
    .astype(float)
)


probabilities = []

for _, row in spread_prices.iterrows():

    cushion = row[
        'Candidate_Cushion'
    ]

    # Because historical errors are home-oriented,
    # flip when model side is away.
    game_row = df_key_value[
        df_key_value['Game']
        ==
        row['Game']
    ].iloc[0]

    model_is_home = (
        row['Model_Side']
        ==
        game_row['Home_Team']
    )

    oriented_global = (
        global_errors
        if model_is_home
        else -global_errors
    )

    global_prob = empirical_cover_prob(
        cushion,
        oriented_global
    )


    bucket = spread_bucket(
        abs(row['Line'])
    )

    bucket_errors = (
        df_hist_spread.loc[
            df_hist_spread[
                'Closing_Home_Spread'
            ]
            .abs()
            .apply(spread_bucket)
            == bucket,
            'Spread_Error'
        ]
        .dropna()
        .astype(float)
    )

    oriented_bucket = (
        bucket_errors
        if model_is_home
        else -bucket_errors
    )

    bucket_prob = empirical_cover_prob(
        cushion,
        oriented_bucket
    )

    conservative_prob = min(
        global_prob,
        bucket_prob
    )

    probabilities.append(
        (
            global_prob,
            bucket_prob,
            conservative_prob
        )
    )


spread_prices[
    [
        'Global_Cover_Prob',
        'Bucket_Cover_Prob',
        'CFB1_Cover_Prob'
    ]
] = probabilities


# --------------------------------------------------
# 10. BREAK-EVEN PROBABILITY
# --------------------------------------------------

def american_break_even(odds):

    odds = float(odds)

    if odds < 0:

        return (
            abs(odds)
            /
            (
                abs(odds) + 100
            )
        )

    else:

        return (
            100
            /
            (
                odds + 100
            )
        )


spread_prices[
    'Break_Even_Prob'
] = (
    spread_prices[
        'American_Odds'
    ].apply(
        american_break_even
    )
)


# --------------------------------------------------
# 11. PROBABILITY EDGE
# --------------------------------------------------

spread_prices[
    'Probability_Edge'
] = (
    spread_prices[
        'CFB1_Cover_Prob'
    ]
    -
    spread_prices[
        'Break_Even_Prob'
    ]
)


# --------------------------------------------------
# 12. KEY-NUMBER TRADEOFF
# --------------------------------------------------

KEYS = [3, 7, 10, 14]


def key_tradeoff(row):

    start = row[
        'Market_Model_Spread'
    ]

    candidate = row[
        'Line'
    ]

    if pd.isna(start) or pd.isna(candidate):
        return ''

    # Primarily relevant when laying points.
    if start >= 0 or candidate >= 0:
        return ''

    start_mag = abs(start)
    cand_mag = abs(candidate)

    # Candidate isn't more aggressive
    if cand_mag <= start_mag:
        return ''

    # Moving OFF a key
    for key in KEYS:

        if (
            abs(start_mag - key) < 0.001
            and cand_mag > key
        ):
            return f'OFF_KEY_{key}'

    # Crossing a key
    crossed = []

    for key in KEYS:

        if (
            start_mag < key
            and cand_mag > key
        ):
            crossed.append(
                str(key)
            )

    if crossed:

        return (
            'CROSSES_'
            +
            '_'.join(crossed)
        )

    # Landing on a key
    for key in KEYS:

        if (
            start_mag < key
            and abs(cand_mag - key) < 0.001
        ):
            return f'LANDS_ON_KEY_{key}'

    return ''


spread_prices[
    'Key_Tradeoff'
] = (
    spread_prices.apply(
        key_tradeoff,
        axis=1
    )
)


# --------------------------------------------------
# 13. KEEP ONLY USEFUL NEIGHBORHOOD
#
# Main line through retained model line,
# plus one half-point around that window.
# --------------------------------------------------

def useful_line(row):

    market = row[
        'Market_Model_Spread'
    ]

    model = row[
        'Retained_Model_Spread'
    ]

    line = row[
        'Line'
    ]

    low = min(
        market,
        model
    ) - 0.5

    high = max(
        market,
        model
    ) + 0.5

    return (
        low <= line <= high
    )


spread_prices = spread_prices[
    spread_prices.apply(
        useful_line,
        axis=1
    )
].copy()


# --------------------------------------------------
# 14. BEST PRICE FOR IDENTICAL LINE
# --------------------------------------------------

spread_prices = (
    spread_prices
    .sort_values(
        'American_Odds',
        ascending=False
    )
    .drop_duplicates(
        subset=[
            'Game',
            'Team',
            'Line'
        ],
        keep='first'
    )
)


# --------------------------------------------------
# 15. REDUCE TO MOST USEFUL LINES PER GAME
#
# Keep:
# - current/main area
# - best probability-edge lines
# - key-number-relevant lines
# --------------------------------------------------

spread_prices[
    'Abs_From_Main'
] = (
    spread_prices['Line']
    -
    spread_prices[
        'Market_Model_Spread'
    ]
).abs()


def choose_shortlist(group):

    # Closest three to current market
    near_main = (
        group
        .sort_values(
            'Abs_From_Main'
        )
        .head(3)
    )

    # Best three by probability edge
    best_edge = (
        group
        .sort_values(
            'Probability_Edge',
            ascending=False
        )
        .head(3)
    )

    # Any key-number line
    key_rows = group[
        group[
            'Key_Tradeoff'
        ] != ''
    ]

    combined = pd.concat(
        [
            near_main,
            best_edge,
            key_rows
        ]
    )

    return (
        combined
        .drop_duplicates()
        .sort_values(
            'Line',
            ascending=False
        )
        .head(6)
    )


df_spread_shortlist = (
    spread_prices
    .groupby(
        'Game',
        group_keys=False
    )
    .apply(
        choose_shortlist
    )
    .reset_index(drop=True)
)


# --------------------------------------------------
# 16. DISPLAY VERSION
# --------------------------------------------------

display_df = (
    df_spread_shortlist.copy()
)

for col in [
    'CFB1_Cover_Prob',
    'Break_Even_Prob',
    'Probability_Edge'
]:

    display_df[col] = (
        display_df[col]
        * 100
    ).round(1)


print(
    "\n" + "=" * 110
)

print(
    "CFB1 FULL-GAME SPREAD SHOPPING LIST"
)

print(
    "=" * 110
)

display(
    display_df[
        [
            'Game',
            'Team',
            'Sportsbook',
            'Line',
            'American_Odds',
            'Candidate_Cushion',
            'CFB1_Cover_Prob',
            'Break_Even_Prob',
            'Probability_Edge',
            'Key_Tradeoff',
            'Decision_Status'
        ]
    ]
    .round(2)
)


# --------------------------------------------------
# 17. SUMMARY
# --------------------------------------------------

print(
    "\nRows in raw pricing pull:",
    len(df_auto_pricing)
)

print(
    "Rows in final spread shopping list:",
    len(df_spread_shortlist)
)

print(
    "\nSMART FILTER COMPLETE — "
    "NO BETS OR KELLY STAKES SELECTED."
)

CFB1 SMART FULL-GAME SPREAD FILTER

CFB1 FULL-GAME SPREAD SHOPPING LIST


/tmp/ipykernel_390/799442248.py:559: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,Game,Team,Sportsbook,Line,American_Odds,Candidate_Cushion,CFB1_Cover_Prob,Break_Even_Prob,Probability_Edge,Key_Tradeoff,Decision_Status
0,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,DraftKings,-30.5,-242,5.62,65.0,70.8,-5.8,CROSSES_10_14,CONFIRMED_BUT_LARGE_SPREAD
1,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,DraftKings,-31.5,-200,4.62,62.9,66.7,-3.7,CROSSES_10_14,CONFIRMED_BUT_LARGE_SPREAD
2,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,DraftKings,-32.5,-181,3.62,60.4,64.4,-4.0,CROSSES_10_14,CONFIRMED_BUT_LARGE_SPREAD
3,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,DraftKings,-33.5,-170,2.62,57.7,63.0,-5.2,CROSSES_10_14,CONFIRMED_BUT_LARGE_SPREAD
4,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,DraftKings,-34.5,-148,1.62,55.6,59.7,-4.1,CROSSES_10_14,CONFIRMED_BUT_LARGE_SPREAD
5,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,DraftKings,-35.5,-126,0.62,52.6,55.8,-3.1,CROSSES_10_14,CONFIRMED_BUT_LARGE_SPREAD
6,Buffalo Bulls at Toledo Rockets,Toledo Rockets,FanDuel,-14.5,-215,6.46,61.6,68.3,-6.6,OFF_KEY_10,NEUTRAL_MARKET
7,Buffalo Bulls at Toledo Rockets,Toledo Rockets,FanDuel,-15.5,-200,5.46,58.8,66.7,-7.8,OFF_KEY_10,NEUTRAL_MARKET
8,Buffalo Bulls at Toledo Rockets,Toledo Rockets,FanDuel,-16.5,-186,4.46,57.0,65.0,-8.1,OFF_KEY_10,NEUTRAL_MARKET
9,Buffalo Bulls at Toledo Rockets,Toledo Rockets,FanDuel,-17.5,-162,3.46,55.9,61.8,-5.9,OFF_KEY_10,NEUTRAL_MARKET



Rows in raw pricing pull: 5416
Rows in final spread shopping list: 47

SMART FILTER COMPLETE — NO BETS OR KELLY STAKES SELECTED.


In [ ]:
# ==========================================
# CELL 56 — HISTORICAL KEY-NUMBER TAX AUDIT
# ==========================================

import numpy as np
import pandas as pd

print("=" * 108)
print("CFB1 HISTORICAL KEY-NUMBER TAX AUDIT")
print("=" * 108)


# --------------------------------------------------
# 1. REQUIRED INPUT
# --------------------------------------------------

if 'df_hist_spread' not in globals():
    raise NameError(
        "df_hist_spread does not exist. "
        "Run Cell 51 first."
    )


# --------------------------------------------------
# 2. BUILD FAVORITE-ORIENTED HISTORICAL DATA
#
# Closing_Home_Spread < 0
#     home team was favorite
#
# Closing_Home_Spread > 0
#     away team was favorite
#
# We convert every game into:
#
# Favorite_Closing_Spread = positive magnitude
# Favorite_Final_Margin   = points favorite won/lost by
#
# Example:
# home -3, wins by 7  -> Favorite_Final_Margin = +7
# away -3, wins by 3  -> Favorite_Final_Margin = +3
# favorite loses by 4 -> Favorite_Final_Margin = -4
# --------------------------------------------------

df_key_hist = df_hist_spread.copy()

# Remove pick'em games because there is no
# meaningful favorite orientation.
df_key_hist = df_key_hist[
    df_key_hist['Closing_Home_Spread'] != 0
].copy()


df_key_hist[
    'Favorite_Closing_Spread'
] = (
    df_key_hist[
        'Closing_Home_Spread'
    ].abs()
)


df_key_hist[
    'Favorite_Final_Margin'
] = np.where(

    df_key_hist[
        'Closing_Home_Spread'
    ] < 0,

    # Home favorite
    df_key_hist[
        'Actual_Home_Margin'
    ],

    # Away favorite
    -df_key_hist[
        'Actual_Home_Margin'
    ]
)


# --------------------------------------------------
# 3. KEY NUMBERS
# --------------------------------------------------

KEY_NUMBERS = [
    3,
    7,
    10,
    14
]


# --------------------------------------------------
# 4. GLOBAL LANDING FREQUENCY
#
# How often did the closing favorite win
# by exactly each key number?
# --------------------------------------------------

global_rows = []

total_games = len(
    df_key_hist
)

for key in KEY_NUMBERS:

    exact_count = (
        df_key_hist[
            'Favorite_Final_Margin'
        ] == key
    ).sum()

    exact_rate = (
        exact_count
        /
        total_games
    )

    global_rows.append({

        'Key_Number':
            key,

        'Historical_Games':
            total_games,

        'Exact_Key_Landings':
            exact_count,

        'Global_Key_Landing_Rate':
            exact_rate
    })


df_key_global = pd.DataFrame(
    global_rows
)


# --------------------------------------------------
# 5. LOCAL KEY-NUMBER SAMPLE
#
# More useful than global rate alone:
#
# Compare games whose closing favorite spread
# was reasonably close to the key.
#
# Example for key 3:
# closing favorite spread between 1 and 5.
#
# This gives us a context closer to a decision
# such as -2.5 vs -3.5.
# --------------------------------------------------

LOCAL_WINDOW = 2.0

local_rows = []

for key in KEY_NUMBERS:

    lower = max(
        0,
        key - LOCAL_WINDOW
    )

    upper = (
        key + LOCAL_WINDOW
    )

    local_sample = df_key_hist[
        (
            df_key_hist[
                'Favorite_Closing_Spread'
            ] >= lower
        )
        &
        (
            df_key_hist[
                'Favorite_Closing_Spread'
            ] <= upper
        )
    ].copy()


    local_n = len(
        local_sample
    )

    exact_count = (
        local_sample[
            'Favorite_Final_Margin'
        ] == key
    ).sum()


    if local_n > 0:

        exact_rate = (
            exact_count
            /
            local_n
        )

    else:

        exact_rate = np.nan


    local_rows.append({

        'Key_Number':
            key,

        'Local_Spread_Low':
            lower,

        'Local_Spread_High':
            upper,

        'Local_Historical_Games':
            local_n,

        'Local_Key_Landings':
            exact_count,

        'Local_Key_Landing_Rate':
            exact_rate
    })


df_key_local = pd.DataFrame(
    local_rows
)


# --------------------------------------------------
# 6. COMBINE GLOBAL + LOCAL
# --------------------------------------------------

df_key_tax = df_key_global.merge(
    df_key_local,
    on='Key_Number',
    how='left',
    validate='one_to_one'
)


# --------------------------------------------------
# 7. CONSERVATIVE V1 KEY TAX
#
# We use the HIGHER of:
# - global exact landing frequency
# - local landing frequency
#
# Reason:
# when deciding whether to give up a key number,
# we do not want to understate its cost.
# --------------------------------------------------

df_key_tax[
    'CFB1_Key_Tax_V1'
] = (
    df_key_tax[
        [
            'Global_Key_Landing_Rate',
            'Local_Key_Landing_Rate'
        ]
    ]
    .max(axis=1)
)


# --------------------------------------------------
# 8. DISPLAY KEY TAX TABLE
# --------------------------------------------------

display_key_tax = (
    df_key_tax.copy()
)

for col in [
    'Global_Key_Landing_Rate',
    'Local_Key_Landing_Rate',
    'CFB1_Key_Tax_V1'
]:

    display_key_tax[col] = (
        display_key_tax[col]
        * 100
    ).round(2)


print(
    "\n" + "=" * 108
)

print(
    "HISTORICAL KEY-NUMBER LANDING RATES"
)

print(
    "=" * 108
)

display(
    display_key_tax[
        [
            'Key_Number',
            'Historical_Games',
            'Exact_Key_Landings',
            'Global_Key_Landing_Rate',
            'Local_Spread_Low',
            'Local_Spread_High',
            'Local_Historical_Games',
            'Local_Key_Landings',
            'Local_Key_Landing_Rate',
            'CFB1_Key_Tax_V1'
        ]
    ]
)


# --------------------------------------------------
# 9. EXACT FAVORITE MARGIN DISTRIBUTION
#
# Useful sanity check:
# show most common winning margins.
# --------------------------------------------------

winning_favorite_margins = (
    df_key_hist[
        df_key_hist[
            'Favorite_Final_Margin'
        ] > 0
    ][
        'Favorite_Final_Margin'
    ]
    .value_counts()
    .sort_values(
        ascending=False
    )
    .head(20)
)


print(
    "\n" + "=" * 108
)

print(
    "MOST COMMON FAVORITE WINNING MARGINS"
)

print(
    "=" * 108
)

print(
    winning_favorite_margins
)


# --------------------------------------------------
# 10. CONNECT TO CELL 55
#
# Determine whether each candidate alternate
# crosses or moves off a key and attach the
# historical key tax.
# --------------------------------------------------

if 'df_spread_shortlist' in globals():

    key_tax_lookup = dict(
        zip(
            df_key_tax[
                'Key_Number'
            ],

            df_key_tax[
                'CFB1_Key_Tax_V1'
            ]
        )
    )


    def extract_key_tax(
        tradeoff
    ):

        if pd.isna(tradeoff):
            return 0.0

        text = str(
            tradeoff
        )

        if text == '':
            return 0.0


        taxes = []


        # ------------------------------------------
        # Moving off a key
        # Example:
        # OFF_KEY_10
        # ------------------------------------------

        if text.startswith(
            'OFF_KEY_'
        ):

            try:

                key = int(
                    text.split('_')[-1]
                )

                taxes.append(
                    key_tax_lookup.get(
                        key,
                        0.0
                    )
                )

            except:
                pass


        # ------------------------------------------
        # Crossing one or more keys
        # Example:
        # CROSSES_3
        # CROSSES_3_7
        # ------------------------------------------

        elif text.startswith(
            'CROSSES_'
        ):

            parts = (
                text
                .replace(
                    'CROSSES_',
                    ''
                )
                .split('_')
            )

            for part in parts:

                try:

                    key = int(
                        part
                    )

                    taxes.append(
                        key_tax_lookup.get(
                            key,
                            0.0
                        )
                    )

                except:
                    pass


        # ------------------------------------------
        # Landing ON a key is not treated as
        # giving up the key.
        #
        # It receives zero tax here.
        # ------------------------------------------

        elif text.startswith(
            'LANDS_ON_KEY_'
        ):

            return 0.0


        return sum(
            taxes
        )


    df_spread_shortlist[
        'Historical_Key_Tax'
    ] = (
        df_spread_shortlist[
            'Key_Tradeoff'
        ].apply(
            extract_key_tax
        )
    )


    # ----------------------------------------------
    # Key-adjusted probability
    #
    # IMPORTANT:
    # This is a diagnostic adjustment.
    #
    # Example:
    # generic model probability = 55%
    # historical key tax = 4%
    #
    # adjusted probability = 51%
    # ----------------------------------------------

    df_spread_shortlist[
        'Key_Adjusted_Cover_Prob'
    ] = (
        df_spread_shortlist[
            'CFB1_Cover_Prob'
        ]
        -
        df_spread_shortlist[
            'Historical_Key_Tax'
        ]
    ).clip(
        lower=0,
        upper=1
    )


    # ----------------------------------------------
    # DISPLAY
    # ----------------------------------------------

    key_examples = (
        df_spread_shortlist[
            df_spread_shortlist[
                'Key_Tradeoff'
            ] != ''
        ]
        .copy()
    )


    if len(
        key_examples
    ) > 0:

        key_display = (
            key_examples.copy()
        )

        for col in [
            'CFB1_Cover_Prob',
            'Historical_Key_Tax',
            'Key_Adjusted_Cover_Prob'
        ]:

            key_display[col] = (
                key_display[col]
                * 100
            ).round(2)


        print(
            "\n" + "=" * 108
        )

        print(
            "CURRENT CFB1 KEY-NUMBER TRADEOFFS"
        )

        print(
            "=" * 108
        )

        display(
            key_display[
                [
                    'Game',
                    'Team',
                    'Sportsbook',
                    'Line',
                    'American_Odds',
                    'Key_Tradeoff',
                    'CFB1_Cover_Prob',
                    'Historical_Key_Tax',
                    'Key_Adjusted_Cover_Prob'
                ]
            ]
        )

    else:

        print(
            "\nNo current Cell 55 rows "
            "cross or move off a key."
        )


# --------------------------------------------------
# 11. VALIDATION
# --------------------------------------------------

print(
    "\n" + "=" * 108
)

print(
    "KEY TAX VALIDATION"
)

print(
    "=" * 108
)

print(
    f"Historical favorite-oriented games: "
    f"{len(df_key_hist)}"
)

print(
    f"Key numbers evaluated: "
    f"{len(df_key_tax)}"
)

print(
    f"Missing V1 key taxes: "
    f"{df_key_tax['CFB1_Key_Tax_V1'].isna().sum()}"
)


print(
    "\n" + "=" * 108
)

print(
    "HISTORICAL KEY-NUMBER TAX AUDIT COMPLETE."
)

print(
    "NO BETS OR KELLY STAKES SELECTED."
)

CFB1 HISTORICAL KEY-NUMBER TAX AUDIT

HISTORICAL KEY-NUMBER LANDING RATES


,Key_Number,Historical_Games,Exact_Key_Landings,Global_Key_Landing_Rate,Local_Spread_Low,Local_Spread_High,Local_Historical_Games,Local_Key_Landings,Local_Key_Landing_Rate,CFB1_Key_Tax_V1
0,3,3720,214,5.75,1.0,5.0,1195,98,8.20,8.20
1,7,3720,199,5.35,5.0,9.0,826,60,7.26,7.26
2,10,3720,126,3.39,8.0,12.0,534,28,5.24,5.24
3,14,3720,114,3.06,12.0,16.0,483,16,3.31,3.31



MOST COMMON FAVORITE WINNING MARGINS
Favorite_Final_Margin
3     214
7     199
10    126
17    124
14    114
21    114
4      87
24     84
28     80
18     73
6      73
8      71
2      70
1      67
31     66
11     56
25     54
20     54
35     53
5      51
Name: count, dtype: int64

CURRENT CFB1 KEY-NUMBER TRADEOFFS


,Game,Team,Sportsbook,Line,American_Odds,Key_Tradeoff,CFB1_Cover_Prob,Historical_Key_Tax,Key_Adjusted_Cover_Prob
0,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,DraftKings,-30.5,-242,CROSSES_10_14,64.96,8.56,56.40
1,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,DraftKings,-31.5,-200,CROSSES_10_14,62.92,8.56,54.37
2,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,DraftKings,-32.5,-181,CROSSES_10_14,60.40,8.56,51.85
3,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,DraftKings,-33.5,-170,CROSSES_10_14,57.75,8.56,49.19
4,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,DraftKings,-34.5,-148,CROSSES_10_14,55.60,8.56,47.05
5,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,DraftKings,-35.5,-126,CROSSES_10_14,52.63,8.56,44.07
6,Buffalo Bulls at Toledo Rockets,Toledo Rockets,FanDuel,-14.5,-215,OFF_KEY_10,61.61,5.24,56.37
7,Buffalo Bulls at Toledo Rockets,Toledo Rockets,FanDuel,-15.5,-200,OFF_KEY_10,58.82,5.24,53.58
8,Buffalo Bulls at Toledo Rockets,Toledo Rockets,FanDuel,-16.5,-186,OFF_KEY_10,56.97,5.24,51.72
9,Buffalo Bulls at Toledo Rockets,Toledo Rockets,FanDuel,-17.5,-162,OFF_KEY_10,55.90,5.24,50.66



KEY TAX VALIDATION
Historical favorite-oriented games: 3720
Key numbers evaluated: 4
Missing V1 key taxes: 0

HISTORICAL KEY-NUMBER TAX AUDIT COMPLETE.
NO BETS OR KELLY STAKES SELECTED.


In [ ]:
# ==========================================
# CELL 57 — FULL-GAME SPREAD EV / PRICING
# ==========================================

import numpy as np
import pandas as pd

print("=" * 115)
print("CFB1 FULL-GAME SPREAD EV / PRICING")
print("=" * 115)


# --------------------------------------------------
# 1. REQUIRED OBJECT
# --------------------------------------------------

if "df_spread_shortlist" not in globals():
    raise NameError(
        "df_spread_shortlist missing. "
        "Run through old Cell 55 first."
    )


df_ev = df_spread_shortlist.copy()


# --------------------------------------------------
# 2. REQUIRED INPUT COLUMNS
# --------------------------------------------------

required_cols = [
    "Game",
    "Team",
    "Sportsbook",
    "Line",
    "American_Odds",
    "Candidate_Cushion"
]

missing = [
    col for col in required_cols
    if col not in df_ev.columns
]

if missing:
    raise KeyError(
        "Old Cell 57 is missing required columns: "
        + ", ".join(missing)
    )


# --------------------------------------------------
# 3. AMERICAN ODDS HELPERS
# --------------------------------------------------

def profit_multiple(odds):

    odds = float(odds)

    if odds > 0:
        return odds / 100.0

    return 100.0 / abs(odds)


def break_even_prob(odds):

    odds = float(odds)

    if odds < 0:
        return (
            abs(odds)
            /
            (abs(odds) + 100.0)
        )

    return (
        100.0
        /
        (odds + 100.0)
    )


df_ev["Profit_Multiple"] = (
    df_ev["American_Odds"]
    .apply(profit_multiple)
)

df_ev["Break_Even_Prob"] = (
    df_ev["American_Odds"]
    .apply(break_even_prob)
)


# --------------------------------------------------
# 4. EMPIRICAL CANDIDATE-LINE PROBABILITIES
#
# Preserve the existing CFB1 probability method.
# Candidate_Cushion is evaluated against the
# historical spread-error distributions created
# upstream.
# --------------------------------------------------

if "global_errors" not in globals():
    raise NameError(
        "global_errors missing. "
        "Run the historical spread probability cells first."
    )


def _model_side_is_home(row):

    if "Home_Team" not in row.index:
        raise KeyError(
            "Home_Team missing from df_spread_shortlist."
        )

    return (
        str(row["Team"])
        ==
        str(row["Home_Team"])
    )


def _probability_triplet(
    cushion,
    model_is_home,
    errors
):

    if pd.isna(cushion):
        return np.nan, np.nan, np.nan

    oriented = (
        errors.astype(float)
        if model_is_home
        else -errors.astype(float)
    )

    adjusted = (
        oriented
        + float(cushion)
    )

    win_prob = (
        adjusted > 0
    ).mean()

    push_prob = (
        adjusted == 0
    ).mean()

    loss_prob = (
        adjusted < 0
    ).mean()

    return (
        float(win_prob),
        float(push_prob),
        float(loss_prob)
    )


global_wins = []
global_pushes = []
global_losses = []

bucket_wins = []
bucket_pushes = []
bucket_losses = []

probability_sources = []


for _, row in df_ev.iterrows():

    cushion = row["Candidate_Cushion"]

    model_is_home = (
        _model_side_is_home(row)
    )

    gw, gp, gl = (
        _probability_triplet(
            cushion,
            model_is_home,
            global_errors
        )
    )

    global_wins.append(gw)
    global_pushes.append(gp)
    global_losses.append(gl)


    # ----------------------------------------------
    # Use spread-bucket history when available and
    # sufficiently populated; otherwise use global.
    # ----------------------------------------------

    bucket_series = None

    if (
        "Spread_Bucket" in row.index
        and
        pd.notna(row["Spread_Bucket"])
        and
        "df_hist_spread" in globals()
        and
        "Spread_Bucket" in df_hist_spread.columns
    ):

        bucket_series = (
            df_hist_spread.loc[
                df_hist_spread["Spread_Bucket"]
                ==
                row["Spread_Bucket"],
                "Spread_Error"
            ]
            if "Spread_Error" in df_hist_spread.columns
            else None
        )


    if (
        bucket_series is not None
        and
        len(bucket_series) > 0
    ):

        bw, bp, bl = (
            _probability_triplet(
                cushion,
                model_is_home,
                bucket_series
            )
        )

    else:

        bw, bp, bl = (
            gw,
            gp,
            gl
        )


    bucket_wins.append(bw)
    bucket_pushes.append(bp)
    bucket_losses.append(bl)


    historical_n = (
        row["Historical_Bucket_N"]
        if "Historical_Bucket_N" in row.index
        else np.nan
    )


    # Existing CFB1 rule:
    # use spread-bucket probability when the
    # historical bucket is sufficiently populated;
    # otherwise fall back to global history.
    if (
        pd.notna(historical_n)
        and
        float(historical_n) >= 300
        and
        bucket_series is not None
        and
        len(bucket_series) > 0
    ):

        probability_sources.append(
            "SPREAD_BUCKET"
        )

    else:

        probability_sources.append(
            "GLOBAL"
        )


df_ev["Global_Win_Prob"] = global_wins
df_ev["Global_Push_Prob"] = global_pushes
df_ev["Global_Loss_Prob"] = global_losses

df_ev["Bucket_Win_Prob"] = bucket_wins
df_ev["Bucket_Push_Prob"] = bucket_pushes
df_ev["Bucket_Loss_Prob"] = bucket_losses

df_ev["Probability_Source"] = (
    probability_sources
)


df_ev["CFB1_Win_Prob"] = np.where(
    df_ev["Probability_Source"]
    ==
    "SPREAD_BUCKET",
    df_ev["Bucket_Win_Prob"],
    df_ev["Global_Win_Prob"]
)

df_ev["CFB1_Push_Prob"] = np.where(
    df_ev["Probability_Source"]
    ==
    "SPREAD_BUCKET",
    df_ev["Bucket_Push_Prob"],
    df_ev["Global_Push_Prob"]
)

df_ev["CFB1_Loss_Prob"] = np.where(
    df_ev["Probability_Source"]
    ==
    "SPREAD_BUCKET",
    df_ev["Bucket_Loss_Prob"],
    df_ev["Global_Loss_Prob"]
)


# Backward-compatible probability field
df_ev["CFB1_Cover_Prob"] = (
    df_ev["CFB1_Win_Prob"]
)


# --------------------------------------------------
# 5. PROBABILITY EDGE
# --------------------------------------------------

df_ev["Probability_Edge"] = (
    df_ev["CFB1_Win_Prob"]
    -
    df_ev["Break_Even_Prob"]
)


# --------------------------------------------------
# 6. EXPECTED VALUE / EXPECTED ROI
#
# Push returns stake:
#
# EV per $1 =
#   P(win) * profit multiple
#   - P(loss)
#
# Cell 57A will perform the dedicated integer-line
# push adjustment.
# --------------------------------------------------

df_ev["EV_Per_Dollar"] = (
    df_ev["CFB1_Win_Prob"]
    *
    df_ev["Profit_Multiple"]
    -
    df_ev["CFB1_Loss_Prob"]
)

df_ev["Expected_ROI"] = (
    df_ev["EV_Per_Dollar"]
)


# --------------------------------------------------
# 7. EV STATUS
# --------------------------------------------------

def ev_status(row):

    roi = row["Expected_ROI"]

    if pd.isna(roi):
        return "NO_EV"

    if roi >= 0.05:
        return "STRONG_PRICE_EDGE"

    if roi >= 0.03:
        return "POSITIVE_EV"

    if roi > 0:
        return "THIN_POSITIVE_EV"

    return "NON_POSITIVE_EV"


df_ev["EV_Status"] = (
    df_ev.apply(
        ev_status,
        axis=1
    )
)


# --------------------------------------------------
# 8. DECISION WARNING
#
# Preserve key-number and market-status warnings.
# --------------------------------------------------

def build_warning(row):

    warnings = []

    if (
        "Key_Tradeoff" in row.index
        and
        pd.notna(row["Key_Tradeoff"])
        and
        str(row["Key_Tradeoff"]).strip() != ""
    ):
        warnings.append(
            str(row["Key_Tradeoff"])
        )

    if (
        "Decision_Status" in row.index
        and
        row["Decision_Status"]
        in [
            "MARKET_CONFLICT",
            "STRONG_MARKET_CONFLICT"
        ]
    ):
        warnings.append(
            str(row["Decision_Status"])
        )

    return " | ".join(warnings)


df_ev["Decision_Warning"] = (
    df_ev.apply(
        build_warning,
        axis=1
    )
)


# --------------------------------------------------
# 9. REMOVE LEGACY STAKING FIELDS
# --------------------------------------------------

legacy_staking_cols = [
    "Full_Kelly",
    "Quarter_Kelly",
    "Quarter_Kelly_Stake",
    "Push_Adjusted_Full_Kelly",
    "Push_Adjusted_Quarter_Kelly",
    "Push_Adjusted_Stake"
]

df_ev.drop(
    columns=[
        col for col in legacy_staking_cols
        if col in df_ev.columns
    ],
    inplace=True
)


# --------------------------------------------------
# 10. DISPLAY
# --------------------------------------------------

display_cols = [
    "Game",
    "Team",
    "Sportsbook",
    "Line",
    "American_Odds",
    "Candidate_Cushion",
    "Probability_Source",
    "CFB1_Win_Prob",
    "CFB1_Push_Prob",
    "CFB1_Loss_Prob",
    "Break_Even_Prob",
    "Expected_ROI",
    "Key_Tradeoff",
    "Decision_Status",
    "Decision_Warning",
    "EV_Status"
]

display_cols = [
    col for col in display_cols
    if col in df_ev.columns
]


display_df = (
    df_ev[
        display_cols
    ]
    .sort_values(
        "Expected_ROI",
        ascending=False
    )
    .reset_index(drop=True)
    .copy()
)


for col in [
    "CFB1_Win_Prob",
    "CFB1_Push_Prob",
    "CFB1_Loss_Prob",
    "Break_Even_Prob",
    "Expected_ROI"
]:
    if col in display_df.columns:
        display_df[col] = (
            display_df[col]
            * 100
        ).round(2)


print("\n" + "=" * 115)
print("CFB1 FULL-GAME SPREAD EV BOARD")
print("=" * 115)

display(display_df)


# --------------------------------------------------
# 11. VALIDATION
# --------------------------------------------------

prob_sum = (
    df_ev["CFB1_Win_Prob"]
    +
    df_ev["CFB1_Push_Prob"]
    +
    df_ev["CFB1_Loss_Prob"]
)

prob_errors = (
    ~np.isclose(
        prob_sum,
        1.0,
        atol=1e-9
    )
).sum()


legacy_remaining = [
    col for col in legacy_staking_cols
    if col in df_ev.columns
]


print("\n" + "=" * 115)
print("CELL 57 VALIDATION")
print("=" * 115)

print(
    "Rows evaluated:",
    len(df_ev)
)

print(
    "Probability rows not summing to 1:",
    int(prob_errors)
)

print(
    "Positive-EV rows:",
    int(
        (df_ev["Expected_ROI"] > 0).sum()
    )
)

print(
    "Negative/zero-EV rows:",
    int(
        (df_ev["Expected_ROI"] <= 0).sum()
    )
)

print(
    "Legacy Kelly/stake columns remaining:",
    legacy_remaining
)


if prob_errors != 0:
    raise ValueError(
        "Probability validation failed."
    )

if legacy_remaining:
    raise ValueError(
        "Legacy Kelly/stake columns remain in df_ev."
    )


print("\nCELL 57 COMPLETE.")
print(
    "CFB1 PRICES FULL-GAME SPREADS WITH "
    "PROBABILITY + BREAK-EVEN + EXPECTED ROI."
)
print(
    "BANKROLL, KELLY, AND STAKE SIZING ARE "
    "NO LONGER PART OF CFB1."
)

CFB1 FULL-GAME SPREAD EV / PRICING

CFB1 FULL-GAME SPREAD EV BOARD


,Game,Team,Sportsbook,Line,American_Odds,Candidate_Cushion,Probability_Source,CFB1_Win_Prob,CFB1_Push_Prob,CFB1_Loss_Prob,Break_Even_Prob,Expected_ROI,Key_Tradeoff,Decision_Status,Decision_Warning,EV_Status
0,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,FanDuel,-39.5,112,1.147840,GLOBAL,54.08,0.0,45.92,47.17,14.64,,CONFIRMED_BUT_LARGE_SPREAD,,STRONG_PRICE_EDGE
1,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,FanDuel,-40.5,120,0.147840,GLOBAL,51.42,0.0,48.58,45.45,13.13,,CONFIRMED_BUT_LARGE_SPREAD,,STRONG_PRICE_EDGE
2,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,FanDuel,-38.5,100,2.147840,GLOBAL,56.49,0.0,43.51,50.00,12.98,,CONFIRMED_BUT_LARGE_SPREAD,,STRONG_PRICE_EDGE
3,Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,FanDuel,-8.5,-114,3.346002,GLOBAL,58.82,0.0,41.18,53.27,10.42,,CONFIRMED,,STRONG_PRICE_EDGE
4,Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,DraftKings,-10.5,106,1.346002,GLOBAL,53.49,0.0,46.51,48.54,10.18,,CONFIRMED,,STRONG_PRICE_EDGE
5,Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,DraftKings,-11.5,116,0.346002,GLOBAL,50.43,0.0,49.57,46.30,8.93,,CONFIRMED,,STRONG_PRICE_EDGE
6,Rice Owls at East Carolina Pirates,East Carolina Pirates,FanDuel,-10.5,-102,1.049153,GLOBAL,54.08,0.0,45.92,50.50,7.09,CROSSES_10,CONFIRMED,CROSSES_10,STRONG_PRICE_EDGE
7,Rice Owls at East Carolina Pirates,East Carolina Pirates,DraftKings,-10.0,-108,1.549153,GLOBAL,55.60,0.0,44.40,51.92,7.09,LANDS_ON_KEY_10,CONFIRMED,LANDS_ON_KEY_10,STRONG_PRICE_EDGE
8,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,DraftKings,-7.5,105,0.627997,GLOBAL,52.23,0.0,47.77,48.78,7.06,,STRONG_MARKET_CONFLICT,STRONG_MARKET_CONFLICT,STRONG_PRICE_EDGE
9,Rice Owls at East Carolina Pirates,East Carolina Pirates,FanDuel,-11.5,108,0.049153,GLOBAL,51.42,0.0,48.58,48.08,6.96,CROSSES_10,CONFIRMED,CROSSES_10,STRONG_PRICE_EDGE



CELL 57 VALIDATION
Rows evaluated: 47
Probability rows not summing to 1: 0
Positive-EV rows: 21
Negative/zero-EV rows: 26
Legacy Kelly/stake columns remaining: []

CELL 57 COMPLETE.
CFB1 PRICES FULL-GAME SPREADS WITH PROBABILITY + BREAK-EVEN + EXPECTED ROI.
BANKROLL, KELLY, AND STAKE SIZING ARE NO LONGER PART OF CFB1.


In [ ]:
# ==========================================
# CELL 57A — INTEGER-LINE PUSH ADJUSTMENT
# ==========================================

import numpy as np
import pandas as pd

print("=" * 115)
print("CFB1 INTEGER-LINE PUSH ADJUSTMENT")
print("=" * 115)


# --------------------------------------------------
# 1. REQUIRED OBJECTS
# --------------------------------------------------

required_objects = [
    "df_ev",
    "df_hist_spread"
]

for obj in required_objects:
    if obj not in globals():
        raise NameError(
            f"{obj} missing. Run the earlier CFB1 cells first."
        )


# --------------------------------------------------
# 2. PUSH WINDOW
# --------------------------------------------------

PUSH_WINDOW = 1.0

print(
    f"\nHistorical push window: "
    f"±{PUSH_WINDOW:.1f} point around candidate spread"
)


# --------------------------------------------------
# 3. INTEGER-LINE HELPER
# --------------------------------------------------

def is_integer_line(value):

    if pd.isna(value):
        return False

    return float(value).is_integer()


# --------------------------------------------------
# 4. PUSH ESTIMATOR
#
# Historical push probability is estimated only
# for integer candidate spreads.
#
# Landing-rate history remains contextual only.
# It is not separately subtracted as a second
# key-number tax.
# --------------------------------------------------

def estimate_push_probability(
    candidate_line,
    model_side_is_home
):

    if not is_integer_line(
        candidate_line
    ):
        return 0.0, 0


    target_margin = abs(
        int(
            round(
                float(candidate_line)
            )
        )
    )


    if "Closing_Home_Spread" not in df_hist_spread.columns:
        raise KeyError(
            "df_hist_spread is missing Closing_Home_Spread."
        )

    if "Actual_Home_Margin" not in df_hist_spread.columns:
        raise KeyError(
            "df_hist_spread is missing Actual_Home_Margin."
        )


    spread_mag = (
        df_hist_spread[
            "Closing_Home_Spread"
        ]
        .astype(float)
        .abs()
    )


    comparable = df_hist_spread[
        (
            spread_mag
            >=
            target_margin - PUSH_WINDOW
        )
        &
        (
            spread_mag
            <=
            target_margin + PUSH_WINDOW
        )
    ].copy()


    if len(comparable) == 0:
        return 0.0, 0


    if model_side_is_home:

        pushes = (
            comparable[
                "Actual_Home_Margin"
            ]
            ==
            target_margin
        ).sum()

    else:

        pushes = (
            comparable[
                "Actual_Home_Margin"
            ]
            ==
            -target_margin
        ).sum()


    push_prob = (
        pushes
        /
        len(comparable)
    )


    return (
        float(push_prob),
        int(len(comparable))
    )


# --------------------------------------------------
# 5. IDENTIFY MODEL-SIDE ORIENTATION
# --------------------------------------------------

def model_side_is_home(row):

    return (
        str(row["Team"])
        ==
        str(row["Home_Team"])
    )


# --------------------------------------------------
# 6. CALCULATE HISTORICAL PUSH PROBABILITY
# --------------------------------------------------

historical_push_probs = []
historical_push_ns = []


for _, row in df_ev.iterrows():

    push_prob, push_n = (
        estimate_push_probability(
            row["Line"],
            model_side_is_home(row)
        )
    )

    historical_push_probs.append(
        push_prob
    )

    historical_push_ns.append(
        push_n
    )


df_ev["Historical_Push_Prob"] = (
    historical_push_probs
)

df_ev["Historical_Push_N"] = (
    historical_push_ns
)


# --------------------------------------------------
# 7. PUSH-ADJUSTED PROBABILITIES
#
# Preserve the directional win/loss relationship
# from Cell 57, then carve the historical push
# probability out proportionally.
#
# Half-point lines remain zero-push.
# --------------------------------------------------

base_nonpush = (
    df_ev["CFB1_Win_Prob"]
    +
    df_ev["CFB1_Loss_Prob"]
)


win_share = np.where(
    base_nonpush > 0,
    df_ev["CFB1_Win_Prob"]
    /
    base_nonpush,
    np.nan
)

loss_share = np.where(
    base_nonpush > 0,
    df_ev["CFB1_Loss_Prob"]
    /
    base_nonpush,
    np.nan
)


remaining_nonpush = (
    1.0
    -
    df_ev["Historical_Push_Prob"]
)


df_ev["Push_Adjusted_Win_Prob"] = (
    win_share
    *
    remaining_nonpush
)

df_ev["Push_Adjusted_Push_Prob"] = (
    df_ev["Historical_Push_Prob"]
)

df_ev["Push_Adjusted_Loss_Prob"] = (
    loss_share
    *
    remaining_nonpush
)


# --------------------------------------------------
# 8. PUSH-ADJUSTED EV / ROI
#
# Push returns stake, so:
#
# EV per $1 =
#   P(win) * profit multiple
#   - P(loss)
# --------------------------------------------------

df_ev["Push_Adjusted_EV_Per_Dollar"] = (
    df_ev["Push_Adjusted_Win_Prob"]
    *
    df_ev["Profit_Multiple"]
    -
    df_ev["Push_Adjusted_Loss_Prob"]
)

df_ev["Push_Adjusted_ROI"] = (
    df_ev["Push_Adjusted_EV_Per_Dollar"]
)


# --------------------------------------------------
# 9. REMOVE ANY LEGACY STAKING FIELDS
# --------------------------------------------------

legacy_staking_cols = [
    "Full_Kelly",
    "Quarter_Kelly",
    "Quarter_Kelly_Stake",
    "Push_Adjusted_Full_Kelly",
    "Push_Adjusted_Quarter_Kelly",
    "Push_Adjusted_Stake"
]


df_ev.drop(
    columns=[
        col for col in legacy_staking_cols
        if col in df_ev.columns
    ],
    inplace=True
)


# --------------------------------------------------
# 10. DISPLAY
# --------------------------------------------------

display_cols = [
    "Game",
    "Team",
    "Sportsbook",
    "Line",
    "American_Odds",

    "Historical_Push_Prob",
    "Historical_Push_N",

    "Push_Adjusted_Win_Prob",
    "Push_Adjusted_Push_Prob",
    "Push_Adjusted_Loss_Prob",

    "Break_Even_Prob",
    "Push_Adjusted_ROI",

    "Key_Tradeoff",
    "Decision_Status",
    "Decision_Warning"
]


display_cols = [
    col for col in display_cols
    if col in df_ev.columns
]


display_df = (
    df_ev[
        display_cols
    ]
    .sort_values(
        "Push_Adjusted_ROI",
        ascending=False
    )
    .reset_index(drop=True)
    .copy()
)


for col in [
    "Historical_Push_Prob",
    "Push_Adjusted_Win_Prob",
    "Push_Adjusted_Push_Prob",
    "Push_Adjusted_Loss_Prob",
    "Break_Even_Prob",
    "Push_Adjusted_ROI"
]:
    if col in display_df.columns:
        display_df[col] = (
            display_df[col]
            * 100
        ).round(2)


print("\n" + "=" * 115)
print("CFB1 PUSH-ADJUSTED EV BOARD")
print("=" * 115)

display(display_df)


# --------------------------------------------------
# 11. VALIDATION
# --------------------------------------------------

prob_sum = (
    df_ev["Push_Adjusted_Win_Prob"]
    +
    df_ev["Push_Adjusted_Push_Prob"]
    +
    df_ev["Push_Adjusted_Loss_Prob"]
)


prob_errors = (
    ~np.isclose(
        prob_sum,
        1.0,
        atol=1e-9
    )
).sum()


integer_rows = (
    df_ev["Line"]
    .apply(is_integer_line)
)


nonzero_push_rows = (
    df_ev["Push_Adjusted_Push_Prob"]
    >
    0
)


legacy_remaining = [
    col for col in legacy_staking_cols
    if col in df_ev.columns
]


print("\n" + "=" * 115)
print("CELL 57A VALIDATION")
print("=" * 115)

print(
    "Rows evaluated:",
    len(df_ev)
)

print(
    "Integer spread rows:",
    int(integer_rows.sum())
)

print(
    "Rows with nonzero push probability:",
    int(nonzero_push_rows.sum())
)

print(
    "Probability rows not summing to 1:",
    int(prob_errors)
)

print(
    "Legacy Kelly/stake columns remaining:",
    legacy_remaining
)


if prob_errors != 0:
    raise ValueError(
        "Push-adjusted probability validation failed."
    )

if legacy_remaining:
    raise ValueError(
        "Legacy Kelly/stake columns remain after Cell 57A."
    )


print("\nCELL 57A COMPLETE.")
print(
    "INTEGER-LINE PUSH RISK IS MODELED SEPARATELY."
)
print(
    "PUSH-ADJUSTED EXPECTED ROI REMAINS IN CFB1."
)
print(
    "KELLY AND STAKE SIZING ARE NOT CALCULATED."
)
print(
    "KEY-NUMBER LANDING RATES REMAIN CONTEXT ONLY — "
    "NO DOUBLE COUNTING."
)

CFB1 INTEGER-LINE PUSH ADJUSTMENT

Historical push window: ±1.0 point around candidate spread

CFB1 PUSH-ADJUSTED EV BOARD


,Game,Team,Sportsbook,Line,American_Odds,Historical_Push_Prob,Historical_Push_N,Push_Adjusted_Win_Prob,Push_Adjusted_Push_Prob,Push_Adjusted_Loss_Prob,Break_Even_Prob,Push_Adjusted_ROI,Key_Tradeoff,Decision_Status,Decision_Warning
0,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,FanDuel,-39.5,112,0.00,0,54.08,0.00,45.92,47.17,14.64,,CONFIRMED_BUT_LARGE_SPREAD,
1,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,FanDuel,-40.5,120,0.00,0,51.42,0.00,48.58,45.45,13.13,,CONFIRMED_BUT_LARGE_SPREAD,
2,Stanford Cardinal at Notre Dame Fighting Irish,Notre Dame Fighting Irish,FanDuel,-38.5,100,0.00,0,56.49,0.00,43.51,50.00,12.98,,CONFIRMED_BUT_LARGE_SPREAD,
3,Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,FanDuel,-8.5,-114,0.00,0,58.82,0.00,41.18,53.27,10.42,,CONFIRMED,
4,Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,DraftKings,-10.5,106,0.00,0,53.49,0.00,46.51,48.54,10.18,,CONFIRMED,
5,Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,DraftKings,-11.5,116,0.00,0,50.43,0.00,49.57,46.30,8.93,,CONFIRMED,
6,Rice Owls at East Carolina Pirates,East Carolina Pirates,FanDuel,-10.5,-102,0.00,0,54.08,0.00,45.92,50.50,7.09,CROSSES_10,CONFIRMED,CROSSES_10
7,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,DraftKings,-7.5,105,0.00,0,52.23,0.00,47.77,48.78,7.06,,STRONG_MARKET_CONFLICT,STRONG_MARKET_CONFLICT
8,Rice Owls at East Carolina Pirates,East Carolina Pirates,FanDuel,-11.5,108,0.00,0,51.42,0.00,48.58,48.08,6.96,CROSSES_10,CONFIRMED,CROSSES_10
9,Rice Owls at East Carolina Pirates,East Carolina Pirates,DraftKings,-10.0,-108,4.21,356,53.26,4.21,42.53,51.92,6.79,LANDS_ON_KEY_10,CONFIRMED,LANDS_ON_KEY_10



CELL 57A VALIDATION
Rows evaluated: 47
Integer spread rows: 2
Rows with nonzero push probability: 2
Probability rows not summing to 1: 0
Legacy Kelly/stake columns remaining: []

CELL 57A COMPLETE.
INTEGER-LINE PUSH RISK IS MODELED SEPARATELY.
PUSH-ADJUSTED EXPECTED ROI REMAINS IN CFB1.
KELLY AND STAKE SIZING ARE NOT CALCULATED.
KEY-NUMBER LANDING RATES REMAIN CONTEXT ONLY — NO DOUBLE COUNTING.


In [ ]:
# ==========================================
# CELL 58 — HISTORICAL TEAM-TOTAL CALIBRATION
# ==========================================

import requests
import pandas as pd
import numpy as np

print("=" * 110)
print("CFB1 HISTORICAL TEAM-TOTAL CALIBRATION")
print("=" * 110)


# --------------------------------------------------
# 1. SETTINGS
# --------------------------------------------------

HISTORICAL_YEARS = [
    2021,
    2022,
    2023,
    2024,
    2025
]

PROVIDER_PRIORITY = [
    "DraftKings",
    "ESPN Bet",
    "Bovada"
]

CFBD_BASE_URL = (
    "https://api.collegefootballdata.com"
)

headers = {
    "Authorization":
        f"Bearer {CFBD_API_KEY.strip()}"
}


# --------------------------------------------------
# 2. CHOOSE ONE PROVIDER WITH BOTH
# SPREAD + CLOSING TOTAL
# --------------------------------------------------

def choose_total_line(game):

    lines = game.get(
        "lines",
        []
    )

    for provider in PROVIDER_PRIORITY:

        for line in lines:

            if (
                line.get("provider") == provider
                and line.get("spread") is not None
                and line.get("overUnder") is not None
            ):

                return line

    return None


# --------------------------------------------------
# 3. DOWNLOAD HISTORICAL GAMES
# --------------------------------------------------

team_total_rows = []


for year in HISTORICAL_YEARS:

    print(
        f"Loading {year}..."
    )

    response = requests.get(

        f"{CFBD_BASE_URL}/lines",

        headers=headers,

        params={
            "year": year,
            "seasonType": "regular"
        },

        timeout=30
    )


    if response.status_code != 200:

        raise ValueError(
            f"CFBD request failed for {year}: "
            f"HTTP {response.status_code}"
        )


    games = response.json()

    print(
        f"  Games returned: {len(games)}"
    )


    for game in games:

        # ------------------------------------------
        # FBS VS FBS ONLY
        # ------------------------------------------

        home_class = str(
            game.get(
                "homeClassification",
                ""
            )
        ).lower()

        away_class = str(
            game.get(
                "awayClassification",
                ""
            )
        ).lower()


        if (
            home_class != "fbs"
            or
            away_class != "fbs"
        ):
            continue


        # ------------------------------------------
        # SCORES REQUIRED
        # ------------------------------------------

        home_score = game.get(
            "homeScore"
        )

        away_score = game.get(
            "awayScore"
        )


        if (
            home_score is None
            or
            away_score is None
        ):
            continue


        # ------------------------------------------
        # SPREAD + TOTAL REQUIRED FROM SAME BOOK
        # ------------------------------------------

        line = choose_total_line(
            game
        )


        if line is None:
            continue


        home_spread = float(
            line["spread"]
        )

        game_total = float(
            line["overUnder"]
        )


        # ------------------------------------------
        # MARKET IMPLIED TEAM TOTALS
        #
        # home expected margin = -home spread
        #
        # Home TT = (Total - Home Spread) / 2
        # Away TT = (Total + Home Spread) / 2
        # ------------------------------------------

        implied_home_tt = (
            game_total
            -
            home_spread
        ) / 2.0


        implied_away_tt = (
            game_total
            +
            home_spread
        ) / 2.0


        # ------------------------------------------
        # ACTUAL TEAM-TOTAL ERRORS
        # ------------------------------------------

        home_tt_error = (
            float(home_score)
            -
            implied_home_tt
        )


        away_tt_error = (
            float(away_score)
            -
            implied_away_tt
        )


        # ------------------------------------------
        # STORE HOME TEAM OBSERVATION
        # ------------------------------------------

        team_total_rows.append({

            "Season":
                year,

            "Week":
                game.get("week"),

            "Game_ID":
                game.get("id"),

            "Team":
                game.get("homeTeam"),

            "Opponent":
                game.get("awayTeam"),

            "Home_Away":
                "HOME",

            "Provider":
                line.get("provider"),

            "Closing_Game_Total":
                game_total,

            "Closing_Home_Spread":
                home_spread,

            "Market_Implied_Team_Total":
                implied_home_tt,

            "Actual_Team_Points":
                float(home_score),

            "Team_Total_Error":
                home_tt_error
        })


        # ------------------------------------------
        # STORE AWAY TEAM OBSERVATION
        # ------------------------------------------

        team_total_rows.append({

            "Season":
                year,

            "Week":
                game.get("week"),

            "Game_ID":
                game.get("id"),

            "Team":
                game.get("awayTeam"),

            "Opponent":
                game.get("homeTeam"),

            "Home_Away":
                "AWAY",

            "Provider":
                line.get("provider"),

            "Closing_Game_Total":
                game_total,

            "Closing_Home_Spread":
                home_spread,

            "Market_Implied_Team_Total":
                implied_away_tt,

            "Actual_Team_Points":
                float(away_score),

            "Team_Total_Error":
                away_tt_error
        })


# --------------------------------------------------
# 4. CREATE DATAFRAME
# --------------------------------------------------

df_hist_team_total = pd.DataFrame(
    team_total_rows
)


# --------------------------------------------------
# 5. VALIDATION
# --------------------------------------------------

print(
    "\n" + "=" * 110
)

print(
    "TEAM-TOTAL DATASET VALIDATION"
)

print(
    "=" * 110
)


print(
    "Team observations:",
    len(df_hist_team_total)
)

print(
    "Unique games:",
    df_hist_team_total[
        'Game_ID'
    ].nunique()
)

print(
    "Missing team-total errors:",
    df_hist_team_total[
        'Team_Total_Error'
    ].isna().sum()
)


print(
    "\nPROVIDER COUNTS"
)

print("-" * 45)

print(
    df_hist_team_total[
        'Provider'
    ].value_counts()
)


print(
    "\nSEASON OBSERVATION COUNTS"
)

print("-" * 45)

print(
    df_hist_team_total[
        'Season'
    ]
    .value_counts()
    .sort_index()
)


# --------------------------------------------------
# 6. ERROR DISTRIBUTION
# --------------------------------------------------

tt_error = (
    df_hist_team_total[
        'Team_Total_Error'
    ]
    .astype(float)
)


print(
    "\n" + "=" * 110
)

print(
    "HISTORICAL TEAM-TOTAL ERROR DISTRIBUTION"
)

print(
    "=" * 110
)


print(
    f"Mean error:          "
    f"{tt_error.mean():.3f}"
)

print(
    f"Median error:        "
    f"{tt_error.median():.3f}"
)

print(
    f"Std deviation:       "
    f"{tt_error.std(ddof=1):.3f}"
)

print(
    f"Mean absolute error: "
    f"{tt_error.abs().mean():.3f}"
)


print(
    "\nTEAM-TOTAL ERROR QUANTILES"
)

print("-" * 45)


print(
    tt_error.quantile(
        [
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    ).round(2)
)


# --------------------------------------------------
# 7. OVER / UNDER / PUSH BALANCE
#
# Relative to market-implied team total.
# --------------------------------------------------

def tt_result(error):

    if error > 0:
        return "OVER"

    elif error < 0:
        return "UNDER"

    else:
        return "PUSH"


df_hist_team_total[
    'TT_Result'
] = (
    df_hist_team_total[
        'Team_Total_Error'
    ].apply(
        tt_result
    )
)


print(
    "\nTEAM-TOTAL RESULT COUNTS"
)

print("-" * 45)


print(
    df_hist_team_total[
        'TT_Result'
    ].value_counts()
)


# --------------------------------------------------
# 8. SEASON STABILITY
# --------------------------------------------------

print(
    "\n" + "=" * 110
)

print(
    "TEAM-TOTAL ERROR BY SEASON"
)

print(
    "=" * 110
)


season_tt_summary = (
    df_hist_team_total
    .groupby(
        'Season'
    )
    .agg(

        Team_Observations=(
            'Team_Total_Error',
            'count'
        ),

        Mean_Error=(
            'Team_Total_Error',
            'mean'
        ),

        Median_Error=(
            'Team_Total_Error',
            'median'
        ),

        Std_Error=(
            'Team_Total_Error',
            'std'
        ),

        MAE=(
            'Team_Total_Error',
            lambda x:
            x.abs().mean()
        )
    )
    .round(3)
)


display(
    season_tt_summary
)


# --------------------------------------------------
# 9. HOME / AWAY CHECK
# --------------------------------------------------

print(
    "\n" + "=" * 110
)

print(
    "TEAM-TOTAL ERROR BY HOME / AWAY"
)

print(
    "=" * 110
)


home_away_summary = (
    df_hist_team_total
    .groupby(
        'Home_Away'
    )
    .agg(

        Observations=(
            'Team_Total_Error',
            'count'
        ),

        Mean_Error=(
            'Team_Total_Error',
            'mean'
        ),

        Median_Error=(
            'Team_Total_Error',
            'median'
        ),

        Std_Error=(
            'Team_Total_Error',
            'std'
        )
    )
    .round(3)
)


display(
    home_away_summary
)


# --------------------------------------------------
# 10. IMPLIED-TOTAL RANGE
# --------------------------------------------------

print(
    "\n" + "=" * 110
)

print(
    "MARKET-IMPLIED TEAM-TOTAL RANGE"
)

print(
    "=" * 110
)


print(
    df_hist_team_total[
        'Market_Implied_Team_Total'
    ]
    .describe(
        percentiles=[
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95
        ]
    )
    .round(2)
)


# --------------------------------------------------
# 11. FINAL STATUS
# --------------------------------------------------

print(
    "\n" + "=" * 110
)

print(
    "CELL 58 COMPLETE."
)

print(
    "HISTORICAL TEAM-TOTAL ERROR "
    "DISTRIBUTION CREATED."
)

print(
    "NO CURRENT TEAM-TOTAL PROBABILITIES, "
    "EV, OR KELLY CALCULATED YET."
)

CFB1 HISTORICAL TEAM-TOTAL CALIBRATION
Loading 2021...
  Games returned: 849
Loading 2022...
  Games returned: 1417
Loading 2023...
  Games returned: 1350
Loading 2024...
  Games returned: 1523
Loading 2025...
  Games returned: 1547

TEAM-TOTAL DATASET VALIDATION
Team observations: 7460
Unique games: 3730
Missing team-total errors: 0

PROVIDER COUNTS
---------------------------------------------
Provider
DraftKings    4108
Bovada        3054
ESPN Bet       298
Name: count, dtype: int64

SEASON OBSERVATION COUNTS
---------------------------------------------
Season
2021    1464
2022    1468
2023    1500
2024    1504
2025    1524
Name: count, dtype: int64

HISTORICAL TEAM-TOTAL ERROR DISTRIBUTION
Mean error:          0.166
Median error:        -0.250
Std deviation:       10.998
Mean absolute error: 8.766

TEAM-TOTAL ERROR QUANTILES
---------------------------------------------
0.05   -17.00
0.10   -13.50
0.25    -7.50
0.50    -0.25
0.75     7.25
0.90    14.75
0.95    19.00
Name: Team_Tot

,Team_Observations,Mean_Error,Median_Error,Std_Error,MAE
Season,,,,,
2021,1464,-0.374,-0.875,11.180,8.959
2022,1468,-0.062,-0.250,10.795,8.567
2023,1500,0.588,0.500,10.967,8.812
2024,1504,0.573,-0.250,11.262,8.929
2025,1524,0.088,-0.750,10.765,8.567



TEAM-TOTAL ERROR BY HOME / AWAY


,Observations,Mean_Error,Median_Error,Std_Error
Home_Away,,,,
AWAY,3730,0.139,-0.50,10.666
HOME,3730,0.194,-0.25,11.323



MARKET-IMPLIED TEAM-TOTAL RANGE
count    7460.00
mean       26.70
std         8.07
min         3.50
5%         13.50
10%        16.50
25%        21.25
50%        26.50
75%        31.75
90%        37.00
95%        40.26
max        59.75
Name: Market_Implied_Team_Total, dtype: float64

CELL 58 COMPLETE.
HISTORICAL TEAM-TOTAL ERROR DISTRIBUTION CREATED.
NO CURRENT TEAM-TOTAL PROBABILITIES, EV, OR KELLY CALCULATED YET.


In [ ]:
# ==========================================
# OLD CELL 59 — ROUTED TEAM-TOTAL PRICING
# + EMPIRICAL PROBABILITY
# DYNAMIC WEEKLY ROUTING
# ==========================================

import numpy as np
import pandas as pd

print("=" * 115)
print("CFB1 ROUTED TEAM-TOTAL PRICING + PROBABILITY")
print("=" * 115)


# --------------------------------------------------
# 1. REQUIRED INPUTS
# --------------------------------------------------

required_objects = [
    "df_auto_pricing",
    "df_key_value",
    "df_hist_team_total"
]

for obj in required_objects:
    if obj not in globals():
        raise NameError(
            f"{obj} missing. "
            "Run the earlier CFB1 cells first."
        )


# --------------------------------------------------
# 2. BUILD EXPLICIT DIRECTIONAL TEAM-TOTAL ROUTES
#
# Dynamic weekly rules:
#
# TEAM_TOTAL_OVER
#   = Model_Side team OVER
#
# OPP_TEAM_TOTAL_UNDER
#   = opponent of Model_Side UNDER
#
# TEAM_TOTAL_REVIEW
#   = nondirectional, so NOT quantitatively priced
# --------------------------------------------------

route_rows = []

for _, row in df_key_value.iterrows():

    game = row["Game"]
    model_side = row["Model_Side"]
    home_team = row["Home_Team"]
    away_team = row["Away_Team"]

    routes = str(
        row["Eligible_Market_Routes"]
    )

    route_list = [
        x.strip()
        for x in routes.split(",")
        if x.strip()
    ]


    # ------------------------------
    # Model-side Team Total OVER
    # ------------------------------

    if "TEAM_TOTAL_OVER" in route_list:

        route_rows.append(
            {
                "Game": game,
                "TT_Target_Team": model_side,
                "TT_Direction": "OVER"
            }
        )


    # ------------------------------
    # Opponent Team Total UNDER
    # ------------------------------

    if "OPP_TEAM_TOTAL_UNDER" in route_list:

        if model_side == home_team:
            opponent = away_team

        elif model_side == away_team:
            opponent = home_team

        else:
            raise ValueError(
                f"Model side does not match "
                f"home/away teams for {game}"
            )

        route_rows.append(
            {
                "Game": game,
                "TT_Target_Team": opponent,
                "TT_Direction": "UNDER"
            }
        )


df_tt_routes = pd.DataFrame(
    route_rows,
    columns=[
        "Game",
        "TT_Target_Team",
        "TT_Direction"
    ]
)


# Remove accidental duplicate routes
df_tt_routes = (
    df_tt_routes
    .drop_duplicates()
    .reset_index(drop=True)
)


# --------------------------------------------------
# 3. VALIDATE ROUTE TABLE
# --------------------------------------------------

duplicate_routes = int(
    df_tt_routes.duplicated(
        subset=[
            "Game",
            "TT_Target_Team",
            "TT_Direction"
        ]
    ).sum()
)

print(
    f"\nExplicit directional routes: "
    f"{len(df_tt_routes)}"
)

print(
    f"Duplicate directional routes: "
    f"{duplicate_routes}"
)

if duplicate_routes > 0:
    raise ValueError(
        "Duplicate directional team-total routes found."
    )


# --------------------------------------------------
# 4. MERGE RETAINED MODEL INFORMATION
# --------------------------------------------------

tt_model = df_key_value[
    [
        "Game",
        "Model_Side",
        "Home_Team",
        "Away_Team",
        "Market_Model_Spread",
        "Retained_Model_Spread",
        "Model_Cushion_Points",
        "Decision_Status"
    ]
].copy()


df_tt_routes = df_tt_routes.merge(
    tt_model,
    on="Game",
    how="left",
    validate="many_to_one"
)


# --------------------------------------------------
# 5. VALIDATE MODEL MATCH
# --------------------------------------------------

missing_model = df_tt_routes[
    df_tt_routes["Model_Side"].isna()
].copy()


if len(missing_model) > 0:

    print("\nUNMATCHED TEAM-TOTAL ROUTES")

    display(
        missing_model[
            [
                "Game",
                "TT_Target_Team",
                "TT_Direction"
            ]
        ]
    )

    raise ValueError(
        "Some team-total routes did not match "
        "the CFB1 model board."
    )


# --------------------------------------------------
# 6. DETERMINE TEAM-TOTAL MODEL ADJUSTMENT
#
# Retained margin disagreement is split 50/50.
#
# Model-side TT OVER:
#     + half retained cushion
#
# Opponent TT UNDER:
#     - half retained cushion
# --------------------------------------------------

def tt_model_adjustment(row):

    cushion = float(
        row["Model_Cushion_Points"]
    )

    target = row[
        "TT_Target_Team"
    ]

    model_side = row[
        "Model_Side"
    ]

    direction = row[
        "TT_Direction"
    ]


    if (
        target == model_side
        and direction == "OVER"
    ):
        return cushion / 2.0


    if (
        target != model_side
        and direction == "UNDER"
    ):
        return -cushion / 2.0


    return np.nan


df_tt_routes[
    "TT_Model_Adjustment"
] = (
    df_tt_routes.apply(
        tt_model_adjustment,
        axis=1
    )
)


# --------------------------------------------------
# 7. STOP ON UNSUPPORTED ROUTING
# --------------------------------------------------

bad_routes = df_tt_routes[
    df_tt_routes[
        "TT_Model_Adjustment"
    ].isna()
].copy()


if len(bad_routes) > 0:

    print(
        "\nUNSUPPORTED TEAM-TOTAL ROUTES"
    )

    display(
        bad_routes[
            [
                "Game",
                "TT_Target_Team",
                "TT_Direction",
                "Model_Side"
            ]
        ]
    )

    raise ValueError(
        "Unsupported team-total direction found. "
        "Do not infer a direction."
    )


# --------------------------------------------------
# 8. CURRENT TEAM-TOTAL PRICES ONLY
# --------------------------------------------------

tt_prices = df_auto_pricing[
    df_auto_pricing[
        "Market_Type"
    ] == "TEAM_TOTAL"
].copy()


# --------------------------------------------------
# 9. MERGE ROUTED GAME / TEAM / DIRECTION
# --------------------------------------------------

tt_candidates = tt_prices.merge(
    df_tt_routes,
    left_on=[
        "Game",
        "Team",
        "Bet_Direction"
    ],
    right_on=[
        "Game",
        "TT_Target_Team",
        "TT_Direction"
    ],
    how="inner"
)


# --------------------------------------------------
# 10. REMOVE EXTREME PRICES
# --------------------------------------------------

tt_candidates = tt_candidates[
    (
        tt_candidates[
            "American_Odds"
        ] >= -250
    )
    &
    (
        tt_candidates[
            "American_Odds"
        ] <= 300
    )
].copy()


# --------------------------------------------------
# 11. FIND MARKET TEAM-TOTAL ANCHOR
# --------------------------------------------------

anchor_rows = []


for (
    game,
    team,
    book
), group in tt_candidates.groupby(
    [
        "Game",
        "Team",
        "Sportsbook"
    ]
):

    featured = group[
        group[
            "Source_Market_Key"
        ] == "team_totals"
    ]


    if len(featured) > 0:

        anchor = (
            featured
            .assign(
                Price_Distance=(
                    featured[
                        "American_Odds"
                    ]
                    + 110
                ).abs()
            )
            .sort_values(
                "Price_Distance"
            )
            .iloc[0]
        )

    else:

        anchor = (
            group
            .assign(
                Price_Distance=(
                    group[
                        "American_Odds"
                    ]
                    + 110
                ).abs()
            )
            .sort_values(
                "Price_Distance"
            )
            .iloc[0]
        )


    anchor_rows.append(
        {
            "Game": game,
            "Team": team,
            "Sportsbook": book,
            "Market_TT_Anchor": float(
                anchor["Line"]
            )
        }
    )


df_tt_anchors = pd.DataFrame(
    anchor_rows
)


tt_candidates = tt_candidates.merge(
    df_tt_anchors,
    on=[
        "Game",
        "Team",
        "Sportsbook"
    ],
    how="left",
    validate="many_to_one"
)


# --------------------------------------------------
# 12. RETAINED TEAM-TOTAL FAIR LINE
# --------------------------------------------------

tt_candidates[
    "Retained_TT_Fair_Line"
] = (
    tt_candidates[
        "Market_TT_Anchor"
    ]
    +
    tt_candidates[
        "TT_Model_Adjustment"
    ]
)


# --------------------------------------------------
# 13. CANDIDATE EDGE
# --------------------------------------------------

def tt_candidate_edge(row):

    if row[
        "TT_Direction"
    ] == "OVER":

        return (
            row[
                "Retained_TT_Fair_Line"
            ]
            -
            row[
                "Line"
            ]
        )

    if row[
        "TT_Direction"
    ] == "UNDER":

        return (
            row[
                "Line"
            ]
            -
            row[
                "Retained_TT_Fair_Line"
            ]
        )

    return np.nan


tt_candidates[
    "TT_Candidate_Edge"
] = (
    tt_candidates.apply(
        tt_candidate_edge,
        axis=1
    )
)


# --------------------------------------------------
# 14. KEEP USEFUL LINE NEIGHBORHOOD
# --------------------------------------------------

tt_candidates = tt_candidates[
    (
        tt_candidates[
            "Line"
        ]
        -
        tt_candidates[
            "Market_TT_Anchor"
        ]
    ).abs() <= 4.0
].copy()


# --------------------------------------------------
# 15. EMPIRICAL TEAM-TOTAL PROBABILITY
# --------------------------------------------------

global_tt_errors = (
    df_hist_team_total[
        "Team_Total_Error"
    ]
    .dropna()
    .astype(float)
)


def empirical_tt_wpl(
    edge,
    direction,
    errors
):

    if direction == "OVER":
        adjusted = errors + edge

    elif direction == "UNDER":
        adjusted = -errors + edge

    else:
        return (
            np.nan,
            np.nan,
            np.nan
        )


    win_prob = (
        adjusted > 0
    ).mean()

    push_prob = (
        adjusted == 0
    ).mean()

    loss_prob = (
        adjusted < 0
    ).mean()


    return (
        win_prob,
        push_prob,
        loss_prob
    )


tt_probs = []


for _, row in tt_candidates.iterrows():

    win, push, loss = empirical_tt_wpl(

        float(
            row[
                "TT_Candidate_Edge"
            ]
        ),

        row[
            "TT_Direction"
        ],

        global_tt_errors
    )


    tt_probs.append(
        (
            win,
            push,
            loss
        )
    )


tt_candidates[
    [
        "TT_Win_Prob",
        "TT_Push_Prob",
        "TT_Loss_Prob"
    ]
] = tt_probs


# --------------------------------------------------
# 16. BREAK-EVEN PROBABILITY
# --------------------------------------------------

def american_break_even(odds):

    odds = float(
        odds
    )

    if odds < 0:

        return (
            abs(odds)
            /
            (
                abs(odds)
                + 100
            )
        )

    return (
        100
        /
        (
            odds
            + 100
        )
    )


tt_candidates[
    "Break_Even_Prob"
] = (
    tt_candidates[
        "American_Odds"
    ].apply(
        american_break_even
    )
)


tt_candidates[
    "Probability_Edge"
] = (
    tt_candidates[
        "TT_Win_Prob"
    ]
    -
    tt_candidates[
        "Break_Even_Prob"
    ]
)


# --------------------------------------------------
# 17. KEEP BEST PRICE FOR SAME LINE
# --------------------------------------------------

tt_candidates = (
    tt_candidates
    .sort_values(
        "American_Odds",
        ascending=False
    )
    .drop_duplicates(
        subset=[
            "Game",
            "Team",
            "TT_Direction",
            "Line"
        ],
        keep="first"
    )
)


# --------------------------------------------------
# 18. COMPACT SHORTLIST
#
# Keep:
# - 2 closest to market anchor
# - 2 best probability edges
# --------------------------------------------------

shortlist_parts = []


for _, group in tt_candidates.groupby(
    [
        "Game",
        "Team",
        "TT_Direction"
    ]
):

    group = group.copy()

    group[
        "Distance_From_Anchor"
    ] = (
        group[
            "Line"
        ]
        -
        group[
            "Market_TT_Anchor"
        ]
    ).abs()


    near_anchor = (
        group
        .sort_values(
            "Distance_From_Anchor"
        )
        .head(2)
    )


    best_edge = (
        group
        .sort_values(
            "Probability_Edge",
            ascending=False
        )
        .head(2)
    )


    combined = (
        pd.concat(
            [
                near_anchor,
                best_edge
            ]
        )
        .drop_duplicates()
    )


    shortlist_parts.append(
        combined
    )


df_tt_shortlist = (
    pd.concat(
        shortlist_parts,
        ignore_index=True
    )
    if shortlist_parts
    else pd.DataFrame()
)


# --------------------------------------------------
# 19. DISPLAY
# --------------------------------------------------

display_tt = (
    df_tt_shortlist.copy()
)


for col in [
    "TT_Win_Prob",
    "TT_Push_Prob",
    "Break_Even_Prob",
    "Probability_Edge"
]:

    display_tt[col] = (
        display_tt[col]
        * 100
    ).round(1)


print(
    "\n" + "=" * 115
)

print(
    "CFB1 TEAM-TOTAL SHOPPING LIST"
)

print(
    "=" * 115
)


display(
    display_tt[
        [
            "Game",
            "Team",
            "TT_Direction",
            "Sportsbook",
            "Line",
            "American_Odds",
            "Market_TT_Anchor",
            "TT_Model_Adjustment",
            "Retained_TT_Fair_Line",
            "TT_Candidate_Edge",
            "TT_Win_Prob",
            "TT_Push_Prob",
            "Break_Even_Prob",
            "Probability_Edge",
            "Decision_Status"
        ]
    ]
    .sort_values(
        [
            "Game",
            "Probability_Edge"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(
        drop=True
    )
    .round(2)
)


# --------------------------------------------------
# 20. VALIDATION
# --------------------------------------------------

print(
    "\n" + "=" * 115
)

print(
    "CELL 59 VALIDATION"
)

print(
    "=" * 115
)


print(
    "Explicit directional routes:",
    len(df_tt_routes)
)

print(
    "Games represented:",
    df_tt_shortlist[
        "Game"
    ].nunique()
    if len(df_tt_shortlist) > 0
    else 0
)

print(
    "Raw TEAM_TOTAL pricing rows:",
    len(tt_prices)
)

print(
    "Final shortlist rows:",
    len(df_tt_shortlist)
)

print(
    "Missing probabilities:",
    (
        df_tt_shortlist[
            "TT_Win_Prob"
        ].isna().sum()
        if len(df_tt_shortlist) > 0
        else 0
    )
)


print(
    "\n" + "=" * 115
)

print(
    "CELL 59 COMPLETE."
)

print(
    "ONLY EXPLICIT CFB1 TEAM-TOTAL DIRECTIONS "
    "WERE EVALUATED."
)

print(
    "GENERIC TEAM_TOTAL_REVIEW ROUTES "
    "REMAIN NONDIRECTIONAL."
)

print(
    "NO TEAM-TOTAL EV OR STAKING "
    "CALCULATED YET."
)

CFB1 ROUTED TEAM-TOTAL PRICING + PROBABILITY

Explicit directional routes: 10
Duplicate directional routes: 0

CFB1 TEAM-TOTAL SHOPPING LIST


,Game,Team,TT_Direction,Sportsbook,Line,American_Odds,Market_TT_Anchor,TT_Model_Adjustment,Retained_TT_Fair_Line,TT_Candidate_Edge,TT_Win_Prob,TT_Push_Prob,Break_Even_Prob,Probability_Edge,Decision_Status
0,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,OVER,FanDuel,45.5,112,44.5,14.31,58.81,13.31,89.6,0.0,47.2,42.4,CONFIRMED_BUT_LARGE_SPREAD
1,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,OVER,DraftKings,48.5,145,44.5,14.31,58.81,10.31,82.8,0.0,40.8,42.0,CONFIRMED_BUT_LARGE_SPREAD
2,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,OVER,DraftKings,44.5,-120,44.5,14.31,58.81,14.31,91.4,0.0,54.5,36.8,CONFIRMED_BUT_LARGE_SPREAD
3,Buffalo Bulls at Toledo Rockets,Toledo Rockets,OVER,FanDuel,38.5,114,37.5,5.48,42.98,4.48,64.6,0.0,46.7,17.9,NEUTRAL_MARKET
4,Buffalo Bulls at Toledo Rockets,Toledo Rockets,OVER,FanDuel,35.5,-146,37.5,5.48,42.98,7.48,74.7,0.0,59.3,15.4,NEUTRAL_MARKET
5,Buffalo Bulls at Toledo Rockets,Toledo Rockets,OVER,DraftKings,37.5,-120,37.5,5.48,42.98,5.48,68.1,0.0,54.5,13.5,NEUTRAL_MARKET
6,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,OVER,FanDuel,42.5,-106,42.5,5.76,48.26,5.76,69.8,0.0,51.5,18.3,CONFIRMED_BUT_LARGE_SPREAD
7,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,OVER,FanDuel,45.5,146,42.5,5.76,48.26,2.77,59.0,0.0,40.7,18.3,CONFIRMED_BUT_LARGE_SPREAD
8,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,OVER,FanDuel,43.5,-102,42.5,5.76,48.26,4.77,66.4,0.0,50.5,15.9,CONFIRMED_BUT_LARGE_SPREAD
9,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,OVER,FanDuel,28.5,136,26.5,6.06,32.56,4.06,63.8,0.0,42.4,21.4,STRONG_MARKET_CONFLICT



CELL 59 VALIDATION
Explicit directional routes: 10
Games represented: 9
Raw TEAM_TOTAL pricing rows: 2606
Final shortlist rows: 32
Missing probabilities: 0

CELL 59 COMPLETE.
ONLY EXPLICIT CFB1 TEAM-TOTAL DIRECTIONS WERE EVALUATED.
GENERIC TEAM_TOTAL_REVIEW ROUTES REMAIN NONDIRECTIONAL.
NO TEAM-TOTAL EV OR STAKING CALCULATED YET.


In [ ]:
# ==========================================
# CELL 59A — TEAM-TOTAL DIRECTION + PUSH VALIDATION
# ==========================================

import numpy as np
import pandas as pd

print("=" * 115)
print("CFB1 TEAM-TOTAL DIRECTION + PUSH VALIDATION")
print("=" * 115)


# --------------------------------------------------
# 1. REQUIRED INPUTS
# --------------------------------------------------

required_objects = [
    "df_tt_shortlist",
    "df_hist_team_total"
]

for obj in required_objects:

    if obj not in globals():
        raise NameError(
            f"{obj} missing. "
            "Run Cell 59 first."
        )


# --------------------------------------------------
# 2. WORKING COPY
# --------------------------------------------------

df_tt_validated = (
    df_tt_shortlist.copy()
)


# --------------------------------------------------
# 3. MODEL-DIRECTION SUPPORT CHECK
#
# OVER route should have POSITIVE adjustment.
# UNDER route should have NEGATIVE adjustment.
#
# If not, keep route visible but do not treat it as
# quantitatively supported.
# --------------------------------------------------

def tt_direction_support(row):

    direction = str(
        row["TT_Direction"]
    ).upper()

    adjustment = float(
        row["TT_Model_Adjustment"]
    )


    if direction == "OVER":

        if adjustment > 0:
            return "SUPPORTED"

        elif adjustment < 0:
            return "OPPOSED"

        else:
            return "NEUTRAL"


    elif direction == "UNDER":

        if adjustment < 0:
            return "SUPPORTED"

        elif adjustment > 0:
            return "OPPOSED"

        else:
            return "NEUTRAL"


    return "CHECK"


df_tt_validated[
    "TT_Model_Direction_Status"
] = (
    df_tt_validated.apply(
        tt_direction_support,
        axis=1
    )
)


# --------------------------------------------------
# 4. QUANTITATIVE ELIGIBILITY
# --------------------------------------------------

df_tt_validated[
    "TT_Quantitative_Status"
] = np.where(

    df_tt_validated[
        "TT_Model_Direction_Status"
    ] == "SUPPORTED",

    "QUANTITATIVE_OK",

    "QUALITATIVE_REVIEW_ONLY"
)


# --------------------------------------------------
# 5. WHOLE-NUMBER TEAM-TOTAL TEST
#
# Team scores are integers.
#
# X.5 line cannot push.
# Whole-number line can.
# --------------------------------------------------

def is_integer_tt(line):

    if pd.isna(line):
        return False

    line = float(line)

    return abs(
        line - round(line)
    ) < 1e-9


df_tt_validated[
    "TT_Is_Integer_Line"
] = (
    df_tt_validated[
        "Line"
    ].apply(
        is_integer_tt
    )
)


# --------------------------------------------------
# 6. HISTORICAL PUSH ESTIMATE FOR INTEGER TT LINES
#
# Use historical team-total observations with
# market-implied team totals within ±1 point.
# --------------------------------------------------

TT_PUSH_WINDOW = 1.0


def estimate_tt_push_probability(row):

    line = float(
        row["Line"]
    )


    # Half-point line cannot push
    if not is_integer_tt(line):

        return pd.Series(
            {
                "TT_Historical_Push_Prob":
                    0.0,

                "TT_Historical_Push_N":
                    0
            }
        )


    comparable = (
        df_hist_team_total[
            (
                df_hist_team_total[
                    "Market_Implied_Team_Total"
                ]
                - line
            ).abs()
            <= TT_PUSH_WINDOW
        ]
        .copy()
    )


    n = len(
        comparable
    )


    if n == 0:

        return pd.Series(
            {
                "TT_Historical_Push_Prob":
                    0.0,

                "TT_Historical_Push_N":
                    0
            }
        )


    push_count = (
        comparable[
            "Actual_Team_Points"
        ] == line
    ).sum()


    push_prob = (
        push_count
        /
        n
    )


    return pd.Series(
        {
            "TT_Historical_Push_Prob":
                push_prob,

            "TT_Historical_Push_N":
                n
        }
    )


df_tt_validated[
    [
        "TT_Historical_Push_Prob",
        "TT_Historical_Push_N"
    ]
] = (
    df_tt_validated.apply(
        estimate_tt_push_probability,
        axis=1
    )
)


# --------------------------------------------------
# 7. REBUILD W/P/L PROBABILITIES
#
# Cell 59 probability is still useful for deciding
# the win-vs-loss balance.
#
# Here we reserve explicit push mass only when a
# whole-number sportsbook line can actually push.
# --------------------------------------------------

def rebuild_tt_wpl(row):

    old_win = float(
        row["TT_Win_Prob"]
    )

    old_loss = float(
        row["TT_Loss_Prob"]
    )

    push = float(
        row["TT_Historical_Push_Prob"]
    )


    # If route is not quantitatively supported,
    # do not produce actionable probabilities.
    if (
        row[
            "TT_Quantitative_Status"
        ]
        != "QUANTITATIVE_OK"
    ):

        return pd.Series(
            {
                "TT_Final_Win_Prob":
                    np.nan,

                "TT_Final_Push_Prob":
                    np.nan,

                "TT_Final_Loss_Prob":
                    np.nan
            }
        )


    # Half-point lines
    if push <= 0:

        nonpush_total = (
            old_win
            +
            old_loss
        )

        if nonpush_total <= 0:

            return pd.Series(
                {
                    "TT_Final_Win_Prob":
                        np.nan,

                    "TT_Final_Push_Prob":
                        0.0,

                    "TT_Final_Loss_Prob":
                        np.nan
                }
            )


        return pd.Series(
            {
                "TT_Final_Win_Prob":
                    old_win
                    /
                    nonpush_total,

                "TT_Final_Push_Prob":
                    0.0,

                "TT_Final_Loss_Prob":
                    old_loss
                    /
                    nonpush_total
            }
        )


    # Whole-number lines:
    # preserve original win/loss ratio
    nonpush_total = (
        old_win
        +
        old_loss
    )


    if nonpush_total <= 0:

        return pd.Series(
            {
                "TT_Final_Win_Prob":
                    np.nan,

                "TT_Final_Push_Prob":
                    push,

                "TT_Final_Loss_Prob":
                    np.nan
            }
        )


    remaining = (
        1.0
        -
        push
    )


    return pd.Series(
        {
            "TT_Final_Win_Prob":
                remaining
                *
                (
                    old_win
                    /
                    nonpush_total
                ),

            "TT_Final_Push_Prob":
                push,

            "TT_Final_Loss_Prob":
                remaining
                *
                (
                    old_loss
                    /
                    nonpush_total
                )
        }
    )


df_tt_validated[
    [
        "TT_Final_Win_Prob",
        "TT_Final_Push_Prob",
        "TT_Final_Loss_Prob"
    ]
] = (
    df_tt_validated.apply(
        rebuild_tt_wpl,
        axis=1
    )
)


# --------------------------------------------------
# 8. DISPLAY VERSION
# --------------------------------------------------

display_tt = (
    df_tt_validated.copy()
)


pct_cols = [
    "TT_Historical_Push_Prob",
    "TT_Final_Win_Prob",
    "TT_Final_Push_Prob",
    "TT_Final_Loss_Prob",
    "Break_Even_Prob",
    "Probability_Edge"
]


for col in pct_cols:

    display_tt[col] = (
        display_tt[col]
        * 100
    ).round(2)


# --------------------------------------------------
# 9. QUANTITATIVE BOARD
# --------------------------------------------------

print(
    "\n" + "=" * 115
)

print(
    "QUANTITATIVELY SUPPORTED TEAM-TOTALS"
)

print(
    "=" * 115
)


quant_board = (
    display_tt[
        display_tt[
            "TT_Quantitative_Status"
        ] == "QUANTITATIVE_OK"
    ]
    .copy()
)


display(
    quant_board[
        [
            "Game",
            "Team",
            "TT_Direction",
            "Sportsbook",
            "Line",
            "American_Odds",

            "TT_Model_Adjustment",
            "TT_Candidate_Edge",

            "TT_Final_Win_Prob",
            "TT_Final_Push_Prob",
            "TT_Final_Loss_Prob",

            "Break_Even_Prob",
            "Decision_Status",

            "TT_Model_Direction_Status",
            "TT_Quantitative_Status"
        ]
    ]
    .sort_values(
        [
            "Game",
            "Line"
        ]
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------
# 10. QUALITATIVE-ONLY BOARD
# --------------------------------------------------

print(
    "\n" + "=" * 115
)

print(
    "QUALITATIVE TEAM-TOTAL REVIEWS"
)

print(
    "=" * 115
)


qual_board = (
    display_tt[
        display_tt[
            "TT_Quantitative_Status"
        ]
        ==
        "QUALITATIVE_REVIEW_ONLY"
    ]
    .copy()
)


if len(
    qual_board
) > 0:

    display(
        qual_board[
            [
                "Game",
                "Team",
                "TT_Direction",
                "Sportsbook",
                "Line",
                "American_Odds",

                "TT_Model_Adjustment",

                "Decision_Status",

                "TT_Model_Direction_Status",
                "TT_Quantitative_Status"
            ]
        ]
        .sort_values(
            [
                "Game",
                "Line"
            ]
        )
        .reset_index(
            drop=True
        )
    )

else:

    print(
        "No qualitative-only rows."
    )


# --------------------------------------------------
# 11. VALIDATION
# --------------------------------------------------

supported = (
    df_tt_validated[
        df_tt_validated[
            "TT_Quantitative_Status"
        ]
        ==
        "QUANTITATIVE_OK"
    ]
)


prob_sum = (
    supported[
        "TT_Final_Win_Prob"
    ]
    +
    supported[
        "TT_Final_Push_Prob"
    ]
    +
    supported[
        "TT_Final_Loss_Prob"
    ]
)


print(
    "\n" + "=" * 115
)

print(
    "CELL 59A VALIDATION"
)

print(
    "=" * 115
)


print(
    "Rows evaluated:",
    len(df_tt_validated)
)

print(
    "Quantitative rows:",
    (
        df_tt_validated[
            "TT_Quantitative_Status"
        ]
        ==
        "QUANTITATIVE_OK"
    ).sum()
)

print(
    "Qualitative-only rows:",
    (
        df_tt_validated[
            "TT_Quantitative_Status"
        ]
        ==
        "QUALITATIVE_REVIEW_ONLY"
    ).sum()
)

print(
    "Integer TT rows:",
    df_tt_validated[
        "TT_Is_Integer_Line"
    ].sum()
)

print(
    "Half-point rows with nonzero final push:",
    (
        (~df_tt_validated[
            "TT_Is_Integer_Line"
        ])
        &
        (
            df_tt_validated[
                "TT_Final_Push_Prob"
            ]
            .fillna(0)
            > 0
        )
    ).sum()
)

print(
    "Supported probability rows not summing to 1:",
    (
        np.abs(
            prob_sum - 1
        ) > 1e-9
    ).sum()
)


print(
    "\n" + "=" * 115
)

print(
    "CELL 59A COMPLETE."
)

print(
    "MODEL-OPPOSED TEAM-TOTAL ROUTES ARE "
    "QUALITATIVE ONLY."
)

print(
    "HALF-POINT TEAM TOTALS NOW HAVE "
    "ZERO PUSH PROBABILITY."
)

print(
    "NO TEAM-TOTAL EV OR KELLY "
    "CALCULATED YET."
)

CFB1 TEAM-TOTAL DIRECTION + PUSH VALIDATION

QUANTITATIVELY SUPPORTED TEAM-TOTALS


,Game,Team,TT_Direction,Sportsbook,Line,American_Odds,TT_Model_Adjustment,TT_Candidate_Edge,TT_Final_Win_Prob,TT_Final_Push_Prob,TT_Final_Loss_Prob,Break_Even_Prob,Decision_Status,TT_Model_Direction_Status,TT_Quantitative_Status
0,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,OVER,DraftKings,44.5,-120,14.310,14.310,91.35,0.0,8.65,54.55,CONFIRMED_BUT_LARGE_SPREAD,SUPPORTED,QUANTITATIVE_OK
1,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,OVER,FanDuel,45.5,112,14.310,13.310,89.58,0.0,10.42,47.17,CONFIRMED_BUT_LARGE_SPREAD,SUPPORTED,QUANTITATIVE_OK
2,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,OVER,DraftKings,48.5,145,14.310,10.310,82.82,0.0,17.18,40.82,CONFIRMED_BUT_LARGE_SPREAD,SUPPORTED,QUANTITATIVE_OK
3,Buffalo Bulls at Toledo Rockets,Toledo Rockets,OVER,FanDuel,35.5,-146,5.480,7.480,74.73,0.0,25.27,59.35,NEUTRAL_MARKET,SUPPORTED,QUANTITATIVE_OK
4,Buffalo Bulls at Toledo Rockets,Toledo Rockets,OVER,DraftKings,37.5,-120,5.480,5.480,68.08,0.0,31.92,54.55,NEUTRAL_MARKET,SUPPORTED,QUANTITATIVE_OK
5,Buffalo Bulls at Toledo Rockets,Toledo Rockets,OVER,FanDuel,38.5,114,5.480,4.480,64.64,0.0,35.36,46.73,NEUTRAL_MARKET,SUPPORTED,QUANTITATIVE_OK
6,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,OVER,FanDuel,42.5,-106,5.765,5.765,69.76,0.0,30.24,51.46,CONFIRMED_BUT_LARGE_SPREAD,SUPPORTED,QUANTITATIVE_OK
7,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,OVER,FanDuel,43.5,-102,5.765,4.765,66.42,0.0,33.58,50.50,CONFIRMED_BUT_LARGE_SPREAD,SUPPORTED,QUANTITATIVE_OK
8,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,OVER,FanDuel,45.5,146,5.765,2.765,58.99,0.0,41.01,40.65,CONFIRMED_BUT_LARGE_SPREAD,SUPPORTED,QUANTITATIVE_OK
9,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,OVER,DraftKings,24.5,-130,6.065,8.065,76.88,0.0,23.12,56.52,STRONG_MARKET_CONFLICT,SUPPORTED,QUANTITATIVE_OK



QUALITATIVE TEAM-TOTAL REVIEWS


,Game,Team,TT_Direction,Sportsbook,Line,American_Odds,TT_Model_Adjustment,Decision_Status,TT_Model_Direction_Status,TT_Quantitative_Status
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,OVER,FanDuel,25.5,-115,-2.275,CONFIRMED,OPPOSED,QUALITATIVE_REVIEW_ONLY
1,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,OVER,FanDuel,26.5,-106,-2.275,CONFIRMED,OPPOSED,QUALITATIVE_REVIEW_ONLY
2,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,OVER,FanDuel,28.5,146,-2.275,CONFIRMED,OPPOSED,QUALITATIVE_REVIEW_ONLY
3,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,OVER,DraftKings,30.5,175,-2.275,CONFIRMED,OPPOSED,QUALITATIVE_REVIEW_ONLY



CELL 59A VALIDATION
Rows evaluated: 32
Quantitative rows: 28
Qualitative-only rows: 4
Integer TT rows: 0
Half-point rows with nonzero final push: 0
Supported probability rows not summing to 1: 0

CELL 59A COMPLETE.
MODEL-OPPOSED TEAM-TOTAL ROUTES ARE QUALITATIVE ONLY.
HALF-POINT TEAM TOTALS NOW HAVE ZERO PUSH PROBABILITY.
NO TEAM-TOTAL EV OR KELLY CALCULATED YET.


In [ ]:
# ==========================================
# CELL 60 — TEAM-TOTAL EV / PRICING
# ==========================================

import numpy as np
import pandas as pd

print("=" * 115)
print("CFB1 TEAM-TOTAL EV / PRICING")
print("=" * 115)


# --------------------------------------------------
# 1. REQUIRED INPUT
# --------------------------------------------------

if "df_tt_validated" not in globals():
    raise NameError(
        "df_tt_validated missing. "
        "Run old Cell 59A first."
    )


# --------------------------------------------------
# 2. QUANTITATIVE ROWS ONLY
# --------------------------------------------------

df_tt_ev = df_tt_validated[
    df_tt_validated[
        "TT_Quantitative_Status"
    ] == "QUANTITATIVE_OK"
].copy()


# --------------------------------------------------
# 3. AMERICAN ODDS -> PROFIT MULTIPLE
# --------------------------------------------------

def profit_multiple(american_odds):

    odds = float(
        american_odds
    )

    if odds < 0:
        return (
            100.0
            /
            abs(odds)
        )

    return (
        odds
        /
        100.0
    )


df_tt_ev[
    "Profit_Multiple"
] = (
    df_tt_ev[
        "American_Odds"
    ].apply(
        profit_multiple
    )
)


# --------------------------------------------------
# 4. BREAK-EVEN PROBABILITY
# --------------------------------------------------

def break_even_prob(american_odds):

    odds = float(
        american_odds
    )

    if odds < 0:
        return (
            abs(odds)
            /
            (
                abs(odds)
                + 100.0
            )
        )

    return (
        100.0
        /
        (
            odds
            + 100.0
        )
    )


df_tt_ev[
    "Break_Even_Prob_Final"
] = (
    df_tt_ev[
        "American_Odds"
    ].apply(
        break_even_prob
    )
)


# --------------------------------------------------
# 5. EXPECTED VALUE / EXPECTED ROI
#
# Push returns stake:
#
# EV per $1 =
#   P(win) * profit multiple
#   - P(loss)
# --------------------------------------------------

df_tt_ev[
    "TT_EV_Per_Dollar"
] = (
    df_tt_ev[
        "TT_Final_Win_Prob"
    ]
    *
    df_tt_ev[
        "Profit_Multiple"
    ]
    -
    df_tt_ev[
        "TT_Final_Loss_Prob"
    ]
)


df_tt_ev[
    "TT_Expected_ROI"
] = (
    df_tt_ev[
        "TT_EV_Per_Dollar"
    ]
)


# --------------------------------------------------
# 6. EV STATUS
# --------------------------------------------------

def tt_ev_status(row):

    ev = float(
        row[
            "TT_Expected_ROI"
        ]
    )

    if ev <= 0:
        return "NEGATIVE_EV"

    if ev < 0.02:
        return "THIN_POSITIVE_EV"

    if ev < 0.05:
        return "POSITIVE_EV"

    return "STRONG_PRICE_EDGE"


df_tt_ev[
    "TT_EV_Status"
] = (
    df_tt_ev.apply(
        tt_ev_status,
        axis=1
    )
)


# --------------------------------------------------
# 7. DECISION WARNING
#
# Pricing never erases earlier CFB1 context.
# --------------------------------------------------

def tt_decision_warning(row):

    warnings = []

    decision = str(
        row.get(
            "Decision_Status",
            ""
        )
    )

    if decision == "STRONG_MARKET_CONFLICT":

        warnings.append(
            "STRONG_MARKET_CONFLICT"
        )

    elif decision == "MARKET_CONFLICT":

        warnings.append(
            "MARKET_CONFLICT"
        )

    elif decision == (
        "CONFLICT_REVIEW_MARKET_SUPPORTS_MODEL"
    ):

        warnings.append(
            "ORIGINAL_MATCHUP_CONFLICT"
        )

    return " | ".join(
        warnings
    )


df_tt_ev[
    "TT_Decision_Warning"
] = (
    df_tt_ev.apply(
        tt_decision_warning,
        axis=1
    )
)


# --------------------------------------------------
# 8. REMOVE LEGACY STAKING FIELDS
# --------------------------------------------------

legacy_staking_cols = [
    "TT_Full_Kelly",
    "TT_Quarter_Kelly",
    "TT_Quarter_Kelly_Stake"
]


df_tt_ev.drop(
    columns=[
        col for col in legacy_staking_cols
        if col in df_tt_ev.columns
    ],
    inplace=True
)


# --------------------------------------------------
# 9. DISPLAY COPY
# --------------------------------------------------

display_tt = (
    df_tt_ev.copy()
)


pct_cols = [
    "TT_Final_Win_Prob",
    "TT_Final_Push_Prob",
    "TT_Final_Loss_Prob",
    "Break_Even_Prob_Final",
    "TT_Expected_ROI"
]


for col in pct_cols:

    if col in display_tt.columns:

        display_tt[col] = (
            display_tt[col]
            * 100
        ).round(2)


# --------------------------------------------------
# 10. MAIN TEAM-TOTAL EV BOARD
#
# Sorted for inspection only.
# Not an automatic betting ranking.
# --------------------------------------------------

display_tt = (
    display_tt
    .sort_values(
        "TT_Expected_ROI",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "\n" + "=" * 115
)

print(
    "CFB1 TEAM-TOTAL EV BOARD"
)

print(
    "=" * 115
)


display(
    display_tt[
        [
            "Game",
            "Team",
            "TT_Direction",
            "Sportsbook",
            "Line",
            "American_Odds",

            "TT_Model_Adjustment",
            "TT_Candidate_Edge",

            "TT_Final_Win_Prob",
            "TT_Final_Push_Prob",
            "TT_Final_Loss_Prob",

            "Break_Even_Prob_Final",
            "TT_Expected_ROI",

            "Decision_Status",
            "TT_Decision_Warning",
            "TT_EV_Status"
        ]
    ]
)


# --------------------------------------------------
# 11. QUALITATIVE-ONLY REMINDER
# --------------------------------------------------

qualitative_rows = (
    df_tt_validated[
        df_tt_validated[
            "TT_Quantitative_Status"
        ]
        ==
        "QUALITATIVE_REVIEW_ONLY"
    ]
    .copy()
)


print(
    "\n" + "=" * 115
)

print(
    "QUALITATIVE-ONLY TEAM-TOTAL ROUTES"
)

print(
    "=" * 115
)


if len(
    qualitative_rows
) > 0:

    display(
        qualitative_rows[
            [
                "Game",
                "Team",
                "TT_Direction",
                "Sportsbook",
                "Line",
                "American_Odds",
                "TT_Model_Adjustment",
                "Decision_Status",
                "TT_Model_Direction_Status"
            ]
        ]
        .reset_index(drop=True)
    )

else:

    print(
        "No qualitative-only team-total routes."
    )


# --------------------------------------------------
# 12. VALIDATION
# --------------------------------------------------

prob_sum = (
    df_tt_ev[
        "TT_Final_Win_Prob"
    ]
    +
    df_tt_ev[
        "TT_Final_Push_Prob"
    ]
    +
    df_tt_ev[
        "TT_Final_Loss_Prob"
    ]
)


prob_errors = (
    np.abs(
        prob_sum - 1
    ) > 1e-9
).sum()


legacy_remaining = [
    col for col in legacy_staking_cols
    if col in df_tt_ev.columns
]


print(
    "\n" + "=" * 115
)

print(
    "CELL 60 VALIDATION"
)

print(
    "=" * 115
)


print(
    "Quantitative rows evaluated:",
    len(df_tt_ev)
)

print(
    "Probability rows not summing to 1:",
    int(prob_errors)
)

print(
    "Positive-EV rows:",
    int(
        (
            df_tt_ev[
                "TT_Expected_ROI"
            ] > 0
        ).sum()
    )
)

print(
    "Negative/zero-EV rows:",
    int(
        (
            df_tt_ev[
                "TT_Expected_ROI"
            ] <= 0
        ).sum()
    )
)

print(
    "Qualitative-only rows excluded from quantitative EV:",
    len(qualitative_rows)
)

print(
    "Legacy Kelly/stake columns remaining:",
    legacy_remaining
)


if prob_errors != 0:
    raise ValueError(
        "Team-total probability validation failed."
    )

if legacy_remaining:
    raise ValueError(
        "Legacy team-total Kelly/stake columns remain."
    )


print(
    "\n" + "=" * 115
)

print(
    "CELL 60 COMPLETE."
)

print(
    "CFB1 PRICES QUANTITATIVE TEAM-TOTAL ROUTES "
    "WITH PROBABILITY + BREAK-EVEN + EXPECTED ROI."
)

print(
    "QUALITATIVE-ONLY ROUTES RECEIVE "
    "NO QUANTITATIVE EV ENDORSEMENT."
)

print(
    "KELLY AND STAKE SIZING ARE NOT CALCULATED."
)

print(
    "NO AUTOMATIC BET SELECTION PERFORMED."
)

CFB1 TEAM-TOTAL EV / PRICING

CFB1 TEAM-TOTAL EV BOARD


,Game,Team,TT_Direction,Sportsbook,Line,American_Odds,TT_Model_Adjustment,TT_Candidate_Edge,TT_Final_Win_Prob,TT_Final_Push_Prob,TT_Final_Loss_Prob,Break_Even_Prob_Final,TT_Expected_ROI,Decision_Status,TT_Decision_Warning,TT_EV_Status
0,Stanford Cardinal at Notre Dame Fighting Irish,Stanford Cardinal,UNDER,FanDuel,5.5,190,-8.325,6.325,72.16,0.0,27.84,34.48,109.26,CONFIRMED_BUT_LARGE_SPREAD,,STRONG_PRICE_EDGE
1,Stanford Cardinal at Notre Dame Fighting Irish,Stanford Cardinal,UNDER,FanDuel,4.5,196,-8.325,5.325,69.34,0.0,30.66,33.78,105.26,CONFIRMED_BUT_LARGE_SPREAD,,STRONG_PRICE_EDGE
2,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,OVER,DraftKings,48.5,145,14.310,10.310,82.82,0.0,17.18,40.82,102.90,CONFIRMED_BUT_LARGE_SPREAD,,STRONG_PRICE_EDGE
3,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,OVER,FanDuel,45.5,112,14.310,13.310,89.58,0.0,10.42,47.17,89.92,CONFIRMED_BUT_LARGE_SPREAD,,STRONG_PRICE_EDGE
4,Stanford Cardinal at Notre Dame Fighting Irish,Stanford Cardinal,UNDER,FanDuel,6.5,142,-8.325,7.325,75.21,0.0,24.79,41.32,82.02,CONFIRMED_BUT_LARGE_SPREAD,,STRONG_PRICE_EDGE
5,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,OVER,DraftKings,44.5,-120,14.310,14.310,91.35,0.0,8.65,54.55,67.48,CONFIRMED_BUT_LARGE_SPREAD,,STRONG_PRICE_EDGE
6,Nevada Wolf Pack at UTEP Miners,Nevada Wolf Pack,OVER,DraftKings,31.5,135,6.675,4.675,65.67,0.0,34.33,42.55,54.33,CONFIRMED,,STRONG_PRICE_EDGE
7,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,OVER,FanDuel,28.5,136,6.065,4.065,63.77,0.0,36.23,42.37,50.49,STRONG_MARKET_CONFLICT,STRONG_MARKET_CONFLICT,STRONG_PRICE_EDGE
8,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,OVER,FanDuel,45.5,146,5.765,2.765,58.99,0.0,41.01,40.65,45.13,CONFIRMED_BUT_LARGE_SPREAD,,STRONG_PRICE_EDGE
9,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,OVER,DraftKings,27.5,110,6.065,5.065,67.41,0.0,32.59,47.62,41.57,STRONG_MARKET_CONFLICT,STRONG_MARKET_CONFLICT,STRONG_PRICE_EDGE



QUALITATIVE-ONLY TEAM-TOTAL ROUTES


,Game,Team,TT_Direction,Sportsbook,Line,American_Odds,TT_Model_Adjustment,Decision_Status,TT_Model_Direction_Status
0,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,OVER,FanDuel,25.5,-115,-2.275,CONFIRMED,OPPOSED
1,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,OVER,FanDuel,26.5,-106,-2.275,CONFIRMED,OPPOSED
2,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,OVER,DraftKings,30.5,175,-2.275,CONFIRMED,OPPOSED
3,Jacksonville State Gamecocks at Kennesaw State...,Jacksonville State Gamecocks,OVER,FanDuel,28.5,146,-2.275,CONFIRMED,OPPOSED



CELL 60 VALIDATION
Quantitative rows evaluated: 28
Probability rows not summing to 1: 0
Positive-EV rows: 28
Negative/zero-EV rows: 0
Qualitative-only rows excluded from quantitative EV: 4
Legacy Kelly/stake columns remaining: []

CELL 60 COMPLETE.
CFB1 PRICES QUANTITATIVE TEAM-TOTAL ROUTES WITH PROBABILITY + BREAK-EVEN + EXPECTED ROI.
QUALITATIVE-ONLY ROUTES RECEIVE NO QUANTITATIVE EV ENDORSEMENT.
KELLY AND STAKE SIZING ARE NOT CALCULATED.
NO AUTOMATIC BET SELECTION PERFORMED.


In [ ]:
# ==========================================
# CELL 61 — COMPACT CFB1 HANDOFF
# ==========================================

import numpy as np
import pandas as pd

print("=" * 115)
print("CFB1 COMPACT HANDOFF")
print("=" * 115)


# --------------------------------------------------
# 1. REQUIRED OBJECTS
# --------------------------------------------------

required_objects = [
    "df_ev",
    "df_tt_ev",
    "df_tt_validated"
]

for obj in required_objects:

    if obj not in globals():
        raise NameError(
            f"{obj} missing. Run the earlier CFB1 cells first."
        )


# --------------------------------------------------
# 2. PRICING-STATUS HELPER
#
# Uses FINAL ROI available for the market.
# --------------------------------------------------

def pricing_status_from_roi(roi):

    if pd.isna(roi):
        return "NO_QUANTITATIVE_PRICE"

    roi = float(roi)

    if roi >= 0.05:
        return "STRONG_PRICE_EDGE"

    if roi >= 0.02:
        return "POSITIVE_EV"

    if roi > 0:
        return "THIN_POSITIVE_EV"

    return "NON_POSITIVE_EV"


# --------------------------------------------------
# 3. FULL-GAME SPREAD EXPORT
#
# Cell 57A push-adjusted values are canonical.
# --------------------------------------------------

required_spread_cols = [
    "Push_Adjusted_Win_Prob",
    "Push_Adjusted_Push_Prob",
    "Push_Adjusted_ROI",
    "Break_Even_Prob"
]

missing_spread_cols = [
    col for col in required_spread_cols
    if col not in df_ev.columns
]

if missing_spread_cols:
    raise ValueError(
        "Old Cell 57A outputs missing: "
        + ", ".join(missing_spread_cols)
    )


spread_export = df_ev.copy()


spread_export[
    "Market_Family"
] = "FULL_GAME_SPREAD"


spread_export[
    "Direction"
] = "SPREAD"


spread_export[
    "Bet_Label"
] = (
    spread_export[
        "Team"
    ].astype(str)
    +
    " "
    +
    spread_export[
        "Line"
    ].astype(str)
)


spread_export[
    "Model_Adjustment"
] = (
    spread_export[
        "Candidate_Cushion"
    ]
)


spread_export[
    "Model_Probability"
] = (
    spread_export[
        "Push_Adjusted_Win_Prob"
    ]
)


spread_export[
    "Push_Probability"
] = (
    spread_export[
        "Push_Adjusted_Push_Prob"
    ]
)


spread_export[
    "Break_Even_Prob"
] = (
    spread_export[
        "Break_Even_Prob"
    ]
)


spread_export[
    "Expected_ROI_Final"
] = (
    spread_export[
        "Push_Adjusted_ROI"
    ]
)


spread_export[
    "Pricing_Status"
] = (
    spread_export[
        "Expected_ROI_Final"
    ].apply(
        pricing_status_from_roi
    )
)


spread_export[
    "Warning"
] = (
    spread_export[
        "Decision_Warning"
    ]
    .fillna("")
)


# --------------------------------------------------
# 4. QUANTITATIVE TEAM-TOTAL EXPORT
# --------------------------------------------------

tt_export = df_tt_ev.copy()


tt_export[
    "Market_Family"
] = "TEAM_TOTAL"


tt_export[
    "Direction"
] = (
    tt_export[
        "TT_Direction"
    ]
)


tt_export[
    "Bet_Label"
] = (
    tt_export[
        "Team"
    ].astype(str)
    +
    " TT "
    +
    tt_export[
        "TT_Direction"
    ].astype(str)
    +
    " "
    +
    tt_export[
        "Line"
    ].astype(str)
)


tt_export[
    "Model_Adjustment"
] = (
    tt_export[
        "TT_Model_Adjustment"
    ]
)


tt_export[
    "Model_Probability"
] = (
    tt_export[
        "TT_Final_Win_Prob"
    ]
)


tt_export[
    "Push_Probability"
] = (
    tt_export[
        "TT_Final_Push_Prob"
    ]
)


tt_export[
    "Break_Even_Prob"
] = (
    tt_export[
        "Break_Even_Prob_Final"
    ]
)


tt_export[
    "Expected_ROI_Final"
] = (
    tt_export[
        "TT_Expected_ROI"
    ]
)


tt_export[
    "Pricing_Status"
] = (
    tt_export[
        "TT_EV_Status"
    ]
)


tt_export[
    "Warning"
] = (
    tt_export[
        "TT_Decision_Warning"
    ]
    .fillna("")
)


# --------------------------------------------------
# 5. QUALITATIVE-ONLY TEAM-TOTAL EXPORT
#
# Preserve these routes for review, but do not
# manufacture probability or EV.
# --------------------------------------------------

qual_tt = (
    df_tt_validated[
        df_tt_validated[
            "TT_Quantitative_Status"
        ]
        ==
        "QUALITATIVE_REVIEW_ONLY"
    ]
    .copy()
)


if len(qual_tt) > 0:

    qual_tt[
        "Market_Family"
    ] = "TEAM_TOTAL"

    qual_tt[
        "Direction"
    ] = (
        qual_tt[
            "TT_Direction"
        ]
    )

    qual_tt[
        "Bet_Label"
    ] = (
        qual_tt[
            "Team"
        ].astype(str)
        +
        " TT "
        +
        qual_tt[
            "TT_Direction"
        ].astype(str)
        +
        " "
        +
        qual_tt[
            "Line"
        ].astype(str)
    )

    qual_tt[
        "Model_Adjustment"
    ] = (
        qual_tt[
            "TT_Model_Adjustment"
        ]
    )

    qual_tt[
        "Model_Probability"
    ] = np.nan

    qual_tt[
        "Push_Probability"
    ] = np.nan

    qual_tt[
        "Break_Even_Prob"
    ] = np.nan

    qual_tt[
        "Expected_ROI_Final"
    ] = np.nan

    qual_tt[
        "Pricing_Status"
    ] = "QUALITATIVE_ONLY"

    qual_tt[
        "Warning"
    ] = "MODEL_DIRECTION_OPPOSES_ROUTE"


# --------------------------------------------------
# 6. STANDARD HANDOFF SCHEMA
#
# No bankroll / Kelly / stake fields.
# --------------------------------------------------

handoff_columns = [
    "Game",
    "Market_Family",
    "Team",
    "Direction",
    "Bet_Label",
    "Sportsbook",
    "Line",
    "American_Odds",

    "Model_Adjustment",
    "Model_Probability",
    "Push_Probability",
    "Break_Even_Prob",
    "Expected_ROI_Final",

    "Decision_Status",
    "Pricing_Status",
    "Warning"
]


spread_export = (
    spread_export[
        handoff_columns
    ]
    .copy()
)


tt_export = (
    tt_export[
        handoff_columns
    ]
    .copy()
)


if len(qual_tt) > 0:

    qual_tt = (
        qual_tt[
            handoff_columns
        ]
        .copy()
    )

else:

    qual_tt = pd.DataFrame(
        columns=handoff_columns
    )


# --------------------------------------------------
# 7. COMPLETE CFB1 HANDOFF
#
# Preserve existing object name for downstream
# compatibility.
# --------------------------------------------------

df_cfb1_gem_handoff = pd.concat(
    [
        spread_export,
        tt_export,
        qual_tt
    ],
    ignore_index=True
)


# --------------------------------------------------
# 8. GEM / ARCHITECT REVIEW SET
#
# Quantitative rows:
#   positive final ROI only
#
# Qualitative rows:
#   preserved for review
#
# This is presentation filtering only.
# Full handoff remains intact.
# --------------------------------------------------

quant_review = (
    df_cfb1_gem_handoff[
        (
            df_cfb1_gem_handoff[
                "Expected_ROI_Final"
            ] > 0
        )
        &
        (
            df_cfb1_gem_handoff[
                "Pricing_Status"
            ]
            !=
            "QUALITATIVE_ONLY"
        )
    ]
    .copy()
)


qual_review = (
    df_cfb1_gem_handoff[
        df_cfb1_gem_handoff[
            "Pricing_Status"
        ]
        ==
        "QUALITATIVE_ONLY"
    ]
    .copy()
)


df_cfb1_gem_review = pd.concat(
    [
        quant_review,
        qual_review
    ],
    ignore_index=True
)


# --------------------------------------------------
# 9. SORT OUTPUTS
# --------------------------------------------------

df_cfb1_gem_handoff = (
    df_cfb1_gem_handoff
    .sort_values(
        [
            "Game",
            "Market_Family",
            "Expected_ROI_Final"
        ],
        ascending=[
            True,
            True,
            False
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)


df_cfb1_gem_review = (
    df_cfb1_gem_review
    .sort_values(
        [
            "Game",
            "Market_Family",
            "Expected_ROI_Final"
        ],
        ascending=[
            True,
            True,
            False
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)


# --------------------------------------------------
# 10. BACKWARD-COMPATIBLE DISPLAY OBJECTS
# --------------------------------------------------

display_handoff = (
    df_cfb1_gem_handoff.copy()
)

display_review = (
    df_cfb1_gem_review.copy()
)


# Convert probabilities / ROI to percentages
# for DISPLAY ONLY.
for display_obj in [
    display_handoff,
    display_review
]:

    for col in [
        "Model_Probability",
        "Push_Probability",
        "Break_Even_Prob",
        "Expected_ROI_Final"
    ]:

        display_obj[col] = (
            display_obj[col]
            .apply(
                lambda x:
                np.nan
                if pd.isna(x)
                else round(
                    float(x) * 100,
                    2
                )
            )
        )


# --------------------------------------------------
# 11. DISPLAY REVIEW CARD
# --------------------------------------------------

print(
    "\n" + "=" * 115
)

print(
    "CFB1 REVIEW CARD"
)

print(
    "=" * 115
)


display(
    display_review[
        [
            "Game",
            "Market_Family",
            "Bet_Label",
            "Sportsbook",
            "American_Odds",

            "Model_Probability",
            "Push_Probability",
            "Break_Even_Prob",
            "Expected_ROI_Final",

            "Decision_Status",
            "Pricing_Status",
            "Warning"
        ]
    ]
)


# --------------------------------------------------
# 12. VALIDATION
# --------------------------------------------------

missing_game_names = (
    df_cfb1_gem_handoff[
        "Game"
    ]
    .isna()
    .sum()
)


legacy_staking_terms = [
    "Kelly",
    "Stake",
    "Bankroll"
]


legacy_columns = [
    col
    for col in df_cfb1_gem_handoff.columns
    if any(
        term.lower()
        in col.lower()
        for term in legacy_staking_terms
    )
]


review_spreads = (
    df_cfb1_gem_review[
        "Market_Family"
    ]
    ==
    "FULL_GAME_SPREAD"
).sum()


review_tt = (
    df_cfb1_gem_review[
        "Market_Family"
    ]
    ==
    "TEAM_TOTAL"
).sum()


qualitative_count = (
    df_cfb1_gem_review[
        "Pricing_Status"
    ]
    ==
    "QUALITATIVE_ONLY"
).sum()


print(
    "\n" + "=" * 115
)

print(
    "CELL 61 VALIDATION"
)

print(
    "=" * 115
)


print(
    "All handoff rows:",
    len(df_cfb1_gem_handoff)
)

print(
    "Review rows:",
    len(df_cfb1_gem_review)
)

print(
    "Spread rows in review:",
    int(review_spreads)
)

print(
    "Team-total rows in review:",
    int(review_tt)
)

print(
    "Qualitative-only rows:",
    int(qualitative_count)
)

print(
    "Missing game names:",
    int(missing_game_names)
)

print(
    "Legacy Kelly/stake/bankroll columns:",
    legacy_columns
)


if missing_game_names != 0:
    raise ValueError(
        "Cell 61 contains rows with missing game names."
    )


if legacy_columns:
    raise ValueError(
        "Legacy staking fields remain in Cell 61 handoff."
    )


print(
    "\n" + "=" * 115
)

print(
    "CELL 61 COMPLETE."
)

print(
    "THIS IS A PRESENTATION / HANDOFF LAYER ONLY."
)

print(
    "CFB1 HANDS OFF PROBABILITIES, PRICES, "
    "EXPECTED ROI, STATUS, AND WARNINGS."
)

print(
    "BANKROLL / KELLY / STAKE SIZING "
    "BELONGS TO THE BET SLIP ARCHITECT."
)

print(
    "NO NEW MODEL SIGNALS OR BET SELECTION WERE CREATED."
)

CFB1 COMPACT HANDOFF

CFB1 REVIEW CARD


,Game,Market_Family,Bet_Label,Sportsbook,American_Odds,Model_Probability,Push_Probability,Break_Even_Prob,Expected_ROI_Final,Decision_Status,Pricing_Status,Warning
0,Ball State Cardinals at Northwestern Wildcats,TEAM_TOTAL,Northwestern Wildcats TT OVER 48.5,DraftKings,145,82.82,0.00,40.82,102.90,CONFIRMED_BUT_LARGE_SPREAD,STRONG_PRICE_EDGE,
1,Ball State Cardinals at Northwestern Wildcats,TEAM_TOTAL,Northwestern Wildcats TT OVER 45.5,FanDuel,112,89.58,0.00,47.17,89.92,CONFIRMED_BUT_LARGE_SPREAD,STRONG_PRICE_EDGE,
2,Ball State Cardinals at Northwestern Wildcats,TEAM_TOTAL,Northwestern Wildcats TT OVER 44.5,DraftKings,-120,91.35,0.00,54.55,67.48,CONFIRMED_BUT_LARGE_SPREAD,STRONG_PRICE_EDGE,
3,Buffalo Bulls at Toledo Rockets,TEAM_TOTAL,Toledo Rockets TT OVER 38.5,FanDuel,114,64.64,0.00,46.73,38.33,NEUTRAL_MARKET,STRONG_PRICE_EDGE,
4,Buffalo Bulls at Toledo Rockets,TEAM_TOTAL,Toledo Rockets TT OVER 35.5,FanDuel,-146,74.73,0.00,59.35,25.92,NEUTRAL_MARKET,STRONG_PRICE_EDGE,
5,Buffalo Bulls at Toledo Rockets,TEAM_TOTAL,Toledo Rockets TT OVER 37.5,DraftKings,-120,68.08,0.00,54.55,24.82,NEUTRAL_MARKET,STRONG_PRICE_EDGE,
6,Charlotte 49ers at North Texas Mean Green,TEAM_TOTAL,North Texas Mean Green TT OVER 45.5,FanDuel,146,58.99,0.00,40.65,45.13,CONFIRMED_BUT_LARGE_SPREAD,STRONG_PRICE_EDGE,
7,Charlotte 49ers at North Texas Mean Green,TEAM_TOTAL,North Texas Mean Green TT OVER 42.5,FanDuel,-106,69.76,0.00,51.46,35.57,CONFIRMED_BUT_LARGE_SPREAD,STRONG_PRICE_EDGE,
8,Charlotte 49ers at North Texas Mean Green,TEAM_TOTAL,North Texas Mean Green TT OVER 43.5,FanDuel,-102,66.42,0.00,50.50,31.54,CONFIRMED_BUT_LARGE_SPREAD,STRONG_PRICE_EDGE,
9,Duke Blue Devils at Georgia Tech Yellow Jackets,FULL_GAME_SPREAD,Duke Blue Devils -7.5,DraftKings,105,52.23,0.00,48.78,7.06,STRONG_MARKET_CONFLICT,STRONG_PRICE_EDGE,STRONG_MARKET_CONFLICT



CELL 61 VALIDATION
All handoff rows: 79
Review rows: 53
Spread rows in review: 21
Team-total rows in review: 32
Qualitative-only rows: 4
Missing game names: 0
Legacy Kelly/stake/bankroll columns: []

CELL 61 COMPLETE.
THIS IS A PRESENTATION / HANDOFF LAYER ONLY.
CFB1 HANDS OFF PROBABILITIES, PRICES, EXPECTED ROI, STATUS, AND WARNINGS.
BANKROLL / KELLY / STAKE SIZING BELONGS TO THE BET SLIP ARCHITECT.
NO NEW MODEL SIGNALS OR BET SELECTION WERE CREATED.


In [ ]:
# ==========================================
# CELL 61B — CFB1 FULL CONTEXT CHECKPOINT
# ==========================================

print("=" * 115)
print("CFB1 FULL CONTEXT V1 CHECKPOINT")
print("=" * 115)

if "df_cfb1_gem_handoff" not in globals():
    raise NameError(
        "df_cfb1_gem_handoff missing. Run Cell 61 first."
    )

# Preserve the richer context layer
CFB1_FULL_CONTEXT_V1 = (
    df_cfb1_gem_handoff.copy()
)

# CSV text version for Gem / archive use
CFB1_FULL_CONTEXT_V1_CSV = (
    CFB1_FULL_CONTEXT_V1
    .to_csv(index=False)
)

print(
    "Rows preserved:",
    len(CFB1_FULL_CONTEXT_V1)
)

print(
    "Games preserved:",
    CFB1_FULL_CONTEXT_V1["Game"].nunique()
)

print(
    "Market families:",
    sorted(
        CFB1_FULL_CONTEXT_V1[
            "Market_Family"
        ].dropna().unique().tolist()
    )
)

print(
    "\nCheckpoint objects created:"
)

print(
    "CFB1_FULL_CONTEXT_V1"
)

print(
    "CFB1_FULL_CONTEXT_V1_CSV"
)

print(
    "\nCELL 61B COMPLETE."
)

print(
    "THE FULL ANALYTICAL HANDOFF IS NOW PRESERVED "
    "BEFORE DECISION-CARD COMPRESSION."
)


CFB1 FULL CONTEXT V1 CHECKPOINT
Rows preserved: 79
Games preserved: 9
Market families: ['FULL_GAME_SPREAD', 'TEAM_TOTAL']

Checkpoint objects created:
CFB1_FULL_CONTEXT_V1
CFB1_FULL_CONTEXT_V1_CSV

CELL 61B COMPLETE.
THE FULL ANALYTICAL HANDOFF IS NOW PRESERVED BEFORE DECISION-CARD COMPRESSION.


In [ ]:
# ==========================================
# CELL 62 — CFB1 DECISION CARD V1
# ==========================================

import numpy as np
import pandas as pd

print("=" * 115)
print("CFB1 DECISION CARD V1")
print("=" * 115)


# --------------------------------------------------
# 1. REQUIRED OBJECTS
# --------------------------------------------------

required_objects = [
    "df_ev",
    "df_tt_ev",
    "df_tt_validated",
    "CFB1_FULL_CONTEXT_V1"
]

for obj in required_objects:
    if obj not in globals():
        raise NameError(
            f"{obj} missing. Run the earlier CFB1 cells first."
        )


# --------------------------------------------------
# 2. REQUIRE 57A PUSH-ADJUSTED SPREAD METRICS
# --------------------------------------------------

required_spread_cols = [
    "Push_Adjusted_Win_Prob",
    "Push_Adjusted_Push_Prob",
    "Push_Adjusted_ROI"
]

missing_cols = [
    c for c in required_spread_cols
    if c not in df_ev.columns
]

if missing_cols:
    raise ValueError(
        "Old Cell 57A metrics are missing: "
        + ", ".join(missing_cols)
    )


# --------------------------------------------------
# 3. BUILD FULL-GAME SPREAD CANDIDATES
# --------------------------------------------------

spread = df_ev.copy()

spread = spread[
    spread["Push_Adjusted_ROI"] > 0
].copy()


spread["Market_Family"] = (
    "FULL_GAME_SPREAD"
)

spread["Direction"] = (
    "SPREAD"
)

spread["Bet_Label"] = (
    spread["Team"].astype(str)
    + " "
    + spread["Line"].astype(str)
)

spread["Model_Probability"] = (
    spread["Push_Adjusted_Win_Prob"]
)

spread["Push_Probability"] = (
    spread["Push_Adjusted_Push_Prob"]
)

spread["Expected_ROI"] = (
    spread["Push_Adjusted_ROI"]
)

spread["Model_Adjustment"] = (
    spread["Candidate_Cushion"]
)

spread["Break_Even"] = (
    spread["Break_Even_Prob"]
)

spread["Model_Status"] = (
    "FULL_GAME_SPREAD_V1"
)

spread["Warning_Final"] = (
    spread["Decision_Warning"]
    .fillna("")
)


# --------------------------------------------------
# 4. CURRENT / MAIN SPREAD ANCHOR
# --------------------------------------------------

spread["Anchor_Line"] = (
    spread["Market_Model_Spread"]
)


# --------------------------------------------------
# 5. CHOOSE PRIMARY + VALUE ALT FOR SPREADS
# --------------------------------------------------

spread_card_parts = []


for (game, team), group in spread.groupby(
    [
        "Game",
        "Team"
    ]
):

    group = group.copy()

    group[
        "Anchor_Distance"
    ] = (
        group["Line"]
        -
        group["Anchor_Line"]
    ).abs()


    primary = (
        group
        .sort_values(
            [
                "Anchor_Distance",
                "American_Odds"
            ],
            ascending=[
                True,
                False
            ]
        )
        .iloc[[0]]
        .copy()
    )

    primary[
        "Card_Role"
    ] = "PRIMARY"

    spread_card_parts.append(
        primary
    )


    remaining = group.drop(
        index=primary.index
    )


    if len(remaining) > 0:

        value_alt = (
            remaining
            .sort_values(
                "Expected_ROI",
                ascending=False
            )
            .iloc[[0]]
            .copy()
        )

        value_alt[
            "Card_Role"
        ] = "VALUE_ALT"

        spread_card_parts.append(
            value_alt
        )


spread_card = (
    pd.concat(
        spread_card_parts,
        ignore_index=True
    )
    if spread_card_parts
    else pd.DataFrame()
)


# --------------------------------------------------
# 6. BUILD QUANTITATIVE TEAM-TOTAL CANDIDATES
# --------------------------------------------------

tt = df_tt_ev[
    df_tt_ev[
        "TT_Expected_ROI"
    ] > 0
].copy()


tt["Market_Family"] = (
    "TEAM_TOTAL"
)

tt["Direction"] = (
    tt["TT_Direction"]
)

tt["Bet_Label"] = (
    tt["Team"].astype(str)
    + " TT "
    + tt["TT_Direction"].astype(str)
    + " "
    + tt["Line"].astype(str)
)

tt["Model_Probability"] = (
    tt["TT_Final_Win_Prob"]
)

tt["Push_Probability"] = (
    tt["TT_Final_Push_Prob"]
)

tt["Expected_ROI"] = (
    tt["TT_Expected_ROI"]
)

tt["Model_Adjustment"] = (
    tt["TT_Model_Adjustment"]
)

tt["Break_Even"] = (
    tt["Break_Even_Prob_Final"]
)

tt["Anchor_Line"] = (
    tt["Market_TT_Anchor"]
)

tt["Model_Status"] = (
    "TEAM_TOTAL_V1_EXPERIMENTAL"
)

tt["Warning_Final"] = (
    tt["TT_Decision_Warning"]
    .fillna("")
)


# --------------------------------------------------
# 7. CHOOSE PRIMARY + VALUE ALT FOR TEAM TOTALS
# --------------------------------------------------

tt_card_parts = []


for (
    game,
    team,
    direction
), group in tt.groupby(
    [
        "Game",
        "Team",
        "TT_Direction"
    ]
):

    group = group.copy()

    group[
        "Anchor_Distance"
    ] = (
        group["Line"]
        -
        group["Anchor_Line"]
    ).abs()


    primary = (
        group
        .sort_values(
            [
                "Anchor_Distance",
                "American_Odds"
            ],
            ascending=[
                True,
                False
            ]
        )
        .iloc[[0]]
        .copy()
    )

    primary[
        "Card_Role"
    ] = "PRIMARY"

    tt_card_parts.append(
        primary
    )


    remaining = group.drop(
        index=primary.index
    )


    if len(remaining) > 0:

        value_alt = (
            remaining
            .sort_values(
                "Expected_ROI",
                ascending=False
            )
            .iloc[[0]]
            .copy()
        )

        value_alt[
            "Card_Role"
        ] = "VALUE_ALT"

        tt_card_parts.append(
            value_alt
        )


tt_card = (
    pd.concat(
        tt_card_parts,
        ignore_index=True
    )
    if tt_card_parts
    else pd.DataFrame()
)


# --------------------------------------------------
# 8. QUALITATIVE-ONLY TEAM TOTALS
#
# One representative line per route.
# No probability or EV.
# --------------------------------------------------

qual = df_tt_validated[
    df_tt_validated[
        "TT_Quantitative_Status"
    ]
    ==
    "QUALITATIVE_REVIEW_ONLY"
].copy()


qual_parts = []


for (
    game,
    team,
    direction
), group in qual.groupby(
    [
        "Game",
        "Team",
        "TT_Direction"
    ]
):

    group = group.copy()

    group[
        "Anchor_Distance"
    ] = (
        group["Line"]
        -
        group["Market_TT_Anchor"]
    ).abs()


    representative = (
        group
        .sort_values(
            "Anchor_Distance"
        )
        .iloc[[0]]
        .copy()
    )


    representative[
        "Market_Family"
    ] = "TEAM_TOTAL"

    representative[
        "Direction"
    ] = representative[
        "TT_Direction"
    ]

    representative[
        "Bet_Label"
    ] = (
        representative[
            "Team"
        ].astype(str)
        +
        " TT "
        +
        representative[
            "TT_Direction"
        ].astype(str)
        +
        " "
        +
        representative[
            "Line"
        ].astype(str)
    )

    representative[
        "Model_Probability"
    ] = np.nan

    representative[
        "Push_Probability"
    ] = np.nan

    representative[
        "Expected_ROI"
    ] = np.nan

    representative[
        "Break_Even"
    ] = np.nan

    representative[
        "Model_Adjustment"
    ] = representative[
        "TT_Model_Adjustment"
    ]

    representative[
        "Model_Status"
    ] = "QUALITATIVE_ONLY"

    representative[
        "Warning_Final"
    ] = (
        "MODEL_DIRECTION_OPPOSES_ROUTE"
    )

    representative[
        "Card_Role"
    ] = "QUALITATIVE_REVIEW"

    qual_parts.append(
        representative
    )


qual_card = (
    pd.concat(
        qual_parts,
        ignore_index=True
    )
    if qual_parts
    else pd.DataFrame()
)


# --------------------------------------------------
# 9. STANDARDIZE OUTPUT COLUMNS
#
# Staking fields intentionally excluded.
# --------------------------------------------------

card_columns = [
    "Game",
    "Market_Family",
    "Card_Role",

    "Team",
    "Direction",
    "Bet_Label",

    "Sportsbook",
    "Line",
    "American_Odds",

    "Model_Adjustment",
    "Model_Probability",
    "Push_Probability",
    "Break_Even",
    "Expected_ROI",

    "Decision_Status",
    "Model_Status",
    "Warning_Final"
]


spread_card = (
    spread_card[
        card_columns
    ]
    .copy()
)


tt_card = (
    tt_card[
        card_columns
    ]
    .copy()
)


if len(qual_card) > 0:

    qual_card = (
        qual_card[
            card_columns
        ]
        .copy()
    )

else:

    qual_card = pd.DataFrame(
        columns=card_columns
    )


# --------------------------------------------------
# 10. COMBINE FINAL DECISION CARD
# --------------------------------------------------

CFB1_DECISION_CARD_V1 = pd.concat(
    [
        spread_card,
        tt_card,
        qual_card
    ],
    ignore_index=True
)


CFB1_DECISION_CARD_V1 = (
    CFB1_DECISION_CARD_V1
    .sort_values(
        [
            "Game",
            "Market_Family",
            "Card_Role"
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------
# 11. CREATE CSV VERSION
# --------------------------------------------------

CFB1_DECISION_CARD_V1_CSV = (
    CFB1_DECISION_CARD_V1
    .to_csv(index=False)
)


# --------------------------------------------------
# 12. HUMAN-FRIENDLY DISPLAY
# --------------------------------------------------

display_card = (
    CFB1_DECISION_CARD_V1.copy()
)


for col in [
    "Model_Probability",
    "Push_Probability",
    "Break_Even",
    "Expected_ROI"
]:

    display_card[col] = (
        display_card[col]
        .apply(
            lambda x:
            np.nan
            if pd.isna(x)
            else round(
                float(x) * 100,
                1
            )
        )
    )


display_card[
    "Model_Adjustment"
] = (
    display_card[
        "Model_Adjustment"
    ].round(2)
)


print(
    "\n" + "=" * 115
)

print(
    "CFB1 DECISION CARD V1"
)

print(
    "=" * 115
)


display(
    display_card[
        [
            "Game",
            "Market_Family",
            "Card_Role",
            "Bet_Label",

            "Sportsbook",
            "American_Odds",

            "Model_Probability",
            "Push_Probability",
            "Break_Even",
            "Expected_ROI",

            "Decision_Status",
            "Model_Status",
            "Warning_Final"
        ]
    ]
)


# --------------------------------------------------
# 13. VALIDATION
# --------------------------------------------------

legacy_terms = [
    "Kelly",
    "Stake",
    "Bankroll"
]


legacy_columns = [
    col
    for col in CFB1_DECISION_CARD_V1.columns
    if any(
        term.lower()
        in col.lower()
        for term in legacy_terms
    )
]


print(
    "\n" + "=" * 115
)

print(
    "CELL 62 VALIDATION"
)

print(
    "=" * 115
)


print(
    "Decision-card rows:",
    len(CFB1_DECISION_CARD_V1)
)

print(
    "Games represented:",
    CFB1_DECISION_CARD_V1[
        "Game"
    ].nunique()
)

print(
    "Primary rows:",
    int(
        (
            CFB1_DECISION_CARD_V1[
                "Card_Role"
            ]
            ==
            "PRIMARY"
        ).sum()
    )
)

print(
    "Value-alt rows:",
    int(
        (
            CFB1_DECISION_CARD_V1[
                "Card_Role"
            ]
            ==
            "VALUE_ALT"
        ).sum()
    )
)

print(
    "Qualitative-review rows:",
    int(
        (
            CFB1_DECISION_CARD_V1[
                "Card_Role"
            ]
            ==
            "QUALITATIVE_REVIEW"
        ).sum()
    )
)

print(
    "Spread rows:",
    int(
        (
            CFB1_DECISION_CARD_V1[
                "Market_Family"
            ]
            ==
            "FULL_GAME_SPREAD"
        ).sum()
    )
)

print(
    "Team-total rows:",
    int(
        (
            CFB1_DECISION_CARD_V1[
                "Market_Family"
            ]
            ==
            "TEAM_TOTAL"
        ).sum()
    )
)

print(
    "Legacy Kelly/stake/bankroll columns:",
    legacy_columns
)


if legacy_columns:
    raise ValueError(
        "Legacy staking fields remain in Decision Card."
    )


print(
    "\nObjects created:"
)

print(
    "CFB1_DECISION_CARD_V1"
)

print(
    "CFB1_DECISION_CARD_V1_CSV"
)


print(
    "\nCELL 62 COMPLETE."
)

print(
    "DECISION CARD IS A COMPACT PRESENTATION LAYER."
)

print(
    "CFB1 PROVIDES MODEL PROBABILITY, PUSH RISK, "
    "BREAK-EVEN, EXPECTED ROI, STATUS, AND WARNINGS."
)

print(
    "BANKROLL / KELLY / STAKE SIZING IS NOT PART "
    "OF THE CFB1 DECISION CARD."
)

print(
    "NO NEW BET SELECTION LOGIC WAS CREATED."
)

CFB1 DECISION CARD V1

CFB1 DECISION CARD V1


,Game,Market_Family,Card_Role,Bet_Label,Sportsbook,American_Odds,Model_Probability,Push_Probability,Break_Even,Expected_ROI,Decision_Status,Model_Status,Warning_Final
0,Ball State Cardinals at Northwestern Wildcats,TEAM_TOTAL,PRIMARY,Northwestern Wildcats TT OVER 44.5,DraftKings,-120,91.4,0.0,54.5,67.5,CONFIRMED_BUT_LARGE_SPREAD,TEAM_TOTAL_V1_EXPERIMENTAL,
1,Ball State Cardinals at Northwestern Wildcats,TEAM_TOTAL,VALUE_ALT,Northwestern Wildcats TT OVER 48.5,DraftKings,145,82.8,0.0,40.8,102.9,CONFIRMED_BUT_LARGE_SPREAD,TEAM_TOTAL_V1_EXPERIMENTAL,
2,Buffalo Bulls at Toledo Rockets,TEAM_TOTAL,PRIMARY,Toledo Rockets TT OVER 37.5,DraftKings,-120,68.1,0.0,54.5,24.8,NEUTRAL_MARKET,TEAM_TOTAL_V1_EXPERIMENTAL,
3,Buffalo Bulls at Toledo Rockets,TEAM_TOTAL,VALUE_ALT,Toledo Rockets TT OVER 38.5,FanDuel,114,64.6,0.0,46.7,38.3,NEUTRAL_MARKET,TEAM_TOTAL_V1_EXPERIMENTAL,
4,Charlotte 49ers at North Texas Mean Green,TEAM_TOTAL,PRIMARY,North Texas Mean Green TT OVER 42.5,FanDuel,-106,69.8,0.0,51.5,35.6,CONFIRMED_BUT_LARGE_SPREAD,TEAM_TOTAL_V1_EXPERIMENTAL,
5,Charlotte 49ers at North Texas Mean Green,TEAM_TOTAL,VALUE_ALT,North Texas Mean Green TT OVER 45.5,FanDuel,146,59.0,0.0,40.7,45.1,CONFIRMED_BUT_LARGE_SPREAD,TEAM_TOTAL_V1_EXPERIMENTAL,
6,Duke Blue Devils at Georgia Tech Yellow Jackets,FULL_GAME_SPREAD,PRIMARY,Duke Blue Devils -3.5,DraftKings,-167,63.0,0.0,62.5,0.8,STRONG_MARKET_CONFLICT,FULL_GAME_SPREAD_V1,STRONG_MARKET_CONFLICT
7,Duke Blue Devils at Georgia Tech Yellow Jackets,FULL_GAME_SPREAD,VALUE_ALT,Duke Blue Devils -7.5,DraftKings,105,52.2,0.0,48.8,7.1,STRONG_MARKET_CONFLICT,FULL_GAME_SPREAD_V1,STRONG_MARKET_CONFLICT
8,Duke Blue Devils at Georgia Tech Yellow Jackets,TEAM_TOTAL,PRIMARY,Duke Blue Devils TT OVER 26.5,DraftKings,-115,70.7,0.0,53.5,32.2,STRONG_MARKET_CONFLICT,TEAM_TOTAL_V1_EXPERIMENTAL,STRONG_MARKET_CONFLICT
9,Duke Blue Devils at Georgia Tech Yellow Jackets,TEAM_TOTAL,VALUE_ALT,Duke Blue Devils TT OVER 28.5,FanDuel,136,63.8,0.0,42.4,50.5,STRONG_MARKET_CONFLICT,TEAM_TOTAL_V1_EXPERIMENTAL,STRONG_MARKET_CONFLICT



CELL 62 VALIDATION
Decision-card rows: 25
Games represented: 9
Primary rows: 12
Value-alt rows: 12
Qualitative-review rows: 1
Spread rows: 8
Team-total rows: 17
Legacy Kelly/stake/bankroll columns: []

Objects created:
CFB1_DECISION_CARD_V1
CFB1_DECISION_CARD_V1_CSV

CELL 62 COMPLETE.
DECISION CARD IS A COMPACT PRESENTATION LAYER.
CFB1 PROVIDES MODEL PROBABILITY, PUSH RISK, BREAK-EVEN, EXPECTED ROI, STATUS, AND WARNINGS.
BANKROLL / KELLY / STAKE SIZING IS NOT PART OF THE CFB1 DECISION CARD.
NO NEW BET SELECTION LOGIC WAS CREATED.


In [ ]:
# ==========================================
# CELL 66 — BUILD CFB1 FULL CONTEXT V2
# ==========================================

import numpy as np
import pandas as pd

print("=" * 115)
print("CFB1 FULL CONTEXT V2")
print("=" * 115)


# --------------------------------------------------
# 1. REQUIRED OBJECTS
# --------------------------------------------------

required_objects = [
    "df_gem_export",
    "df_phase1_shrinkage",
    "df_key_value",
    "CFB1_FULL_CONTEXT_V1",
    "CFB1_DECISION_CARD_V1"
]

for obj in required_objects:
    if obj not in globals():
        raise NameError(
            f"{obj} missing. "
            "Run the earlier CFB1 cells first."
        )


# --------------------------------------------------
# 2. GAME-LEVEL DECISION / MARKET CONTEXT
#
# df_gem_export already contains the cleanest
# Phase 1 + Module 3 + Covers + routing story.
# --------------------------------------------------

decision_cols = [
    "Game",
    "Model_Side",
    "Review_Class",
    "Phase1_Reason",
    "M3_Support",
    "M3_Conflict",

    "Favorite_Spread_Size",
    "Spread_Status",

    "Open_Home_Spread",
    "Current_Home_Spread",

    "Movement_Toward",
    "Movement_Class",

    "Key_Numbers_Crossed",
    "Market_Relationship",

    "Decision_Status",
    "Eligible_Market_Routes",
    "Decision_Note"
]


game_context = (
    df_gem_export[
        decision_cols
    ]
    .copy()
)


# --------------------------------------------------
# 3. ADD MOVEMENT SIZE / KEY CROSS BOOLEAN
#
# These live in df_final_decision when available.
# --------------------------------------------------

if "df_final_decision" in globals():

    extra_market_cols = [
        "Game",
        "Movement_Size",
        "Key_Cross"
    ]

    available_extra = [
        c for c in extra_market_cols
        if c in df_final_decision.columns
    ]

    if "Game" in available_extra:

        extras = (
            df_final_decision[
                available_extra
            ]
            .drop_duplicates(
                subset=["Game"]
            )
            .copy()
        )

        game_context = (
            game_context.merge(
                extras,
                on="Game",
                how="left",
                validate="one_to_one"
            )
        )


# --------------------------------------------------
# 4. SELECT RICH PHASE-1 MODEL / MATCHUP CONTEXT
#
# We do not need every development column.
# Keep the fields most useful for explaining
# why a game qualified and what matchup signals exist.
# --------------------------------------------------

phase1_cols = [
    "Game",
    "Home_Team",
    "Away_Team",
    "Kickoff_ET",

    # Current market snapshot
    "DK_Spread",
    "FD_Spread",
    "DK_Total",
    "FD_Total",

    # EPA
    "Home_Off_EPA",
    "Home_Def_EPA",
    "Home_Net_EPA_Margin",
    "Away_Off_EPA",
    "Away_Def_EPA",
    "Away_Net_EPA_Margin",
    "Net_EPA_Edge_Home",
    "EPA_Edge_Value",
    "EPA_Edge_Team",
    "EPA_Trigger",

    # FEI family
    "Home_FEI",
    "Home_OFEI",
    "Home_DFEI",
    "Home_SFEI",
    "Home_OAY",
    "Home_DAY",

    "Away_FEI",
    "Away_OFEI",
    "Away_DFEI",
    "Away_SFEI",
    "Away_OAY",
    "Away_DAY",

    "FEI_Model_Diff_Home",
    "FEI_Model_Favorite",
    "FEI_Model_Margin",
    "FEI_Model_Home_Spread",
    "FEI_Edge_Conservative",

    # Module 3 matchup flags
    "Away_Havoc_Trap",
    "Home_Havoc_Trap",

    "Away_LineYards_Push",
    "Home_LineYards_Push",

    "Away_Explosiveness_Fraud",
    "Home_Explosiveness_Fraud",

    "Away_OAY_DAY_Edge",
    "Home_OAY_DAY_Edge",

    # Spread/key environment
    "Consensus_Spread",
    "Nearest_Key_Number",
    "Distance_From_Key",
    "Within_0.5_Key",
    "Within_1.0_Key",
    "Spread_Size_Bucket",

    # Raw / retained model
    "Consensus_Home_Spread",
    "Market_Home_Margin",
    "EPA_Model_Home_Margin",
    "EPA_Model_Home_Spread",
    "FEI_Model_Home_Margin",
    "Blended_Home_Margin_Raw",
    "Blended_Home_Spread_Raw",
    "Blended_Edge_Home_Raw",
    "Edge_Retained_40",
    "Shrunk_Home_Spread_40",
    "Raw_Edge_Size"
]


phase1_available = [
    c for c in phase1_cols
    if c in df_phase1_shrinkage.columns
]


phase1_context = (
    df_phase1_shrinkage[
        phase1_available
    ]
    .drop_duplicates(
        subset=["Game"]
    )
    .copy()
)


game_context = (
    game_context.merge(
        phase1_context,
        on="Game",
        how="left",
        validate="one_to_one"
    )
)


# --------------------------------------------------
# 5. KEY-NUMBER / RETAINED-LINE CONTEXT
# --------------------------------------------------

key_cols = [
    "Game",
    "Market_Model_Spread",
    "Retained_Model_Spread",
    "Model_Cushion_Points",
    "Keys_Between_Market_And_Model",
    "Key_Barrier_Present",
    "Next_Adverse_Key",
    "Line_Value_Status",
    "Key_Tradeoff_Note"
]


key_available = [
    c for c in key_cols
    if c in df_key_value.columns
]


key_context = (
    df_key_value[
        key_available
    ]
    .drop_duplicates(
        subset=["Game"]
    )
    .copy()
)


game_context = (
    game_context.merge(
        key_context,
        on="Game",
        how="left",
        validate="one_to_one"
    )
)


# --------------------------------------------------
# 6. FLAG WHETHER GAME IS ON CURRENT DECISION CARD
# --------------------------------------------------

decision_card_games = set(
    CFB1_DECISION_CARD_V1[
        "Game"
    ]
    .dropna()
    .unique()
)


game_context[
    "Decision_Card_Included"
] = (
    game_context[
        "Game"
    ].isin(
        decision_card_games
    )
)


game_context[
    "Context_Usage"
] = np.where(
    game_context[
        "Decision_Card_Included"
    ],
    "CURRENT_DECISION_CARD_GAME",
    "BACKGROUND_PHASE1_ONLY"
)


# --------------------------------------------------
# 7. NORMALIZE FIELD NAMES GEM EXPECTED
#
# Keep originals too, but create explicit aliases.
# --------------------------------------------------

game_context[
    "Phase_1_Reason"
] = (
    game_context[
        "Phase1_Reason"
    ]
)


game_context[
    "Module_3_Support_Conflict"
] = (
    game_context[
        "M3_Support"
    ].fillna("")
    .astype(str)
    +
    np.where(
        game_context[
            "M3_Conflict"
        ].fillna("").astype(str).str.len() > 0,
        " | CONFLICT: "
        +
        game_context[
            "M3_Conflict"
        ].fillna("").astype(str),
        ""
    )
)


game_context[
    "Late_Market_Movement"
] = (
    "Toward="
    +
    game_context[
        "Movement_Toward"
    ].fillna("UNKNOWN").astype(str)
    +
    "; Class="
    +
    game_context[
        "Movement_Class"
    ].fillna("UNKNOWN").astype(str)
)


# --------------------------------------------------
# 8. PRICE / EV CONTEXT FROM V1
#
# CFB1_FULL_CONTEXT_V1 is still useful.
# We merge the richer game-level information onto
# every available pricing row.
# --------------------------------------------------

pricing_context = (
    CFB1_FULL_CONTEXT_V1.copy()
)


CFB1_FULL_CONTEXT_V2 = (
    pricing_context.merge(
        game_context,
        on="Game",
        how="left",
        validate="many_to_one",
        suffixes=("", "_GameContext")
    )
)


# --------------------------------------------------
# 9. PRESERVE PHASE-1 GAMES THAT HAVE NO PRICING ROW
#
# Example: a derivatives-first game that never reached
# the current quantitative pricing card.
# --------------------------------------------------

priced_games = set(
    CFB1_FULL_CONTEXT_V2[
        "Game"
    ]
    .dropna()
    .unique()
)


missing_pricing_games = (
    game_context[
        ~game_context[
            "Game"
        ].isin(
            priced_games
        )
    ]
    .copy()
)


if len(missing_pricing_games) > 0:

    # Create blank pricing columns so these rows
    # can coexist in one CSV.
    for col in CFB1_FULL_CONTEXT_V2.columns:

        if col not in missing_pricing_games.columns:
            missing_pricing_games[col] = np.nan


    missing_pricing_games[
        "Market_Family"
    ] = "NO_CURRENT_PRICING_ROW"

    missing_pricing_games[
        "Pricing_Status"
    ] = "BACKGROUND_CONTEXT_ONLY"

    missing_pricing_games[
        "Warning"
    ] = (
        "NOT_ON_CURRENT_DECISION_CARD"
    )


    missing_pricing_games = (
        missing_pricing_games[
            CFB1_FULL_CONTEXT_V2.columns
        ]
    )


    CFB1_FULL_CONTEXT_V2 = pd.concat(
        [
            CFB1_FULL_CONTEXT_V2,
            missing_pricing_games
        ],
        ignore_index=True
    )


# --------------------------------------------------
# 10. SORT
# --------------------------------------------------

CFB1_FULL_CONTEXT_V2 = (
    CFB1_FULL_CONTEXT_V2
    .sort_values(
        [
            "Decision_Card_Included",
            "Game",
            "Market_Family"
        ],
        ascending=[
            False,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)


# --------------------------------------------------
# 11. CREATE CSV TEXT + FILE
# --------------------------------------------------

CFB1_FULL_CONTEXT_V2_CSV = (
    CFB1_FULL_CONTEXT_V2
    .to_csv(
        index=False
    )
)


FULL_CONTEXT_V2_FILE = (
    "cfb1_full_context_v2.csv"
)


CFB1_FULL_CONTEXT_V2.to_csv(
    FULL_CONTEXT_V2_FILE,
    index=False
)


# --------------------------------------------------
# 12. COMPACT VALIDATION VIEW
# --------------------------------------------------

validation_cols = [
    "Game",
    "Decision_Card_Included",
    "Context_Usage",

    "Model_Side",
    "Review_Class",
    "Phase_1_Reason",
    "Module_3_Support_Conflict",

    "Key_Numbers_Crossed",
    "Market_Relationship",
    "Late_Market_Movement",

    "Decision_Status",
    "Eligible_Market_Routes"
]


game_validation = (
    CFB1_FULL_CONTEXT_V2[
        validation_cols
    ]
    .drop_duplicates(
        subset=["Game"]
    )
    .reset_index(drop=True)
)


print(
    "\n" + "=" * 115
)

print(
    "CFB1 FULL CONTEXT V2 — GAME LEVEL CHECK"
)

print(
    "=" * 115
)


display(
    game_validation
)


# --------------------------------------------------
# 13. VALIDATION
# --------------------------------------------------

print(
    "\n" + "=" * 115
)

print(
    "CELL 66 VALIDATION"
)

print(
    "=" * 115
)


print(
    "Full-context V2 rows:",
    len(CFB1_FULL_CONTEXT_V2)
)

print(
    "Phase 1 games represented:",
    CFB1_FULL_CONTEXT_V2[
        "Game"
    ].nunique()
)

print(
    "Decision-card games represented:",
    CFB1_FULL_CONTEXT_V2[
        CFB1_FULL_CONTEXT_V2[
            "Decision_Card_Included"
        ] == True
    ][
        "Game"
    ].nunique()
)

print(
    "Background-only Phase 1 games:",
    CFB1_FULL_CONTEXT_V2[
        CFB1_FULL_CONTEXT_V2[
            "Decision_Card_Included"
        ] == False
    ][
        "Game"
    ].nunique()
)


for col in [
    "Phase_1_Reason",
    "Module_3_Support_Conflict",
    "Eligible_Market_Routes",
    "Key_Numbers_Crossed",
    "Market_Relationship",
    "Late_Market_Movement"
]:

    print(
        f"Missing {col}:",
        game_validation[
            col
        ].isna().sum()
    )


print(
    "\nFile created:",
    FULL_CONTEXT_V2_FILE
)

print(
    "\nCELL 66 COMPLETE."
)

print(
    "CFB1_FULL_CONTEXT_V2 NOW CONTAINS "
    "PHASE 1 + MODULE 3 + MARKET + KEY + "
    "MODEL + PRICING CONTEXT."
)

print(
    "BACKGROUND PHASE-1 GAMES ARE RETAINED "
    "BUT FLAGGED AS NOT ON THE CURRENT DECISION CARD."
)

CFB1 FULL CONTEXT V2

CFB1 FULL CONTEXT V2 — GAME LEVEL CHECK


,Game,Decision_Card_Included,Context_Usage,Model_Side,Review_Class,Phase_1_Reason,Module_3_Support_Conflict,Key_Numbers_Crossed,Market_Relationship,Late_Market_Movement,Decision_Status,Eligible_Market_Routes
0,Ball State Cardinals at Northwestern Wildcats,True,CURRENT_DECISION_CARD_GAME,Northwestern Wildcats,MODEL_PLUS_MATCHUP,"MODULE3_SUPPORT, DUAL_TOP25_MODEL_STRENGTH",Northwestern Wildcats +OAY_DAY,7,CONFIRMS_MODEL,Toward=Northwestern Wildcats; Class=MODERATE,CONFIRMED_BUT_LARGE_SPREAD,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT..."
1,Buffalo Bulls at Toledo Rockets,True,CURRENT_DECISION_CARD_GAME,Toledo Rockets,MATCHUP_SUPPORTED,MODULE3_SUPPORT,Toledo Rockets +OAY_DAY,,NEUTRAL,Toward=NONE; Class=NO_MOVE,NEUTRAL_MARKET,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT..."
2,Charlotte 49ers at North Texas Mean Green,True,CURRENT_DECISION_CARD_GAME,North Texas Mean Green,MATCHUP_SUPPORTED,MODULE3_SUPPORT,North Texas Mean Green +OAY_DAY,14,CONFIRMS_MODEL,Toward=North Texas Mean Green; Class=MAJOR,CONFIRMED_BUT_LARGE_SPREAD,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT..."
3,Duke Blue Devils at Georgia Tech Yellow Jackets,True,CURRENT_DECISION_CARD_GAME,Duke Blue Devils,MATCHUP_SUPPORTED,MODULE3_SUPPORT,Duke Blue Devils +OAY_DAY,3,CONFLICTS_WITH_MODEL,Toward=Georgia Tech Yellow Jackets; Class=MODE...,STRONG_MARKET_CONFLICT,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT..."
4,Jacksonville State Gamecocks at Kennesaw State...,True,CURRENT_DECISION_CARD_GAME,Jacksonville State Gamecocks,MATCHUP_SUPPORTED,MODULE3_SUPPORT,Jacksonville State Gamecocks +OAY_DAY,7,CONFIRMS_MODEL,Toward=Jacksonville State Gamecocks; Class=MOD...,CONFIRMED,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT..."
5,Nevada Wolf Pack at UTEP Miners,True,CURRENT_DECISION_CARD_GAME,Nevada Wolf Pack,MODEL_PLUS_MATCHUP,"MODULE3_SUPPORT, DUAL_TOP25_MODEL_STRENGTH",Nevada Wolf Pack +OAY_DAY,,CONFIRMS_MODEL,Toward=Nevada Wolf Pack; Class=MODERATE,CONFIRMED,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT..."
6,Rice Owls at East Carolina Pirates,True,CURRENT_DECISION_CARD_GAME,East Carolina Pirates,MATCHUP_SUPPORTED,MODULE3_SUPPORT,Rice Owls -HAVOC_TRAP,7,CONFIRMS_MODEL,Toward=East Carolina Pirates; Class=MODERATE,CONFIRMED,"FULL_GAME_SPREAD, OPP_TEAM_TOTAL_UNDER, OPP_1H..."
7,Stanford Cardinal at Notre Dame Fighting Irish,True,CURRENT_DECISION_CARD_GAME,Notre Dame Fighting Irish,MODEL_PLUS_MATCHUP,"MODULE3_SUPPORT, DUAL_TOP25_MODEL_STRENGTH",Stanford Cardinal -HAVOC_TRAP,,CONFIRMS_MODEL,Toward=Notre Dame Fighting Irish; Class=MAJOR,CONFIRMED_BUT_LARGE_SPREAD,"FULL_GAME_SPREAD, OPP_TEAM_TOTAL_UNDER, OPP_1H..."
8,Virginia Tech Hokies at California Golden Bears,True,CURRENT_DECISION_CARD_GAME,Virginia Tech Hokies,MATCHUP_SUPPORTED,MODULE3_SUPPORT,California Golden Bears -HAVOC_TRAP,,CONFIRMS_MODEL,Toward=Virginia Tech Hokies; Class=MAJOR,CONFIRMED,"FULL_GAME_SPREAD, OPP_TEAM_TOTAL_UNDER, OPP_1H..."
9,Air Force Falcons at Northern Illinois Huskies,False,BACKGROUND_PHASE1_ONLY,Air Force Falcons,STRONG_MODEL_AGREEMENT,DUAL_TOP25_MODEL_STRENGTH,,7,CONFLICTS_WITH_MODEL,Toward=Northern Illinois Huskies; Class=MODERATE,STRONG_MARKET_CONFLICT,



CELL 66 VALIDATION
Full-context V2 rows: 84
Phase 1 games represented: 14
Decision-card games represented: 9
Background-only Phase 1 games: 5
Missing Phase_1_Reason: 0
Missing Module_3_Support_Conflict: 0
Missing Eligible_Market_Routes: 0
Missing Key_Numbers_Crossed: 0
Missing Market_Relationship: 0
Missing Late_Market_Movement: 0

File created: cfb1_full_context_v2.csv

CELL 66 COMPLETE.
CFB1_FULL_CONTEXT_V2 NOW CONTAINS PHASE 1 + MODULE 3 + MARKET + KEY + MODEL + PRICING CONTEXT.
BACKGROUND PHASE-1 GAMES ARE RETAINED BUT FLAGGED AS NOT ON THE CURRENT DECISION CARD.


In [ ]:
# ==========================================
# CELL 67 — FINAL SYNCHRONIZED CFB1 HANDOFF
# ==========================================

import numpy as np
import pandas as pd
from google.colab import files

print("=" * 115)
print("CFB1 FINAL SYNCHRONIZED HANDOFF")
print("=" * 115)


# --------------------------------------------------
# 1. REQUIRED OBJECTS
# --------------------------------------------------

required_objects = [
    "CFB1_FULL_CONTEXT_V2",
    "CFB1_DECISION_CARD_V1",
    "df_ev"
]

for obj in required_objects:

    if obj not in globals():

        raise NameError(
            f"{obj} missing. "
            "Run the earlier CFB1 cells first."
        )


# --------------------------------------------------
# 2. START FROM FULL CONTEXT V2
# --------------------------------------------------

final_context = (
    CFB1_FULL_CONTEXT_V2.copy()
)


# --------------------------------------------------
# 3. BUILD CANONICAL 57A SPREAD LOOKUP
#
# Old Cell 57A owns the final push-adjusted
# spread probability and ROI calculations.
#
# No Kelly / bankroll / stake fields belong here.
# --------------------------------------------------

spread_fix_cols = [
    "Game",
    "Team",
    "Sportsbook",
    "Line",
    "American_Odds",

    "Candidate_Cushion",
    "Break_Even_Prob",

    "Push_Adjusted_Win_Prob",
    "Push_Adjusted_Push_Prob",
    "Push_Adjusted_ROI",

    "Decision_Warning"
]


missing_57a = [
    c for c in spread_fix_cols
    if c not in df_ev.columns
]


if missing_57a:

    raise KeyError(
        "Missing required old Cell 57A columns in df_ev: "
        + ", ".join(missing_57a)
    )


spread_fix = (
    df_ev[
        spread_fix_cols
    ]
    .copy()
)


# --------------------------------------------------
# 4. VERIFY LOOKUP KEYS ARE UNIQUE
# --------------------------------------------------

match_keys = [
    "Game",
    "Team",
    "Sportsbook",
    "Line",
    "American_Odds"
]


dupes = (
    spread_fix
    .duplicated(
        subset=match_keys,
        keep=False
    )
)


if dupes.any():

    print(
        spread_fix.loc[
            dupes,
            match_keys
        ]
        .sort_values(
            match_keys
        )
    )

    raise ValueError(
        "Duplicate spread pricing keys found. "
        "Stopping rather than applying ambiguous corrections."
    )


# --------------------------------------------------
# 5. RENAME 57A VALUES FOR SAFE MERGE
# --------------------------------------------------

spread_fix = spread_fix.rename(
    columns={

        "Candidate_Cushion":
            "_57A_Model_Adjustment",

        "Break_Even_Prob":
            "_57A_Break_Even",

        "Push_Adjusted_Win_Prob":
            "_57A_Model_Probability",

        "Push_Adjusted_Push_Prob":
            "_57A_Push_Probability",

        "Push_Adjusted_ROI":
            "_57A_Expected_ROI",

        "Decision_Warning":
            "_57A_Warning"
    }
)


# --------------------------------------------------
# 6. MERGE 57A VALUES ONTO FULL CONTEXT
# --------------------------------------------------

final_context = final_context.merge(
    spread_fix,
    on=match_keys,
    how="left",
    validate="many_to_one"
)


# --------------------------------------------------
# 7. REPLACE SPREAD METRICS WITH CANONICAL
#    57A VALUES
# --------------------------------------------------

is_spread = (
    final_context[
        "Market_Family"
    ]
    ==
    "FULL_GAME_SPREAD"
)


spread_metric_map = {

    "Model_Adjustment":
        "_57A_Model_Adjustment",

    "Model_Probability":
        "_57A_Model_Probability",

    "Push_Probability":
        "_57A_Push_Probability",

    "Break_Even_Prob":
        "_57A_Break_Even",

    "Expected_ROI_Final":
        "_57A_Expected_ROI"
}


for target_col, source_col in spread_metric_map.items():

    # Create target column if necessary.
    if target_col not in final_context.columns:
        final_context[target_col] = np.nan

    mask = (
        is_spread
        &
        final_context[
            source_col
        ].notna()
    )

    final_context.loc[
        mask,
        target_col
    ] = final_context.loc[
        mask,
        source_col
    ]


# Warning only when 57A actually supplies one.
if "Warning" not in final_context.columns:
    final_context["Warning"] = ""


mask_warning = (
    is_spread
    &
    final_context[
        "_57A_Warning"
    ].notna()
    &
    (
        final_context[
            "_57A_Warning"
        ].astype(str).str.strip()
        !=
        ""
    )
)


final_context.loc[
    mask_warning,
    "Warning"
] = final_context.loc[
    mask_warning,
    "_57A_Warning"
]


# --------------------------------------------------
# 8. REMOVE TEMPORARY 57A COLUMNS
# --------------------------------------------------

temp_cols = [
    c for c in final_context.columns
    if c.startswith("_57A_")
]


final_context = final_context.drop(
    columns=temp_cols
)


# --------------------------------------------------
# 9. REMOVE ANY LEGACY STAKING FIELDS
#
# Defensive cleanup in case an old object survived
# in memory from an earlier notebook run.
# --------------------------------------------------

legacy_terms = [
    "kelly",
    "stake",
    "bankroll"
]


legacy_columns_before = [
    c for c in final_context.columns
    if any(
        term in c.lower()
        for term in legacy_terms
    )
]


if legacy_columns_before:

    final_context = final_context.drop(
        columns=legacy_columns_before
    )


# --------------------------------------------------
# 10. CLEAN CONTEXT TEXT
# --------------------------------------------------

def clean_m3(row):

    support = str(
        row.get(
            "M3_Support",
            ""
        )
    ).strip()

    conflict = str(
        row.get(
            "M3_Conflict",
            ""
        )
    ).strip()


    if support.lower() == "nan":
        support = ""

    if conflict.lower() == "nan":
        conflict = ""


    if support and conflict:
        return (
            f"{support} | "
            f"CONFLICT: {conflict}"
        )

    if support:
        return support

    if conflict:
        return (
            f"CONFLICT: {conflict}"
        )

    return "NONE"


final_context[
    "Module_3_Support_Conflict"
] = final_context.apply(
    clean_m3,
    axis=1
)


# Explicit NONE is clearer than a blank key field.
final_context[
    "Key_Numbers_Crossed"
] = (
    final_context[
        "Key_Numbers_Crossed"
    ]
    .replace(
        "",
        np.nan
    )
    .fillna(
        "NONE"
    )
)


# --------------------------------------------------
# 11. CREATE CANONICAL FINAL OBJECT
# --------------------------------------------------

CFB1_FULL_CONTEXT_FINAL = (
    final_context.copy()
)


FULL_CONTEXT_FILE = (
    "cfb1_full_context_final.csv"
)


CFB1_FULL_CONTEXT_FINAL.to_csv(
    FULL_CONTEXT_FILE,
    index=False
)


# --------------------------------------------------
# 12. VERIFY DECISION CARD SPREADS MATCH
#     FULL CONTEXT
#
# Synchronize only CFB1-owned analytical fields:
#
# - model adjustment
# - model probability
# - push probability
# - break-even probability
# - expected ROI
# --------------------------------------------------

card_spreads = (
    CFB1_DECISION_CARD_V1[
        CFB1_DECISION_CARD_V1[
            "Market_Family"
        ]
        ==
        "FULL_GAME_SPREAD"
    ]
    .copy()
)


compare_keys = [
    "Game",
    "Team",
    "Sportsbook",
    "Line",
    "American_Odds"
]


context_spreads = (
    CFB1_FULL_CONTEXT_FINAL[
        CFB1_FULL_CONTEXT_FINAL[
            "Market_Family"
        ]
        ==
        "FULL_GAME_SPREAD"
    ][
        compare_keys
        +
        [
            "Model_Adjustment",
            "Model_Probability",
            "Push_Probability",
            "Break_Even_Prob",
            "Expected_ROI_Final"
        ]
    ]
    .copy()
)


check = card_spreads.merge(
    context_spreads,
    on=compare_keys,
    how="left",
    suffixes=(
        "_Card",
        "_Context"
    ),
    validate="one_to_one"
)


# --------------------------------------------------
# 13. SYNCHRONIZATION TESTS
# --------------------------------------------------

check[
    "Model_Adjustment_Match"
] = np.isclose(
    check[
        "Model_Adjustment_Card"
    ].astype(float),
    check[
        "Model_Adjustment_Context"
    ].astype(float),
    equal_nan=True
)


check[
    "Probability_Match"
] = np.isclose(
    check[
        "Model_Probability_Card"
    ].astype(float),
    check[
        "Model_Probability_Context"
    ].astype(float),
    equal_nan=True
)


check[
    "Push_Probability_Match"
] = np.isclose(
    check[
        "Push_Probability_Card"
    ].astype(float),
    check[
        "Push_Probability_Context"
    ].astype(float),
    equal_nan=True
)


check[
    "Break_Even_Match"
] = np.isclose(
    check[
        "Break_Even"
    ].astype(float),
    check[
        "Break_Even_Prob"
    ].astype(float),
    equal_nan=True
)


check[
    "ROI_Match"
] = np.isclose(
    check[
        "Expected_ROI"
    ].astype(float),
    check[
        "Expected_ROI_Final"
    ].astype(float),
    equal_nan=True
)


sync_columns = [
    "Model_Adjustment_Match",
    "Probability_Match",
    "Push_Probability_Match",
    "Break_Even_Match",
    "ROI_Match"
]


all_synced = (
    check[
        sync_columns
    ]
    .all()
    .all()
)


if not all_synced:

    print(
        "\nSPREAD SYNCHRONIZATION FAILURE"
    )

    display(
        check[
            ~check[
                sync_columns
            ].all(axis=1)
        ]
    )

    raise ValueError(
        "Decision Card and Full Context spread "
        "metrics are not fully synchronized."
    )


# --------------------------------------------------
# 14. BUILD COPY / PASTE DECISION CARD
# --------------------------------------------------

copy_card = (
    CFB1_DECISION_CARD_V1.copy()
)


for col in [
    "Model_Probability",
    "Push_Probability",
    "Break_Even",
    "Expected_ROI"
]:

    if col in copy_card.columns:

        copy_card[col] = (
            copy_card[col]
            .apply(
                lambda x:
                ""
                if pd.isna(x)
                else f"{x * 100:.1f}%"
            )
        )


if "Model_Adjustment" in copy_card.columns:

    copy_card[
        "Model_Adjustment"
    ] = (
        copy_card[
            "Model_Adjustment"
        ]
        .apply(
            lambda x:
            ""
            if pd.isna(x)
            else f"{x:.2f}"
        )
    )


decision_card_text = (
    "CFB1_DECISION_CARD_V1\n\n"
    +
    copy_card.to_csv(
        index=False
    )
)


# --------------------------------------------------
# 15. FINAL LEGACY-FIELD VALIDATION
# --------------------------------------------------

legacy_final_columns = [
    c
    for c in CFB1_FULL_CONTEXT_FINAL.columns
    if any(
        term in c.lower()
        for term in legacy_terms
    )
]


legacy_card_columns = [
    c
    for c in CFB1_DECISION_CARD_V1.columns
    if any(
        term in c.lower()
        for term in legacy_terms
    )
]


if legacy_final_columns:

    raise ValueError(
        "Legacy staking fields remain in "
        "CFB1_FULL_CONTEXT_FINAL: "
        + ", ".join(
            legacy_final_columns
        )
    )


if legacy_card_columns:

    raise ValueError(
        "Legacy staking fields remain in "
        "CFB1_DECISION_CARD_V1: "
        + ", ".join(
            legacy_card_columns
        )
    )


# --------------------------------------------------
# 16. FINAL VALIDATION SUMMARY
# --------------------------------------------------

print(
    "\nFULL CONTEXT FILE READY"
)

print(
    "-" * 60
)

print(
    "File:",
    FULL_CONTEXT_FILE
)

print(
    "Rows:",
    len(
        CFB1_FULL_CONTEXT_FINAL
    )
)

print(
    "Phase 1 games:",
    CFB1_FULL_CONTEXT_FINAL[
        "Game"
    ].nunique()
)

print(
    "Current Decision Card games:",
    CFB1_FULL_CONTEXT_FINAL.loc[
        CFB1_FULL_CONTEXT_FINAL[
            "Decision_Card_Included"
        ]
        ==
        True,
        "Game"
    ].nunique()
)

print(
    "Background-only games:",
    CFB1_FULL_CONTEXT_FINAL.loc[
        CFB1_FULL_CONTEXT_FINAL[
            "Decision_Card_Included"
        ]
        ==
        False,
        "Game"
    ].nunique()
)

print(
    "Decision Card spread rows checked:",
    len(
        card_spreads
    )
)

print(
    "57A spread synchronization:",
    "PASS"
    if all_synced
    else "FAIL"
)

print(
    "Legacy Kelly/stake/bankroll columns "
    "in Full Context:",
    legacy_final_columns
)

print(
    "Legacy Kelly/stake/bankroll columns "
    "in Decision Card:",
    legacy_card_columns
)


# --------------------------------------------------
# 17. DOWNLOAD FULL CONTEXT FILE
# --------------------------------------------------

print(
    "\nStarting full-context CSV download..."
)


files.download(
    FULL_CONTEXT_FILE
)


# --------------------------------------------------
# 18. PRINT COPY / PASTE DECISION CARD
# --------------------------------------------------

print(
    "\n" + "=" * 115
)

print(
    "COPY EVERYTHING BELOW THIS LINE "
    "INTO THE CFB1 GEM"
)

print(
    "=" * 115
)

print(
    decision_card_text
)


print(
    "\n" + "=" * 115
)

print(
    "CELL 67 COMPLETE."
)

print(
    "1. Upload cfb1_full_context_final.csv "
    "to the CFB1 Gem."
)

print(
    "2. Copy/paste the Decision Card printed above."
)

print(
    "3. Decision Card and Full Context use "
    "synchronized CFB1 spread calculations."
)

print(
    "4. CFB1 hands off probability, push risk, "
    "break-even, expected ROI, status, and context."
)

print(
    "5. Bankroll, Kelly, and stake sizing are "
    "reserved for the Bet Slip Architect."
)

CFB1 FINAL SYNCHRONIZED HANDOFF

FULL CONTEXT FILE READY
------------------------------------------------------------
File: cfb1_full_context_final.csv
Rows: 84
Phase 1 games: 14
Current Decision Card games: 9
Background-only games: 5
Decision Card spread rows checked: 8
57A spread synchronization: PASS
Legacy Kelly/stake/bankroll columns in Full Context: []
Legacy Kelly/stake/bankroll columns in Decision Card: []

Starting full-context CSV download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


COPY EVERYTHING BELOW THIS LINE INTO THE CFB1 GEM
CFB1_DECISION_CARD_V1

Game,Market_Family,Card_Role,Team,Direction,Bet_Label,Sportsbook,Line,American_Odds,Model_Adjustment,Model_Probability,Push_Probability,Break_Even,Expected_ROI,Decision_Status,Model_Status,Warning_Final
Ball State Cardinals at Northwestern Wildcats,TEAM_TOTAL,PRIMARY,Northwestern Wildcats,OVER,Northwestern Wildcats TT OVER 44.5,DraftKings,44.5,-120,14.31,91.4%,0.0%,54.5%,67.5%,CONFIRMED_BUT_LARGE_SPREAD,TEAM_TOTAL_V1_EXPERIMENTAL,
Ball State Cardinals at Northwestern Wildcats,TEAM_TOTAL,VALUE_ALT,Northwestern Wildcats,OVER,Northwestern Wildcats TT OVER 48.5,DraftKings,48.5,145,14.31,82.8%,0.0%,40.8%,102.9%,CONFIRMED_BUT_LARGE_SPREAD,TEAM_TOTAL_V1_EXPERIMENTAL,
Buffalo Bulls at Toledo Rockets,TEAM_TOTAL,PRIMARY,Toledo Rockets,OVER,Toledo Rockets TT OVER 37.5,DraftKings,37.5,-120,5.48,68.1%,0.0%,54.5%,24.8%,NEUTRAL_MARKET,TEAM_TOTAL_V1_EXPERIMENTAL,
Buffalo Bulls at Toledo Rockets,TEAM_TOTAL,VALUE_ALT,Toledo Rocket

In [ ]:
# ==========================================
# CELL 69 — CFB1 SELECTIVE LIVE SPREAD CHECKER
# ==========================================

import pandas as pd
import numpy as np

print("=" * 115)
print("CFB1 SELECTIVE LIVE SPREAD CHECKER")
print("=" * 115)


# ==================================================
# 1. REQUIRED OBJECTS
#
# This cell defines a reusable recalculation
# function. It does NOT automatically run a game.
# ==================================================

required = [
    "df_ev",
    "df_hist_spread",
    "spread_error",
    "global_errors",
    "spread_bucket",
    "empirical_probability",
    "estimate_push_probability",
    "CFB1_FULL_CONTEXT_FINAL"
]


for name in required:

    if name not in globals():

        raise NameError(
            f"{name} is missing. "
            "Run the existing CFB1 spread-model "
            "and final-handoff cells first."
        )


# ==================================================
# 2. ODDS HELPERS
# ==================================================

def live_american_break_even(
    american_odds
):

    odds = float(
        american_odds
    )

    if odds < 0:

        return (
            abs(odds)
            /
            (
                abs(odds)
                + 100.0
            )
        )

    return (
        100.0
        /
        (
            odds
            + 100.0
        )
    )


def live_profit_multiple(
    american_odds
):

    odds = float(
        american_odds
    )

    if odds < 0:

        return (
            100.0
            /
            abs(odds)
        )

    return (
        odds
        /
        100.0
    )


# ==================================================
# 3. REUSABLE LIVE SPREAD RECALC FUNCTION
# ==================================================

def cfb1_live_spread_recalc(
    game,
    team,
    sportsbook,
    line,
    american_odds,
    display_result=True
):

    # ----------------------------------------------
    # A. FROZEN GAME / MODEL INFORMATION
    # ----------------------------------------------

    game_rows = df_ev[
        (df_ev["Game"] == game)
        &
        (df_ev["Team"] == team)
    ].copy()


    if len(game_rows) == 0:

        raise ValueError(
            f"No df_ev rows found for "
            f"{team} in {game}."
        )


    retained_values = (
        game_rows[
            "Retained_Model_Spread"
        ]
        .dropna()
        .unique()
    )


    if len(retained_values) != 1:

        raise ValueError(
            "Expected exactly one retained "
            "model spread for this game/team."
        )


    retained_model_line = float(
        retained_values[0]
    )


    # ----------------------------------------------
    # B. HOME / AWAY ORIENTATION
    # ----------------------------------------------

    ctx = (
        CFB1_FULL_CONTEXT_FINAL[
            CFB1_FULL_CONTEXT_FINAL[
                "Game"
            ] == game
        ]
        .drop_duplicates(
            subset=["Game"]
        )
    )


    if len(ctx) != 1:

        raise ValueError(
            "Could not uniquely identify "
            "game context."
        )


    home_team = (
        ctx.iloc[0][
            "Home_Team"
        ]
    )

    away_team = (
        ctx.iloc[0][
            "Away_Team"
        ]
    )


    if team == home_team:

        model_is_home = True

    elif team == away_team:

        model_is_home = False

    else:

        raise ValueError(
            "Team does not match the home "
            "or away team in Full Context."
        )


    # ----------------------------------------------
    # C. LIVE MODEL CUSHION
    # ----------------------------------------------

    line = float(
        line
    )

    american_odds = int(
        american_odds
    )


    cushion = (
        line
        -
        retained_model_line
    )


    # ----------------------------------------------
    # D. GLOBAL EMPIRICAL PROBABILITY
    # ----------------------------------------------

    (
        global_win,
        global_push,
        global_loss
    ) = empirical_probability(
        cushion,
        model_is_home,
        global_errors
    )


    global_win = float(
        global_win
    )

    global_push = float(
        global_push
    )

    global_loss = float(
        global_loss
    )


    # ----------------------------------------------
    # E. SPREAD-BUCKET EMPIRICAL PROBABILITY
    # ----------------------------------------------

    live_bucket = spread_bucket(
        abs(line)
    )


    hist_bucket_labels = (
        df_hist_spread[
            "Closing_Home_Spread"
        ]
        .abs()
        .apply(
            spread_bucket
        )
    )


    bucket_mask = (
        hist_bucket_labels
        ==
        live_bucket
    )


    bucket_errors_live = (
        spread_error.loc[
            bucket_mask
        ]
    )


    if len(
        bucket_errors_live
    ) == 0:

        raise ValueError(
            f"No historical observations "
            f"for spread bucket {live_bucket}."
        )


    (
        bucket_win,
        bucket_push,
        bucket_loss
    ) = empirical_probability(
        cushion,
        model_is_home,
        bucket_errors_live
    )


    bucket_win = float(
        bucket_win
    )

    bucket_push = float(
        bucket_push
    )

    bucket_loss = float(
        bucket_loss
    )


    # ----------------------------------------------
    # F. CFB1 V1 CONSERVATIVE PROBABILITY SOURCE
    #
    # Same rule as production spread pricing:
    # use the lower win estimate.
    # ----------------------------------------------

    if global_win <= bucket_win:

        cfb1_win = (
            global_win
        )

        cfb1_push = (
            global_push
        )

        cfb1_loss = (
            global_loss
        )

        probability_source = (
            "GLOBAL"
        )

    else:

        cfb1_win = (
            bucket_win
        )

        cfb1_push = (
            bucket_push
        )

        cfb1_loss = (
            bucket_loss
        )

        probability_source = (
            "SPREAD_BUCKET"
        )


    # ----------------------------------------------
    # G. HISTORICAL INTEGER-LINE PUSH ESTIMATE
    #
    # Same estimator used by old Cell 57A.
    # Half-point lines should return zero.
    # ----------------------------------------------

    (
        historical_push_prob,
        historical_push_n
    ) = estimate_push_probability(
        line,
        model_is_home
    )


    historical_push_prob = float(
        historical_push_prob
    )

    historical_push_n = int(
        historical_push_n
    )


    # ----------------------------------------------
    # H. CELL 57A PUSH ADJUSTMENT
    #
    # Preserve the base Win/Loss ratio and allocate
    # the remaining probability mass around the
    # separately estimated historical push rate.
    # ----------------------------------------------

    base_nonpush = (
        cfb1_win
        +
        cfb1_loss
    )


    if base_nonpush <= 0:

        raise ValueError(
            "Base win/loss probability mass "
            "is zero or negative."
        )


    win_share = (
        cfb1_win
        /
        base_nonpush
    )


    loss_share = (
        cfb1_loss
        /
        base_nonpush
    )


    remaining_mass = (
        1.0
        -
        historical_push_prob
    )


    push_adjusted_win = (
        win_share
        *
        remaining_mass
    )


    push_adjusted_push = (
        historical_push_prob
    )


    push_adjusted_loss = (
        loss_share
        *
        remaining_mass
    )


    # ----------------------------------------------
    # I. ODDS / BREAK-EVEN / EXPECTED ROI
    # ----------------------------------------------

    break_even = (
        live_american_break_even(
            american_odds
        )
    )


    profit_multiple = (
        live_profit_multiple(
            american_odds
        )
    )


    expected_roi = (
        push_adjusted_win
        *
        profit_multiple
        -
        push_adjusted_loss
    )


    probability_edge = (
        push_adjusted_win
        -
        break_even
    )


    # ----------------------------------------------
    # J. EV STATUS
    #
    # Same thresholds used by CFB1 spread pricing.
    # ----------------------------------------------

    if expected_roi >= 0.05:

        ev_status = (
            "STRONG_PRICE_EDGE"
        )

    elif expected_roi >= 0.03:

        ev_status = (
            "POSITIVE_EV"
        )

    elif expected_roi > 0:

        ev_status = (
            "THIN_POSITIVE_EV"
        )

    else:

        ev_status = (
            "NON_POSITIVE_EV"
        )


    # ----------------------------------------------
    # K. PROBABILITY VALIDATION
    # ----------------------------------------------

    probability_sum = (
        push_adjusted_win
        +
        push_adjusted_push
        +
        push_adjusted_loss
    )


    if not np.isclose(
        probability_sum,
        1.0,
        atol=1e-10
    ):

        raise ValueError(
            "Adjusted Win + Push + Loss "
            "does not equal 1."
        )


    # ----------------------------------------------
    # L. RESULT PACKET
    # ----------------------------------------------

    result = {

        "Game":
            game,

        "Team":
            team,

        "Sportsbook":
            sportsbook,

        "Live_Line":
            line,

        "Live_Odds":
            american_odds,

        "Retained_Model_Line":
            retained_model_line,

        "Candidate_Cushion":
            cushion,

        "Spread_Bucket":
            live_bucket,

        "Historical_Bucket_N":
            len(
                bucket_errors_live
            ),

        "Global_Win_Prob":
            global_win,

        "Bucket_Win_Prob":
            bucket_win,

        "CFB1_Base_Win_Prob":
            cfb1_win,

        "Probability_Source":
            probability_source,

        "Historical_Push_Prob":
            historical_push_prob,

        "Historical_Push_N":
            historical_push_n,

        "Push_Adjusted_Win_Prob":
            push_adjusted_win,

        "Push_Adjusted_Push_Prob":
            push_adjusted_push,

        "Push_Adjusted_Loss_Prob":
            push_adjusted_loss,

        "Break_Even_Prob":
            break_even,

        "Probability_Edge":
            probability_edge,

        "Push_Adjusted_ROI":
            expected_roi,

        "EV_Status":
            ev_status
    }


    # ----------------------------------------------
    # M. OPTIONAL DISPLAY
    # ----------------------------------------------

    if display_result:

        result_df = pd.DataFrame(
            [result]
        )


        print(
            "\n" + "=" * 115
        )

        print(
            "LIVE CFB1 SPREAD RESULT"
        )

        print(
            "=" * 115
        )


        print(
            f"Game:                 "
            f"{result['Game']}"
        )

        print(
            f"Route:                "
            f"{result['Team']} "
            f"{result['Live_Line']:+.1f}"
        )

        print(
            f"Sportsbook:           "
            f"{result['Sportsbook']}"
        )

        print(
            f"Price:                "
            f"{result['Live_Odds']:+d}"
        )

        print(
            f"Retained model line:  "
            f"{result['Retained_Model_Line']:+.3f}"
        )

        print(
            f"Model cushion:        "
            f"{result['Candidate_Cushion']:.3f}"
        )

        print(
            f"Spread bucket:        "
            f"{result['Spread_Bucket']} "
            f"(N={result['Historical_Bucket_N']})"
        )

        print(
            f"Global probability:   "
            f"{result['Global_Win_Prob']:.1%}"
        )

        print(
            f"Bucket probability:   "
            f"{result['Bucket_Win_Prob']:.1%}"
        )

        print(
            f"CFB1 base probability:"
            f" {result['CFB1_Base_Win_Prob']:.1%} "
            f"[{result['Probability_Source']}]"
        )

        print(
            f"Historical push:      "
            f"{result['Historical_Push_Prob']:.1%} "
            f"(N={result['Historical_Push_N']})"
        )

        print(
            f"Adjusted win:         "
            f"{result['Push_Adjusted_Win_Prob']:.1%}"
        )

        print(
            f"Adjusted push:        "
            f"{result['Push_Adjusted_Push_Prob']:.1%}"
        )

        print(
            f"Adjusted loss:        "
            f"{result['Push_Adjusted_Loss_Prob']:.1%}"
        )

        print(
            f"Break-even:           "
            f"{result['Break_Even_Prob']:.1%}"
        )

        print(
            f"Probability edge:     "
            f"{result['Probability_Edge']:.1%}"
        )

        print(
            f"Expected ROI:         "
            f"{result['Push_Adjusted_ROI']:.1%}"
        )

        print(
            f"Pricing status:       "
            f"{result['EV_Status']}"
        )


        print(
            "\nFULL RESULT DATAFRAME"
        )

        print(
            "-" * 80
        )

        display(
            result_df
        )


    return result


# ==================================================
# 4. VALIDATION
# ==================================================

legacy_terms = [
    "kelly",
    "stake",
    "bankroll"
]


function_text_fields = [
    "Push_Adjusted_Win_Prob",
    "Push_Adjusted_Push_Prob",
    "Push_Adjusted_Loss_Prob",
    "Break_Even_Prob",
    "Push_Adjusted_ROI",
    "EV_Status"
]


print(
    "\n" + "=" * 115
)

print(
    "CELL 69 VALIDATION"
)

print(
    "=" * 115
)


print(
    "Reusable function created:",
    "cfb1_live_spread_recalc"
)

print(
    "Automatic live target executed:",
    False
)

print(
    "Staking/Kelly calculation included:",
    False
)

print(
    "Core output fields:",
    function_text_fields
)


print(
    "\nCELL 69 COMPLETE."
)

print(
    "NO GAME WAS RECALCULATED AUTOMATICALLY."
)

print(
    "CALL cfb1_live_spread_recalc(...) "
    "ONLY WHEN A CURRENT LINE OR PRICE NEEDS REVIEW."
)

print(
    "THE FROZEN CFB1 MODEL REMAINS UNCHANGED."
)

print(
    "BANKROLL / KELLY / STAKE SIZING "
    "BELONGS TO THE BET SLIP ARCHITECT."
)

CFB1 SELECTIVE LIVE SPREAD CHECKER

CELL 69 VALIDATION
Reusable function created: cfb1_live_spread_recalc
Automatic live target executed: False
Staking/Kelly calculation included: False
Core output fields: ['Push_Adjusted_Win_Prob', 'Push_Adjusted_Push_Prob', 'Push_Adjusted_Loss_Prob', 'Break_Even_Prob', 'Push_Adjusted_ROI', 'EV_Status']

CELL 69 COMPLETE.
NO GAME WAS RECALCULATED AUTOMATICALLY.
CALL cfb1_live_spread_recalc(...) ONLY WHEN A CURRENT LINE OR PRICE NEEDS REVIEW.
THE FROZEN CFB1 MODEL REMAINS UNCHANGED.
BANKROLL / KELLY / STAKE SIZING BELONGS TO THE BET SLIP ARCHITECT.


In [ ]:
# ============================================================
# CELL 73 — BUILD CFB1 -> BET SLIP ARCHITECT HANDOFF
# ============================================================

import pandas as pd
import numpy as np

print("=" * 115)
print("CFB1 -> BET SLIP ARCHITECT HANDOFF")
print("=" * 115)


# ------------------------------------------------------------
# 1. REQUIRED OBJECTS
# ------------------------------------------------------------

required_objects = [
    "CFB1_DECISION_CARD_V1",
    "CFB1_FULL_CONTEXT_FINAL",
]

for obj in required_objects:

    if obj not in globals():

        raise NameError(
            f"{obj} is missing. "
            "Run the final CFB1 handoff cells first."
        )


# ------------------------------------------------------------
# 2. START WITH THE ACTUAL DECISION CARD
#
# CFB1_DECISION_CARD_V1 is the authoritative compact
# quantitative / qualitative presentation layer.
# ------------------------------------------------------------

architect = (
    CFB1_DECISION_CARD_V1.copy()
)


# ------------------------------------------------------------
# 3. GAME-LEVEL CONTEXT FROM FULL CONTEXT FINAL
# ------------------------------------------------------------

context_cols = [
    "Game",

    "Model_Side",
    "Review_Class",

    "Phase_1_Reason",
    "Module_3_Support_Conflict",

    "Open_Home_Spread",
    "Current_Home_Spread",
    "Movement_Size",
    "Movement_Toward",
    "Movement_Class",

    "Key_Numbers_Crossed",
    "Market_Relationship",

    "Eligible_Market_Routes",
    "Decision_Note",
    "Spread_Status",
]


available_context_cols = [
    c for c in context_cols
    if c in CFB1_FULL_CONTEXT_FINAL.columns
]


game_context = (
    CFB1_FULL_CONTEXT_FINAL[
        available_context_cols
    ]
    .drop_duplicates(
        subset=["Game"]
    )
    .copy()
)


# ------------------------------------------------------------
# 4. MERGE GAME CONTEXT ONTO DECISION CARD
# ------------------------------------------------------------

architect = architect.merge(
    game_context,
    on="Game",
    how="left",
    validate="many_to_one",
    suffixes=(
        "",
        "_GameContext"
    )
)


# ------------------------------------------------------------
# 5. STANDARDIZE FIELD NAMES FOR ARCHITECT
#
# Preserve CFB1 meaning.
# Do not create staking fields.
# ------------------------------------------------------------

rename_map = {

    "Break_Even":
        "Break_Even_Prob",

    "Warning_Final":
        "Warning",

    "Phase_1_Reason":
        "Phase1_Reason",

    "Module_3_Support_Conflict":
        "Module3_Support_Conflict",
}


architect = architect.rename(
    columns={
        old: new
        for old, new in rename_map.items()
        if old in architect.columns
    }
)


# ------------------------------------------------------------
# 6. FINAL ARCHITECT HANDOFF SCHEMA
#
# These are CFB1-owned outputs/context only.
#
# Architect owns:
# - bankroll
# - Kelly
# - stake sizing
# - promo sizing
# - parlay / SGP construction
# - correlation / portfolio decisions
# ------------------------------------------------------------

handoff_cols = [
    "Game",
    "Model_Side",

    "Market_Family",
    "Card_Role",

    "Team",
    "Direction",
    "Bet_Label",

    "Sportsbook",
    "Line",
    "American_Odds",

    "Model_Adjustment",
    "Model_Probability",
    "Push_Probability",
    "Break_Even_Prob",
    "Expected_ROI",

    "Model_Status",
    "Decision_Status",
    "Warning",

    "Phase1_Reason",
    "Module3_Support_Conflict",

    "Spread_Status",
    "Market_Relationship",

    "Open_Home_Spread",
    "Current_Home_Spread",
    "Movement_Size",
    "Movement_Toward",
    "Movement_Class",

    "Key_Numbers_Crossed",

    "Eligible_Market_Routes",
    "Decision_Note",
]


handoff_cols = [
    c for c in handoff_cols
    if c in architect.columns
]


CFB1_ARCHITECT_HANDOFF = (
    architect[
        handoff_cols
    ]
    .copy()
)


# ------------------------------------------------------------
# 7. CLEAN TEXT FIELDS
# ------------------------------------------------------------

text_cols = [
    "Model_Side",
    "Model_Status",
    "Decision_Status",
    "Warning",

    "Phase1_Reason",
    "Module3_Support_Conflict",

    "Spread_Status",
    "Market_Relationship",

    "Movement_Toward",
    "Movement_Class",

    "Key_Numbers_Crossed",

    "Eligible_Market_Routes",
    "Decision_Note",
]


for col in text_cols:

    if col in CFB1_ARCHITECT_HANDOFF.columns:

        CFB1_ARCHITECT_HANDOFF[
            col
        ] = (
            CFB1_ARCHITECT_HANDOFF[
                col
            ]
            .fillna("")
        )


# ------------------------------------------------------------
# 8. VALIDATE QUALITATIVE-ONLY ROWS
#
# QUALITATIVE_REVIEW must not receive quantitative
# probability / ROI endorsement.
# ------------------------------------------------------------

qual_mask = (
    CFB1_ARCHITECT_HANDOFF[
        "Card_Role"
    ]
    ==
    "QUALITATIVE_REVIEW"
)


quant_cols_for_qual_check = [
    "Model_Probability",
    "Push_Probability",
    "Break_Even_Prob",
    "Expected_ROI"
]


qualitative_quant_errors = 0


if qual_mask.any():

    for col in quant_cols_for_qual_check:

        if col in CFB1_ARCHITECT_HANDOFF.columns:

            qualitative_quant_errors += int(
                CFB1_ARCHITECT_HANDOFF.loc[
                    qual_mask,
                    col
                ]
                .notna()
                .sum()
            )


# ------------------------------------------------------------
# 9. VALIDATE NO STAKING FIELDS
# ------------------------------------------------------------

legacy_terms = [
    "kelly",
    "stake",
    "bankroll"
]


legacy_columns = [
    col
    for col in CFB1_ARCHITECT_HANDOFF.columns
    if any(
        term in col.lower()
        for term in legacy_terms
    )
]


# ------------------------------------------------------------
# 10. VALIDATE CORE ARCHITECT FIELDS
# ------------------------------------------------------------

required_handoff_cols = [
    "Game",
    "Market_Family",
    "Card_Role",

    "Team",
    "Direction",
    "Bet_Label",

    "Sportsbook",
    "Line",
    "American_Odds",

    "Model_Probability",
    "Push_Probability",
    "Break_Even_Prob",
    "Expected_ROI",

    "Model_Status",
    "Decision_Status",
    "Warning",

    "Eligible_Market_Routes"
]


missing_handoff_cols = [
    col
    for col in required_handoff_cols
    if col not in CFB1_ARCHITECT_HANDOFF.columns
]


# ------------------------------------------------------------
# 11. VALIDATION SUMMARY
# ------------------------------------------------------------

print(
    "\n" + "=" * 115
)

print(
    "CELL 73 VALIDATION"
)

print(
    "=" * 115
)


print(
    "Handoff rows:",
    len(
        CFB1_ARCHITECT_HANDOFF
    )
)

print(
    "Games represented:",
    CFB1_ARCHITECT_HANDOFF[
        "Game"
    ].nunique()
)


print(
    "Primary rows:",
    int(
        (
            CFB1_ARCHITECT_HANDOFF[
                "Card_Role"
            ]
            ==
            "PRIMARY"
        ).sum()
    )
)


print(
    "Value-alt rows:",
    int(
        (
            CFB1_ARCHITECT_HANDOFF[
                "Card_Role"
            ]
            ==
            "VALUE_ALT"
        ).sum()
    )
)


print(
    "Qualitative-review rows:",
    int(
        qual_mask.sum()
    )
)


print(
    "Qualitative rows with quantitative metrics:",
    qualitative_quant_errors
)


print(
    "Legacy Kelly/stake/bankroll columns:",
    legacy_columns
)


print(
    "Missing required handoff columns:",
    missing_handoff_cols
)


# ------------------------------------------------------------
# 12. HARD STOPS
# ------------------------------------------------------------

if qualitative_quant_errors != 0:

    raise ValueError(
        "QUALITATIVE_REVIEW rows contain "
        "quantitative probability or ROI values."
    )


if legacy_columns:

    raise ValueError(
        "Legacy Kelly/stake/bankroll fields "
        "remain in Architect handoff."
    )


if missing_handoff_cols:

    raise ValueError(
        "Architect handoff is missing required fields: "
        + ", ".join(
            missing_handoff_cols
        )
    )


# ------------------------------------------------------------
# 13. DISPLAY HANDOFF PREVIEW
# ------------------------------------------------------------

print(
    "\n" + "=" * 115
)

print(
    "ARCHITECT HANDOFF PREVIEW"
)

print(
    "=" * 115
)


display(
    CFB1_ARCHITECT_HANDOFF
)


# ------------------------------------------------------------
# 14. EXPORT
# ------------------------------------------------------------

ARCHITECT_HANDOFF_FILE = (
    "cfb1_architect_handoff.csv"
)


CFB1_ARCHITECT_HANDOFF.to_csv(
    ARCHITECT_HANDOFF_FILE,
    index=False
)


print(
    "\n" + "=" * 115
)

print(
    "EXPORT COMPLETE"
)

print(
    "=" * 115
)


print(
    "Saved:",
    ARCHITECT_HANDOFF_FILE
)


print(
    "\nCELL 73 COMPLETE."
)

print(
    "CFB1 -> ARCHITECT HANDOFF CONTAINS "
    "CFB1 MODEL OUTPUT AND CONTEXT ONLY."
)

print(
    "CFB1 PROVIDES PROBABILITY, PUSH RISK, "
    "BREAK-EVEN, EXPECTED ROI, ROUTING, "
    "STATUS, AND MARKET CONTEXT."
)

print(
    "BANKROLL, KELLY, STAKE SIZING, PROMOS, "
    "PARLAYS, SGPs, CORRELATION, AND PORTFOLIO "
    "DECISIONS BELONG TO THE BET SLIP ARCHITECT."
)

print(
    "NO NEW HANDICAPPING CALCULATIONS WERE PERFORMED."
)

CFB1 -> BET SLIP ARCHITECT HANDOFF

CELL 73 VALIDATION
Handoff rows: 25
Games represented: 9
Primary rows: 12
Value-alt rows: 12
Qualitative-review rows: 1
Qualitative rows with quantitative metrics: 0
Legacy Kelly/stake/bankroll columns: []
Missing required handoff columns: []

ARCHITECT HANDOFF PREVIEW


,Game,Model_Side,Market_Family,Card_Role,Team,Direction,Bet_Label,Sportsbook,Line,American_Odds,...,Spread_Status,Market_Relationship,Open_Home_Spread,Current_Home_Spread,Movement_Size,Movement_Toward,Movement_Class,Key_Numbers_Crossed,Eligible_Market_Routes,Decision_Note
0,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,TEAM_TOTAL,PRIMARY,Northwestern Wildcats,OVER,Northwestern Wildcats TT OVER 44.5,DraftKings,44.5,-120,...,DERIVATIVES_FIRST_20.5_PLUS,CONFIRMS_MODEL,-6.5,-7.5,1.0,Northwestern Wildcats,MODERATE,7,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...","Market supports the model side, but spread siz..."
1,Ball State Cardinals at Northwestern Wildcats,Northwestern Wildcats,TEAM_TOTAL,VALUE_ALT,Northwestern Wildcats,OVER,Northwestern Wildcats TT OVER 48.5,DraftKings,48.5,145,...,DERIVATIVES_FIRST_20.5_PLUS,CONFIRMS_MODEL,-6.5,-7.5,1.0,Northwestern Wildcats,MODERATE,7,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...","Market supports the model side, but spread siz..."
2,Buffalo Bulls at Toledo Rockets,Toledo Rockets,TEAM_TOTAL,PRIMARY,Toledo Rockets,OVER,Toledo Rockets TT OVER 37.5,DraftKings,37.5,-120,...,DERIVATIVES_FIRST_20.5_PLUS,NEUTRAL,-10.0,-10.0,0.0,NONE,NO_MOVE,NONE,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",Market movement provides no additional confirm...
3,Buffalo Bulls at Toledo Rockets,Toledo Rockets,TEAM_TOTAL,VALUE_ALT,Toledo Rockets,OVER,Toledo Rockets TT OVER 38.5,FanDuel,38.5,114,...,DERIVATIVES_FIRST_20.5_PLUS,NEUTRAL,-10.0,-10.0,0.0,NONE,NO_MOVE,NONE,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",Market movement provides no additional confirm...
4,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,TEAM_TOTAL,PRIMARY,North Texas Mean Green,OVER,North Texas Mean Green TT OVER 42.5,FanDuel,42.5,-106,...,DERIVATIVES_FIRST_20.5_PLUS,CONFIRMS_MODEL,-13.5,-16.0,2.5,North Texas Mean Green,MAJOR,14,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...","Market supports the model side, but spread siz..."
5,Charlotte 49ers at North Texas Mean Green,North Texas Mean Green,TEAM_TOTAL,VALUE_ALT,North Texas Mean Green,OVER,North Texas Mean Green TT OVER 45.5,FanDuel,45.5,146,...,DERIVATIVES_FIRST_20.5_PLUS,CONFIRMS_MODEL,-13.5,-16.0,2.5,North Texas Mean Green,MAJOR,14,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...","Market supports the model side, but spread siz..."
6,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,FULL_GAME_SPREAD,PRIMARY,Duke Blue Devils,SPREAD,Duke Blue Devils -3.5,DraftKings,-3.5,-167,...,STANDARD_REVIEW,CONFLICTS_WITH_MODEL,-2.5,-4.0,1.5,Georgia Tech Yellow Jackets,MODERATE,3,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",Major and/or key-number movement is against th...
7,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,FULL_GAME_SPREAD,VALUE_ALT,Duke Blue Devils,SPREAD,Duke Blue Devils -7.5,DraftKings,-7.5,105,...,STANDARD_REVIEW,CONFLICTS_WITH_MODEL,-2.5,-4.0,1.5,Georgia Tech Yellow Jackets,MODERATE,3,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",Major and/or key-number movement is against th...
8,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,TEAM_TOTAL,PRIMARY,Duke Blue Devils,OVER,Duke Blue Devils TT OVER 26.5,DraftKings,26.5,-115,...,STANDARD_REVIEW,CONFLICTS_WITH_MODEL,-2.5,-4.0,1.5,Georgia Tech Yellow Jackets,MODERATE,3,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",Major and/or key-number movement is against th...
9,Duke Blue Devils at Georgia Tech Yellow Jackets,Duke Blue Devils,TEAM_TOTAL,VALUE_ALT,Duke Blue Devils,OVER,Duke Blue Devils TT OVER 28.5,FanDuel,28.5,136,...,STANDARD_REVIEW,CONFLICTS_WITH_MODEL,-2.5,-4.0,1.5,Georgia Tech Yellow Jackets,MODERATE,3,"FULL_GAME_SPREAD, TEAM_TOTAL_OVER, 1H_TEAM_TOT...",Major and/or key-number movement is against th...



EXPORT COMPLETE
Saved: cfb1_architect_handoff.csv

CELL 73 COMPLETE.
CFB1 -> ARCHITECT HANDOFF CONTAINS CFB1 MODEL OUTPUT AND CONTEXT ONLY.
CFB1 PROVIDES PROBABILITY, PUSH RISK, BREAK-EVEN, EXPECTED ROI, ROUTING, STATUS, AND MARKET CONTEXT.
BANKROLL, KELLY, STAKE SIZING, PROMOS, PARLAYS, SGPs, CORRELATION, AND PORTFOLIO DECISIONS BELONG TO THE BET SLIP ARCHITECT.
NO NEW HANDICAPPING CALCULATIONS WERE PERFORMED.


In [ ]:
# ==========================================
# CELL 73C — LOAD FROZEN FULL CONTEXT CSV
# ==========================================

import os
import glob
import pandas as pd

print("=" * 100)
print("LOAD FROZEN CFB1 FULL CONTEXT")
print("=" * 100)

# Look for the saved Full Context CSV in /content
matches = sorted(
    glob.glob("/content/*cfb1_full_context_final*.csv")
)

if not matches:
    raise FileNotFoundError(
        "No cfb1_full_context_final CSV found in /content."
    )

print("\nMatching files found:")
for i, path in enumerate(matches, start=1):
    print(f"{i}. {path}")

# Use the first matching file
FULL_CONTEXT_PATH = matches[0]

print(f"\nLoading:\n{FULL_CONTEXT_PATH}")

CFB1_FULL_CONTEXT_FINAL = pd.read_csv(FULL_CONTEXT_PATH)

print("\nLOAD COMPLETE")
print("-" * 50)
print(f"Rows: {len(CFB1_FULL_CONTEXT_FINAL)}")
print(f"Columns: {len(CFB1_FULL_CONTEXT_FINAL.columns)}")

if "Game" in CFB1_FULL_CONTEXT_FINAL.columns:
    print(
        f"Unique games: "
        f"{CFB1_FULL_CONTEXT_FINAL['Game'].nunique()}"
    )

print("\nFirst 10 column names:")
print(
    CFB1_FULL_CONTEXT_FINAL.columns.tolist()[:10]
)

print("\nCELL 73C COMPLETE.")
print("Frozen Thursday data restored.")
print("No API calls were made.")

LOAD FROZEN CFB1 FULL CONTEXT

Matching files found:
1. /content/cfb1_full_context_final.csv

Loading:
/content/cfb1_full_context_final.csv

LOAD COMPLETE
--------------------------------------------------
Rows: 84
Columns: 106
Unique games: 14

First 10 column names:
['Game', 'Market_Family', 'Team', 'Direction', 'Bet_Label', 'Sportsbook', 'Line', 'American_Odds', 'Model_Adjustment', 'Model_Probability']

CELL 73C COMPLETE.
Frozen Thursday data restored.
No API calls were made.
